# 04f Extraction, batch f: eleven studies coded from their PDFs and one full-text exclusion

Design log.
- Batch f of notebook 04 (plan v2 B4), coded under the frozen schema (schema/record_schema.yaml, v1) and the coding rules, as batches a to e were. The eleven reports: RAISED (2610.06401), AdvSim2Real (2610.08773), LTBD (2610.11634), TRACER (2610.19133, Findings of EMNLP 2026), AgentSafeEvo (2610.19422), EDIC (2610.22874, EMNLP 2026), Ignored, Not Refused (2610.23254), Heads of Security (2610.37657, version 2), LLMON (2610.38120), SecPO (2610.41998) and Treating the Model as an Untrusted Component (2611.11571).
- Each record's extraction_scope names the tables and rows coded. As before, detectors and system-level designs are left out and the scope says so (amendment 4): Prompt Guard 2 in AgentSafeEvo, and the deterministic monitors, human approval, taint tracking and the detector stack in 2611.11571, whose coded part is the in-prompt guardrail with and without datamarking on two real models. Results whose attacker writes the user's own input are left out too (the AgentHarm and Agent-SafetyBench columns of AgentSafeEvo, the Direct and Multi-turn columns of LLMON's red-teaming table).
- Coding decisions worth knowing when reading these records (reading aids for notebook 05, not findings):
  - LTBD's StruQ-Full and SecAlign-LoRA rows (and the uncoded prompting and StruQ-LoRA rows) match the DefensiveTokens report's Table 3 value for value, while its No Defense and DefensiveToken rows were re-measured and differ; the paper does not say the rows are copied. Those evaluations carry measured_by defensivetokens_aisec2025 and their comparisons are X with unreported conditions, so they grade U.
  - TRACER's attack success is a conditional answer flip on GSM8K pooled over three payload families, so R1 is coded as a proxy.
  - AdvSim2Real and AgentSafeEvo co-evolve an attacker with the agent and are coded adversarial_rl; SecPO's adversarial training is a per-step search with API models, not a learned attacker, so axis 2 stays preference_optimisation with distillation and the search is described in the components.
  - Heads of Security is coded on axis 1 as chat roles plus an internal intervention (learned per-head attention gates keyed to the trust channel). Its GPT-OSS sweep (Table 13) is coded at every stable learning rate; the paper does not say which SecAlign checkpoint its Tables 1 and 2 use.
  - Ignored, Not Refused is an evaluation study; its two prompt blocks were run on subsets of InjecAgent while the baseline uses every case, so those comparisons grade B.
  - Paper facts that name commercial models in judge fields: Heads of Security (LLMail utility judge), SecPO (AlpacaEval2 and SEP utility judge) and the Claudini attack name in Heads of Security are kept as printed.
- Full-text exclusion found while coding: Position, Role or Label? (2610.33201) was included at title and abstract as a measurement of signal dependence. Its full text is a benchmark of instruction conflicts on off-the-shelf models; no defence is added, trained or compared, and its role-against-label tests concern the operator line, which is a system-against-user conflict outside the review. It is excluded as benchmark only. Stage 2b checks the anchor in the PDF and carries the decision into schema/fulltext_decisions.csv, schema/screening_decisions.csv, reports/screening.csv, reports/defences.csv, reports/fulltext_check.csv and reports/prisma_counts.json, as batch d did. The number of included studies falls from 136 to 135.
- Outputs: reports/records/ (eleven files), reports/extraction_04f_check.csv, reports/extraction_04f_summary.csv, reports/extraction_batches.csv (rows for batch f added, earlier batches kept), reports/decision_04f.txt.
- CPU; no network calls beyond pip; model calls: 0.

## Stage 0. Session bootstrap

In [ ]:
import os, sys, subprocess, importlib
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/SourceMap_Research"
    REPO_DIR = PROJECT_ROOT + "/sourcemap-research"
except ImportError:
    PROJECT_ROOT = os.environ["SOURCEMAP_PROJECT_ROOT"]
    REPO_DIR = os.environ["SOURCEMAP_REPO_ROOT"]
os.environ["SOURCEMAP_PROJECT_ROOT"] = PROJECT_ROOT
os.environ["SOURCEMAP_REPO_ROOT"] = REPO_DIR
os.chdir(REPO_DIR)
if "src" not in sys.path:
    sys.path.insert(0, "src")
for module, package in (("yaml", "pyyaml"), ("pypdf", "pypdf"), ("pypdfium2", "pypdfium2")):
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=False)
import pypdf
if int(pypdf.__version__.split(".")[0]) < 4:                 # the column-preserving extraction needs pypdf 4 or later
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pypdf>=4"], check=False)
    print("pypdf upgraded; restart the runtime if the next stage reports an old version")
print(subprocess.run(["git", "pull", "--ff-only"], capture_output=True, text=True).stdout.strip() or "pull: nothing to report")
print("cwd:", os.getcwd(), "| pypdf", pypdf.__version__)

## Stage 1. The 11 records

In [ ]:
Path("reports/records").mkdir(parents=True, exist_ok=True)       # %%writefile does not create folders

`raised_arxiv2026` (2610.06401)

In [ ]:
%%writefile reports/records/raised_arxiv2026.yaml
report:
  id: raised_arxiv2026
  slug: '2610.06401'
  title: 'RAISED: Self-Distillation for Robustness to Prompt Injection in LLM Agents'
  first_author: Mohamed Dhouib
  venue: arXiv preprint (under review)
  peer_reviewed: 'no'
  arxiv: '2610.06401'
  version_read: arXiv v2
  first_date: '2026-10-05'
  read: F
  role: proposes
  code_release: not checked
  extraction_scope: 'Sections 6 and 7: Table 1 rows Vanilla, Prompt Sandwiching, Spotlighting, SecAlign++ and RAISED on both models; the abstract claim (PromptGuard 2, a detector, is left out; benign agentic Table 2, general
    Table 3, the Qwen appendix tables and the ablations not coded)'
defences:
- id: raised_gem
  name: RAISED
  origin: this_report
  base_checkpoint: Gemma-4-12B-it
  axis1:
  - no_signal
  axis2:
  - distillation
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: yes_exact_targets
  page: 1
  components:
  - the model generates its own tool-use scenarios, stressing cases where finishing the task needs legitimate guidance from tool outputs
  - student trained by token-level KL to match the frozen teacher's clean-context distribution on both clean and injected variants
  layout: agent loop; tool responses carry the injection
  layout_compat: &id001
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: a training framework that combines self-generation and self- distillation
  pages:
    components: 7
- id: sand_gem
  name: Prompt Sandwiching
  origin: other_report
  base_checkpoint: Gemma-4-12B-it
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  layout: agent loop; tool responses carry the injection
  layout_compat: *id001
  anchor: we evaluate Prompt Sandwiching Learn Prompting
- id: spot_gem
  name: Spotlighting
  origin: other_report
  base_checkpoint: Gemma-4-12B-it
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  layout: agent loop; tool responses carry the injection
  layout_compat: *id001
  anchor: Spotlighting Hines et al.
- id: sapp_gem
  name: SecAlign++ (trained by the authors, three seeds)
  origin: other_report
  base_checkpoint: Gemma-4-12B-it
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  layout: agent loop; tool responses carry the injection
  layout_compat: *id001
  anchor: Trained defenses are run over three seeds
- id: raised_nem
  name: RAISED
  origin: this_report
  base_checkpoint: Nemotron-3-Nano-30B-A3B
  axis1:
  - no_signal
  axis2:
  - distillation
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: yes_exact_targets
  page: 1
  components:
  - the model generates its own tool-use scenarios, stressing cases where finishing the task needs legitimate guidance from tool outputs
  - student trained by token-level KL to match the frozen teacher's clean-context distribution on both clean and injected variants
  layout: agent loop; tool responses carry the injection
  layout_compat: *id001
  anchor: a training framework that combines self-generation and self- distillation
  pages:
    components: 7
- id: sand_nem
  name: Prompt Sandwiching
  origin: other_report
  base_checkpoint: Nemotron-3-Nano-30B-A3B
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  layout: agent loop; tool responses carry the injection
  layout_compat: *id001
  anchor: we evaluate Prompt Sandwiching Learn Prompting
- id: spot_nem
  name: Spotlighting
  origin: other_report
  base_checkpoint: Nemotron-3-Nano-30B-A3B
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  layout: agent loop; tool responses carry the injection
  layout_compat: *id001
  anchor: Spotlighting Hines et al.
- id: sapp_nem
  name: SecAlign++ (trained by the authors, three seeds)
  origin: other_report
  base_checkpoint: Nemotron-3-Nano-30B-A3B
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  layout: agent loop; tool responses carry the injection
  layout_compat: *id001
  anchor: Trained defenses are run over three seeds
evaluations:
- id: ev_gem_van_iab
  defence_id: none:Gemma-4-12B-it
  benchmark: InjecAgent, Base setting (Direct Harm and Data Stealing)
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent attack success
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  sample_size: 1,054
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_gem_van_iae
  defence_id: none:Gemma-4-12B-it
  benchmark: InjecAgent, Enhanced setting
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent attack success
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  sample_size: 1,054
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_gem_van_adc
  defence_id: none:Gemma-4-12B-it
  benchmark: AgentDojo, four environments, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_gem_van_ada
  defence_id: none:Gemma-4-12B-it
  benchmark: AgentDojo, four environments, prompt-injection attacks
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_gem_van_dyc
  defence_id: none:Gemma-4-12B-it
  benchmark: AgentDyn, three suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDyn checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_gem_van_dya
  defence_id: none:Gemma-4-12B-it
  benchmark: AgentDyn, three suites, injections in third-party content
  attacker_type: static
  attacker_access: black_box
  judge: AgentDyn checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_gem_sand_iab
  defence_id: sand_gem
  benchmark: InjecAgent, Base setting (Direct Harm and Data Stealing)
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent attack success
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  sample_size: 1,054
  pages:
    benchmark: 7
- id: ev_gem_sand_iae
  defence_id: sand_gem
  benchmark: InjecAgent, Enhanced setting
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent attack success
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  sample_size: 1,054
  pages:
    benchmark: 7
- id: ev_gem_sand_adc
  defence_id: sand_gem
  benchmark: AgentDojo, four environments, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_gem_sand_ada
  defence_id: sand_gem
  benchmark: AgentDojo, four environments, prompt-injection attacks
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_gem_sand_dyc
  defence_id: sand_gem
  benchmark: AgentDyn, three suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDyn checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_gem_sand_dya
  defence_id: sand_gem
  benchmark: AgentDyn, three suites, injections in third-party content
  attacker_type: static
  attacker_access: black_box
  judge: AgentDyn checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_gem_spot_iab
  defence_id: spot_gem
  benchmark: InjecAgent, Base setting (Direct Harm and Data Stealing)
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent attack success
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  sample_size: 1,054
  pages:
    benchmark: 7
- id: ev_gem_spot_iae
  defence_id: spot_gem
  benchmark: InjecAgent, Enhanced setting
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent attack success
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  sample_size: 1,054
  pages:
    benchmark: 7
- id: ev_gem_spot_adc
  defence_id: spot_gem
  benchmark: AgentDojo, four environments, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_gem_spot_ada
  defence_id: spot_gem
  benchmark: AgentDojo, four environments, prompt-injection attacks
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_gem_spot_dyc
  defence_id: spot_gem
  benchmark: AgentDyn, three suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDyn checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_gem_spot_dya
  defence_id: spot_gem
  benchmark: AgentDyn, three suites, injections in third-party content
  attacker_type: static
  attacker_access: black_box
  judge: AgentDyn checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_gem_sapp_iab
  defence_id: sapp_gem
  benchmark: InjecAgent, Base setting (Direct Harm and Data Stealing)
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent attack success
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  sample_size: 1,054
  pages:
    benchmark: 7
- id: ev_gem_sapp_iae
  defence_id: sapp_gem
  benchmark: InjecAgent, Enhanced setting
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent attack success
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  sample_size: 1,054
  pages:
    benchmark: 7
- id: ev_gem_sapp_adc
  defence_id: sapp_gem
  benchmark: AgentDojo, four environments, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_gem_sapp_ada
  defence_id: sapp_gem
  benchmark: AgentDojo, four environments, prompt-injection attacks
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_gem_sapp_dyc
  defence_id: sapp_gem
  benchmark: AgentDyn, three suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDyn checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_gem_sapp_dya
  defence_id: sapp_gem
  benchmark: AgentDyn, three suites, injections in third-party content
  attacker_type: static
  attacker_access: black_box
  judge: AgentDyn checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_gem_raised_iab
  defence_id: raised_gem
  benchmark: InjecAgent, Base setting (Direct Harm and Data Stealing)
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent attack success
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  sample_size: 1,054
  pages:
    benchmark: 7
- id: ev_gem_raised_iae
  defence_id: raised_gem
  benchmark: InjecAgent, Enhanced setting
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent attack success
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  sample_size: 1,054
  pages:
    benchmark: 7
- id: ev_gem_raised_adc
  defence_id: raised_gem
  benchmark: AgentDojo, four environments, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_gem_raised_ada
  defence_id: raised_gem
  benchmark: AgentDojo, four environments, prompt-injection attacks
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_gem_raised_dyc
  defence_id: raised_gem
  benchmark: AgentDyn, three suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDyn checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_gem_raised_dya
  defence_id: raised_gem
  benchmark: AgentDyn, three suites, injections in third-party content
  attacker_type: static
  attacker_access: black_box
  judge: AgentDyn checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_nem_van_iab
  defence_id: none:Nemotron-3-Nano-30B-A3B
  benchmark: InjecAgent, Base setting (Direct Harm and Data Stealing)
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent attack success
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  sample_size: 1,054
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_nem_van_iae
  defence_id: none:Nemotron-3-Nano-30B-A3B
  benchmark: InjecAgent, Enhanced setting
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent attack success
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  sample_size: 1,054
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_nem_van_adc
  defence_id: none:Nemotron-3-Nano-30B-A3B
  benchmark: AgentDojo, four environments, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_nem_van_ada
  defence_id: none:Nemotron-3-Nano-30B-A3B
  benchmark: AgentDojo, four environments, prompt-injection attacks
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_nem_van_dyc
  defence_id: none:Nemotron-3-Nano-30B-A3B
  benchmark: AgentDyn, three suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDyn checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_nem_van_dya
  defence_id: none:Nemotron-3-Nano-30B-A3B
  benchmark: AgentDyn, three suites, injections in third-party content
  attacker_type: static
  attacker_access: black_box
  judge: AgentDyn checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_nem_sand_iab
  defence_id: sand_nem
  benchmark: InjecAgent, Base setting (Direct Harm and Data Stealing)
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent attack success
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  sample_size: 1,054
  pages:
    benchmark: 7
- id: ev_nem_sand_iae
  defence_id: sand_nem
  benchmark: InjecAgent, Enhanced setting
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent attack success
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  sample_size: 1,054
  pages:
    benchmark: 7
- id: ev_nem_sand_adc
  defence_id: sand_nem
  benchmark: AgentDojo, four environments, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_nem_sand_ada
  defence_id: sand_nem
  benchmark: AgentDojo, four environments, prompt-injection attacks
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_nem_sand_dyc
  defence_id: sand_nem
  benchmark: AgentDyn, three suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDyn checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_nem_sand_dya
  defence_id: sand_nem
  benchmark: AgentDyn, three suites, injections in third-party content
  attacker_type: static
  attacker_access: black_box
  judge: AgentDyn checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_nem_spot_iab
  defence_id: spot_nem
  benchmark: InjecAgent, Base setting (Direct Harm and Data Stealing)
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent attack success
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  sample_size: 1,054
  pages:
    benchmark: 7
- id: ev_nem_spot_iae
  defence_id: spot_nem
  benchmark: InjecAgent, Enhanced setting
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent attack success
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  sample_size: 1,054
  pages:
    benchmark: 7
- id: ev_nem_spot_adc
  defence_id: spot_nem
  benchmark: AgentDojo, four environments, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_nem_spot_ada
  defence_id: spot_nem
  benchmark: AgentDojo, four environments, prompt-injection attacks
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_nem_spot_dyc
  defence_id: spot_nem
  benchmark: AgentDyn, three suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDyn checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_nem_spot_dya
  defence_id: spot_nem
  benchmark: AgentDyn, three suites, injections in third-party content
  attacker_type: static
  attacker_access: black_box
  judge: AgentDyn checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_nem_sapp_iab
  defence_id: sapp_nem
  benchmark: InjecAgent, Base setting (Direct Harm and Data Stealing)
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent attack success
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  sample_size: 1,054
  pages:
    benchmark: 7
- id: ev_nem_sapp_iae
  defence_id: sapp_nem
  benchmark: InjecAgent, Enhanced setting
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent attack success
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  sample_size: 1,054
  pages:
    benchmark: 7
- id: ev_nem_sapp_adc
  defence_id: sapp_nem
  benchmark: AgentDojo, four environments, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_nem_sapp_ada
  defence_id: sapp_nem
  benchmark: AgentDojo, four environments, prompt-injection attacks
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_nem_sapp_dyc
  defence_id: sapp_nem
  benchmark: AgentDyn, three suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDyn checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_nem_sapp_dya
  defence_id: sapp_nem
  benchmark: AgentDyn, three suites, injections in third-party content
  attacker_type: static
  attacker_access: black_box
  judge: AgentDyn checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_nem_raised_iab
  defence_id: raised_nem
  benchmark: InjecAgent, Base setting (Direct Harm and Data Stealing)
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent attack success
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  sample_size: 1,054
  pages:
    benchmark: 7
- id: ev_nem_raised_iae
  defence_id: raised_nem
  benchmark: InjecAgent, Enhanced setting
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent attack success
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  sample_size: 1,054
  pages:
    benchmark: 7
- id: ev_nem_raised_adc
  defence_id: raised_nem
  benchmark: AgentDojo, four environments, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_nem_raised_ada
  defence_id: raised_nem
  benchmark: AgentDojo, four environments, prompt-injection attacks
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_nem_raised_dyc
  defence_id: raised_nem
  benchmark: AgentDyn, three suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDyn checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_nem_raised_dya
  defence_id: raised_nem
  benchmark: AgentDyn, three suites, injections in third-party content
  attacker_type: static
  attacker_access: black_box
  judge: AgentDyn checkers
  layout_same_as_defence: 'yes'
  measured_by: raised_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
results:
- id: r_gem_van_iab
  evaluation_id: ev_gem_van_iab
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR (Base)
  value: '1.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_gem_van_iae
  evaluation_id: ev_gem_van_iae
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR (Enhanced)
  value: '0.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_gem_van_adu
  evaluation_id: ev_gem_van_adc
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDojo U
  value: '84.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_gem_van_aduatk
  evaluation_id: ev_gem_van_ada
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo Uatk
  value: '64.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_gem_van_adasr
  evaluation_id: ev_gem_van_ada
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '39.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_gem_van_dyu
  evaluation_id: ev_gem_van_dyc
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDyn U
  value: '61.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_gem_van_dyuatk
  evaluation_id: ev_gem_van_dya
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDyn Uatk
  value: '55.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_gem_van_dyasr
  evaluation_id: ev_gem_van_dya
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDyn ASR
  value: '53.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_gem_sand_iab
  evaluation_id: ev_gem_sand_iab
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR (Base)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_gem_sand_iae
  evaluation_id: ev_gem_sand_iae
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR (Enhanced)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_gem_sand_adu
  evaluation_id: ev_gem_sand_adc
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDojo U
  value: '83.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_gem_sand_aduatk
  evaluation_id: ev_gem_sand_ada
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo Uatk
  value: '64.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_gem_sand_adasr
  evaluation_id: ev_gem_sand_ada
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '34.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_gem_sand_dyu
  evaluation_id: ev_gem_sand_dyc
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDyn U
  value: '60.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_gem_sand_dyuatk
  evaluation_id: ev_gem_sand_dya
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDyn Uatk
  value: '52.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_gem_sand_dyasr
  evaluation_id: ev_gem_sand_dya
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDyn ASR
  value: '56.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_gem_spot_iab
  evaluation_id: ev_gem_spot_iab
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR (Base)
  value: '1.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_gem_spot_iae
  evaluation_id: ev_gem_spot_iae
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR (Enhanced)
  value: '0.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_gem_spot_adu
  evaluation_id: ev_gem_spot_adc
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDojo U
  value: '80.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_gem_spot_aduatk
  evaluation_id: ev_gem_spot_ada
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo Uatk
  value: '55.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_gem_spot_adasr
  evaluation_id: ev_gem_spot_ada
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '43.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_gem_spot_dyu
  evaluation_id: ev_gem_spot_dyc
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDyn U
  value: '61.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_gem_spot_dyuatk
  evaluation_id: ev_gem_spot_dya
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDyn Uatk
  value: '55.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_gem_spot_dyasr
  evaluation_id: ev_gem_spot_dya
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDyn ASR
  value: '54.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_gem_sapp_iab
  evaluation_id: ev_gem_sapp_iab
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR (Base)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: ± 0.0 over three seeds
- id: r_gem_sapp_iae
  evaluation_id: ev_gem_sapp_iae
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR (Enhanced)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: ± 0.0 over three seeds
- id: r_gem_sapp_adu
  evaluation_id: ev_gem_sapp_adc
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDojo U
  value: '77.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: ± 2.2 over three seeds
- id: r_gem_sapp_aduatk
  evaluation_id: ev_gem_sapp_ada
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo Uatk
  value: '74.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: ± 0.7 over three seeds
- id: r_gem_sapp_adasr
  evaluation_id: ev_gem_sapp_ada
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '1.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: ± 0.2 over three seeds
- id: r_gem_sapp_dyu
  evaluation_id: ev_gem_sapp_dyc
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDyn U
  value: '45.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: ± 0.8 over three seeds
- id: r_gem_sapp_dyuatk
  evaluation_id: ev_gem_sapp_dya
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDyn Uatk
  value: '46.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: ± 1.4 over three seeds
- id: r_gem_sapp_dyasr
  evaluation_id: ev_gem_sapp_dya
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDyn ASR
  value: '1.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: ± 0.1 over three seeds
- id: r_gem_raised_iab
  evaluation_id: ev_gem_raised_iab
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR (Base)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: ± 0.0 over three seeds
- id: r_gem_raised_iae
  evaluation_id: ev_gem_raised_iae
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR (Enhanced)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: ± 0.0 over three seeds
- id: r_gem_raised_adu
  evaluation_id: ev_gem_raised_adc
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDojo U
  value: '83.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: ± 0.8 over three seeds
- id: r_gem_raised_aduatk
  evaluation_id: ev_gem_raised_ada
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo Uatk
  value: '80.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: ± 0.5 over three seeds
- id: r_gem_raised_adasr
  evaluation_id: ev_gem_raised_ada
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '1.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: ± 0.3 over three seeds
- id: r_gem_raised_dyu
  evaluation_id: ev_gem_raised_dyc
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDyn U
  value: '60.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: ± 2.4 over three seeds
- id: r_gem_raised_dyuatk
  evaluation_id: ev_gem_raised_dya
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDyn Uatk
  value: '55.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: ± 0.3 over three seeds
- id: r_gem_raised_dyasr
  evaluation_id: ev_gem_raised_dya
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDyn ASR
  value: '1.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: ± 1.8 over three seeds
- id: r_nem_van_iab
  evaluation_id: ev_nem_van_iab
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR (Base)
  value: '26.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_nem_van_iae
  evaluation_id: ev_nem_van_iae
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR (Enhanced)
  value: '33.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_nem_van_adu
  evaluation_id: ev_nem_van_adc
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDojo U
  value: '48.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_nem_van_aduatk
  evaluation_id: ev_nem_van_ada
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo Uatk
  value: '39.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_nem_van_adasr
  evaluation_id: ev_nem_van_ada
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '11.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_nem_van_dyu
  evaluation_id: ev_nem_van_dyc
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDyn U
  value: '25.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_nem_van_dyuatk
  evaluation_id: ev_nem_van_dya
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDyn Uatk
  value: '19.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_nem_van_dyasr
  evaluation_id: ev_nem_van_dya
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDyn ASR
  value: '17.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_nem_sand_iab
  evaluation_id: ev_nem_sand_iab
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR (Base)
  value: '7.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_nem_sand_iae
  evaluation_id: ev_nem_sand_iae
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR (Enhanced)
  value: '8.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_nem_sand_adu
  evaluation_id: ev_nem_sand_adc
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDojo U
  value: '44.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_nem_sand_aduatk
  evaluation_id: ev_nem_sand_ada
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo Uatk
  value: '39.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_nem_sand_adasr
  evaluation_id: ev_nem_sand_ada
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '6.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_nem_sand_dyu
  evaluation_id: ev_nem_sand_dyc
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDyn U
  value: '25.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_nem_sand_dyuatk
  evaluation_id: ev_nem_sand_dya
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDyn Uatk
  value: '18.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_nem_sand_dyasr
  evaluation_id: ev_nem_sand_dya
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDyn ASR
  value: '11.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_nem_spot_iab
  evaluation_id: ev_nem_spot_iab
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR (Base)
  value: '28.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_nem_spot_iae
  evaluation_id: ev_nem_spot_iae
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR (Enhanced)
  value: '35.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_nem_spot_adu
  evaluation_id: ev_nem_spot_adc
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDojo U
  value: '47.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_nem_spot_aduatk
  evaluation_id: ev_nem_spot_ada
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo Uatk
  value: '38.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_nem_spot_adasr
  evaluation_id: ev_nem_spot_ada
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '11.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_nem_spot_dyu
  evaluation_id: ev_nem_spot_dyc
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDyn U
  value: '23.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_nem_spot_dyuatk
  evaluation_id: ev_nem_spot_dya
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDyn Uatk
  value: '20.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_nem_spot_dyasr
  evaluation_id: ev_nem_spot_dya
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDyn ASR
  value: '17.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_nem_sapp_iab
  evaluation_id: ev_nem_sapp_iab
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR (Base)
  value: '6.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: ± 0.9 over three seeds
- id: r_nem_sapp_iae
  evaluation_id: ev_nem_sapp_iae
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR (Enhanced)
  value: '8.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: ± 1.4 over three seeds
- id: r_nem_sapp_adu
  evaluation_id: ev_nem_sapp_adc
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDojo U
  value: '50.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: ± 1.0 over three seeds
- id: r_nem_sapp_aduatk
  evaluation_id: ev_nem_sapp_ada
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo Uatk
  value: '47.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: ± 0.7 over three seeds
- id: r_nem_sapp_adasr
  evaluation_id: ev_nem_sapp_ada
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '4.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: ± 0.7 over three seeds
- id: r_nem_sapp_dyu
  evaluation_id: ev_nem_sapp_dyc
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDyn U
  value: '22.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: ± 0.8 over three seeds
- id: r_nem_sapp_dyuatk
  evaluation_id: ev_nem_sapp_dya
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDyn Uatk
  value: '20.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: ± 0.5 over three seeds
- id: r_nem_sapp_dyasr
  evaluation_id: ev_nem_sapp_dya
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDyn ASR
  value: '7.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: ± 0.3 over three seeds
- id: r_nem_raised_iab
  evaluation_id: ev_nem_raised_iab
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR (Base)
  value: '3.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: ± 0.5 over three seeds
- id: r_nem_raised_iae
  evaluation_id: ev_nem_raised_iae
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR (Enhanced)
  value: '1.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: ± 0.6 over three seeds
- id: r_nem_raised_adu
  evaluation_id: ev_nem_raised_adc
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDojo U
  value: '50.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: ± 1.3 over three seeds
- id: r_nem_raised_aduatk
  evaluation_id: ev_nem_raised_ada
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo Uatk
  value: '46.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: ± 0.6 over three seeds
- id: r_nem_raised_adasr
  evaluation_id: ev_nem_raised_ada
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: ± 0.1 over three seeds
- id: r_nem_raised_dyu
  evaluation_id: ev_nem_raised_dyc
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDyn U
  value: '25.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: ± 0.0 over three seeds
- id: r_nem_raised_dyuatk
  evaluation_id: ev_nem_raised_dya
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDyn Uatk
  value: '25.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: ± 1.2 over three seeds
- id: r_nem_raised_dyasr
  evaluation_id: ev_nem_raised_dya
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDyn ASR
  value: '1.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: ± 0.6 over three seeds
comparisons:
- id: c_gem_sand_iab
  left: ev_gem_sand_iab
  right: ev_gem_van_iab
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gem_sand_iae
  left: ev_gem_sand_iae
  right: ev_gem_van_iae
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gem_sand_adc
  left: ev_gem_sand_adc
  right: ev_gem_van_adc
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gem_sand_ada
  left: ev_gem_sand_ada
  right: ev_gem_van_ada
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gem_sand_dyc
  left: ev_gem_sand_dyc
  right: ev_gem_van_dyc
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gem_sand_dya
  left: ev_gem_sand_dya
  right: ev_gem_van_dya
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gem_spot_iab
  left: ev_gem_spot_iab
  right: ev_gem_van_iab
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gem_spot_iae
  left: ev_gem_spot_iae
  right: ev_gem_van_iae
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gem_spot_adc
  left: ev_gem_spot_adc
  right: ev_gem_van_adc
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gem_spot_ada
  left: ev_gem_spot_ada
  right: ev_gem_van_ada
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gem_spot_dyc
  left: ev_gem_spot_dyc
  right: ev_gem_van_dyc
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gem_spot_dya
  left: ev_gem_spot_dya
  right: ev_gem_van_dya
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gem_sapp_iab
  left: ev_gem_sapp_iab
  right: ev_gem_van_iab
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gem_sapp_iae
  left: ev_gem_sapp_iae
  right: ev_gem_van_iae
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gem_sapp_adc
  left: ev_gem_sapp_adc
  right: ev_gem_van_adc
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gem_sapp_ada
  left: ev_gem_sapp_ada
  right: ev_gem_van_ada
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gem_sapp_dyc
  left: ev_gem_sapp_dyc
  right: ev_gem_van_dyc
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gem_sapp_dya
  left: ev_gem_sapp_dya
  right: ev_gem_van_dya
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gem_raised_iab
  left: ev_gem_raised_iab
  right: ev_gem_van_iab
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gem_raised_iae
  left: ev_gem_raised_iae
  right: ev_gem_van_iae
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gem_raised_adc
  left: ev_gem_raised_adc
  right: ev_gem_van_adc
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gem_raised_ada
  left: ev_gem_raised_ada
  right: ev_gem_van_ada
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gem_raised_dyc
  left: ev_gem_raised_dyc
  right: ev_gem_van_dyc
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gem_raised_dya
  left: ev_gem_raised_dya
  right: ev_gem_van_dya
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gem_raised_sapp_iab
  left: ev_gem_raised_iab
  right: ev_gem_sapp_iab
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gem_raised_sapp_iae
  left: ev_gem_raised_iae
  right: ev_gem_sapp_iae
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gem_raised_sapp_adc
  left: ev_gem_raised_adc
  right: ev_gem_sapp_adc
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gem_raised_sapp_ada
  left: ev_gem_raised_ada
  right: ev_gem_sapp_ada
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gem_raised_sapp_dyc
  left: ev_gem_raised_dyc
  right: ev_gem_sapp_dyc
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gem_raised_sapp_dya
  left: ev_gem_raised_dya
  right: ev_gem_sapp_dya
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_nem_sand_iab
  left: ev_nem_sand_iab
  right: ev_nem_van_iab
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_nem_sand_iae
  left: ev_nem_sand_iae
  right: ev_nem_van_iae
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_nem_sand_adc
  left: ev_nem_sand_adc
  right: ev_nem_van_adc
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_nem_sand_ada
  left: ev_nem_sand_ada
  right: ev_nem_van_ada
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_nem_sand_dyc
  left: ev_nem_sand_dyc
  right: ev_nem_van_dyc
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_nem_sand_dya
  left: ev_nem_sand_dya
  right: ev_nem_van_dya
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_nem_spot_iab
  left: ev_nem_spot_iab
  right: ev_nem_van_iab
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_nem_spot_iae
  left: ev_nem_spot_iae
  right: ev_nem_van_iae
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_nem_spot_adc
  left: ev_nem_spot_adc
  right: ev_nem_van_adc
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_nem_spot_ada
  left: ev_nem_spot_ada
  right: ev_nem_van_ada
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_nem_spot_dyc
  left: ev_nem_spot_dyc
  right: ev_nem_van_dyc
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_nem_spot_dya
  left: ev_nem_spot_dya
  right: ev_nem_van_dya
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_nem_sapp_iab
  left: ev_nem_sapp_iab
  right: ev_nem_van_iab
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_nem_sapp_iae
  left: ev_nem_sapp_iae
  right: ev_nem_van_iae
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_nem_sapp_adc
  left: ev_nem_sapp_adc
  right: ev_nem_van_adc
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_nem_sapp_ada
  left: ev_nem_sapp_ada
  right: ev_nem_van_ada
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_nem_sapp_dyc
  left: ev_nem_sapp_dyc
  right: ev_nem_van_dyc
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_nem_sapp_dya
  left: ev_nem_sapp_dya
  right: ev_nem_van_dya
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_nem_raised_iab
  left: ev_nem_raised_iab
  right: ev_nem_van_iab
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_nem_raised_iae
  left: ev_nem_raised_iae
  right: ev_nem_van_iae
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_nem_raised_adc
  left: ev_nem_raised_adc
  right: ev_nem_van_adc
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_nem_raised_ada
  left: ev_nem_raised_ada
  right: ev_nem_van_ada
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_nem_raised_dyc
  left: ev_nem_raised_dyc
  right: ev_nem_van_dyc
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_nem_raised_dya
  left: ev_nem_raised_dya
  right: ev_nem_van_dya
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_nem_raised_sapp_iab
  left: ev_nem_raised_iab
  right: ev_nem_sapp_iab
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_nem_raised_sapp_iae
  left: ev_nem_raised_iae
  right: ev_nem_sapp_iae
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_nem_raised_sapp_adc
  left: ev_nem_raised_adc
  right: ev_nem_sapp_adc
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_nem_raised_sapp_ada
  left: ev_nem_raised_ada
  right: ev_nem_sapp_ada
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_nem_raised_sapp_dyc
  left: ev_nem_raised_dyc
  right: ev_nem_sapp_dyc
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_nem_raised_sapp_dya
  left: ev_nem_raised_dya
  right: ev_nem_sapp_dya
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
claims:
- id: k1
  text: RAISED substantially reduces attack success while, unlike prior trained defences, preserving utility
  location: abstract
  anchor: substantially reduces the attack success rate of prompt in- jections in tool responses
  jobs:
  - R1
  - R2
  supported_by:
  - c_gem_raised_iab
  - c_gem_raised_iae
  - c_gem_raised_adc
  - c_gem_raised_ada
  - c_gem_raised_dyc
  - c_gem_raised_dya
  - c_gem_raised_sapp_iab
  - c_gem_raised_sapp_iae
  - c_gem_raised_sapp_adc
  - c_gem_raised_sapp_ada
  - c_gem_raised_sapp_dyc
  - c_gem_raised_sapp_dya
  - c_nem_raised_iab
  - c_nem_raised_iae
  - c_nem_raised_adc
  - c_nem_raised_ada
  - c_nem_raised_dyc
  - c_nem_raised_dya
  - c_nem_raised_sapp_iab
  - c_nem_raised_sapp_iae
  - c_nem_raised_sapp_adc
  - c_nem_raised_sapp_ada
  - c_nem_raised_sapp_dyc
  - c_nem_raised_sapp_dya
  page: 1
coverage:
  R1: direct
  R2clean: direct
  R2attack: direct
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 1
    R2clean: Table 1 (U)
    R2attack: Table 1 (Uatk)
schema_gaps:
- where: defences raised_*
  note: trains_carry_through is coded yes_exact_targets because the self-generated scenarios require acting on legitimate guidance in tool outputs and the targets are the frozen model's own clean-context predictions (pp. 1,
    7)

`advsim2real_arxiv2026` (2610.08773)

In [ ]:
%%writefile reports/records/advsim2real_arxiv2026.yaml
report:
  id: advsim2real_arxiv2026
  slug: '2610.08773'
  title: 'AdvSim2Real: Training Web Agents Against Adaptive Prompt Injection in a Web World Model'
  first_author: Sarim Hashmi
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2610.08773'
  version_read: arXiv v1
  first_date: '2026-10-06'
  read: F
  role: proposes
  code_release: stated as released in the abstract; not checked
  extraction_scope: 'Section 5: Table 1 rows Base, Capability iter 3 and Robust iter 3 (clean and Adv v1 to v3) and Table 3 rows Base and Robust iter 3 (Kimi-K3 adversary); the abstract claim (the other checkpoints, the hosted
    9B reference, Table 2 browser runs, which test capability only, and the ablations not coded)'
defences:
- id: robust3
  name: 'AdvSim2Real (Robust iter 3: curriculum, then adversary-executor co-evolution)'
  origin: this_report
  base_checkpoint: Qwen3.5-4B
  axis1:
  - no_signal
  axis2:
  - adversarial_rl
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 1
  components:
  - 'Stage 1: a curriculum model proposes tasks the agent solves about half the time; the agent trains on fresh rollouts in the world model'
  - 'Stage 2: an adversary is rewarded only for success flips (a judged success turned into a failure by one timed injection); the agent trains on new and earlier attacks plus clean tasks'
  - LoRA per role, group-normalised advantages
  layout: web agent acting on pages predicted by a frozen web world model (WebWorld-14B); injections appear in the page
  layout_compat: &id001
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: co-evolves a task curriculum, an injection adversary, and the agent
  pages:
    components: 6
- id: cap3
  name: AdvSim2Real Stage 1 only (Capability iter 3, no injections in training)
  origin: this_report
  base_checkpoint: Qwen3.5-4B
  axis1:
  - no_signal
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  layout: web agent acting on pages predicted by a frozen web world model (WebWorld-14B); injections appear in the page
  layout_compat: *id001
  anchor: Capability iter t is the executor E after t Stage-1 rounds
evaluations:
- id: ev_base_clean
  defence_id: none:Qwen3.5-4B
  benchmark: 150 web tasks in WebWorld-14B, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: frozen LLM judge over the world-model trajectory
  layout_same_as_defence: 'yes'
  measured_by: advsim2real_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '150'
  baseline_kind: off_the_shelf
- id: ev_base_v1
  defence_id: none:Qwen3.5-4B
  benchmark: 150 web tasks in WebWorld-14B, one reactive page injection from learned adversary Adv v1
  attacker_type: rl
  attacker_access: black_box
  judge: frozen LLM judge over the world-model trajectory
  layout_same_as_defence: 'yes'
  measured_by: advsim2real_arxiv2026
  page: 7
  attacker_budget: one timed injection per episode; adversary trained during Stage 2 against the Robust lineage
  decoding: not reported
  sample_size: '150'
  baseline_kind: off_the_shelf
- id: ev_base_v2
  defence_id: none:Qwen3.5-4B
  benchmark: 150 web tasks in WebWorld-14B, one reactive page injection from learned adversary Adv v2
  attacker_type: rl
  attacker_access: black_box
  judge: frozen LLM judge over the world-model trajectory
  layout_same_as_defence: 'yes'
  measured_by: advsim2real_arxiv2026
  page: 7
  attacker_budget: one timed injection per episode; adversary trained during Stage 2 against the Robust lineage
  decoding: not reported
  sample_size: '150'
  baseline_kind: off_the_shelf
- id: ev_base_v3
  defence_id: none:Qwen3.5-4B
  benchmark: 150 web tasks in WebWorld-14B, one reactive page injection from learned adversary Adv v3
  attacker_type: rl
  attacker_access: black_box
  judge: frozen LLM judge over the world-model trajectory
  layout_same_as_defence: 'yes'
  measured_by: advsim2real_arxiv2026
  page: 7
  attacker_budget: one timed injection per episode; adversary trained during Stage 2 against the Robust lineage
  decoding: not reported
  sample_size: '150'
  baseline_kind: off_the_shelf
- id: ev_cap3_clean
  defence_id: cap3
  benchmark: 150 web tasks in WebWorld-14B, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: frozen LLM judge over the world-model trajectory
  layout_same_as_defence: 'yes'
  measured_by: advsim2real_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '150'
- id: ev_cap3_v1
  defence_id: cap3
  benchmark: 150 web tasks in WebWorld-14B, one reactive page injection from learned adversary Adv v1
  attacker_type: rl
  attacker_access: black_box
  judge: frozen LLM judge over the world-model trajectory
  layout_same_as_defence: 'yes'
  measured_by: advsim2real_arxiv2026
  page: 7
  attacker_budget: one timed injection per episode; adversary trained during Stage 2 against the Robust lineage
  decoding: not reported
  sample_size: '150'
- id: ev_cap3_v2
  defence_id: cap3
  benchmark: 150 web tasks in WebWorld-14B, one reactive page injection from learned adversary Adv v2
  attacker_type: rl
  attacker_access: black_box
  judge: frozen LLM judge over the world-model trajectory
  layout_same_as_defence: 'yes'
  measured_by: advsim2real_arxiv2026
  page: 7
  attacker_budget: one timed injection per episode; adversary trained during Stage 2 against the Robust lineage
  decoding: not reported
  sample_size: '150'
- id: ev_cap3_v3
  defence_id: cap3
  benchmark: 150 web tasks in WebWorld-14B, one reactive page injection from learned adversary Adv v3
  attacker_type: rl
  attacker_access: black_box
  judge: frozen LLM judge over the world-model trajectory
  layout_same_as_defence: 'yes'
  measured_by: advsim2real_arxiv2026
  page: 7
  attacker_budget: one timed injection per episode; adversary trained during Stage 2 against the Robust lineage
  decoding: not reported
  sample_size: '150'
- id: ev_robust3_clean
  defence_id: robust3
  benchmark: 150 web tasks in WebWorld-14B, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: frozen LLM judge over the world-model trajectory
  layout_same_as_defence: 'yes'
  measured_by: advsim2real_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '150'
- id: ev_robust3_v1
  defence_id: robust3
  benchmark: 150 web tasks in WebWorld-14B, one reactive page injection from learned adversary Adv v1
  attacker_type: rl
  attacker_access: black_box
  judge: frozen LLM judge over the world-model trajectory
  layout_same_as_defence: 'yes'
  measured_by: advsim2real_arxiv2026
  page: 7
  attacker_budget: one timed injection per episode; adversary trained during Stage 2 against the Robust lineage
  decoding: not reported
  sample_size: '150'
- id: ev_robust3_v2
  defence_id: robust3
  benchmark: 150 web tasks in WebWorld-14B, one reactive page injection from learned adversary Adv v2
  attacker_type: rl
  attacker_access: black_box
  judge: frozen LLM judge over the world-model trajectory
  layout_same_as_defence: 'yes'
  measured_by: advsim2real_arxiv2026
  page: 7
  attacker_budget: one timed injection per episode; adversary trained during Stage 2 against the Robust lineage
  decoding: not reported
  sample_size: '150'
- id: ev_robust3_v3
  defence_id: robust3
  benchmark: 150 web tasks in WebWorld-14B, one reactive page injection from learned adversary Adv v3
  attacker_type: rl
  attacker_access: black_box
  judge: frozen LLM judge over the world-model trajectory
  layout_same_as_defence: 'yes'
  measured_by: advsim2real_arxiv2026
  page: 7
  attacker_budget: one timed injection per episode; adversary trained during Stage 2 against the Robust lineage
  decoding: not reported
  sample_size: '150'
- id: ev_base_kimi
  defence_id: none:Qwen3.5-4B
  benchmark: 150 web tasks in WebWorld-14B, one injection written by the Kimi-K3 adversary (never seen in training)
  attacker_type: adaptive
  attacker_access: black_box
  judge: frozen LLM judge over the world-model trajectory
  layout_same_as_defence: 'yes'
  measured_by: advsim2real_arxiv2026
  page: 8
  attacker_budget: one injection per episode, same prompt and observation as the learned adversary
  decoding: not reported
  sample_size: '150'
  baseline_kind: off_the_shelf
- id: ev_robust3_kimi
  defence_id: robust3
  benchmark: 150 web tasks in WebWorld-14B, one injection written by the Kimi-K3 adversary (never seen in training)
  attacker_type: adaptive
  attacker_access: black_box
  judge: frozen LLM judge over the world-model trajectory
  layout_same_as_defence: 'yes'
  measured_by: advsim2real_arxiv2026
  page: 8
  attacker_budget: one injection per episode, same prompt and observation as the learned adversary
  decoding: not reported
  sample_size: '150'
results:
- id: r_base_clean
  evaluation_id: ev_base_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Clean judged completion
  value: '74.89'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: sample SD 1.39 over three rollout seeds
- id: r_base_v1
  evaluation_id: ev_base_v1
  quantity: R2attack
  coverage: direct
  metric_as_printed: Judged completion under Adv v1
  value: '51.11'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: sample SD 2.34 over three rollout seeds
- id: r_base_v2
  evaluation_id: ev_base_v2
  quantity: R2attack
  coverage: direct
  metric_as_printed: Judged completion under Adv v2
  value: '48.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: sample SD 4.16 over three rollout seeds
- id: r_base_v3
  evaluation_id: ev_base_v3
  quantity: R2attack
  coverage: direct
  metric_as_printed: Judged completion under Adv v3
  value: '45.11'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: sample SD 1.54 over three rollout seeds
- id: r_cap3_clean
  evaluation_id: ev_cap3_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Clean judged completion
  value: '79.33'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: sample SD 1.76 over three rollout seeds
- id: r_cap3_v1
  evaluation_id: ev_cap3_v1
  quantity: R2attack
  coverage: direct
  metric_as_printed: Judged completion under Adv v1
  value: '57.11'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: sample SD 1.39 over three rollout seeds
- id: r_cap3_v2
  evaluation_id: ev_cap3_v2
  quantity: R2attack
  coverage: direct
  metric_as_printed: Judged completion under Adv v2
  value: '52.35'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: sample SD 4.12 over three rollout seeds
- id: r_cap3_v3
  evaluation_id: ev_cap3_v3
  quantity: R2attack
  coverage: direct
  metric_as_printed: Judged completion under Adv v3
  value: '53.56'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: sample SD 3.67 over three rollout seeds
- id: r_robust3_clean
  evaluation_id: ev_robust3_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Clean judged completion
  value: '81.33'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: sample SD 2.31 over three rollout seeds
- id: r_robust3_v1
  evaluation_id: ev_robust3_v1
  quantity: R2attack
  coverage: direct
  metric_as_printed: Judged completion under Adv v1
  value: '62.89'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: sample SD 4.73 over three rollout seeds
- id: r_robust3_v2
  evaluation_id: ev_robust3_v2
  quantity: R2attack
  coverage: direct
  metric_as_printed: Judged completion under Adv v2
  value: '54.89'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: sample SD 3.91 over three rollout seeds
- id: r_robust3_v3
  evaluation_id: ev_robust3_v3
  quantity: R2attack
  coverage: direct
  metric_as_printed: Judged completion under Adv v3
  value: '54.67'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
  spread: sample SD 3.53 over three rollout seeds
- id: r_base_kimi
  evaluation_id: ev_base_kimi
  quantity: R2attack
  coverage: direct
  metric_as_printed: Judged completion under Kimi-K3
  value: '23.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
  spread: sample SD 3.30 over three rollout seeds
- id: r_robust3_kimi
  evaluation_id: ev_robust3_kimi
  quantity: R2attack
  coverage: direct
  metric_as_printed: Judged completion under Kimi-K3
  value: '30.72'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
  spread: sample SD 3.04 over three rollout seeds
comparisons:
- id: c_cap3_clean
  left: ev_cap3_clean
  right: ev_base_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_cap3_v1
  left: ev_cap3_v1
  right: ev_base_v1
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_cap3_v2
  left: ev_cap3_v2
  right: ev_base_v2
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_cap3_v3
  left: ev_cap3_v3
  right: ev_base_v3
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_robust3_clean
  left: ev_robust3_clean
  right: ev_base_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_robust3_v1
  left: ev_robust3_v1
  right: ev_base_v1
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_robust3_v2
  left: ev_robust3_v2
  right: ev_base_v2
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_robust3_v3
  left: ev_robust3_v3
  right: ev_base_v3
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_stage2_clean
  left: ev_robust3_clean
  right: ev_cap3_clean
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: Stage 2 adversarial co-evolution added after Stage 1
  page: 7
- id: c_stage2_v1
  left: ev_robust3_v1
  right: ev_cap3_v1
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: Stage 2 adversarial co-evolution added after Stage 1
  page: 7
- id: c_stage2_v2
  left: ev_robust3_v2
  right: ev_cap3_v2
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: Stage 2 adversarial co-evolution added after Stage 1
  page: 7
- id: c_stage2_v3
  left: ev_robust3_v3
  right: ev_cap3_v3
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: Stage 2 adversarial co-evolution added after Stage 1
  page: 7
- id: c_robust3_kimi
  left: ev_robust3_kimi
  right: ev_base_kimi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
claims:
- id: k1
  text: AdvSim2Real raises completion under an unseen frontier-model adversary by 33.6% relative to the base agent
  location: abstract
  anchor: raises com- pletion under this unseen adversary by 33.6% relative
  jobs:
  - R2
  supported_by:
  - c_robust3_kimi
  - c_robust3_v1
  - c_robust3_clean
  page: 1
coverage:
  R1: not_reported
  R2clean: direct
  R2attack: direct
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: attack success is not reported; the adversary's success flips are a training reward
    R2clean: Table 1 (clean)
    R2attack: Tables 1 and 3 (completion under injection)
schema_gaps:
- where: evaluations ev_*
  note: all coded results are measured inside a frozen web world model with an LLM judge, not in a real browser; the browser runs of Table 2 cover clean capability only (p. 8)
- where: evaluations ev_*_v1, ev_*_v2, ev_*_v3
  note: Adv v1 to v3 were trained against the Robust lineage, so they are adaptive for Robust iter 3 and transferred for Base and Capability iter 3

`ltbd_arxiv2026` (2610.11634)

In [ ]:
%%writefile reports/records/ltbd_arxiv2026.yaml
report:
  id: ltbd_arxiv2026
  slug: '2610.11634'
  title: 'LTBD: Learnable Trust-Boundary Delimiters for Prompt Injection Defense'
  first_author: Luman Zhao
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2610.11634'
  version_read: arXiv v1
  first_date: '2026-10-08'
  read: F
  role: proposes
  code_release: not stated
  extraction_scope: 'Section 3: Table 1 rows No Defense, DefensiveToken, LTBD, StruQ-Full and SecAlign-LoRA for all four models and all six columns; Table 2 (ablation); Table 3 (Delimiter-Spoof); the abstract and introduction
    claims (TextGrad, Reminder, Sandwich and StruQ-LoRA rows not coded)'
defences:
- id: ltbd_l3
  name: LTBD (learnable trust-boundary delimiters)
  origin: this_report
  base_checkpoint: Llama3-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - soft_tokens_frozen
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: measured_other_checkpoint
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 2
  signal_dependence_detail: 'Table 2: Prefix + CE trains the same four tokens as a prefix instead of at the region boundaries (a separately trained checkpoint)'
  components:
  - four new delimiter tokens [INST-START], [INST-END], [DATA-START], [DATA-END]; only their embeddings are trained, the LLM is frozen
  - 'loss: cross-entropy towards the frozen model''s own response on the clean input, plus a DPO-style term against its response on the attacked input (lambda 0.3, beta 0.1)'
  - 'training data: Cleaned Alpaca, half the samples injected with Ignore, Completion or Delimiter-Spoof'
  layout: four learned delimiter tokens wrap the trusted instruction and the untrusted data in one input sequence; chat template use not stated
  layout_compat: &id001
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: we introduce two pairs of special delimiter tokens
  pages:
    components: 2
    isolation: 3
- id: dt_l3
  name: DefensiveToken (run by the authors)
  origin: other_report
  base_checkpoint: Llama3-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - soft_tokens_frozen
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  layout: as described in its own report; not restated here
  anchor: DefensiveToken [16] learns lightweight defensive token embeddings
- id: struq_l3
  name: StruQ-Full (row matching the DefensiveTokens report, Table 3)
  origin: other_report
  base_checkpoint: Llama3-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  layout: as described in its own report
  anchor: We also compare against stronger training-based methods
- id: sec_l3
  name: SecAlign-LoRA (row matching the DefensiveTokens report, Table 3)
  origin: other_report
  base_checkpoint: Llama3-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  layout: as described in its own report
  anchor: We also compare against stronger training-based methods
- id: pre_l3
  name: LTBD tokens as a prefix, cross-entropy only (Prefix + CE)
  origin: this_report
  base_checkpoint: Llama3-8B-Instruct
  axis1:
  - no_signal
  axis2:
  - soft_tokens_frozen
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  layout: four learned tokens placed before the input
  anchor: Ablation study on trust boundary placement and loss type
- id: bce_l3
  name: LTBD at the boundaries, cross-entropy only (Boundary + CE)
  origin: this_report
  base_checkpoint: Llama3-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - soft_tokens_frozen
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  layout: four learned delimiter tokens wrap the trusted instruction and the untrusted data in one input sequence; chat template use not stated
  anchor: Ablation study on trust boundary placement and loss type
- id: ltbd_l31
  name: LTBD (learnable trust-boundary delimiters)
  origin: this_report
  base_checkpoint: Llama3.1-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - soft_tokens_frozen
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: measured_other_checkpoint
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 2
  signal_dependence_detail: 'Table 2: Prefix + CE trains the same four tokens as a prefix instead of at the region boundaries (a separately trained checkpoint)'
  components:
  - four new delimiter tokens [INST-START], [INST-END], [DATA-START], [DATA-END]; only their embeddings are trained, the LLM is frozen
  - 'loss: cross-entropy towards the frozen model''s own response on the clean input, plus a DPO-style term against its response on the attacked input (lambda 0.3, beta 0.1)'
  - 'training data: Cleaned Alpaca, half the samples injected with Ignore, Completion or Delimiter-Spoof'
  layout: four learned delimiter tokens wrap the trusted instruction and the untrusted data in one input sequence; chat template use not stated
  layout_compat: *id001
  anchor: we introduce two pairs of special delimiter tokens
  pages:
    components: 2
    isolation: 3
- id: dt_l31
  name: DefensiveToken (run by the authors)
  origin: other_report
  base_checkpoint: Llama3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - soft_tokens_frozen
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  layout: as described in its own report; not restated here
  anchor: DefensiveToken [16] learns lightweight defensive token embeddings
- id: struq_l31
  name: StruQ-Full (row matching the DefensiveTokens report, Table 3)
  origin: other_report
  base_checkpoint: Llama3.1-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  layout: as described in its own report
  anchor: We also compare against stronger training-based methods
- id: sec_l31
  name: SecAlign-LoRA (row matching the DefensiveTokens report, Table 3)
  origin: other_report
  base_checkpoint: Llama3.1-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  layout: as described in its own report
  anchor: We also compare against stronger training-based methods
- id: pre_l31
  name: LTBD tokens as a prefix, cross-entropy only (Prefix + CE)
  origin: this_report
  base_checkpoint: Llama3.1-8B-Instruct
  axis1:
  - no_signal
  axis2:
  - soft_tokens_frozen
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  layout: four learned tokens placed before the input
  anchor: Ablation study on trust boundary placement and loss type
- id: bce_l31
  name: LTBD at the boundaries, cross-entropy only (Boundary + CE)
  origin: this_report
  base_checkpoint: Llama3.1-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - soft_tokens_frozen
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  layout: four learned delimiter tokens wrap the trusted instruction and the untrusted data in one input sequence; chat template use not stated
  anchor: Ablation study on trust boundary placement and loss type
- id: ltbd_f3
  name: LTBD (learnable trust-boundary delimiters)
  origin: this_report
  base_checkpoint: Falcon3-7B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - soft_tokens_frozen
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: measured_other_checkpoint
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 2
  signal_dependence_detail: 'Table 2: Prefix + CE trains the same four tokens as a prefix instead of at the region boundaries (a separately trained checkpoint)'
  components:
  - four new delimiter tokens [INST-START], [INST-END], [DATA-START], [DATA-END]; only their embeddings are trained, the LLM is frozen
  - 'loss: cross-entropy towards the frozen model''s own response on the clean input, plus a DPO-style term against its response on the attacked input (lambda 0.3, beta 0.1)'
  - 'training data: Cleaned Alpaca, half the samples injected with Ignore, Completion or Delimiter-Spoof'
  layout: four learned delimiter tokens wrap the trusted instruction and the untrusted data in one input sequence; chat template use not stated
  layout_compat: *id001
  anchor: we introduce two pairs of special delimiter tokens
  pages:
    components: 2
    isolation: 3
- id: dt_f3
  name: DefensiveToken (run by the authors)
  origin: other_report
  base_checkpoint: Falcon3-7B-Instruct
  axis1:
  - chat_roles
  axis2:
  - soft_tokens_frozen
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  layout: as described in its own report; not restated here
  anchor: DefensiveToken [16] learns lightweight defensive token embeddings
- id: struq_f3
  name: StruQ-Full (row matching the DefensiveTokens report, Table 3)
  origin: other_report
  base_checkpoint: Falcon3-7B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  layout: as described in its own report
  anchor: We also compare against stronger training-based methods
- id: sec_f3
  name: SecAlign-LoRA (row matching the DefensiveTokens report, Table 3)
  origin: other_report
  base_checkpoint: Falcon3-7B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  layout: as described in its own report
  anchor: We also compare against stronger training-based methods
- id: ltbd_q25
  name: LTBD (learnable trust-boundary delimiters)
  origin: this_report
  base_checkpoint: Qwen2.5-7B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - soft_tokens_frozen
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: measured_other_checkpoint
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 2
  signal_dependence_detail: 'Table 2: Prefix + CE trains the same four tokens as a prefix instead of at the region boundaries (a separately trained checkpoint)'
  components:
  - four new delimiter tokens [INST-START], [INST-END], [DATA-START], [DATA-END]; only their embeddings are trained, the LLM is frozen
  - 'loss: cross-entropy towards the frozen model''s own response on the clean input, plus a DPO-style term against its response on the attacked input (lambda 0.3, beta 0.1)'
  - 'training data: Cleaned Alpaca, half the samples injected with Ignore, Completion or Delimiter-Spoof'
  layout: four learned delimiter tokens wrap the trusted instruction and the untrusted data in one input sequence; chat template use not stated
  layout_compat: *id001
  anchor: we introduce two pairs of special delimiter tokens
  pages:
    components: 2
    isolation: 3
- id: dt_q25
  name: DefensiveToken (run by the authors)
  origin: other_report
  base_checkpoint: Qwen2.5-7B-Instruct
  axis1:
  - chat_roles
  axis2:
  - soft_tokens_frozen
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  layout: as described in its own report; not restated here
  anchor: DefensiveToken [16] learns lightweight defensive token embeddings
- id: struq_q25
  name: StruQ-Full (row matching the DefensiveTokens report, Table 3)
  origin: other_report
  base_checkpoint: Qwen2.5-7B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  layout: as described in its own report
  anchor: We also compare against stronger training-based methods
- id: sec_q25
  name: SecAlign-LoRA (row matching the DefensiveTokens report, Table 3)
  origin: other_report
  base_checkpoint: Qwen2.5-7B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  layout: as described in its own report
  anchor: We also compare against stronger training-based methods
evaluations:
- id: ev_l3_none_alp_win
  defence_id: none:Llama3-8B-Instruct
  benchmark: AlpacaEval2 on AlpacaFarm, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  baseline_kind: off_the_shelf
- id: ev_l3_none_alp_asr
  defence_id: none:Llama3-8B-Instruct
  benchmark: AlpacaFarm with Ignore, Completion and Ignore-Completion (StruQ protocol)
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  attacker_budget: worst of Ignore, Completion and Ignore-Completion
  baseline_kind: off_the_shelf
- id: ev_l3_none_sep_win
  defence_id: none:Llama3-8B-Instruct
  benchmark: SEP, AlpacaEval2 protocol, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  baseline_kind: off_the_shelf
- id: ev_l3_none_sep_asr
  defence_id: none:Llama3-8B-Instruct
  benchmark: SEP with its instance-specific injected instructions
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  baseline_kind: off_the_shelf
- id: ev_l3_none_tt_asr
  defence_id: none:Llama3-8B-Instruct
  benchmark: TaskTracker, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  baseline_kind: off_the_shelf
- id: ev_l3_none_cse_asr
  defence_id: none:Llama3-8B-Instruct
  benchmark: CyberSecEval2 prompt injection, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  baseline_kind: off_the_shelf
- id: ev_l3_dt_alp_win
  defence_id: dt_l3
  benchmark: AlpacaEval2 on AlpacaFarm, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l3_dt_alp_asr
  defence_id: dt_l3
  benchmark: AlpacaFarm with Ignore, Completion and Ignore-Completion (StruQ protocol)
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  attacker_budget: worst of Ignore, Completion and Ignore-Completion
- id: ev_l3_dt_sep_win
  defence_id: dt_l3
  benchmark: SEP, AlpacaEval2 protocol, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l3_dt_sep_asr
  defence_id: dt_l3
  benchmark: SEP with its instance-specific injected instructions
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l3_dt_tt_asr
  defence_id: dt_l3
  benchmark: TaskTracker, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l3_dt_cse_asr
  defence_id: dt_l3
  benchmark: CyberSecEval2 prompt injection, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l3_ltbd_alp_win
  defence_id: ltbd_l3
  benchmark: AlpacaEval2 on AlpacaFarm, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l3_ltbd_alp_asr
  defence_id: ltbd_l3
  benchmark: AlpacaFarm with Ignore, Completion and Ignore-Completion (StruQ protocol)
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  attacker_budget: worst of Ignore, Completion and Ignore-Completion
- id: ev_l3_ltbd_sep_win
  defence_id: ltbd_l3
  benchmark: SEP, AlpacaEval2 protocol, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l3_ltbd_sep_asr
  defence_id: ltbd_l3
  benchmark: SEP with its instance-specific injected instructions
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l3_ltbd_tt_asr
  defence_id: ltbd_l3
  benchmark: TaskTracker, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l3_ltbd_cse_asr
  defence_id: ltbd_l3
  benchmark: CyberSecEval2 prompt injection, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l3_struq_alp_win
  defence_id: struq_l3
  benchmark: AlpacaEval2 on AlpacaFarm, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l3_struq_alp_asr
  defence_id: struq_l3
  benchmark: AlpacaFarm with Ignore, Completion and Ignore-Completion (StruQ protocol)
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
  attacker_budget: worst of Ignore, Completion and Ignore-Completion
- id: ev_l3_struq_sep_win
  defence_id: struq_l3
  benchmark: SEP, AlpacaEval2 protocol, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l3_struq_sep_asr
  defence_id: struq_l3
  benchmark: SEP with its instance-specific injected instructions
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l3_struq_tt_asr
  defence_id: struq_l3
  benchmark: TaskTracker, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l3_struq_cse_asr
  defence_id: struq_l3
  benchmark: CyberSecEval2 prompt injection, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l3_sec_alp_win
  defence_id: sec_l3
  benchmark: AlpacaEval2 on AlpacaFarm, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l3_sec_alp_asr
  defence_id: sec_l3
  benchmark: AlpacaFarm with Ignore, Completion and Ignore-Completion (StruQ protocol)
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
  attacker_budget: worst of Ignore, Completion and Ignore-Completion
- id: ev_l3_sec_sep_win
  defence_id: sec_l3
  benchmark: SEP, AlpacaEval2 protocol, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l3_sec_sep_asr
  defence_id: sec_l3
  benchmark: SEP with its instance-specific injected instructions
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l3_sec_tt_asr
  defence_id: sec_l3
  benchmark: TaskTracker, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l3_sec_cse_asr
  defence_id: sec_l3
  benchmark: CyberSecEval2 prompt injection, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l31_none_alp_win
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: AlpacaEval2 on AlpacaFarm, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  baseline_kind: off_the_shelf
- id: ev_l31_none_alp_asr
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: AlpacaFarm with Ignore, Completion and Ignore-Completion (StruQ protocol)
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  attacker_budget: worst of Ignore, Completion and Ignore-Completion
  baseline_kind: off_the_shelf
- id: ev_l31_none_sep_win
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: SEP, AlpacaEval2 protocol, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  baseline_kind: off_the_shelf
- id: ev_l31_none_sep_asr
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: SEP with its instance-specific injected instructions
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  baseline_kind: off_the_shelf
- id: ev_l31_none_tt_asr
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: TaskTracker, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  baseline_kind: off_the_shelf
- id: ev_l31_none_cse_asr
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: CyberSecEval2 prompt injection, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  baseline_kind: off_the_shelf
- id: ev_l31_dt_alp_win
  defence_id: dt_l31
  benchmark: AlpacaEval2 on AlpacaFarm, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l31_dt_alp_asr
  defence_id: dt_l31
  benchmark: AlpacaFarm with Ignore, Completion and Ignore-Completion (StruQ protocol)
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  attacker_budget: worst of Ignore, Completion and Ignore-Completion
- id: ev_l31_dt_sep_win
  defence_id: dt_l31
  benchmark: SEP, AlpacaEval2 protocol, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l31_dt_sep_asr
  defence_id: dt_l31
  benchmark: SEP with its instance-specific injected instructions
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l31_dt_tt_asr
  defence_id: dt_l31
  benchmark: TaskTracker, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l31_dt_cse_asr
  defence_id: dt_l31
  benchmark: CyberSecEval2 prompt injection, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l31_ltbd_alp_win
  defence_id: ltbd_l31
  benchmark: AlpacaEval2 on AlpacaFarm, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l31_ltbd_alp_asr
  defence_id: ltbd_l31
  benchmark: AlpacaFarm with Ignore, Completion and Ignore-Completion (StruQ protocol)
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  attacker_budget: worst of Ignore, Completion and Ignore-Completion
- id: ev_l31_ltbd_sep_win
  defence_id: ltbd_l31
  benchmark: SEP, AlpacaEval2 protocol, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l31_ltbd_sep_asr
  defence_id: ltbd_l31
  benchmark: SEP with its instance-specific injected instructions
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l31_ltbd_tt_asr
  defence_id: ltbd_l31
  benchmark: TaskTracker, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l31_ltbd_cse_asr
  defence_id: ltbd_l31
  benchmark: CyberSecEval2 prompt injection, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l31_struq_alp_win
  defence_id: struq_l31
  benchmark: AlpacaEval2 on AlpacaFarm, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l31_struq_alp_asr
  defence_id: struq_l31
  benchmark: AlpacaFarm with Ignore, Completion and Ignore-Completion (StruQ protocol)
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
  attacker_budget: worst of Ignore, Completion and Ignore-Completion
- id: ev_l31_struq_sep_win
  defence_id: struq_l31
  benchmark: SEP, AlpacaEval2 protocol, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l31_struq_sep_asr
  defence_id: struq_l31
  benchmark: SEP with its instance-specific injected instructions
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l31_struq_tt_asr
  defence_id: struq_l31
  benchmark: TaskTracker, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l31_struq_cse_asr
  defence_id: struq_l31
  benchmark: CyberSecEval2 prompt injection, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l31_sec_alp_win
  defence_id: sec_l31
  benchmark: AlpacaEval2 on AlpacaFarm, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l31_sec_alp_asr
  defence_id: sec_l31
  benchmark: AlpacaFarm with Ignore, Completion and Ignore-Completion (StruQ protocol)
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
  attacker_budget: worst of Ignore, Completion and Ignore-Completion
- id: ev_l31_sec_sep_win
  defence_id: sec_l31
  benchmark: SEP, AlpacaEval2 protocol, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l31_sec_sep_asr
  defence_id: sec_l31
  benchmark: SEP with its instance-specific injected instructions
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l31_sec_tt_asr
  defence_id: sec_l31
  benchmark: TaskTracker, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l31_sec_cse_asr
  defence_id: sec_l31
  benchmark: CyberSecEval2 prompt injection, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_f3_none_alp_win
  defence_id: none:Falcon3-7B-Instruct
  benchmark: AlpacaEval2 on AlpacaFarm, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  baseline_kind: off_the_shelf
- id: ev_f3_none_alp_asr
  defence_id: none:Falcon3-7B-Instruct
  benchmark: AlpacaFarm with Ignore, Completion and Ignore-Completion (StruQ protocol)
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  attacker_budget: worst of Ignore, Completion and Ignore-Completion
  baseline_kind: off_the_shelf
- id: ev_f3_none_sep_win
  defence_id: none:Falcon3-7B-Instruct
  benchmark: SEP, AlpacaEval2 protocol, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  baseline_kind: off_the_shelf
- id: ev_f3_none_sep_asr
  defence_id: none:Falcon3-7B-Instruct
  benchmark: SEP with its instance-specific injected instructions
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  baseline_kind: off_the_shelf
- id: ev_f3_none_tt_asr
  defence_id: none:Falcon3-7B-Instruct
  benchmark: TaskTracker, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  baseline_kind: off_the_shelf
- id: ev_f3_none_cse_asr
  defence_id: none:Falcon3-7B-Instruct
  benchmark: CyberSecEval2 prompt injection, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  baseline_kind: off_the_shelf
- id: ev_f3_dt_alp_win
  defence_id: dt_f3
  benchmark: AlpacaEval2 on AlpacaFarm, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_f3_dt_alp_asr
  defence_id: dt_f3
  benchmark: AlpacaFarm with Ignore, Completion and Ignore-Completion (StruQ protocol)
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  attacker_budget: worst of Ignore, Completion and Ignore-Completion
- id: ev_f3_dt_sep_win
  defence_id: dt_f3
  benchmark: SEP, AlpacaEval2 protocol, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_f3_dt_sep_asr
  defence_id: dt_f3
  benchmark: SEP with its instance-specific injected instructions
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_f3_dt_tt_asr
  defence_id: dt_f3
  benchmark: TaskTracker, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_f3_dt_cse_asr
  defence_id: dt_f3
  benchmark: CyberSecEval2 prompt injection, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_f3_ltbd_alp_win
  defence_id: ltbd_f3
  benchmark: AlpacaEval2 on AlpacaFarm, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_f3_ltbd_alp_asr
  defence_id: ltbd_f3
  benchmark: AlpacaFarm with Ignore, Completion and Ignore-Completion (StruQ protocol)
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  attacker_budget: worst of Ignore, Completion and Ignore-Completion
- id: ev_f3_ltbd_sep_win
  defence_id: ltbd_f3
  benchmark: SEP, AlpacaEval2 protocol, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_f3_ltbd_sep_asr
  defence_id: ltbd_f3
  benchmark: SEP with its instance-specific injected instructions
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_f3_ltbd_tt_asr
  defence_id: ltbd_f3
  benchmark: TaskTracker, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_f3_ltbd_cse_asr
  defence_id: ltbd_f3
  benchmark: CyberSecEval2 prompt injection, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_f3_struq_alp_win
  defence_id: struq_f3
  benchmark: AlpacaEval2 on AlpacaFarm, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_f3_struq_alp_asr
  defence_id: struq_f3
  benchmark: AlpacaFarm with Ignore, Completion and Ignore-Completion (StruQ protocol)
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
  attacker_budget: worst of Ignore, Completion and Ignore-Completion
- id: ev_f3_struq_sep_win
  defence_id: struq_f3
  benchmark: SEP, AlpacaEval2 protocol, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_f3_struq_sep_asr
  defence_id: struq_f3
  benchmark: SEP with its instance-specific injected instructions
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_f3_struq_tt_asr
  defence_id: struq_f3
  benchmark: TaskTracker, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_f3_struq_cse_asr
  defence_id: struq_f3
  benchmark: CyberSecEval2 prompt injection, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_f3_sec_alp_win
  defence_id: sec_f3
  benchmark: AlpacaEval2 on AlpacaFarm, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_f3_sec_alp_asr
  defence_id: sec_f3
  benchmark: AlpacaFarm with Ignore, Completion and Ignore-Completion (StruQ protocol)
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
  attacker_budget: worst of Ignore, Completion and Ignore-Completion
- id: ev_f3_sec_sep_win
  defence_id: sec_f3
  benchmark: SEP, AlpacaEval2 protocol, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_f3_sec_sep_asr
  defence_id: sec_f3
  benchmark: SEP with its instance-specific injected instructions
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_f3_sec_tt_asr
  defence_id: sec_f3
  benchmark: TaskTracker, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_f3_sec_cse_asr
  defence_id: sec_f3
  benchmark: CyberSecEval2 prompt injection, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_q25_none_alp_win
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: AlpacaEval2 on AlpacaFarm, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  baseline_kind: off_the_shelf
- id: ev_q25_none_alp_asr
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: AlpacaFarm with Ignore, Completion and Ignore-Completion (StruQ protocol)
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  attacker_budget: worst of Ignore, Completion and Ignore-Completion
  baseline_kind: off_the_shelf
- id: ev_q25_none_sep_win
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: SEP, AlpacaEval2 protocol, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  baseline_kind: off_the_shelf
- id: ev_q25_none_sep_asr
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: SEP with its instance-specific injected instructions
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  baseline_kind: off_the_shelf
- id: ev_q25_none_tt_asr
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: TaskTracker, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  baseline_kind: off_the_shelf
- id: ev_q25_none_cse_asr
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: CyberSecEval2 prompt injection, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  baseline_kind: off_the_shelf
- id: ev_q25_dt_alp_win
  defence_id: dt_q25
  benchmark: AlpacaEval2 on AlpacaFarm, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_q25_dt_alp_asr
  defence_id: dt_q25
  benchmark: AlpacaFarm with Ignore, Completion and Ignore-Completion (StruQ protocol)
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  attacker_budget: worst of Ignore, Completion and Ignore-Completion
- id: ev_q25_dt_sep_win
  defence_id: dt_q25
  benchmark: SEP, AlpacaEval2 protocol, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_q25_dt_sep_asr
  defence_id: dt_q25
  benchmark: SEP with its instance-specific injected instructions
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_q25_dt_tt_asr
  defence_id: dt_q25
  benchmark: TaskTracker, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_q25_dt_cse_asr
  defence_id: dt_q25
  benchmark: CyberSecEval2 prompt injection, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_q25_ltbd_alp_win
  defence_id: ltbd_q25
  benchmark: AlpacaEval2 on AlpacaFarm, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_q25_ltbd_alp_asr
  defence_id: ltbd_q25
  benchmark: AlpacaFarm with Ignore, Completion and Ignore-Completion (StruQ protocol)
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  attacker_budget: worst of Ignore, Completion and Ignore-Completion
- id: ev_q25_ltbd_sep_win
  defence_id: ltbd_q25
  benchmark: SEP, AlpacaEval2 protocol, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_q25_ltbd_sep_asr
  defence_id: ltbd_q25
  benchmark: SEP with its instance-specific injected instructions
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_q25_ltbd_tt_asr
  defence_id: ltbd_q25
  benchmark: TaskTracker, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_q25_ltbd_cse_asr
  defence_id: ltbd_q25
  benchmark: CyberSecEval2 prompt injection, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_q25_struq_alp_win
  defence_id: struq_q25
  benchmark: AlpacaEval2 on AlpacaFarm, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_q25_struq_alp_asr
  defence_id: struq_q25
  benchmark: AlpacaFarm with Ignore, Completion and Ignore-Completion (StruQ protocol)
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
  attacker_budget: worst of Ignore, Completion and Ignore-Completion
- id: ev_q25_struq_sep_win
  defence_id: struq_q25
  benchmark: SEP, AlpacaEval2 protocol, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_q25_struq_sep_asr
  defence_id: struq_q25
  benchmark: SEP with its instance-specific injected instructions
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_q25_struq_tt_asr
  defence_id: struq_q25
  benchmark: TaskTracker, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_q25_struq_cse_asr
  defence_id: struq_q25
  benchmark: CyberSecEval2 prompt injection, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_q25_sec_alp_win
  defence_id: sec_q25
  benchmark: AlpacaEval2 on AlpacaFarm, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_q25_sec_alp_asr
  defence_id: sec_q25
  benchmark: AlpacaFarm with Ignore, Completion and Ignore-Completion (StruQ protocol)
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
  attacker_budget: worst of Ignore, Completion and Ignore-Completion
- id: ev_q25_sec_sep_win
  defence_id: sec_q25
  benchmark: SEP, AlpacaEval2 protocol, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_q25_sec_sep_asr
  defence_id: sec_q25
  benchmark: SEP with its instance-specific injected instructions
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_q25_sec_tt_asr
  defence_id: sec_q25
  benchmark: TaskTracker, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_q25_sec_cse_asr
  defence_id: sec_q25
  benchmark: CyberSecEval2 prompt injection, benchmark-defined attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: defensivetokens_aisec2025
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l3_pre_alp_win
  defence_id: pre_l3
  benchmark: AlpacaEval2 on AlpacaFarm, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l3_pre_alp_asr
  defence_id: pre_l3
  benchmark: AlpacaFarm with Ignore, Completion and Ignore-Completion (StruQ protocol)
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  attacker_budget: worst of Ignore, Completion and Ignore-Completion
- id: ev_l3_pre_sep_asr
  defence_id: pre_l3
  benchmark: SEP with its instance-specific injected instructions
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l3_bce_alp_win
  defence_id: bce_l3
  benchmark: AlpacaEval2 on AlpacaFarm, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l3_bce_alp_asr
  defence_id: bce_l3
  benchmark: AlpacaFarm with Ignore, Completion and Ignore-Completion (StruQ protocol)
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  attacker_budget: worst of Ignore, Completion and Ignore-Completion
- id: ev_l3_bce_sep_asr
  defence_id: bce_l3
  benchmark: SEP with its instance-specific injected instructions
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l31_pre_alp_win
  defence_id: pre_l31
  benchmark: AlpacaEval2 on AlpacaFarm, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l31_pre_alp_asr
  defence_id: pre_l31
  benchmark: AlpacaFarm with Ignore, Completion and Ignore-Completion (StruQ protocol)
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  attacker_budget: worst of Ignore, Completion and Ignore-Completion
- id: ev_l31_pre_sep_asr
  defence_id: pre_l31
  benchmark: SEP with its instance-specific injected instructions
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l31_bce_alp_win
  defence_id: bce_l31
  benchmark: AlpacaEval2 on AlpacaFarm, no injection
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o win rate against GPT-4 Turbo references
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l31_bce_alp_asr
  defence_id: bce_l31
  benchmark: AlpacaFarm with Ignore, Completion and Ignore-Completion (StruQ protocol)
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
  attacker_budget: worst of Ignore, Completion and Ignore-Completion
- id: ev_l31_bce_sep_asr
  defence_id: bce_l31
  benchmark: SEP with its instance-specific injected instructions
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 3
  decoding: not reported
  pages:
    judge: 4
- id: ev_l3_ltbd_plain
  defence_id: ltbd_l3
  benchmark: benchmark not stated; plain injection
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 4
  decoding: not reported
- id: ev_l3_ltbd_spoof
  defence_id: ltbd_l3
  benchmark: benchmark not stated; Delimiter-Spoof (five forged-delimiter templates around the injection)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 4
  attacker_budget: five fixed templates written with knowledge of the delimiters; worst case of the five reported
  decoding: not reported
- id: ev_l31_ltbd_plain
  defence_id: ltbd_l31
  benchmark: benchmark not stated; plain injection
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 4
  decoding: not reported
- id: ev_l31_ltbd_spoof
  defence_id: ltbd_l31
  benchmark: benchmark not stated; Delimiter-Spoof (five forged-delimiter templates around the injection)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 4
  attacker_budget: five fixed templates written with knowledge of the delimiters; worst case of the five reported
  decoding: not reported
- id: ev_f3_ltbd_plain
  defence_id: ltbd_f3
  benchmark: benchmark not stated; plain injection
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 4
  decoding: not reported
- id: ev_f3_ltbd_spoof
  defence_id: ltbd_f3
  benchmark: benchmark not stated; Delimiter-Spoof (five forged-delimiter templates around the injection)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 4
  attacker_budget: five fixed templates written with knowledge of the delimiters; worst case of the five reported
  decoding: not reported
- id: ev_q25_ltbd_plain
  defence_id: ltbd_q25
  benchmark: benchmark not stated; plain injection
  attacker_type: static
  attacker_access: black_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 4
  decoding: not reported
- id: ev_q25_ltbd_spoof
  defence_id: ltbd_q25
  benchmark: benchmark not stated; Delimiter-Spoof (five forged-delimiter templates around the injection)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: 'attack success: the reply follows the injected instruction rather than the task; judge not named'
  layout_same_as_defence: 'yes'
  measured_by: ltbd_arxiv2026
  page: 4
  attacker_budget: five fixed templates written with knowledge of the delimiters; worst case of the five reported
  decoding: not reported
results:
- id: r_l3_none_alp_win
  evaluation_id: ev_l3_none_alp_win
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaFarm WinRate
  value: '33.82'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l3_none_alp_asr
  evaluation_id: ev_l3_none_alp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR (worst case)
  value: '55.29'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l3_none_sep_win
  evaluation_id: ev_l3_none_sep_win
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP WinRate
  value: '59.72'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l3_none_sep_asr
  evaluation_id: ev_l3_none_sep_asr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '74.47'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l3_none_tt_asr
  evaluation_id: ev_l3_none_tt_asr
  quantity: R1
  coverage: direct
  metric_as_printed: TaskTracker ASR
  value: '16.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l3_none_cse_asr
  evaluation_id: ev_l3_none_cse_asr
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval2 ASR
  value: '45.45'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l3_dt_alp_win
  evaluation_id: ev_l3_dt_alp_win
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaFarm WinRate
  value: '33.46'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l3_dt_alp_asr
  evaluation_id: ev_l3_dt_alp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR (worst case)
  value: '0.48'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l3_dt_sep_win
  evaluation_id: ev_l3_dt_sep_win
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP WinRate
  value: '58.60'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l3_dt_sep_asr
  evaluation_id: ev_l3_dt_sep_asr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '3.20'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l3_dt_tt_asr
  evaluation_id: ev_l3_dt_tt_asr
  quantity: R1
  coverage: direct
  metric_as_printed: TaskTracker ASR
  value: '0.27'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l3_dt_cse_asr
  evaluation_id: ev_l3_dt_cse_asr
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval2 ASR
  value: '3.64'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l3_ltbd_alp_win
  evaluation_id: ev_l3_ltbd_alp_win
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaFarm WinRate
  value: '35.73'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l3_ltbd_alp_asr
  evaluation_id: ev_l3_ltbd_alp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR (worst case)
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l3_ltbd_sep_win
  evaluation_id: ev_l3_ltbd_sep_win
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP WinRate
  value: '60.55'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l3_ltbd_sep_asr
  evaluation_id: ev_l3_ltbd_sep_asr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '2.04'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l3_ltbd_tt_asr
  evaluation_id: ev_l3_ltbd_tt_asr
  quantity: R1
  coverage: direct
  metric_as_printed: TaskTracker ASR
  value: '0.13'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l3_ltbd_cse_asr
  evaluation_id: ev_l3_ltbd_cse_asr
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval2 ASR
  value: '7.27'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l3_struq_alp_win
  evaluation_id: ev_l3_struq_alp_win
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaFarm WinRate
  value: '27.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l3_struq_alp_asr
  evaluation_id: ev_l3_struq_alp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR (worst case)
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l3_struq_sep_win
  evaluation_id: ev_l3_struq_sep_win
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP WinRate
  value: '51.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l3_struq_sep_asr
  evaluation_id: ev_l3_struq_sep_asr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '0.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l3_struq_tt_asr
  evaluation_id: ev_l3_struq_tt_asr
  quantity: R1
  coverage: direct
  metric_as_printed: TaskTracker ASR
  value: '0.23'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l3_struq_cse_asr
  evaluation_id: ev_l3_struq_cse_asr
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval2 ASR
  value: '10.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l3_sec_alp_win
  evaluation_id: ev_l3_sec_alp_win
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaFarm WinRate
  value: '27.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l3_sec_alp_asr
  evaluation_id: ev_l3_sec_alp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR (worst case)
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l3_sec_sep_win
  evaluation_id: ev_l3_sec_sep_win
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP WinRate
  value: '47.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l3_sec_sep_asr
  evaluation_id: ev_l3_sec_sep_asr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '3.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l3_sec_tt_asr
  evaluation_id: ev_l3_sec_tt_asr
  quantity: R1
  coverage: direct
  metric_as_printed: TaskTracker ASR
  value: '0.18'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l3_sec_cse_asr
  evaluation_id: ev_l3_sec_cse_asr
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval2 ASR
  value: '18.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l31_none_alp_win
  evaluation_id: ev_l31_none_alp_win
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaFarm WinRate
  value: '31.37'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l31_none_alp_asr
  evaluation_id: ev_l31_none_alp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR (worst case)
  value: '74.52'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l31_none_sep_win
  evaluation_id: ev_l31_none_sep_win
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP WinRate
  value: '59.04'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l31_none_sep_asr
  evaluation_id: ev_l31_none_sep_asr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '75.04'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l31_none_tt_asr
  evaluation_id: ev_l31_none_tt_asr
  quantity: R1
  coverage: direct
  metric_as_printed: TaskTracker ASR
  value: '26.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l31_none_cse_asr
  evaluation_id: ev_l31_none_cse_asr
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval2 ASR
  value: '38.18'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l31_dt_alp_win
  evaluation_id: ev_l31_dt_alp_win
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaFarm WinRate
  value: '27.94'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l31_dt_alp_asr
  evaluation_id: ev_l31_dt_alp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR (worst case)
  value: '0.48'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l31_dt_sep_win
  evaluation_id: ev_l31_dt_sep_win
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP WinRate
  value: '57.90'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l31_dt_sep_asr
  evaluation_id: ev_l31_dt_sep_asr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '2.81'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l31_dt_tt_asr
  evaluation_id: ev_l31_dt_tt_asr
  quantity: R1
  coverage: direct
  metric_as_printed: TaskTracker ASR
  value: '0.19'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l31_dt_cse_asr
  evaluation_id: ev_l31_dt_cse_asr
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval2 ASR
  value: '7.27'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l31_ltbd_alp_win
  evaluation_id: ev_l31_ltbd_alp_win
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaFarm WinRate
  value: '30.98'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l31_ltbd_alp_asr
  evaluation_id: ev_l31_ltbd_alp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR (worst case)
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l31_ltbd_sep_win
  evaluation_id: ev_l31_ltbd_sep_win
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP WinRate
  value: '54.92'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l31_ltbd_sep_asr
  evaluation_id: ev_l31_ltbd_sep_asr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '2.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l31_ltbd_tt_asr
  evaluation_id: ev_l31_ltbd_tt_asr
  quantity: R1
  coverage: direct
  metric_as_printed: TaskTracker ASR
  value: '0.19'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l31_ltbd_cse_asr
  evaluation_id: ev_l31_ltbd_cse_asr
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval2 ASR
  value: '7.27'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l31_struq_alp_win
  evaluation_id: ev_l31_struq_alp_win
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaFarm WinRate
  value: '28.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l31_struq_alp_asr
  evaluation_id: ev_l31_struq_alp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR (worst case)
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l31_struq_sep_win
  evaluation_id: ev_l31_struq_sep_win
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP WinRate
  value: '52.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l31_struq_sep_asr
  evaluation_id: ev_l31_struq_sep_asr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '0.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l31_struq_tt_asr
  evaluation_id: ev_l31_struq_tt_asr
  quantity: R1
  coverage: direct
  metric_as_printed: TaskTracker ASR
  value: '0.18'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l31_struq_cse_asr
  evaluation_id: ev_l31_struq_cse_asr
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval2 ASR
  value: '10.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l31_sec_alp_win
  evaluation_id: ev_l31_sec_alp_win
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaFarm WinRate
  value: '27.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l31_sec_alp_asr
  evaluation_id: ev_l31_sec_alp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR (worst case)
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l31_sec_sep_win
  evaluation_id: ev_l31_sec_sep_win
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP WinRate
  value: '50.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l31_sec_sep_asr
  evaluation_id: ev_l31_sec_sep_asr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '2.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l31_sec_tt_asr
  evaluation_id: ev_l31_sec_tt_asr
  quantity: R1
  coverage: direct
  metric_as_printed: TaskTracker ASR
  value: '0.19'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l31_sec_cse_asr
  evaluation_id: ev_l31_sec_cse_asr
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval2 ASR
  value: '5.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_f3_none_alp_win
  evaluation_id: ev_f3_none_alp_win
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaFarm WinRate
  value: '35.97'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_f3_none_alp_asr
  evaluation_id: ev_f3_none_alp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR (worst case)
  value: '89.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_f3_none_sep_win
  evaluation_id: ev_f3_none_sep_win
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP WinRate
  value: '55.74'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_f3_none_sep_asr
  evaluation_id: ev_f3_none_sep_asr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '78.49'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_f3_none_tt_asr
  evaluation_id: ev_f3_none_tt_asr
  quantity: R1
  coverage: direct
  metric_as_printed: TaskTracker ASR
  value: '27.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_f3_none_cse_asr
  evaluation_id: ev_f3_none_cse_asr
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval2 ASR
  value: '34.55'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_f3_dt_alp_win
  evaluation_id: ev_f3_dt_alp_win
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaFarm WinRate
  value: '35.13'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_f3_dt_alp_asr
  evaluation_id: ev_f3_dt_alp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR (worst case)
  value: '4.81'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_f3_dt_sep_win
  evaluation_id: ev_f3_dt_sep_win
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP WinRate
  value: '56.60'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_f3_dt_sep_asr
  evaluation_id: ev_f3_dt_sep_asr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '6.70'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_f3_dt_tt_asr
  evaluation_id: ev_f3_dt_tt_asr
  quantity: R1
  coverage: direct
  metric_as_printed: TaskTracker ASR
  value: '0.27'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_f3_dt_cse_asr
  evaluation_id: ev_f3_dt_cse_asr
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval2 ASR
  value: '12.73'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_f3_ltbd_alp_win
  evaluation_id: ev_f3_ltbd_alp_win
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaFarm WinRate
  value: '36.42'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_f3_ltbd_alp_asr
  evaluation_id: ev_f3_ltbd_alp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR (worst case)
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_f3_ltbd_sep_win
  evaluation_id: ev_f3_ltbd_sep_win
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP WinRate
  value: '57.09'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_f3_ltbd_sep_asr
  evaluation_id: ev_f3_ltbd_sep_asr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '1.57'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_f3_ltbd_tt_asr
  evaluation_id: ev_f3_ltbd_tt_asr
  quantity: R1
  coverage: direct
  metric_as_printed: TaskTracker ASR
  value: '0.11'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_f3_ltbd_cse_asr
  evaluation_id: ev_f3_ltbd_cse_asr
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval2 ASR
  value: '1.82'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_f3_struq_alp_win
  evaluation_id: ev_f3_struq_alp_win
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaFarm WinRate
  value: '25.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_f3_struq_alp_asr
  evaluation_id: ev_f3_struq_alp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR (worst case)
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_f3_struq_sep_win
  evaluation_id: ev_f3_struq_sep_win
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP WinRate
  value: '31.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_f3_struq_sep_asr
  evaluation_id: ev_f3_struq_sep_asr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '2.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_f3_struq_tt_asr
  evaluation_id: ev_f3_struq_tt_asr
  quantity: R1
  coverage: direct
  metric_as_printed: TaskTracker ASR
  value: '0.20'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_f3_struq_cse_asr
  evaluation_id: ev_f3_struq_cse_asr
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval2 ASR
  value: '7.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_f3_sec_alp_win
  evaluation_id: ev_f3_sec_alp_win
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaFarm WinRate
  value: '27.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_f3_sec_alp_asr
  evaluation_id: ev_f3_sec_alp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR (worst case)
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_f3_sec_sep_win
  evaluation_id: ev_f3_sec_sep_win
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP WinRate
  value: '46.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_f3_sec_sep_asr
  evaluation_id: ev_f3_sec_sep_asr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '35.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_f3_sec_tt_asr
  evaluation_id: ev_f3_sec_tt_asr
  quantity: R1
  coverage: direct
  metric_as_printed: TaskTracker ASR
  value: '1.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_f3_sec_cse_asr
  evaluation_id: ev_f3_sec_cse_asr
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval2 ASR
  value: '29.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_q25_none_alp_win
  evaluation_id: ev_q25_none_alp_win
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaFarm WinRate
  value: '37.71'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_q25_none_alp_asr
  evaluation_id: ev_q25_none_alp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR (worst case)
  value: '93.75'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_q25_none_sep_win
  evaluation_id: ev_q25_none_sep_win
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP WinRate
  value: '62.86'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_q25_none_sep_asr
  evaluation_id: ev_q25_none_sep_asr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '86.57'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_q25_none_tt_asr
  evaluation_id: ev_q25_none_tt_asr
  quantity: R1
  coverage: direct
  metric_as_printed: TaskTracker ASR
  value: '37.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_q25_none_cse_asr
  evaluation_id: ev_q25_none_cse_asr
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval2 ASR
  value: '50.91'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_q25_dt_alp_win
  evaluation_id: ev_q25_dt_alp_win
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaFarm WinRate
  value: '40.16'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_q25_dt_alp_asr
  evaluation_id: ev_q25_dt_alp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR (worst case)
  value: '0.96'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_q25_dt_sep_win
  evaluation_id: ev_q25_dt_sep_win
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP WinRate
  value: '59.63'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_q25_dt_sep_asr
  evaluation_id: ev_q25_dt_sep_asr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '4.40'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_q25_dt_tt_asr
  evaluation_id: ev_q25_dt_tt_asr
  quantity: R1
  coverage: direct
  metric_as_printed: TaskTracker ASR
  value: '0.25'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_q25_dt_cse_asr
  evaluation_id: ev_q25_dt_cse_asr
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval2 ASR
  value: '20.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_q25_ltbd_alp_win
  evaluation_id: ev_q25_ltbd_alp_win
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaFarm WinRate
  value: '41.71'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_q25_ltbd_alp_asr
  evaluation_id: ev_q25_ltbd_alp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR (worst case)
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_q25_ltbd_sep_win
  evaluation_id: ev_q25_ltbd_sep_win
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP WinRate
  value: '63.90'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_q25_ltbd_sep_asr
  evaluation_id: ev_q25_ltbd_sep_asr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '1.86'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_q25_ltbd_tt_asr
  evaluation_id: ev_q25_ltbd_tt_asr
  quantity: R1
  coverage: direct
  metric_as_printed: TaskTracker ASR
  value: '0.14'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_q25_ltbd_cse_asr
  evaluation_id: ev_q25_ltbd_cse_asr
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval2 ASR
  value: '12.73'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_q25_struq_alp_win
  evaluation_id: ev_q25_struq_alp_win
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaFarm WinRate
  value: '31.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_q25_struq_alp_asr
  evaluation_id: ev_q25_struq_alp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR (worst case)
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_q25_struq_sep_win
  evaluation_id: ev_q25_struq_sep_win
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP WinRate
  value: '50.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_q25_struq_sep_asr
  evaluation_id: ev_q25_struq_sep_asr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '2.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_q25_struq_tt_asr
  evaluation_id: ev_q25_struq_tt_asr
  quantity: R1
  coverage: direct
  metric_as_printed: TaskTracker ASR
  value: '0.20'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_q25_struq_cse_asr
  evaluation_id: ev_q25_struq_cse_asr
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval2 ASR
  value: '3.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_q25_sec_alp_win
  evaluation_id: ev_q25_sec_alp_win
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaFarm WinRate
  value: '32.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_q25_sec_alp_asr
  evaluation_id: ev_q25_sec_alp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR (worst case)
  value: '1.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_q25_sec_sep_win
  evaluation_id: ev_q25_sec_sep_win
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP WinRate
  value: '50.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_q25_sec_sep_asr
  evaluation_id: ev_q25_sec_sep_asr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '14.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_q25_sec_tt_asr
  evaluation_id: ev_q25_sec_tt_asr
  quantity: R1
  coverage: direct
  metric_as_printed: TaskTracker ASR
  value: '0.57'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_q25_sec_cse_asr
  evaluation_id: ev_q25_sec_cse_asr
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval2 ASR
  value: '20.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_l3_pre_alp_win
  evaluation_id: ev_l3_pre_alp_win
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaFarm WinRate
  value: '27.60'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 2
- id: r_l3_pre_alp_asr
  evaluation_id: ev_l3_pre_alp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR (worst case)
  value: '6.73'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 2
- id: r_l3_pre_sep_asr
  evaluation_id: ev_l3_pre_sep_asr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '15.43'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 2
- id: r_l3_bce_alp_win
  evaluation_id: ev_l3_bce_alp_win
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaFarm WinRate
  value: '34.51'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 2
- id: r_l3_bce_alp_asr
  evaluation_id: ev_l3_bce_alp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR (worst case)
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 2
- id: r_l3_bce_sep_asr
  evaluation_id: ev_l3_bce_sep_asr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '2.34'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 2
- id: r_l31_pre_alp_win
  evaluation_id: ev_l31_pre_alp_win
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaFarm WinRate
  value: '27.50'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 2
- id: r_l31_pre_alp_asr
  evaluation_id: ev_l31_pre_alp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR (worst case)
  value: '23.08'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 2
- id: r_l31_pre_sep_asr
  evaluation_id: ev_l31_pre_sep_asr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '33.81'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 2
- id: r_l31_bce_alp_win
  evaluation_id: ev_l31_bce_alp_win
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaFarm WinRate
  value: '30.98'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 2
- id: r_l31_bce_alp_asr
  evaluation_id: ev_l31_bce_alp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR (worst case)
  value: '0.48'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 2
- id: r_l31_bce_sep_asr
  evaluation_id: ev_l31_bce_sep_asr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '3.43'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 2
- id: r_l3_ltbd_plain
  evaluation_id: ev_l3_ltbd_plain
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, plain injection
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_l3_ltbd_spoof
  evaluation_id: ev_l3_ltbd_spoof
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Delimiter-Spoof (worst case)
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_l31_ltbd_plain
  evaluation_id: ev_l31_ltbd_plain
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, plain injection
  value: '0.48'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_l31_ltbd_spoof
  evaluation_id: ev_l31_ltbd_spoof
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Delimiter-Spoof (worst case)
  value: '0.48'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_f3_ltbd_plain
  evaluation_id: ev_f3_ltbd_plain
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, plain injection
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_f3_ltbd_spoof
  evaluation_id: ev_f3_ltbd_spoof
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Delimiter-Spoof (worst case)
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_q25_ltbd_plain
  evaluation_id: ev_q25_ltbd_plain
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, plain injection
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_q25_ltbd_spoof
  evaluation_id: ev_q25_ltbd_spoof
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Delimiter-Spoof (worst case)
  value: '0.48'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
comparisons:
- id: c_l3_alp_win_ltbd_none
  left: ev_l3_ltbd_alp_win
  right: ev_l3_none_alp_win
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_l3_alp_win_ltbd_dt
  left: ev_l3_ltbd_alp_win
  right: ev_l3_dt_alp_win
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: 'the defence: LTBD against DefensiveToken on the same base, both run by the authors'
  page: 3
- id: c_l3_alp_win_ltbd_struq
  left: ev_l3_ltbd_alp_win
  right: ev_l3_struq_alp_win
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_l3_alp_win_ltbd_sec
  left: ev_l3_ltbd_alp_win
  right: ev_l3_sec_alp_win
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_l3_alp_asr_ltbd_none
  left: ev_l3_ltbd_alp_asr
  right: ev_l3_none_alp_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_l3_alp_asr_ltbd_dt
  left: ev_l3_ltbd_alp_asr
  right: ev_l3_dt_alp_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: 'the defence: LTBD against DefensiveToken on the same base, both run by the authors'
  page: 3
- id: c_l3_alp_asr_ltbd_struq
  left: ev_l3_ltbd_alp_asr
  right: ev_l3_struq_alp_asr
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: unreported
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_l3_alp_asr_ltbd_sec
  left: ev_l3_ltbd_alp_asr
  right: ev_l3_sec_alp_asr
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: unreported
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_l3_sep_win_ltbd_none
  left: ev_l3_ltbd_sep_win
  right: ev_l3_none_sep_win
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_l3_sep_win_ltbd_dt
  left: ev_l3_ltbd_sep_win
  right: ev_l3_dt_sep_win
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: 'the defence: LTBD against DefensiveToken on the same base, both run by the authors'
  page: 3
- id: c_l3_sep_win_ltbd_struq
  left: ev_l3_ltbd_sep_win
  right: ev_l3_struq_sep_win
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_l3_sep_win_ltbd_sec
  left: ev_l3_ltbd_sep_win
  right: ev_l3_sec_sep_win
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_l3_sep_asr_ltbd_none
  left: ev_l3_ltbd_sep_asr
  right: ev_l3_none_sep_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_l3_sep_asr_ltbd_dt
  left: ev_l3_ltbd_sep_asr
  right: ev_l3_dt_sep_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: 'the defence: LTBD against DefensiveToken on the same base, both run by the authors'
  page: 3
- id: c_l3_sep_asr_ltbd_struq
  left: ev_l3_ltbd_sep_asr
  right: ev_l3_struq_sep_asr
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: unreported
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_l3_sep_asr_ltbd_sec
  left: ev_l3_ltbd_sep_asr
  right: ev_l3_sec_sep_asr
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: unreported
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_l3_tt_asr_ltbd_none
  left: ev_l3_ltbd_tt_asr
  right: ev_l3_none_tt_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_l3_tt_asr_ltbd_dt
  left: ev_l3_ltbd_tt_asr
  right: ev_l3_dt_tt_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: 'the defence: LTBD against DefensiveToken on the same base, both run by the authors'
  page: 3
- id: c_l3_tt_asr_ltbd_struq
  left: ev_l3_ltbd_tt_asr
  right: ev_l3_struq_tt_asr
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: unreported
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_l3_tt_asr_ltbd_sec
  left: ev_l3_ltbd_tt_asr
  right: ev_l3_sec_tt_asr
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: unreported
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_l3_cse_asr_ltbd_none
  left: ev_l3_ltbd_cse_asr
  right: ev_l3_none_cse_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_l3_cse_asr_ltbd_dt
  left: ev_l3_ltbd_cse_asr
  right: ev_l3_dt_cse_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: 'the defence: LTBD against DefensiveToken on the same base, both run by the authors'
  page: 3
- id: c_l3_cse_asr_ltbd_struq
  left: ev_l3_ltbd_cse_asr
  right: ev_l3_struq_cse_asr
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: unreported
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_l3_cse_asr_ltbd_sec
  left: ev_l3_ltbd_cse_asr
  right: ev_l3_sec_cse_asr
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: unreported
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_l31_alp_win_ltbd_none
  left: ev_l31_ltbd_alp_win
  right: ev_l31_none_alp_win
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_l31_alp_win_ltbd_dt
  left: ev_l31_ltbd_alp_win
  right: ev_l31_dt_alp_win
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: 'the defence: LTBD against DefensiveToken on the same base, both run by the authors'
  page: 3
- id: c_l31_alp_win_ltbd_struq
  left: ev_l31_ltbd_alp_win
  right: ev_l31_struq_alp_win
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_l31_alp_win_ltbd_sec
  left: ev_l31_ltbd_alp_win
  right: ev_l31_sec_alp_win
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_l31_alp_asr_ltbd_none
  left: ev_l31_ltbd_alp_asr
  right: ev_l31_none_alp_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_l31_alp_asr_ltbd_dt
  left: ev_l31_ltbd_alp_asr
  right: ev_l31_dt_alp_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: 'the defence: LTBD against DefensiveToken on the same base, both run by the authors'
  page: 3
- id: c_l31_alp_asr_ltbd_struq
  left: ev_l31_ltbd_alp_asr
  right: ev_l31_struq_alp_asr
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: unreported
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_l31_alp_asr_ltbd_sec
  left: ev_l31_ltbd_alp_asr
  right: ev_l31_sec_alp_asr
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: unreported
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_l31_sep_win_ltbd_none
  left: ev_l31_ltbd_sep_win
  right: ev_l31_none_sep_win
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_l31_sep_win_ltbd_dt
  left: ev_l31_ltbd_sep_win
  right: ev_l31_dt_sep_win
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: 'the defence: LTBD against DefensiveToken on the same base, both run by the authors'
  page: 3
- id: c_l31_sep_win_ltbd_struq
  left: ev_l31_ltbd_sep_win
  right: ev_l31_struq_sep_win
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_l31_sep_win_ltbd_sec
  left: ev_l31_ltbd_sep_win
  right: ev_l31_sec_sep_win
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_l31_sep_asr_ltbd_none
  left: ev_l31_ltbd_sep_asr
  right: ev_l31_none_sep_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_l31_sep_asr_ltbd_dt
  left: ev_l31_ltbd_sep_asr
  right: ev_l31_dt_sep_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: 'the defence: LTBD against DefensiveToken on the same base, both run by the authors'
  page: 3
- id: c_l31_sep_asr_ltbd_struq
  left: ev_l31_ltbd_sep_asr
  right: ev_l31_struq_sep_asr
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: unreported
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_l31_sep_asr_ltbd_sec
  left: ev_l31_ltbd_sep_asr
  right: ev_l31_sec_sep_asr
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: unreported
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_l31_tt_asr_ltbd_none
  left: ev_l31_ltbd_tt_asr
  right: ev_l31_none_tt_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_l31_tt_asr_ltbd_dt
  left: ev_l31_ltbd_tt_asr
  right: ev_l31_dt_tt_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: 'the defence: LTBD against DefensiveToken on the same base, both run by the authors'
  page: 3
- id: c_l31_tt_asr_ltbd_struq
  left: ev_l31_ltbd_tt_asr
  right: ev_l31_struq_tt_asr
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: unreported
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_l31_tt_asr_ltbd_sec
  left: ev_l31_ltbd_tt_asr
  right: ev_l31_sec_tt_asr
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: unreported
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_l31_cse_asr_ltbd_none
  left: ev_l31_ltbd_cse_asr
  right: ev_l31_none_cse_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_l31_cse_asr_ltbd_dt
  left: ev_l31_ltbd_cse_asr
  right: ev_l31_dt_cse_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: 'the defence: LTBD against DefensiveToken on the same base, both run by the authors'
  page: 3
- id: c_l31_cse_asr_ltbd_struq
  left: ev_l31_ltbd_cse_asr
  right: ev_l31_struq_cse_asr
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: unreported
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_l31_cse_asr_ltbd_sec
  left: ev_l31_ltbd_cse_asr
  right: ev_l31_sec_cse_asr
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: unreported
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_f3_alp_win_ltbd_none
  left: ev_f3_ltbd_alp_win
  right: ev_f3_none_alp_win
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_f3_alp_win_ltbd_dt
  left: ev_f3_ltbd_alp_win
  right: ev_f3_dt_alp_win
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: 'the defence: LTBD against DefensiveToken on the same base, both run by the authors'
  page: 3
- id: c_f3_alp_win_ltbd_struq
  left: ev_f3_ltbd_alp_win
  right: ev_f3_struq_alp_win
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_f3_alp_win_ltbd_sec
  left: ev_f3_ltbd_alp_win
  right: ev_f3_sec_alp_win
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_f3_alp_asr_ltbd_none
  left: ev_f3_ltbd_alp_asr
  right: ev_f3_none_alp_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_f3_alp_asr_ltbd_dt
  left: ev_f3_ltbd_alp_asr
  right: ev_f3_dt_alp_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: 'the defence: LTBD against DefensiveToken on the same base, both run by the authors'
  page: 3
- id: c_f3_alp_asr_ltbd_struq
  left: ev_f3_ltbd_alp_asr
  right: ev_f3_struq_alp_asr
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: unreported
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_f3_alp_asr_ltbd_sec
  left: ev_f3_ltbd_alp_asr
  right: ev_f3_sec_alp_asr
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: unreported
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_f3_sep_win_ltbd_none
  left: ev_f3_ltbd_sep_win
  right: ev_f3_none_sep_win
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_f3_sep_win_ltbd_dt
  left: ev_f3_ltbd_sep_win
  right: ev_f3_dt_sep_win
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: 'the defence: LTBD against DefensiveToken on the same base, both run by the authors'
  page: 3
- id: c_f3_sep_win_ltbd_struq
  left: ev_f3_ltbd_sep_win
  right: ev_f3_struq_sep_win
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_f3_sep_win_ltbd_sec
  left: ev_f3_ltbd_sep_win
  right: ev_f3_sec_sep_win
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_f3_sep_asr_ltbd_none
  left: ev_f3_ltbd_sep_asr
  right: ev_f3_none_sep_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_f3_sep_asr_ltbd_dt
  left: ev_f3_ltbd_sep_asr
  right: ev_f3_dt_sep_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: 'the defence: LTBD against DefensiveToken on the same base, both run by the authors'
  page: 3
- id: c_f3_sep_asr_ltbd_struq
  left: ev_f3_ltbd_sep_asr
  right: ev_f3_struq_sep_asr
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: unreported
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_f3_sep_asr_ltbd_sec
  left: ev_f3_ltbd_sep_asr
  right: ev_f3_sec_sep_asr
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: unreported
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_f3_tt_asr_ltbd_none
  left: ev_f3_ltbd_tt_asr
  right: ev_f3_none_tt_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_f3_tt_asr_ltbd_dt
  left: ev_f3_ltbd_tt_asr
  right: ev_f3_dt_tt_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: 'the defence: LTBD against DefensiveToken on the same base, both run by the authors'
  page: 3
- id: c_f3_tt_asr_ltbd_struq
  left: ev_f3_ltbd_tt_asr
  right: ev_f3_struq_tt_asr
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: unreported
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_f3_tt_asr_ltbd_sec
  left: ev_f3_ltbd_tt_asr
  right: ev_f3_sec_tt_asr
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: unreported
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_f3_cse_asr_ltbd_none
  left: ev_f3_ltbd_cse_asr
  right: ev_f3_none_cse_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_f3_cse_asr_ltbd_dt
  left: ev_f3_ltbd_cse_asr
  right: ev_f3_dt_cse_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: 'the defence: LTBD against DefensiveToken on the same base, both run by the authors'
  page: 3
- id: c_f3_cse_asr_ltbd_struq
  left: ev_f3_ltbd_cse_asr
  right: ev_f3_struq_cse_asr
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: unreported
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_f3_cse_asr_ltbd_sec
  left: ev_f3_ltbd_cse_asr
  right: ev_f3_sec_cse_asr
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: unreported
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_q25_alp_win_ltbd_none
  left: ev_q25_ltbd_alp_win
  right: ev_q25_none_alp_win
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_q25_alp_win_ltbd_dt
  left: ev_q25_ltbd_alp_win
  right: ev_q25_dt_alp_win
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: 'the defence: LTBD against DefensiveToken on the same base, both run by the authors'
  page: 3
- id: c_q25_alp_win_ltbd_struq
  left: ev_q25_ltbd_alp_win
  right: ev_q25_struq_alp_win
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_q25_alp_win_ltbd_sec
  left: ev_q25_ltbd_alp_win
  right: ev_q25_sec_alp_win
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_q25_alp_asr_ltbd_none
  left: ev_q25_ltbd_alp_asr
  right: ev_q25_none_alp_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_q25_alp_asr_ltbd_dt
  left: ev_q25_ltbd_alp_asr
  right: ev_q25_dt_alp_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: 'the defence: LTBD against DefensiveToken on the same base, both run by the authors'
  page: 3
- id: c_q25_alp_asr_ltbd_struq
  left: ev_q25_ltbd_alp_asr
  right: ev_q25_struq_alp_asr
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: unreported
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_q25_alp_asr_ltbd_sec
  left: ev_q25_ltbd_alp_asr
  right: ev_q25_sec_alp_asr
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: unreported
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_q25_sep_win_ltbd_none
  left: ev_q25_ltbd_sep_win
  right: ev_q25_none_sep_win
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_q25_sep_win_ltbd_dt
  left: ev_q25_ltbd_sep_win
  right: ev_q25_dt_sep_win
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: 'the defence: LTBD against DefensiveToken on the same base, both run by the authors'
  page: 3
- id: c_q25_sep_win_ltbd_struq
  left: ev_q25_ltbd_sep_win
  right: ev_q25_struq_sep_win
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_q25_sep_win_ltbd_sec
  left: ev_q25_ltbd_sep_win
  right: ev_q25_sec_sep_win
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_q25_sep_asr_ltbd_none
  left: ev_q25_ltbd_sep_asr
  right: ev_q25_none_sep_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_q25_sep_asr_ltbd_dt
  left: ev_q25_ltbd_sep_asr
  right: ev_q25_dt_sep_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: 'the defence: LTBD against DefensiveToken on the same base, both run by the authors'
  page: 3
- id: c_q25_sep_asr_ltbd_struq
  left: ev_q25_ltbd_sep_asr
  right: ev_q25_struq_sep_asr
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: unreported
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_q25_sep_asr_ltbd_sec
  left: ev_q25_ltbd_sep_asr
  right: ev_q25_sec_sep_asr
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: unreported
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_q25_tt_asr_ltbd_none
  left: ev_q25_ltbd_tt_asr
  right: ev_q25_none_tt_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_q25_tt_asr_ltbd_dt
  left: ev_q25_ltbd_tt_asr
  right: ev_q25_dt_tt_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: 'the defence: LTBD against DefensiveToken on the same base, both run by the authors'
  page: 3
- id: c_q25_tt_asr_ltbd_struq
  left: ev_q25_ltbd_tt_asr
  right: ev_q25_struq_tt_asr
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: unreported
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_q25_tt_asr_ltbd_sec
  left: ev_q25_ltbd_tt_asr
  right: ev_q25_sec_tt_asr
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: unreported
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_q25_cse_asr_ltbd_none
  left: ev_q25_ltbd_cse_asr
  right: ev_q25_none_cse_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_q25_cse_asr_ltbd_dt
  left: ev_q25_ltbd_cse_asr
  right: ev_q25_dt_cse_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: 'the defence: LTBD against DefensiveToken on the same base, both run by the authors'
  page: 3
- id: c_q25_cse_asr_ltbd_struq
  left: ev_q25_ltbd_cse_asr
  right: ev_q25_struq_cse_asr
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: unreported
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_q25_cse_asr_ltbd_sec
  left: ev_q25_ltbd_cse_asr
  right: ev_q25_sec_cse_asr
  purpose: X
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: unreported
  c6_decoding: unreported
  varies: 'the defence: LTBD (this report) against a training-based row whose values match the DefensiveTokens report'
  page: 3
- id: c_l3_alp_win_place
  left: ev_l3_bce_alp_win
  right: ev_l3_pre_alp_win
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: 'where the four learned tokens sit: at the instruction and data boundaries against a prefix (both cross-entropy only)'
  page: 3
- id: c_l3_alp_win_pref
  left: ev_l3_ltbd_alp_win
  right: ev_l3_bce_alp_win
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the preference term added to the cross-entropy loss
  page: 3
- id: c_l3_alp_asr_place
  left: ev_l3_bce_alp_asr
  right: ev_l3_pre_alp_asr
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: 'where the four learned tokens sit: at the instruction and data boundaries against a prefix (both cross-entropy only)'
  page: 3
- id: c_l3_alp_asr_pref
  left: ev_l3_ltbd_alp_asr
  right: ev_l3_bce_alp_asr
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the preference term added to the cross-entropy loss
  page: 3
- id: c_l3_sep_asr_place
  left: ev_l3_bce_sep_asr
  right: ev_l3_pre_sep_asr
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: 'where the four learned tokens sit: at the instruction and data boundaries against a prefix (both cross-entropy only)'
  page: 3
- id: c_l3_sep_asr_pref
  left: ev_l3_ltbd_sep_asr
  right: ev_l3_bce_sep_asr
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the preference term added to the cross-entropy loss
  page: 3
- id: c_l31_alp_win_place
  left: ev_l31_bce_alp_win
  right: ev_l31_pre_alp_win
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: 'where the four learned tokens sit: at the instruction and data boundaries against a prefix (both cross-entropy only)'
  page: 3
- id: c_l31_alp_win_pref
  left: ev_l31_ltbd_alp_win
  right: ev_l31_bce_alp_win
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the preference term added to the cross-entropy loss
  page: 3
- id: c_l31_alp_asr_place
  left: ev_l31_bce_alp_asr
  right: ev_l31_pre_alp_asr
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: 'where the four learned tokens sit: at the instruction and data boundaries against a prefix (both cross-entropy only)'
  page: 3
- id: c_l31_alp_asr_pref
  left: ev_l31_ltbd_alp_asr
  right: ev_l31_bce_alp_asr
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the preference term added to the cross-entropy loss
  page: 3
- id: c_l31_sep_asr_place
  left: ev_l31_bce_sep_asr
  right: ev_l31_pre_sep_asr
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: 'where the four learned tokens sit: at the instruction and data boundaries against a prefix (both cross-entropy only)'
  page: 3
- id: c_l31_sep_asr_pref
  left: ev_l31_ltbd_sep_asr
  right: ev_l31_bce_sep_asr
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the preference term added to the cross-entropy loss
  page: 3
- id: c_l3_spoof
  left: ev_l3_ltbd_spoof
  right: ev_l3_ltbd_plain
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched_unstated
  c4_attacker: manipulated
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: 'the attack: Delimiter-Spoof against plain injection'
  page: 4
- id: c_l31_spoof
  left: ev_l31_ltbd_spoof
  right: ev_l31_ltbd_plain
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched_unstated
  c4_attacker: manipulated
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: 'the attack: Delimiter-Spoof against plain injection'
  page: 4
- id: c_f3_spoof
  left: ev_f3_ltbd_spoof
  right: ev_f3_ltbd_plain
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched_unstated
  c4_attacker: manipulated
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: 'the attack: Delimiter-Spoof against plain injection'
  page: 4
- id: c_q25_spoof
  left: ev_q25_ltbd_spoof
  right: ev_q25_ltbd_plain
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched_unstated
  c4_attacker: manipulated
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: 'the attack: Delimiter-Spoof against plain injection'
  page: 4
claims:
- id: k1
  text: LTBD reaches 0.00% ASR on AlpacaFarm and 0.11 to 0.19% on TaskTracker
  location: abstract
  anchor: LTBD achieves 0.00% ASR on AlpacaFarm
  jobs:
  - R1
  supported_by:
  - c_l3_alp_asr_ltbd_none
  - c_l3_tt_asr_ltbd_none
  - c_l31_alp_asr_ltbd_none
  - c_l31_tt_asr_ltbd_none
  - c_f3_alp_asr_ltbd_none
  - c_f3_tt_asr_ltbd_none
  - c_q25_alp_asr_ltbd_none
  - c_q25_tt_asr_ltbd_none
  page: 1
- id: k2
  text: LTBD substantially outperforms inference-time defences and remains competitive with training-based ones
  location: introduction
  anchor: LTBD substantially outperforms existing inference-time defenses and remains competitive with training-based approaches
  jobs:
  - R1
  supported_by:
  - c_l3_alp_asr_ltbd_dt
  - c_l3_alp_asr_ltbd_struq
  - c_l3_alp_asr_ltbd_sec
  - c_l3_sep_asr_ltbd_dt
  - c_l3_sep_asr_ltbd_struq
  - c_l3_sep_asr_ltbd_sec
  - c_l3_tt_asr_ltbd_dt
  - c_l3_tt_asr_ltbd_struq
  - c_l3_tt_asr_ltbd_sec
  - c_l3_cse_asr_ltbd_dt
  - c_l3_cse_asr_ltbd_struq
  - c_l3_cse_asr_ltbd_sec
  - c_l31_alp_asr_ltbd_dt
  - c_l31_alp_asr_ltbd_struq
  - c_l31_alp_asr_ltbd_sec
  - c_l31_sep_asr_ltbd_dt
  - c_l31_sep_asr_ltbd_struq
  - c_l31_sep_asr_ltbd_sec
  - c_l31_tt_asr_ltbd_dt
  - c_l31_tt_asr_ltbd_struq
  - c_l31_tt_asr_ltbd_sec
  - c_l31_cse_asr_ltbd_dt
  - c_l31_cse_asr_ltbd_struq
  - c_l31_cse_asr_ltbd_sec
  - c_f3_alp_asr_ltbd_dt
  - c_f3_alp_asr_ltbd_struq
  - c_f3_alp_asr_ltbd_sec
  - c_f3_sep_asr_ltbd_dt
  - c_f3_sep_asr_ltbd_struq
  - c_f3_sep_asr_ltbd_sec
  - c_f3_tt_asr_ltbd_dt
  - c_f3_tt_asr_ltbd_struq
  - c_f3_tt_asr_ltbd_sec
  - c_f3_cse_asr_ltbd_dt
  - c_f3_cse_asr_ltbd_struq
  - c_f3_cse_asr_ltbd_sec
  - c_q25_alp_asr_ltbd_dt
  - c_q25_alp_asr_ltbd_struq
  - c_q25_alp_asr_ltbd_sec
  - c_q25_sep_asr_ltbd_dt
  - c_q25_sep_asr_ltbd_struq
  - c_q25_sep_asr_ltbd_sec
  - c_q25_tt_asr_ltbd_dt
  - c_q25_tt_asr_ltbd_struq
  - c_q25_tt_asr_ltbd_sec
  - c_q25_cse_asr_ltbd_dt
  - c_q25_cse_asr_ltbd_struq
  - c_q25_cse_asr_ltbd_sec
  page: 1
  contradicted_by:
  - r_l3_ltbd_cse_asr
- id: k3
  text: LTBD preserves benign-task utility
  location: abstract
  anchor: while preserving benign- task utility
  jobs:
  - general_utility
  - R2
  supported_by:
  - c_l3_alp_win_ltbd_none
  - c_l3_sep_win_ltbd_none
  - c_l31_alp_win_ltbd_none
  - c_l31_sep_win_ltbd_none
  - c_f3_alp_win_ltbd_none
  - c_f3_sep_win_ltbd_none
  - c_q25_alp_win_ltbd_none
  - c_q25_sep_win_ltbd_none
  page: 1
  contradicted_by:
  - r_l31_ltbd_sep_win
- id: k4
  text: LTBD remains effective under adaptive attacks by an attacker who knows the defence
  location: abstract
  anchor: LTBD also remains effective under adaptive attacks
  jobs:
  - R1
  supported_by:
  - c_l3_spoof
  - c_l31_spoof
  - c_f3_spoof
  - c_q25_spoof
  page: 1
coverage:
  R1: direct
  R2clean: proxy
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Tables 1 and 3
    R2clean: Table 1 (SEP WinRate)
    general_utility: Table 1 (AlpacaFarm WinRate)
    R2attack: not reported
schema_gaps:
- where: evaluations ev_*_struq_*, ev_*_sec_*
  note: the StruQ-Full and SecAlign-LoRA rows (and the uncoded TextGrad, Reminder, Sandwich and StruQ-LoRA rows) match the DefensiveTokens report's Table 3 value for value, at one decimal; the No Defense and DefensiveToken
    rows are at two decimals and differ from that report (Llama3-8B No Defense WinRate 33.82 here against 26.5 there). The text says the comparison is under comparable conditions and does not say the rows are copied, so measured_by
    is set to that report and the X comparisons carry unreported conditions
- where: evaluations ev_*_ltbd_spoof
  note: the Delimiter-Spoof templates are also used as training augmentations (pp. 3 and 4), so the adaptive test uses attacks the delimiters were trained on; no optimisation attack is run
- where: evaluations ev_*_ltbd_plain
  note: Table 3 does not name its benchmark; its Llama3.1-8B plain-injection ASR (0.48) differs from the Table 1 AlpacaFarm value (0.00)
- where: defences ltbd_*
  note: 'axis 2 lists both soft_tokens_frozen and preference_optimisation: only the four token embeddings are trained, with a DPO-style term in the loss'

`tracer_emnlp2026` (2610.19133)

In [ ]:
%%writefile reports/records/tracer_emnlp2026.yaml
report:
  id: tracer_emnlp2026
  slug: '2610.19133'
  title: 'TRACER: Trace-Regularized Training Against Indirect Prompt Injection'
  first_author: Kshitij Mishra
  venue: Findings of EMNLP 2026
  peer_reviewed: 'yes'
  doi: 10.18653/2026.findings-emnlp.2124
  arxiv: '2610.19133'
  version_read: arXiv v1
  first_date: '2026-10-13'
  read: F
  role: proposes
  code_release: https://github.com/Mishrakshitij/TRACER.git (linked in the abstract)
  extraction_scope: 'Sections 5 and appendices A, D to F: Table 8 rows for the 3B base, RAG, final tool, multi-framing and agent models and the 7B base, epoch-1 and final RAG models; Table 1 (a) and (b); Table 5 (follow-up-trained
    snapshots); Table 6; Table 9 clean accuracy (base and final); Table 11; the abstract claims (Table 7 diagnostics, Tables 9, 10 and 12 intermediate checkpoints and Figures 3 to 12 not coded)'
defences:
- id: rag3
  name: TRACER, RAG specialist (3B, final checkpoint)
  origin: this_report
  base_checkpoint: Qwen2.5-3B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  - distillation
  isolation: ablation_present
  signal_dependence: measured_other_checkpoint
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  signal_dependence_detail: 'Table 11: RAG wrapper and RAG system prompt removed one at a time and together; the full condition reads 5.5% against the 6.1% of the main checkpoint and the text does not say whether the variants
    were retrained'
  components: &id001
  - 'channel-sampled three-turn game: clean answer, the same payload rendered as a follow-up, RAG document, tool output or agent message, then the post-attack answer'
  - GRPO on a composite post-attack reward (answer, process, anchor retention, evidence-gated corrigibility, length)
  - 'ATRC: KL and token loss pulling the attacked reasoning towards committed spans of the clean trace when the injected evidence is invalid'
  - 4-bit QLoRA, 2,000 GSM8K training tasks, 15% benign episodes
  layout: RAG document wrapper with a RAG-specific system prompt
  layout_compat: &id002
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: The optimizer itself is standard GRPO
  pages:
    components: 4
    isolation: 8
- id: tool3
  name: TRACER, tool-output specialist (3B, final checkpoint)
  origin: this_report
  base_checkpoint: Qwen2.5-3B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  - distillation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  components: *id001
  layout: tool-response wrapper with provenance metadata
  layout_compat: *id002
  anchor: The optimizer itself is standard GRPO
  pages:
    components: 4
    isolation: 8
- id: multi3
  name: TRACER, joint RAG and tool training (3B, final checkpoint)
  origin: this_report
  base_checkpoint: Qwen2.5-3B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  - distillation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  components: *id001
  layout: RAG and tool wrappers alternated
  layout_compat: *id002
  anchor: The optimizer itself is standard GRPO
  pages:
    components: 4
    isolation: 8
- id: agent3
  name: TRACER, agent-message specialist (3B, final checkpoint)
  origin: this_report
  base_checkpoint: Qwen2.5-3B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  - distillation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  components: *id001
  layout: agent-message wrapper
  layout_compat: *id002
  anchor: The optimizer itself is standard GRPO
  pages:
    components: 4
    isolation: 8
- id: rag7e1
  name: TRACER, RAG specialist (7B, after one epoch)
  origin: this_report
  base_checkpoint: Qwen2.5-7B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  - distillation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  components: *id001
  layout: RAG document wrapper with a RAG-specific system prompt
  layout_compat: *id002
  anchor: The optimizer itself is standard GRPO
  pages:
    components: 4
    isolation: 8
- id: rag7
  name: TRACER, RAG specialist (7B, final checkpoint)
  origin: this_report
  base_checkpoint: Qwen2.5-7B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  - distillation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  components: *id001
  layout: RAG document wrapper with a RAG-specific system prompt
  layout_compat: *id002
  anchor: The optimizer itself is standard GRPO
  pages:
    components: 4
    isolation: 8
- id: fu3
  name: TRACER trained on follow-up attacks only (six intermediate 3B snapshots, pooled)
  origin: this_report
  base_checkpoint: Qwen2.5-3B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  - distillation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  layout: follow-up user message
  anchor: six intermediate model snapshots trained only against follow-up-style attacks
- id: abl_ans
  name: channel exposure with an answer-only reward (3B, RAG)
  origin: this_report
  base_checkpoint: Qwen2.5-3B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  anchor: answer-only training reaches 21.3% ASR
- id: abl_comp
  name: channel exposure with the composite reward, no ATRC (3B, RAG)
  origin: this_report
  base_checkpoint: Qwen2.5-3B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  anchor: the composite reward lowers this to 13.8%
- id: t6
  name: TRACER (checkpoint used for the mixed-evidence test not named)
  origin: this_report
  base_checkpoint: Qwen2.5-3B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  - distillation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 13
  anchor: Table 6 shows that TRACER retains high clean/evidence accuracy
- id: t11
  name: TRACER RAG specialist, Table 11 runs
  origin: this_report
  base_checkpoint: Qwen2.5-3B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  - distillation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 18
  anchor: Both the RAG wrapper and the RAG-specific system prompt matter
evaluations:
- id: ev_base3_fu
  defence_id: none:Qwen2.5-3B-Instruct
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as a follow-up user message
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'no'
  measured_by: tracer_emnlp2026
  page: 17
  denominator: clean-solved examples
  decoding: not reported
  sample_size: '200'
  baseline_kind: off_the_shelf
- id: ev_base3_rag
  defence_id: none:Qwen2.5-3B-Instruct
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as a retrieved document
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'no'
  measured_by: tracer_emnlp2026
  page: 17
  denominator: clean-solved examples
  decoding: not reported
  sample_size: '200'
  baseline_kind: off_the_shelf
- id: ev_base3_tool
  defence_id: none:Qwen2.5-3B-Instruct
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as a tool output
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'no'
  measured_by: tracer_emnlp2026
  page: 17
  denominator: clean-solved examples
  decoding: not reported
  sample_size: '200'
  baseline_kind: off_the_shelf
- id: ev_base3_agent
  defence_id: none:Qwen2.5-3B-Instruct
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as an agent message
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'no'
  measured_by: tracer_emnlp2026
  page: 17
  denominator: clean-solved examples
  decoding: not reported
  sample_size: '200'
  baseline_kind: off_the_shelf
- id: ev_rag3_fu
  defence_id: rag3
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as a follow-up user message
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'no'
  measured_by: tracer_emnlp2026
  page: 17
  denominator: clean-solved examples
  decoding: not reported
  sample_size: '200'
- id: ev_rag3_rag
  defence_id: rag3
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as a retrieved document
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'yes'
  measured_by: tracer_emnlp2026
  page: 17
  denominator: clean-solved examples
  decoding: not reported
  sample_size: '200'
- id: ev_rag3_tool
  defence_id: rag3
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as a tool output
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'no'
  measured_by: tracer_emnlp2026
  page: 17
  denominator: clean-solved examples
  decoding: not reported
  sample_size: '200'
- id: ev_rag3_agent
  defence_id: rag3
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as an agent message
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'no'
  measured_by: tracer_emnlp2026
  page: 17
  denominator: clean-solved examples
  decoding: not reported
  sample_size: '200'
- id: ev_tool3_fu
  defence_id: tool3
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as a follow-up user message
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'no'
  measured_by: tracer_emnlp2026
  page: 17
  denominator: clean-solved examples
  decoding: not reported
  sample_size: '200'
- id: ev_tool3_rag
  defence_id: tool3
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as a retrieved document
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'no'
  measured_by: tracer_emnlp2026
  page: 17
  denominator: clean-solved examples
  decoding: not reported
  sample_size: '200'
- id: ev_tool3_tool
  defence_id: tool3
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as a tool output
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'yes'
  measured_by: tracer_emnlp2026
  page: 17
  denominator: clean-solved examples
  decoding: not reported
  sample_size: '200'
- id: ev_tool3_agent
  defence_id: tool3
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as an agent message
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'no'
  measured_by: tracer_emnlp2026
  page: 17
  denominator: clean-solved examples
  decoding: not reported
  sample_size: '200'
- id: ev_multi3_fu
  defence_id: multi3
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as a follow-up user message
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'no'
  measured_by: tracer_emnlp2026
  page: 17
  denominator: clean-solved examples
  decoding: not reported
  sample_size: '200'
- id: ev_multi3_rag
  defence_id: multi3
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as a retrieved document
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'yes'
  measured_by: tracer_emnlp2026
  page: 17
  denominator: clean-solved examples
  decoding: not reported
  sample_size: '200'
- id: ev_multi3_tool
  defence_id: multi3
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as a tool output
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'yes'
  measured_by: tracer_emnlp2026
  page: 17
  denominator: clean-solved examples
  decoding: not reported
  sample_size: '200'
- id: ev_multi3_agent
  defence_id: multi3
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as an agent message
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'no'
  measured_by: tracer_emnlp2026
  page: 17
  denominator: clean-solved examples
  decoding: not reported
  sample_size: '200'
- id: ev_agent3_fu
  defence_id: agent3
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as a follow-up user message
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'no'
  measured_by: tracer_emnlp2026
  page: 17
  denominator: clean-solved examples
  decoding: not reported
  sample_size: '200'
- id: ev_agent3_rag
  defence_id: agent3
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as a retrieved document
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'no'
  measured_by: tracer_emnlp2026
  page: 17
  denominator: clean-solved examples
  decoding: not reported
  sample_size: '200'
- id: ev_agent3_tool
  defence_id: agent3
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as a tool output
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'no'
  measured_by: tracer_emnlp2026
  page: 17
  denominator: clean-solved examples
  decoding: not reported
  sample_size: '200'
- id: ev_agent3_agent
  defence_id: agent3
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as an agent message
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'yes'
  measured_by: tracer_emnlp2026
  page: 17
  denominator: clean-solved examples
  decoding: not reported
  sample_size: '200'
- id: ev_base7_fu
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as a follow-up user message
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'no'
  measured_by: tracer_emnlp2026
  page: 17
  denominator: clean-solved examples
  decoding: not reported
  sample_size: '200'
  baseline_kind: off_the_shelf
- id: ev_base7_rag
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as a retrieved document
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'no'
  measured_by: tracer_emnlp2026
  page: 17
  denominator: clean-solved examples
  decoding: not reported
  sample_size: '200'
  baseline_kind: off_the_shelf
- id: ev_base7_tool
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as a tool output
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'no'
  measured_by: tracer_emnlp2026
  page: 17
  denominator: clean-solved examples
  decoding: not reported
  sample_size: '200'
  baseline_kind: off_the_shelf
- id: ev_base7_agent
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as an agent message
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'no'
  measured_by: tracer_emnlp2026
  page: 17
  denominator: clean-solved examples
  decoding: not reported
  sample_size: '200'
  baseline_kind: off_the_shelf
- id: ev_rag7e1_fu
  defence_id: rag7e1
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as a follow-up user message
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'no'
  measured_by: tracer_emnlp2026
  page: 17
  denominator: clean-solved examples
  decoding: not reported
  sample_size: '200'
- id: ev_rag7e1_rag
  defence_id: rag7e1
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as a retrieved document
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'yes'
  measured_by: tracer_emnlp2026
  page: 17
  denominator: clean-solved examples
  decoding: not reported
  sample_size: '200'
- id: ev_rag7e1_tool
  defence_id: rag7e1
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as a tool output
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'no'
  measured_by: tracer_emnlp2026
  page: 17
  denominator: clean-solved examples
  decoding: not reported
  sample_size: '200'
- id: ev_rag7e1_agent
  defence_id: rag7e1
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as an agent message
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'no'
  measured_by: tracer_emnlp2026
  page: 17
  denominator: clean-solved examples
  decoding: not reported
  sample_size: '200'
- id: ev_rag7_fu
  defence_id: rag7
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as a follow-up user message
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'no'
  measured_by: tracer_emnlp2026
  page: 17
  denominator: clean-solved examples
  decoding: not reported
  sample_size: '200'
- id: ev_rag7_rag
  defence_id: rag7
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as a retrieved document
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'yes'
  measured_by: tracer_emnlp2026
  page: 17
  denominator: clean-solved examples
  decoding: not reported
  sample_size: '200'
- id: ev_rag7_tool
  defence_id: rag7
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as a tool output
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'no'
  measured_by: tracer_emnlp2026
  page: 17
  denominator: clean-solved examples
  decoding: not reported
  sample_size: '200'
- id: ev_rag7_agent
  defence_id: rag7
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as an agent message
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'no'
  measured_by: tracer_emnlp2026
  page: 17
  denominator: clean-solved examples
  decoding: not reported
  sample_size: '200'
- id: ev_fu3_fu
  defence_id: fu3
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as a follow-up user message
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'yes'
  measured_by: tracer_emnlp2026
  page: 16
  denominator: clean-solved examples pooled over six snapshots (960)
  decoding: not reported
- id: ev_fu3_rag
  defence_id: fu3
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as a retrieved document
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'no'
  measured_by: tracer_emnlp2026
  page: 16
  denominator: clean-solved examples pooled over six snapshots (960)
  decoding: not reported
- id: ev_base3_para
  defence_id: none:Qwen2.5-3B-Instruct
  benchmark: GSM8K, RAG wrapper, held-out paraphrases of the RAG templates
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'yes'
  measured_by: tracer_emnlp2026
  page: 8
  denominator: clean-solved examples
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_rag3_para
  defence_id: rag3
  benchmark: GSM8K, RAG wrapper, held-out paraphrases of the RAG templates
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'yes'
  measured_by: tracer_emnlp2026
  page: 8
  denominator: clean-solved examples
  decoding: not reported
  sample_size: 160 clean-solved (TRACER)
- id: ev_base3_chan
  defence_id: none:Qwen2.5-3B-Instruct
  benchmark: GSM8K, RAG wrapper, channel-aware escalation (stronger wrapper provenance cues)
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'yes'
  measured_by: tracer_emnlp2026
  page: 8
  denominator: clean-solved examples
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_rag3_chan
  defence_id: rag3
  benchmark: GSM8K, RAG wrapper, channel-aware escalation (stronger wrapper provenance cues)
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'yes'
  measured_by: tracer_emnlp2026
  page: 8
  denominator: clean-solved examples
  decoding: not reported
- id: ev_base3_defaware
  defence_id: none:Qwen2.5-3B-Instruct
  benchmark: GSM8K, RAG wrapper, defence-aware suite that references the defence's expected behaviour
  attacker_type: adaptive
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'yes'
  measured_by: tracer_emnlp2026
  page: 8
  attacker_budget: fixed templates written against the trained behaviour
  denominator: clean-solved examples
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_rag3_defaware
  defence_id: rag3
  benchmark: GSM8K, RAG wrapper, defence-aware suite that references the defence's expected behaviour
  attacker_type: adaptive
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'yes'
  measured_by: tracer_emnlp2026
  page: 8
  attacker_budget: fixed templates written against the trained behaviour
  denominator: clean-solved examples
  decoding: not reported
  sample_size: 160 clean-solved (TRACER)
- id: ev_abl_ans
  defence_id: abl_ans
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as a retrieved document
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'yes'
  measured_by: tracer_emnlp2026
  page: 8
  denominator: clean-solved examples
  decoding: not reported
- id: ev_abl_comp
  defence_id: abl_comp
  benchmark: GSM8K, 200 evaluation tasks; reasoning-rewrite, preemptive-answer and prompt-injection payloads (disjoint tasks) rendered as a retrieved document
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'yes'
  measured_by: tracer_emnlp2026
  page: 8
  denominator: clean-solved examples
  decoding: not reported
- id: ev_t6base_rag
  defence_id: none:Qwen2.5-3B-Instruct
  benchmark: GSM8K with valid task evidence and an embedded injection in a retrieved document
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'yes'
  measured_by: tracer_emnlp2026
  page: 16
  denominator: correct clean-evidence responses
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_t6_rag
  defence_id: t6
  benchmark: GSM8K with valid task evidence and an embedded injection in a retrieved document
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'yes'
  measured_by: tracer_emnlp2026
  page: 16
  denominator: correct clean-evidence responses
  decoding: not reported
- id: ev_t6base_tool
  defence_id: none:Qwen2.5-3B-Instruct
  benchmark: GSM8K with valid task evidence and an embedded injection in a tool output
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'yes'
  measured_by: tracer_emnlp2026
  page: 16
  denominator: correct clean-evidence responses
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_t6_tool
  defence_id: t6
  benchmark: GSM8K with valid task evidence and an embedded injection in a tool output
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'yes'
  measured_by: tracer_emnlp2026
  page: 16
  denominator: correct clean-evidence responses
  decoding: not reported
- id: ev_t6base_agent
  defence_id: none:Qwen2.5-3B-Instruct
  benchmark: GSM8K with valid task evidence and an embedded injection in an agent message
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'yes'
  measured_by: tracer_emnlp2026
  page: 16
  denominator: correct clean-evidence responses
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_t6_agent
  defence_id: t6
  benchmark: GSM8K with valid task evidence and an embedded injection in an agent message
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'yes'
  measured_by: tracer_emnlp2026
  page: 16
  denominator: correct clean-evidence responses
  decoding: not reported
- id: ev_base3_clean
  defence_id: none:Qwen2.5-3B-Instruct
  benchmark: GSM8K, 200 evaluation tasks, no payload
  attacker_type: none
  attacker_access: not_applicable
  judge: final answer against the gold answer
  layout_same_as_defence: 'yes'
  measured_by: tracer_emnlp2026
  page: 17
  decoding: not reported
  sample_size: '200'
  baseline_kind: off_the_shelf
- id: ev_tool3_clean
  defence_id: tool3
  benchmark: GSM8K, 200 evaluation tasks, no payload
  attacker_type: none
  attacker_access: not_applicable
  judge: final answer against the gold answer
  layout_same_as_defence: 'yes'
  measured_by: tracer_emnlp2026
  page: 17
  decoding: not reported
  sample_size: '200'
- id: ev_t11_full
  defence_id: t11
  benchmark: GSM8K, RAG evaluation, RAG system prompt and RAG wrapper
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'yes'
  measured_by: tracer_emnlp2026
  page: 18
  denominator: clean-solved examples
  decoding: not reported
- id: ev_t11_wrap
  defence_id: t11
  benchmark: GSM8K, RAG evaluation, default system prompt, RAG wrapper
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'no'
  measured_by: tracer_emnlp2026
  page: 18
  denominator: clean-solved examples
  decoding: not reported
- id: ev_t11_prompt
  defence_id: t11
  benchmark: GSM8K, RAG evaluation, RAG system prompt, no wrapper
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'no'
  measured_by: tracer_emnlp2026
  page: 18
  denominator: clean-solved examples
  decoding: not reported
- id: ev_t11_neither
  defence_id: t11
  benchmark: GSM8K, RAG evaluation, default system prompt, no wrapper
  attacker_type: static
  attacker_access: black_box
  judge: final answer against the gold answer; an attack succeeds when a correct clean answer turns wrong after the payload
  layout_same_as_defence: 'no'
  measured_by: tracer_emnlp2026
  page: 18
  denominator: clean-solved examples
  decoding: not reported
results:
- id: r_base3_fu
  evaluation_id: ev_base3_fu
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, a follow-up user message
  value: '41.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 8
- id: r_base3_rag
  evaluation_id: ev_base3_rag
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, a retrieved document
  value: '65.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 8
- id: r_base3_tool
  evaluation_id: ev_base3_tool
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, a tool output
  value: '59.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 8
- id: r_base3_agent
  evaluation_id: ev_base3_agent
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, an agent message
  value: '46.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 8
- id: r_rag3_fu
  evaluation_id: ev_rag3_fu
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, a follow-up user message
  value: '14.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 8
- id: r_rag3_rag
  evaluation_id: ev_rag3_rag
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, a retrieved document
  value: '6.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 8
- id: r_rag3_tool
  evaluation_id: ev_rag3_tool
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, a tool output
  value: '25.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 8
- id: r_rag3_agent
  evaluation_id: ev_rag3_agent
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, an agent message
  value: '26.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 8
- id: r_tool3_fu
  evaluation_id: ev_tool3_fu
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, a follow-up user message
  value: '20.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 8
- id: r_tool3_rag
  evaluation_id: ev_tool3_rag
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, a retrieved document
  value: '20.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 8
- id: r_tool3_tool
  evaluation_id: ev_tool3_tool
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, a tool output
  value: '15.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 8
- id: r_tool3_agent
  evaluation_id: ev_tool3_agent
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, an agent message
  value: '15.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 8
- id: r_multi3_fu
  evaluation_id: ev_multi3_fu
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, a follow-up user message
  value: '11.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 8
- id: r_multi3_rag
  evaluation_id: ev_multi3_rag
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, a retrieved document
  value: '5.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 8
- id: r_multi3_tool
  evaluation_id: ev_multi3_tool
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, a tool output
  value: '21.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 8
- id: r_multi3_agent
  evaluation_id: ev_multi3_agent
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, an agent message
  value: '17.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 8
- id: r_agent3_fu
  evaluation_id: ev_agent3_fu
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, a follow-up user message
  value: '19.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 8
- id: r_agent3_rag
  evaluation_id: ev_agent3_rag
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, a retrieved document
  value: '15.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 8
- id: r_agent3_tool
  evaluation_id: ev_agent3_tool
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, a tool output
  value: '26.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 8
- id: r_agent3_agent
  evaluation_id: ev_agent3_agent
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, an agent message
  value: '18.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 8
- id: r_base7_fu
  evaluation_id: ev_base7_fu
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, a follow-up user message
  value: '9.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 8
- id: r_base7_rag
  evaluation_id: ev_base7_rag
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, a retrieved document
  value: '34.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 8
- id: r_base7_tool
  evaluation_id: ev_base7_tool
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, a tool output
  value: '36.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 8
- id: r_base7_agent
  evaluation_id: ev_base7_agent
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, an agent message
  value: '16.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 8
- id: r_rag7e1_fu
  evaluation_id: ev_rag7e1_fu
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, a follow-up user message
  value: '5.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 8
- id: r_rag7e1_rag
  evaluation_id: ev_rag7e1_rag
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, a retrieved document
  value: '10.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 8
- id: r_rag7e1_tool
  evaluation_id: ev_rag7e1_tool
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, a tool output
  value: '13.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 8
- id: r_rag7e1_agent
  evaluation_id: ev_rag7e1_agent
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, an agent message
  value: '8.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 8
- id: r_rag7_fu
  evaluation_id: ev_rag7_fu
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, a follow-up user message
  value: '5.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 8
- id: r_rag7_rag
  evaluation_id: ev_rag7_rag
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, a retrieved document
  value: '13.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 8
- id: r_rag7_tool
  evaluation_id: ev_rag7_tool
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, a tool output
  value: '18.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 8
- id: r_rag7_agent
  evaluation_id: ev_rag7_agent
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, an agent message
  value: '6.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 8
- id: r_fu3_fu
  evaluation_id: ev_fu3_fu
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, follow-up (average over six snapshots)
  value: '18.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 5
  spread: Wilson 95% CI [15.7, 20.6]
- id: r_fu3_rag
  evaluation_id: ev_fu3_rag
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, RAG document (average over six snapshots)
  value: '53.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 5
  spread: Wilson 95% CI [49.9, 56.2]
- id: r_base3_para
  evaluation_id: ev_base3_para
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, held-out paraphrases of the RAG templates
  value: '62.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1 (a)
- id: r_rag3_para
  evaluation_id: ev_rag3_para
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, held-out paraphrases of the RAG templates
  value: '10.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1 (a)
- id: r_base3_chan
  evaluation_id: ev_base3_chan
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, channel-aware escalation (stronger wrapper provenance cues)
  value: '68.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1 (a)
- id: r_rag3_chan
  evaluation_id: ev_rag3_chan
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, channel-aware escalation (stronger wrapper provenance cues)
  value: '18.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1 (a)
- id: r_base3_defaware
  evaluation_id: ev_base3_defaware
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, defence-aware suite that references the defence's expected behaviour
  value: '70.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1 (a)
- id: r_rag3_defaware
  evaluation_id: ev_rag3_defaware
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, defence-aware suite that references the defence's expected behaviour
  value: '23.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1 (a)
- id: r_abl_ans
  evaluation_id: ev_abl_ans
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, retrieved document
  value: '21.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1 (b)
- id: r_abl_comp
  evaluation_id: ev_abl_comp
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, retrieved document
  value: '13.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1 (b)
- id: r_t6base_rag
  evaluation_id: ev_t6base_rag
  quantity: R1
  coverage: proxy
  metric_as_printed: flip rate
  value: '58.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 6
- id: r_t6_rag
  evaluation_id: ev_t6_rag
  quantity: R1
  coverage: proxy
  metric_as_printed: flip rate
  value: '13.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 6
- id: r_t6_rag_acc
  evaluation_id: ev_t6_rag
  quantity: R2attack
  coverage: direct
  metric_as_printed: evidence accuracy (valid evidence plus injection)
  value: '83.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 6
- id: r_t6base_tool
  evaluation_id: ev_t6base_tool
  quantity: R1
  coverage: proxy
  metric_as_printed: flip rate
  value: '52.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 6
- id: r_t6_tool
  evaluation_id: ev_t6_tool
  quantity: R1
  coverage: proxy
  metric_as_printed: flip rate
  value: '17.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 6
- id: r_t6_tool_acc
  evaluation_id: ev_t6_tool
  quantity: R2attack
  coverage: direct
  metric_as_printed: evidence accuracy (valid evidence plus injection)
  value: '82.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 6
- id: r_t6base_agent
  evaluation_id: ev_t6base_agent
  quantity: R1
  coverage: proxy
  metric_as_printed: flip rate
  value: '46.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 6
- id: r_t6_agent
  evaluation_id: ev_t6_agent
  quantity: R1
  coverage: proxy
  metric_as_printed: flip rate
  value: '16.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 6
- id: r_t6_agent_acc
  evaluation_id: ev_t6_agent
  quantity: R2attack
  coverage: direct
  metric_as_printed: evidence accuracy (valid evidence plus injection)
  value: '81.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 6
- id: r_base3_clean
  evaluation_id: ev_base3_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean accuracy
  value: '64.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 9
- id: r_tool3_clean
  evaluation_id: ev_tool3_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean accuracy
  value: '77.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 9
- id: r_t11_full
  evaluation_id: ev_t11_full
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, RAG system prompt and RAG wrapper
  value: '5.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 11
- id: r_t11_wrap
  evaluation_id: ev_t11_wrap
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, default system prompt, RAG wrapper
  value: '10.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 11
- id: r_t11_prompt
  evaluation_id: ev_t11_prompt
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, RAG system prompt, no wrapper
  value: '11.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 11
- id: r_t11_neither
  evaluation_id: ev_t11_neither
  quantity: R1
  coverage: proxy
  metric_as_printed: conditional ASR, default system prompt, no wrapper
  value: '60.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 11
comparisons:
- id: c_rag3_fu
  left: ev_rag3_fu
  right: ev_base3_fu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_rag3_rag
  left: ev_rag3_rag
  right: ev_base3_rag
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_rag3_tool
  left: ev_rag3_tool
  right: ev_base3_tool
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_rag3_agent
  left: ev_rag3_agent
  right: ev_base3_agent
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_tool3_fu
  left: ev_tool3_fu
  right: ev_base3_fu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_tool3_rag
  left: ev_tool3_rag
  right: ev_base3_rag
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_tool3_tool
  left: ev_tool3_tool
  right: ev_base3_tool
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_tool3_agent
  left: ev_tool3_agent
  right: ev_base3_agent
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_multi3_fu
  left: ev_multi3_fu
  right: ev_base3_fu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_multi3_rag
  left: ev_multi3_rag
  right: ev_base3_rag
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_multi3_tool
  left: ev_multi3_tool
  right: ev_base3_tool
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_multi3_agent
  left: ev_multi3_agent
  right: ev_base3_agent
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_agent3_fu
  left: ev_agent3_fu
  right: ev_base3_fu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_agent3_rag
  left: ev_agent3_rag
  right: ev_base3_rag
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_agent3_tool
  left: ev_agent3_tool
  right: ev_base3_tool
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_agent3_agent
  left: ev_agent3_agent
  right: ev_base3_agent
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_rag7e1_fu
  left: ev_rag7e1_fu
  right: ev_base7_fu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_rag7e1_rag
  left: ev_rag7e1_rag
  right: ev_base7_rag
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_rag7e1_tool
  left: ev_rag7e1_tool
  right: ev_base7_tool
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_rag7e1_agent
  left: ev_rag7e1_agent
  right: ev_base7_agent
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_rag7_fu
  left: ev_rag7_fu
  right: ev_base7_fu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_rag7_rag
  left: ev_rag7_rag
  right: ev_base7_rag
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_rag7_tool
  left: ev_rag7_tool
  right: ev_base7_tool
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_rag7_agent
  left: ev_rag7_agent
  right: ev_base7_agent
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_fu3_frame
  left: ev_fu3_rag
  right: ev_fu3_fu
  purpose: M
  c1_base: matched
  c2_layout: manipulated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the channel: the same payloads rendered as a retrieved document instead of a follow-up message'
  page: 16
- id: c_rag3_para
  left: ev_rag3_para
  right: ev_base3_para
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_rag3_chan
  left: ev_rag3_chan
  right: ev_base3_chan
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_rag3_defaware
  left: ev_rag3_defaware
  right: ev_base3_defaware
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_abl_reward
  left: ev_abl_comp
  right: ev_abl_ans
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the composite reward in place of the answer-only reward
  page: 8
- id: c_abl_atrc
  left: ev_rag3_rag
  right: ev_abl_comp
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: ATRC added to the composite reward
  page: 8
- id: c_t6_rag
  left: ev_t6_rag
  right: ev_t6base_rag
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_t6_tool
  left: ev_t6_tool
  right: ev_t6base_tool
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_t6_agent
  left: ev_t6_agent
  right: ev_t6base_agent
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_tool3_clean
  left: ev_tool3_clean
  right: ev_base3_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_t11_wrap
  left: ev_t11_wrap
  right: ev_t11_full
  purpose: M
  c1_base: matched
  c2_layout: manipulated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the signal: default system prompt, RAG wrapper instead of both'
  page: 18
- id: c_t11_prompt
  left: ev_t11_prompt
  right: ev_t11_full
  purpose: M
  c1_base: matched
  c2_layout: manipulated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the signal: RAG system prompt, no wrapper instead of both'
  page: 18
- id: c_t11_neither
  left: ev_t11_neither
  right: ev_t11_full
  purpose: M
  c1_base: matched
  c2_layout: manipulated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the signal: default system prompt, no wrapper instead of both'
  page: 18
claims:
- id: k1
  text: the same attack succeeds 18% of the time as a follow-up message but 53% in retrieved context
  location: abstract
  anchor: the same attack succeeds 18% of the time as a follow-up
  jobs:
  - R1
  supported_by:
  - c_fu3_frame
  page: 1
- id: k2
  text: TRACER reduces RAG-framed ASR from 65.5% to 6.1% on GSM8K with Qwen2.5-3B-Instruct while keeping 78 to 82% clean accuracy
  location: abstract
  anchor: TRACER reduces RAG-framed ASR from 65.5% to 6.1%
  jobs:
  - R1
  - R2
  supported_by:
  - c_rag3_rag
  - c_rag3_para
  page: 1
- id: k3
  text: joint RAG and tool training reaches 13.9% average ASR across framings at 3B; one epoch at 7B reaches 9.4%
  location: abstract
  anchor: Joint RAG+tool training achieves 13.9% average ASR across framings
  jobs:
  - R1
  supported_by:
  - c_multi3_fu
  - c_multi3_rag
  - c_multi3_tool
  - c_multi3_agent
  - c_rag7e1_fu
  - c_rag7e1_rag
  - c_rag7e1_tool
  - c_rag7e1_agent
  page: 1
coverage:
  R1: proxy
  R2clean: direct
  R2attack: proxy
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: conditional answer flip (correct clean answer turned wrong), Tables 1, 5, 6, 8 and 11
    R2clean: Table 9 (tool training only; RAG clean accuracy is a 78 to 82% range in the text and Figure 4)
    R2attack: Table 6 evidence accuracy
schema_gaps:
- where: results r_* (R1)
  note: ASR is a conditional answer flip on GSM8K pooled over three payload families (reasoning rewrite, false verified answer, explicit instruction); it does not check that an injected instruction was carried out, so R1 is
    coded as a proxy
- where: evaluations ev_base3_rag, ev_rag3_rag
  note: the base RAG ASR is 65.5 in Tables 1, 3, 7 and 8 but 65.6 (99/151) in Table 5, whose caption notes reporting-granularity differences; the TRACER RAG denominator is 164 in Table 5 and 160 for the held-out and defence-aware
    runs
- where: claims k2
  note: the 78 to 82% clean accuracy is a range read from Figure 4 and the text; no exact clean accuracy is given for the RAG checkpoint
- where: claims k3
  note: the 13.9% 3B average appears only in the text; it equals the mean of the four Table 8 values for the multi-framing model; the 7B 9.4% is the epoch-1 checkpoint, the final 7B checkpoint averages 10.8%
- where: defences t6
  note: the paper does not say which TRACER checkpoint was used for the mixed-evidence test (Table 6), and gives no base evidence accuracy

`agentsafeevo_arxiv2026` (2610.19422)

In [ ]:
%%writefile reports/records/agentsafeevo_arxiv2026.yaml
report:
  id: agentsafeevo_arxiv2026
  slug: '2610.19422'
  title: 'AgentSafeEvo: Adversarial Co-Evolution for LLM Agent Safety'
  first_author: Chenxu Wang
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2610.19422'
  version_read: arXiv v1
  first_date: '2026-10-14'
  read: F
  role: proposes
  code_release: https://github.com/lalalamdbf/AgentSafeEvo and Hugging Face collection (linked on p. 1)
  extraction_scope: 'Section 4: Table 1 AgentDojo and MSB columns for Qwen3-8B, Spotlighting, Static-AT and AgentSafeEvo; Table 2; Table 3 MSB, ALFWorld and ToolHop columns; the abstract claim (Agent-SafetyBench and AgentHarm
    columns, which pool direct, harmful-request and indirect risks, Prompt Guard 2, a detector, and Figure 3, whose generated tasks mix six attack categories including direct injection and jailbreaks, not coded)'
defences:
- id: ase
  name: AgentSafeEvo (attacker-defender co-evolution with learnability weighting)
  origin: this_report
  base_checkpoint: Qwen3-8B
  axis1:
  - no_signal
  axis2:
  - adversarial_rl
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 1
  components:
  - 'attacker (Qwen3-8B, SFT-initialised on about 10,000 adversarial and 10,000 benign tasks) writes adversarial agent tasks: user query, tool definitions and environment'
  - defender trained with GRPO on safety and utility rewards from a Qwen3.8-27B reward model; attacker rewarded when the defender's reward falls
  - 'learnability-guided task weighting: tasks with more within-task spread in defender reward get larger weight in both updates'
  layout: multi-turn tool-calling agent; attacks reach it through user queries, tool descriptions or environment observations
  layout_compat: &id001
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: a co-evolutionary reinforcement learning framework for agent safety
  pages:
    components: 7
    isolation: 10
- id: static
  name: Static-AT (fixed SFT attacker in the loop, defender trained with GRPO, no weighting)
  origin: this_report
  base_checkpoint: Qwen3-8B
  axis1:
  - no_signal
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  layout: multi-turn tool-calling agent; attacks reach it through user queries, tool descriptions or environment observations
  layout_compat: *id001
  anchor: Static Adversarial Training (Static-AT), which keeps the SFT attacker fixed
- id: spot
  name: Spotlighting, delimiting variant (tool outputs wrapped in « », system prompt marks them untrusted)
  origin: other_report
  base_checkpoint: Qwen3-8B
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  layout: multi-turn tool-calling agent; attacks reach it through user queries, tool descriptions or environment observations; tool outputs wrapped in delimiters
  layout_compat: *id001
  anchor: the prompt-based defense Spotlighting with Delimiting
- id: nolw
  name: co-evolution without learnability weighting
  origin: this_report
  base_checkpoint: Qwen3-8B
  axis1:
  - no_signal
  axis2:
  - adversarial_rl
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 10
  layout: multi-turn tool-calling agent; attacks reach it through user queries, tool descriptions or environment observations
  anchor: We compare AgentSafeEvo with four ablated variants
- id: nodw
  name: AgentSafeEvo without defender weighting (attacker weighted only)
  origin: this_report
  base_checkpoint: Qwen3-8B
  axis1:
  - no_signal
  axis2:
  - adversarial_rl
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 10
  layout: multi-turn tool-calling agent; attacks reach it through user queries, tool descriptions or environment observations
  anchor: We compare AgentSafeEvo with four ablated variants
- id: noaw
  name: AgentSafeEvo without attacker weighting (defender weighted only)
  origin: this_report
  base_checkpoint: Qwen3-8B
  axis1:
  - no_signal
  axis2:
  - adversarial_rl
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 10
  layout: multi-turn tool-calling agent; attacks reach it through user queries, tool descriptions or environment observations
  anchor: We compare AgentSafeEvo with four ablated variants
evaluations:
- id: ev_base_ad_asr
  defence_id: none:Qwen3-8B
  benchmark: AgentDojo (97 user tasks, 629 security cases; injection attack not named)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo injection-task check
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 8
  decoding: temperature 0.6, top_p 0.95
  pages:
    benchmark: 25
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_base_ad_ua
  defence_id: none:Qwen3-8B
  benchmark: AgentDojo (97 user tasks, 629 security cases; injection attack not named)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo user-task check under attack
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 8
  decoding: temperature 0.6, top_p 0.95
  pages:
    benchmark: 25
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_base_msb_asr
  defence_id: none:Qwen3-8B
  benchmark: MCP Security Bench (over 2,000 attacks, 12 types on tool signatures, parameters, responses and retrieval)
  attacker_type: static
  attacker_access: black_box
  judge: MSB attack check
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 8
  decoding: temperature 0.6, top_p 0.95
  pages:
    benchmark: 25
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_base_msb_nrp
  defence_id: none:Qwen3-8B
  benchmark: MCP Security Bench (over 2,000 attacks, 12 types on tool signatures, parameters, responses and retrieval)
  attacker_type: static
  attacker_access: black_box
  judge: MSB net resilient performance
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 8
  decoding: temperature 0.6, top_p 0.95
  pages:
    benchmark: 25
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_spot_ad_asr
  defence_id: spot
  benchmark: AgentDojo (97 user tasks, 629 security cases; injection attack not named)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo injection-task check
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 8
  decoding: temperature 0.6, top_p 0.95
  pages:
    benchmark: 25
    decoding: 24
- id: ev_spot_ad_ua
  defence_id: spot
  benchmark: AgentDojo (97 user tasks, 629 security cases; injection attack not named)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo user-task check under attack
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 8
  decoding: temperature 0.6, top_p 0.95
  pages:
    benchmark: 25
    decoding: 24
- id: ev_spot_msb_asr
  defence_id: spot
  benchmark: MCP Security Bench (over 2,000 attacks, 12 types on tool signatures, parameters, responses and retrieval)
  attacker_type: static
  attacker_access: black_box
  judge: MSB attack check
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 8
  decoding: temperature 0.6, top_p 0.95
  pages:
    benchmark: 25
    decoding: 24
- id: ev_spot_msb_nrp
  defence_id: spot
  benchmark: MCP Security Bench (over 2,000 attacks, 12 types on tool signatures, parameters, responses and retrieval)
  attacker_type: static
  attacker_access: black_box
  judge: MSB net resilient performance
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 8
  decoding: temperature 0.6, top_p 0.95
  pages:
    benchmark: 25
    decoding: 24
- id: ev_static_ad_asr
  defence_id: static
  benchmark: AgentDojo (97 user tasks, 629 security cases; injection attack not named)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo injection-task check
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 8
  decoding: temperature 0.6, top_p 0.95
  pages:
    benchmark: 25
    decoding: 24
- id: ev_static_ad_ua
  defence_id: static
  benchmark: AgentDojo (97 user tasks, 629 security cases; injection attack not named)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo user-task check under attack
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 8
  decoding: temperature 0.6, top_p 0.95
  pages:
    benchmark: 25
    decoding: 24
- id: ev_static_msb_asr
  defence_id: static
  benchmark: MCP Security Bench (over 2,000 attacks, 12 types on tool signatures, parameters, responses and retrieval)
  attacker_type: static
  attacker_access: black_box
  judge: MSB attack check
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 8
  decoding: temperature 0.6, top_p 0.95
  pages:
    benchmark: 25
    decoding: 24
- id: ev_static_msb_nrp
  defence_id: static
  benchmark: MCP Security Bench (over 2,000 attacks, 12 types on tool signatures, parameters, responses and retrieval)
  attacker_type: static
  attacker_access: black_box
  judge: MSB net resilient performance
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 8
  decoding: temperature 0.6, top_p 0.95
  pages:
    benchmark: 25
    decoding: 24
- id: ev_ase_ad_asr
  defence_id: ase
  benchmark: AgentDojo (97 user tasks, 629 security cases; injection attack not named)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo injection-task check
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 8
  decoding: temperature 0.6, top_p 0.95
  pages:
    benchmark: 25
    decoding: 24
- id: ev_ase_ad_ua
  defence_id: ase
  benchmark: AgentDojo (97 user tasks, 629 security cases; injection attack not named)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo user-task check under attack
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 8
  decoding: temperature 0.6, top_p 0.95
  pages:
    benchmark: 25
    decoding: 24
- id: ev_ase_msb_asr
  defence_id: ase
  benchmark: MCP Security Bench (over 2,000 attacks, 12 types on tool signatures, parameters, responses and retrieval)
  attacker_type: static
  attacker_access: black_box
  judge: MSB attack check
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 8
  decoding: temperature 0.6, top_p 0.95
  pages:
    benchmark: 25
    decoding: 24
- id: ev_ase_msb_nrp
  defence_id: ase
  benchmark: MCP Security Bench (over 2,000 attacks, 12 types on tool signatures, parameters, responses and retrieval)
  attacker_type: static
  attacker_access: black_box
  judge: MSB net resilient performance
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 8
  decoding: temperature 0.6, top_p 0.95
  pages:
    benchmark: 25
    decoding: 24
- id: ev_base_bfcl
  defence_id: none:Qwen3-8B
  benchmark: BFCL-V3
  attacker_type: none
  attacker_access: not_applicable
  judge: function-call accuracy
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 8
  decoding: temperature 0.6, top_p 0.95
  pages:
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_base_alf
  defence_id: none:Qwen3-8B
  benchmark: ALFWorld (AgentBench-FC setting)
  attacker_type: none
  attacker_access: not_applicable
  judge: goal completion
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 8
  decoding: temperature 0.6, top_p 0.95
  pages:
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_base_th
  defence_id: none:Qwen3-8B
  benchmark: ToolHop
  attacker_type: none
  attacker_access: not_applicable
  judge: answer accuracy
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 8
  decoding: temperature 0.6, top_p 0.95
  pages:
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_spot_bfcl
  defence_id: spot
  benchmark: BFCL-V3
  attacker_type: none
  attacker_access: not_applicable
  judge: function-call accuracy
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 8
  decoding: temperature 0.6, top_p 0.95
  pages:
    decoding: 24
- id: ev_spot_alf
  defence_id: spot
  benchmark: ALFWorld (AgentBench-FC setting)
  attacker_type: none
  attacker_access: not_applicable
  judge: goal completion
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 8
  decoding: temperature 0.6, top_p 0.95
  pages:
    decoding: 24
- id: ev_spot_th
  defence_id: spot
  benchmark: ToolHop
  attacker_type: none
  attacker_access: not_applicable
  judge: answer accuracy
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 8
  decoding: temperature 0.6, top_p 0.95
  pages:
    decoding: 24
- id: ev_static_bfcl
  defence_id: static
  benchmark: BFCL-V3
  attacker_type: none
  attacker_access: not_applicable
  judge: function-call accuracy
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 8
  decoding: temperature 0.6, top_p 0.95
  pages:
    decoding: 24
- id: ev_static_alf
  defence_id: static
  benchmark: ALFWorld (AgentBench-FC setting)
  attacker_type: none
  attacker_access: not_applicable
  judge: goal completion
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 8
  decoding: temperature 0.6, top_p 0.95
  pages:
    decoding: 24
- id: ev_static_th
  defence_id: static
  benchmark: ToolHop
  attacker_type: none
  attacker_access: not_applicable
  judge: answer accuracy
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 8
  decoding: temperature 0.6, top_p 0.95
  pages:
    decoding: 24
- id: ev_ase_bfcl
  defence_id: ase
  benchmark: BFCL-V3
  attacker_type: none
  attacker_access: not_applicable
  judge: function-call accuracy
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 8
  decoding: temperature 0.6, top_p 0.95
  pages:
    decoding: 24
- id: ev_ase_alf
  defence_id: ase
  benchmark: ALFWorld (AgentBench-FC setting)
  attacker_type: none
  attacker_access: not_applicable
  judge: goal completion
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 8
  decoding: temperature 0.6, top_p 0.95
  pages:
    decoding: 24
- id: ev_ase_th
  defence_id: ase
  benchmark: ToolHop
  attacker_type: none
  attacker_access: not_applicable
  judge: answer accuracy
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 8
  decoding: temperature 0.6, top_p 0.95
  pages:
    decoding: 24
- id: ev_nolw_msb_asr
  defence_id: nolw
  benchmark: MCP Security Bench (over 2,000 attacks, 12 types on tool signatures, parameters, responses and retrieval)
  attacker_type: static
  attacker_access: black_box
  judge: MSB attack check
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 10
  decoding: temperature 0.6, top_p 0.95
- id: ev_nolw_msb_nrp
  defence_id: nolw
  benchmark: MCP Security Bench (over 2,000 attacks, 12 types on tool signatures, parameters, responses and retrieval)
  attacker_type: static
  attacker_access: black_box
  judge: MSB net resilient performance
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 10
  decoding: temperature 0.6, top_p 0.95
- id: ev_nolw_alf
  defence_id: nolw
  benchmark: ALFWorld (AgentBench-FC setting)
  attacker_type: none
  attacker_access: not_applicable
  judge: goal completion
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 10
  decoding: temperature 0.6, top_p 0.95
- id: ev_nolw_th
  defence_id: nolw
  benchmark: ToolHop
  attacker_type: none
  attacker_access: not_applicable
  judge: answer accuracy
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 10
  decoding: temperature 0.6, top_p 0.95
- id: ev_nodw_msb_asr
  defence_id: nodw
  benchmark: MCP Security Bench (over 2,000 attacks, 12 types on tool signatures, parameters, responses and retrieval)
  attacker_type: static
  attacker_access: black_box
  judge: MSB attack check
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 10
  decoding: temperature 0.6, top_p 0.95
- id: ev_nodw_msb_nrp
  defence_id: nodw
  benchmark: MCP Security Bench (over 2,000 attacks, 12 types on tool signatures, parameters, responses and retrieval)
  attacker_type: static
  attacker_access: black_box
  judge: MSB net resilient performance
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 10
  decoding: temperature 0.6, top_p 0.95
- id: ev_nodw_alf
  defence_id: nodw
  benchmark: ALFWorld (AgentBench-FC setting)
  attacker_type: none
  attacker_access: not_applicable
  judge: goal completion
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 10
  decoding: temperature 0.6, top_p 0.95
- id: ev_nodw_th
  defence_id: nodw
  benchmark: ToolHop
  attacker_type: none
  attacker_access: not_applicable
  judge: answer accuracy
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 10
  decoding: temperature 0.6, top_p 0.95
- id: ev_noaw_msb_asr
  defence_id: noaw
  benchmark: MCP Security Bench (over 2,000 attacks, 12 types on tool signatures, parameters, responses and retrieval)
  attacker_type: static
  attacker_access: black_box
  judge: MSB attack check
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 10
  decoding: temperature 0.6, top_p 0.95
- id: ev_noaw_msb_nrp
  defence_id: noaw
  benchmark: MCP Security Bench (over 2,000 attacks, 12 types on tool signatures, parameters, responses and retrieval)
  attacker_type: static
  attacker_access: black_box
  judge: MSB net resilient performance
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 10
  decoding: temperature 0.6, top_p 0.95
- id: ev_noaw_alf
  defence_id: noaw
  benchmark: ALFWorld (AgentBench-FC setting)
  attacker_type: none
  attacker_access: not_applicable
  judge: goal completion
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 10
  decoding: temperature 0.6, top_p 0.95
- id: ev_noaw_th
  defence_id: noaw
  benchmark: ToolHop
  attacker_type: none
  attacker_access: not_applicable
  judge: answer accuracy
  layout_same_as_defence: 'yes'
  measured_by: agentsafeevo_arxiv2026
  page: 10
  decoding: temperature 0.6, top_p 0.95
results:
- id: r_base_ad_asr
  evaluation_id: ev_base_ad_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '22.20'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: 'SD over three independent runs: 0.57'
- id: r_base_ad_ua
  evaluation_id: ev_base_ad_ua
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo utility under attack
  value: '53.64'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: 'SD over three independent runs: 0.90'
- id: r_base_msb_asr
  evaluation_id: ev_base_msb_asr
  quantity: R1
  coverage: direct
  metric_as_printed: MSB ASR
  value: '60.43'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: 'SD over three independent runs: 0.97'
- id: r_base_msb_nrp
  evaluation_id: ev_base_msb_nrp
  quantity: other
  coverage: direct
  metric_as_printed: MSB NRP
  value: '28.12'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: 'SD over three independent runs: 1.30'
- id: r_spot_ad_asr
  evaluation_id: ev_spot_ad_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '19.25'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: 'SD over three independent runs: 1.04'
- id: r_spot_ad_ua
  evaluation_id: ev_spot_ad_ua
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo utility under attack
  value: '57.82'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: 'SD over three independent runs: 1.11'
- id: r_spot_msb_asr
  evaluation_id: ev_spot_msb_asr
  quantity: R1
  coverage: direct
  metric_as_printed: MSB ASR
  value: '58.53'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: 'SD over three independent runs: 0.73'
- id: r_spot_msb_nrp
  evaluation_id: ev_spot_msb_nrp
  quantity: other
  coverage: direct
  metric_as_printed: MSB NRP
  value: '29.55'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: 'SD over three independent runs: 0.46'
- id: r_static_ad_asr
  evaluation_id: ev_static_ad_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '6.71'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: 'SD over three independent runs: 0.43'
- id: r_static_ad_ua
  evaluation_id: ev_static_ad_ua
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo utility under attack
  value: '61.43'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: 'SD over three independent runs: 0.15'
- id: r_static_msb_asr
  evaluation_id: ev_static_msb_asr
  quantity: R1
  coverage: direct
  metric_as_printed: MSB ASR
  value: '38.64'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: 'SD over three independent runs: 2.24'
- id: r_static_msb_nrp
  evaluation_id: ev_static_msb_nrp
  quantity: other
  coverage: direct
  metric_as_printed: MSB NRP
  value: '40.64'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: 'SD over three independent runs: 0.93'
- id: r_ase_ad_asr
  evaluation_id: ev_ase_ad_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '6.18'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: 'SD over three independent runs: 0.18'
- id: r_ase_ad_ua
  evaluation_id: ev_ase_ad_ua
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo utility under attack
  value: '66.52'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: 'SD over three independent runs: 1.43'
- id: r_ase_msb_asr
  evaluation_id: ev_ase_msb_asr
  quantity: R1
  coverage: direct
  metric_as_printed: MSB ASR
  value: '31.25'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: 'SD over three independent runs: 1.24'
- id: r_ase_msb_nrp
  evaluation_id: ev_ase_msb_nrp
  quantity: other
  coverage: direct
  metric_as_printed: MSB NRP
  value: '46.00'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: 'SD over three independent runs: 0.54'
- id: r_base_bfcl
  evaluation_id: ev_base_bfcl
  quantity: general_utility
  coverage: direct
  metric_as_printed: BFCL-V3 accuracy
  value: '66.13'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
  spread: 'SD over three independent runs: 0.22'
- id: r_base_alf
  evaluation_id: ev_base_alf
  quantity: general_utility
  coverage: direct
  metric_as_printed: ALFWorld (AgentBench-FC setting) accuracy
  value: '30.28'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
  spread: 'SD over three independent runs: 4.59'
- id: r_base_th
  evaluation_id: ev_base_th
  quantity: general_utility
  coverage: direct
  metric_as_printed: ToolHop accuracy
  value: '21.84'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
  spread: 'SD over three independent runs: 1.23'
- id: r_spot_bfcl
  evaluation_id: ev_spot_bfcl
  quantity: general_utility
  coverage: direct
  metric_as_printed: BFCL-V3 accuracy
  value: '66.08'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
  spread: 'SD over three independent runs: 0.37'
- id: r_spot_alf
  evaluation_id: ev_spot_alf
  quantity: general_utility
  coverage: direct
  metric_as_printed: ALFWorld (AgentBench-FC setting) accuracy
  value: '32.13'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
  spread: 'SD over three independent runs: 2.69'
- id: r_spot_th
  evaluation_id: ev_spot_th
  quantity: general_utility
  coverage: direct
  metric_as_printed: ToolHop accuracy
  value: '25.76'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
  spread: 'SD over three independent runs: 0.15'
- id: r_static_bfcl
  evaluation_id: ev_static_bfcl
  quantity: general_utility
  coverage: direct
  metric_as_printed: BFCL-V3 accuracy
  value: '66.73'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
  spread: 'SD over three independent runs: 0.50'
- id: r_static_alf
  evaluation_id: ev_static_alf
  quantity: general_utility
  coverage: direct
  metric_as_printed: ALFWorld (AgentBench-FC setting) accuracy
  value: '30.89'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
  spread: 'SD over three independent runs: 0.43'
- id: r_static_th
  evaluation_id: ev_static_th
  quantity: general_utility
  coverage: direct
  metric_as_printed: ToolHop accuracy
  value: '39.43'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
  spread: 'SD over three independent runs: 0.09'
- id: r_ase_bfcl
  evaluation_id: ev_ase_bfcl
  quantity: general_utility
  coverage: direct
  metric_as_printed: BFCL-V3 accuracy
  value: '65.98'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
  spread: 'SD over three independent runs: 0.14'
- id: r_ase_alf
  evaluation_id: ev_ase_alf
  quantity: general_utility
  coverage: direct
  metric_as_printed: ALFWorld (AgentBench-FC setting) accuracy
  value: '43.72'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
  spread: 'SD over three independent runs: 3.03'
- id: r_ase_th
  evaluation_id: ev_ase_th
  quantity: general_utility
  coverage: direct
  metric_as_printed: ToolHop accuracy
  value: '48.04'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
  spread: 'SD over three independent runs: 1.40'
- id: r_nolw_msb_asr
  evaluation_id: ev_nolw_msb_asr
  quantity: R1
  coverage: direct
  metric_as_printed: MSB ASR
  value: '32.89'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
  spread: 'SD over three independent runs: 0.66'
- id: r_nolw_msb_nrp
  evaluation_id: ev_nolw_msb_nrp
  quantity: other
  coverage: direct
  metric_as_printed: MSB NRP
  value: '41.48'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
  spread: 'SD over three independent runs: 0.21'
- id: r_nolw_alf
  evaluation_id: ev_nolw_alf
  quantity: general_utility
  coverage: direct
  metric_as_printed: ALFWorld (AgentBench-FC setting) accuracy
  value: '36.70'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
  spread: 'SD over three independent runs: 0.75'
- id: r_nolw_th
  evaluation_id: ev_nolw_th
  quantity: general_utility
  coverage: direct
  metric_as_printed: ToolHop accuracy
  value: '40.07'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
  spread: 'SD over three independent runs: 0.58'
- id: r_nodw_msb_asr
  evaluation_id: ev_nodw_msb_asr
  quantity: R1
  coverage: direct
  metric_as_printed: MSB ASR
  value: '28.47'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
  spread: 'SD over three independent runs: 1.03'
- id: r_nodw_msb_nrp
  evaluation_id: ev_nodw_msb_nrp
  quantity: other
  coverage: direct
  metric_as_printed: MSB NRP
  value: '43.04'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
  spread: 'SD over three independent runs: 0.73'
- id: r_nodw_alf
  evaluation_id: ev_nodw_alf
  quantity: general_utility
  coverage: direct
  metric_as_printed: ALFWorld (AgentBench-FC setting) accuracy
  value: '29.05'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
  spread: 'SD over three independent runs: 0.43'
- id: r_nodw_th
  evaluation_id: ev_nodw_th
  quantity: general_utility
  coverage: direct
  metric_as_printed: ToolHop accuracy
  value: '40.80'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
  spread: 'SD over three independent runs: 0.22'
- id: r_noaw_msb_asr
  evaluation_id: ev_noaw_msb_asr
  quantity: R1
  coverage: direct
  metric_as_printed: MSB ASR
  value: '34.92'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
  spread: 'SD over three independent runs: 0.89'
- id: r_noaw_msb_nrp
  evaluation_id: ev_noaw_msb_nrp
  quantity: other
  coverage: direct
  metric_as_printed: MSB NRP
  value: '41.87'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
  spread: 'SD over three independent runs: 0.33'
- id: r_noaw_alf
  evaluation_id: ev_noaw_alf
  quantity: general_utility
  coverage: direct
  metric_as_printed: ALFWorld (AgentBench-FC setting) accuracy
  value: '34.25'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
  spread: 'SD over three independent runs: 1.89'
- id: r_noaw_th
  evaluation_id: ev_noaw_th
  quantity: general_utility
  coverage: direct
  metric_as_printed: ToolHop accuracy
  value: '40.63'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
  spread: 'SD over three independent runs: 0.29'
comparisons:
- id: c_ase_ad_asr
  left: ev_ase_ad_asr
  right: ev_base_ad_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_ase_ad_ua
  left: ev_ase_ad_ua
  right: ev_base_ad_ua
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_ase_msb_asr
  left: ev_ase_msb_asr
  right: ev_base_msb_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_ase_msb_nrp
  left: ev_ase_msb_nrp
  right: ev_base_msb_nrp
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_ase_bfcl
  left: ev_ase_bfcl
  right: ev_base_bfcl
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_ase_alf
  left: ev_ase_alf
  right: ev_base_alf
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_ase_th
  left: ev_ase_th
  right: ev_base_th
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_static_ad_asr
  left: ev_static_ad_asr
  right: ev_base_ad_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_static_ad_ua
  left: ev_static_ad_ua
  right: ev_base_ad_ua
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_static_msb_asr
  left: ev_static_msb_asr
  right: ev_base_msb_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_static_msb_nrp
  left: ev_static_msb_nrp
  right: ev_base_msb_nrp
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_static_bfcl
  left: ev_static_bfcl
  right: ev_base_bfcl
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_static_alf
  left: ev_static_alf
  right: ev_base_alf
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_static_th
  left: ev_static_th
  right: ev_base_th
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_spot_ad_asr
  left: ev_spot_ad_asr
  right: ev_base_ad_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_spot_ad_ua
  left: ev_spot_ad_ua
  right: ev_base_ad_ua
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_spot_msb_asr
  left: ev_spot_msb_asr
  right: ev_base_msb_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_spot_msb_nrp
  left: ev_spot_msb_nrp
  right: ev_base_msb_nrp
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_spot_bfcl
  left: ev_spot_bfcl
  right: ev_base_bfcl
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_spot_alf
  left: ev_spot_alf
  right: ev_base_alf
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_spot_th
  left: ev_spot_th
  right: ev_base_th
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_ase_static_ad_asr
  left: ev_ase_ad_asr
  right: ev_static_ad_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: co-evolved attacker with weighting against a fixed attacker (Static-AT)'
  page: 8
- id: c_ase_static_ad_ua
  left: ev_ase_ad_ua
  right: ev_static_ad_ua
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: co-evolved attacker with weighting against a fixed attacker (Static-AT)'
  page: 8
- id: c_ase_static_msb_asr
  left: ev_ase_msb_asr
  right: ev_static_msb_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: co-evolved attacker with weighting against a fixed attacker (Static-AT)'
  page: 8
- id: c_ase_static_msb_nrp
  left: ev_ase_msb_nrp
  right: ev_static_msb_nrp
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: co-evolved attacker with weighting against a fixed attacker (Static-AT)'
  page: 8
- id: c_ase_static_bfcl
  left: ev_ase_bfcl
  right: ev_static_bfcl
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: co-evolved attacker with weighting against a fixed attacker (Static-AT)'
  page: 8
- id: c_ase_static_alf
  left: ev_ase_alf
  right: ev_static_alf
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: co-evolved attacker with weighting against a fixed attacker (Static-AT)'
  page: 8
- id: c_ase_static_th
  left: ev_ase_th
  right: ev_static_th
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: co-evolved attacker with weighting against a fixed attacker (Static-AT)'
  page: 8
- id: c_coevo_msb_asr
  left: ev_nolw_msb_asr
  right: ev_static_msb_asr
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: attacker updates (co-evolution) without task weighting, against a fixed attacker
  page: 10
- id: c_weight_msb_asr
  left: ev_ase_msb_asr
  right: ev_nolw_msb_asr
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: learnability-guided task weighting on both policies
  page: 10
- id: c_dw_msb_asr
  left: ev_ase_msb_asr
  right: ev_nodw_msb_asr
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: defender weighting added to attacker weighting
  page: 10
- id: c_aw_msb_asr
  left: ev_ase_msb_asr
  right: ev_noaw_msb_asr
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: attacker weighting added to defender weighting
  page: 10
- id: c_coevo_msb_nrp
  left: ev_nolw_msb_nrp
  right: ev_static_msb_nrp
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: attacker updates (co-evolution) without task weighting, against a fixed attacker
  page: 10
- id: c_weight_msb_nrp
  left: ev_ase_msb_nrp
  right: ev_nolw_msb_nrp
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: learnability-guided task weighting on both policies
  page: 10
- id: c_dw_msb_nrp
  left: ev_ase_msb_nrp
  right: ev_nodw_msb_nrp
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: defender weighting added to attacker weighting
  page: 10
- id: c_aw_msb_nrp
  left: ev_ase_msb_nrp
  right: ev_noaw_msb_nrp
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: attacker weighting added to defender weighting
  page: 10
- id: c_coevo_alf
  left: ev_nolw_alf
  right: ev_static_alf
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: attacker updates (co-evolution) without task weighting, against a fixed attacker
  page: 10
- id: c_weight_alf
  left: ev_ase_alf
  right: ev_nolw_alf
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: learnability-guided task weighting on both policies
  page: 10
- id: c_dw_alf
  left: ev_ase_alf
  right: ev_nodw_alf
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: defender weighting added to attacker weighting
  page: 10
- id: c_aw_alf
  left: ev_ase_alf
  right: ev_noaw_alf
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: attacker weighting added to defender weighting
  page: 10
- id: c_coevo_th
  left: ev_nolw_th
  right: ev_static_th
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: attacker updates (co-evolution) without task weighting, against a fixed attacker
  page: 10
- id: c_weight_th
  left: ev_ase_th
  right: ev_nolw_th
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: learnability-guided task weighting on both policies
  page: 10
- id: c_dw_th
  left: ev_ase_th
  right: ev_nodw_th
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: defender weighting added to attacker weighting
  page: 10
- id: c_aw_th
  left: ev_ase_th
  right: ev_noaw_th
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: attacker weighting added to defender weighting
  page: 10
claims:
- id: k1
  text: AgentSafeEvo improves adversarial robustness while keeping general agent capabilities
  location: abstract
  anchor: our method improves adversarial robustness while maintaining general agent capabilities
  jobs:
  - R1
  - R2
  - general_utility
  supported_by:
  - c_ase_ad_asr
  - c_ase_ad_ua
  - c_ase_msb_asr
  - c_ase_msb_nrp
  - c_ase_bfcl
  - c_ase_alf
  - c_ase_th
  page: 1
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: direct
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 1 (AgentDojo and MSB ASR)
    R2attack: Table 1 (AgentDojo utility under attack)
    general_utility: Tables 2 and 3
    R2clean: AgentDojo clean utility not reported
schema_gaps:
- where: results r_*_msb_nrp
  note: NRP combines task performance under attack with attack success in one score; coded as quantity other
- where: evaluations ev_*_msb_*
  note: MSB pools attacks on tool signatures, parameters, responses and retrieval results; tool-description attacks are counted with the rest
- where: evaluations ev_*_ad_*
  note: the AgentDojo injection attack and the AgentDojo version are not named
- where: defences ase, static
  note: training tasks are generated across six attack categories, including direct injection and jailbreaks; the reward model is Qwen3.8-27B

`edic_emnlp2026` (2610.22874)

In [ ]:
%%writefile reports/records/edic_emnlp2026.yaml
report:
  id: edic_emnlp2026
  slug: '2610.22874'
  title: 'Identify, Classify, Act: Per-Instruction Controlled Compliance for LLMs over External Data'
  first_author: Zesheng Shi
  venue: EMNLP 2026 (main conference)
  peer_reviewed: 'yes'
  doi: 10.18653/2026.emnlp-main.1092
  arxiv: '2610.22874'
  version_read: arXiv v1
  first_date: '2026-10-16'
  read: F
  role: proposes
  code_release: https://github.com/yingyingxia666/EDIC (linked in the abstract)
  extraction_scope: 'Sections 5 and 6: Table 1 (all rows and columns), Table 2 (trigger), Table 3 (data composition; H-R, B-F, DJ-U and DJ-ASR) and Table 6 (NotInject); the abstract claim (Table 4, Tables 5, 7 and 8 and Figures
    3 to 6 not coded)'
defences:
- id: edic_q3
  name: EDIC (judge-guided per-instruction compliance training, classification trigger on)
  origin: this_report
  base_checkpoint: Qwen3-4B
  axis1:
  - no_signal
  axis2:
  - sft
  - preference_optimisation
  - distillation
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  components: &id001
  - policy SFT on chain-of-thought responses written by GPT-5.4 that identify each embedded instruction, label it beneficial or harmful, and act on the labels
  - generative judge (SFT then GRPO) decides per instruction whether the output followed it
  - 'policy GRPO on a decomposed reward: credit per beneficial instruction followed, penalty per harmful instruction followed, plus a structure bonus'
  - training documents mix beneficial and harmful instructions in four scene families (email, API docs, medical guidance, knowledge evidence)
  layout: user query with external data in the prompt; tool returns for AgentDojo and InjecAgent
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: The RL reward decomposes across instructions
  pages:
    components: 4
    isolation: 6
- id: notrig_q3
  name: EDIC with the classification trigger off at inference (same checkpoint)
  origin: this_report
  base_checkpoint: Qwen3-4B
  axis1:
  - no_signal
  axis2:
  - sft
  - preference_optimisation
  - distillation
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  anchor: This makes fine-grained classification an inference-time switch
- id: secalign_q3
  name: SecAlign (how it was obtained for this backbone is not stated)
  origin: other_report
  base_checkpoint: Qwen3-4B
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  anchor: We compare EDIC against Base model and five prompt-injection defense
- id: struq_q3
  name: StruQ (how it was obtained for this backbone is not stated)
  origin: other_report
  base_checkpoint: Qwen3-4B
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  anchor: We compare EDIC against Base model and five prompt-injection defense
- id: reasalign_q3
  name: ReasAlign (how it was obtained for this backbone is not stated)
  origin: other_report
  base_checkpoint: Qwen3-4B
  axis1:
  - delimiters_markers
  axis2:
  - sft
  - distillation
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  anchor: We compare EDIC against Base model and five prompt-injection defense
- id: edic_l31
  name: EDIC (judge-guided per-instruction compliance training, classification trigger on)
  origin: this_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - no_signal
  axis2:
  - sft
  - preference_optimisation
  - distillation
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  components: *id001
  layout: user query with external data in the prompt; tool returns for AgentDojo and InjecAgent
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: The RL reward decomposes across instructions
  pages:
    components: 4
    isolation: 6
- id: notrig_l31
  name: EDIC with the classification trigger off at inference (same checkpoint)
  origin: this_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - no_signal
  axis2:
  - sft
  - preference_optimisation
  - distillation
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  anchor: This makes fine-grained classification an inference-time switch
- id: secalign_l31
  name: SecAlign (how it was obtained for this backbone is not stated)
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  anchor: We compare EDIC against Base model and five prompt-injection defense
- id: struq_l31
  name: StruQ (how it was obtained for this backbone is not stated)
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  anchor: We compare EDIC against Base model and five prompt-injection defense
- id: reasalign_l31
  name: ReasAlign (how it was obtained for this backbone is not stated)
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  - distillation
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  anchor: We compare EDIC against Base model and five prompt-injection defense
- id: metasec_l31
  name: Meta-SecAlign (checkpoint not named)
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  anchor: We compare EDIC against Base model and five prompt-injection defense
- id: honly_q3
  name: EDIC trained on harmful-only documents
  origin: this_report
  base_checkpoint: Qwen3-4B
  axis1:
  - no_signal
  axis2:
  - sft
  - preference_optimisation
  - distillation
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  anchor: Data composition ablation (Qwen3-4B, with
- id: bonly_q3
  name: EDIC trained on benign-only documents
  origin: this_report
  base_checkpoint: Qwen3-4B
  axis1:
  - no_signal
  axis2:
  - sft
  - preference_optimisation
  - distillation
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  anchor: Data composition ablation (Qwen3-4B, with
- id: sepctx_q3
  name: EDIC trained with beneficial and harmful instructions in separate documents
  origin: this_report
  base_checkpoint: Qwen3-4B
  axis1:
  - no_signal
  axis2:
  - sft
  - preference_optimisation
  - distillation
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  anchor: Data composition ablation (Qwen3-4B, with
evaluations:
- id: ev_q3_none_arena
  defence_id: none:Qwen3-4B
  benchmark: Arena-Hard-Auto (750 hard prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: DeepSeek-Chat against GPT-4o-variant baselines
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
    judge: 13
  baseline_kind: off_the_shelf
- id: ev_q3_none_mtb
  defence_id: none:Qwen3-4B
  benchmark: MT-Bench (80 questions, FastChat)
  attacker_type: none
  attacker_access: not_applicable
  judge: DeepSeek-Chat with GPT-4 reference answers
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
    judge: 13
  baseline_kind: off_the_shelf
- id: ev_q3_none_dju
  defence_id: none:Qwen3-4B
  benchmark: AgentDojo v1, important_instructions, banking, slack, travel and workspace (629 attacked cases)
  attacker_type: static
  attacker_access: black_box
  judge: suite user-task checker on the attacked pass
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
  baseline_kind: off_the_shelf
- id: ev_q3_none_djasr
  defence_id: none:Qwen3-4B
  benchmark: AgentDojo v1, important_instructions, banking, slack, travel and workspace (629 attacked cases)
  attacker_type: static
  attacker_access: black_box
  judge: suite attack-success checker
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
  baseline_kind: off_the_shelf
- id: ev_q3_none_cyb
  defence_id: none:Qwen3-4B
  benchmark: CyberSecEval 2 prompt injection (250 samples, inspect_evals)
  attacker_type: static
  attacker_access: black_box
  judge: DeepSeek-Chat
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
  baseline_kind: off_the_shelf
- id: ev_q3_none_inj
  defence_id: none:Qwen3-4B
  benchmark: InjecAgent, base and enhanced settings, 200 per DH and DS split
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent output check
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
  baseline_kind: off_the_shelf
- id: ev_q3_none_hr
  defence_id: none:Qwen3-4B
  benchmark: held-out mixed-instruction data from the authors' four scenes (150 validation samples per scene)
  attacker_type: static
  attacker_access: black_box
  judge: judge not named
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
  baseline_kind: off_the_shelf
- id: ev_q3_none_bf
  defence_id: none:Qwen3-4B
  benchmark: held-out mixed-instruction data from the authors' four scenes (150 validation samples per scene)
  attacker_type: static
  attacker_access: black_box
  judge: judge not named
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
  baseline_kind: off_the_shelf
- id: ev_q3_secalign_arena
  defence_id: secalign_q3
  benchmark: Arena-Hard-Auto (750 hard prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: DeepSeek-Chat against GPT-4o-variant baselines
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
    judge: 13
- id: ev_q3_secalign_mtb
  defence_id: secalign_q3
  benchmark: MT-Bench (80 questions, FastChat)
  attacker_type: none
  attacker_access: not_applicable
  judge: DeepSeek-Chat with GPT-4 reference answers
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
    judge: 13
- id: ev_q3_secalign_dju
  defence_id: secalign_q3
  benchmark: AgentDojo v1, important_instructions, banking, slack, travel and workspace (629 attacked cases)
  attacker_type: static
  attacker_access: black_box
  judge: suite user-task checker on the attacked pass
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_q3_secalign_djasr
  defence_id: secalign_q3
  benchmark: AgentDojo v1, important_instructions, banking, slack, travel and workspace (629 attacked cases)
  attacker_type: static
  attacker_access: black_box
  judge: suite attack-success checker
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_q3_secalign_cyb
  defence_id: secalign_q3
  benchmark: CyberSecEval 2 prompt injection (250 samples, inspect_evals)
  attacker_type: static
  attacker_access: black_box
  judge: DeepSeek-Chat
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_q3_secalign_inj
  defence_id: secalign_q3
  benchmark: InjecAgent, base and enhanced settings, 200 per DH and DS split
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent output check
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_q3_secalign_hr
  defence_id: secalign_q3
  benchmark: held-out mixed-instruction data from the authors' four scenes (150 validation samples per scene)
  attacker_type: static
  attacker_access: black_box
  judge: judge not named
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
- id: ev_q3_secalign_bf
  defence_id: secalign_q3
  benchmark: held-out mixed-instruction data from the authors' four scenes (150 validation samples per scene)
  attacker_type: static
  attacker_access: black_box
  judge: judge not named
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
- id: ev_q3_struq_dju
  defence_id: struq_q3
  benchmark: AgentDojo v1, important_instructions, banking, slack, travel and workspace (629 attacked cases)
  attacker_type: static
  attacker_access: black_box
  judge: suite user-task checker on the attacked pass
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_q3_struq_djasr
  defence_id: struq_q3
  benchmark: AgentDojo v1, important_instructions, banking, slack, travel and workspace (629 attacked cases)
  attacker_type: static
  attacker_access: black_box
  judge: suite attack-success checker
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_q3_struq_cyb
  defence_id: struq_q3
  benchmark: CyberSecEval 2 prompt injection (250 samples, inspect_evals)
  attacker_type: static
  attacker_access: black_box
  judge: DeepSeek-Chat
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_q3_struq_inj
  defence_id: struq_q3
  benchmark: InjecAgent, base and enhanced settings, 200 per DH and DS split
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent output check
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_q3_struq_hr
  defence_id: struq_q3
  benchmark: held-out mixed-instruction data from the authors' four scenes (150 validation samples per scene)
  attacker_type: static
  attacker_access: black_box
  judge: judge not named
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
- id: ev_q3_struq_bf
  defence_id: struq_q3
  benchmark: held-out mixed-instruction data from the authors' four scenes (150 validation samples per scene)
  attacker_type: static
  attacker_access: black_box
  judge: judge not named
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
- id: ev_q3_reasalign_arena
  defence_id: reasalign_q3
  benchmark: Arena-Hard-Auto (750 hard prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: DeepSeek-Chat against GPT-4o-variant baselines
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
    judge: 13
- id: ev_q3_reasalign_mtb
  defence_id: reasalign_q3
  benchmark: MT-Bench (80 questions, FastChat)
  attacker_type: none
  attacker_access: not_applicable
  judge: DeepSeek-Chat with GPT-4 reference answers
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
    judge: 13
- id: ev_q3_reasalign_dju
  defence_id: reasalign_q3
  benchmark: AgentDojo v1, important_instructions, banking, slack, travel and workspace (629 attacked cases)
  attacker_type: static
  attacker_access: black_box
  judge: suite user-task checker on the attacked pass
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_q3_reasalign_djasr
  defence_id: reasalign_q3
  benchmark: AgentDojo v1, important_instructions, banking, slack, travel and workspace (629 attacked cases)
  attacker_type: static
  attacker_access: black_box
  judge: suite attack-success checker
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_q3_reasalign_cyb
  defence_id: reasalign_q3
  benchmark: CyberSecEval 2 prompt injection (250 samples, inspect_evals)
  attacker_type: static
  attacker_access: black_box
  judge: DeepSeek-Chat
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_q3_reasalign_inj
  defence_id: reasalign_q3
  benchmark: InjecAgent, base and enhanced settings, 200 per DH and DS split
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent output check
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_q3_reasalign_hr
  defence_id: reasalign_q3
  benchmark: held-out mixed-instruction data from the authors' four scenes (150 validation samples per scene)
  attacker_type: static
  attacker_access: black_box
  judge: judge not named
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
- id: ev_q3_reasalign_bf
  defence_id: reasalign_q3
  benchmark: held-out mixed-instruction data from the authors' four scenes (150 validation samples per scene)
  attacker_type: static
  attacker_access: black_box
  judge: judge not named
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
- id: ev_q3_edic_arena
  defence_id: edic_q3
  benchmark: Arena-Hard-Auto (750 hard prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: DeepSeek-Chat against GPT-4o-variant baselines
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
    judge: 13
- id: ev_q3_edic_mtb
  defence_id: edic_q3
  benchmark: MT-Bench (80 questions, FastChat)
  attacker_type: none
  attacker_access: not_applicable
  judge: DeepSeek-Chat with GPT-4 reference answers
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
    judge: 13
- id: ev_q3_edic_dju
  defence_id: edic_q3
  benchmark: AgentDojo v1, important_instructions, banking, slack, travel and workspace (629 attacked cases)
  attacker_type: static
  attacker_access: black_box
  judge: suite user-task checker on the attacked pass
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_q3_edic_djasr
  defence_id: edic_q3
  benchmark: AgentDojo v1, important_instructions, banking, slack, travel and workspace (629 attacked cases)
  attacker_type: static
  attacker_access: black_box
  judge: suite attack-success checker
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_q3_edic_cyb
  defence_id: edic_q3
  benchmark: CyberSecEval 2 prompt injection (250 samples, inspect_evals)
  attacker_type: static
  attacker_access: black_box
  judge: DeepSeek-Chat
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_q3_edic_inj
  defence_id: edic_q3
  benchmark: InjecAgent, base and enhanced settings, 200 per DH and DS split
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent output check
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_q3_edic_hr
  defence_id: edic_q3
  benchmark: held-out mixed-instruction data from the authors' four scenes (150 validation samples per scene)
  attacker_type: static
  attacker_access: black_box
  judge: judge not named
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
- id: ev_q3_edic_bf
  defence_id: edic_q3
  benchmark: held-out mixed-instruction data from the authors' four scenes (150 validation samples per scene)
  attacker_type: static
  attacker_access: black_box
  judge: judge not named
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
- id: ev_q3_notrig_arena
  defence_id: notrig_q3
  benchmark: Arena-Hard-Auto (750 hard prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: DeepSeek-Chat against GPT-4o-variant baselines
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
    judge: 13
- id: ev_q3_notrig_mtb
  defence_id: notrig_q3
  benchmark: MT-Bench (80 questions, FastChat)
  attacker_type: none
  attacker_access: not_applicable
  judge: DeepSeek-Chat with GPT-4 reference answers
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
    judge: 13
- id: ev_q3_notrig_dju
  defence_id: notrig_q3
  benchmark: AgentDojo v1, important_instructions, banking, slack, travel and workspace (629 attacked cases)
  attacker_type: static
  attacker_access: black_box
  judge: suite user-task checker on the attacked pass
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_q3_notrig_djasr
  defence_id: notrig_q3
  benchmark: AgentDojo v1, important_instructions, banking, slack, travel and workspace (629 attacked cases)
  attacker_type: static
  attacker_access: black_box
  judge: suite attack-success checker
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_q3_notrig_cyb
  defence_id: notrig_q3
  benchmark: CyberSecEval 2 prompt injection (250 samples, inspect_evals)
  attacker_type: static
  attacker_access: black_box
  judge: DeepSeek-Chat
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_q3_notrig_inj
  defence_id: notrig_q3
  benchmark: InjecAgent, base and enhanced settings, 200 per DH and DS split
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent output check
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_q3_notrig_hr
  defence_id: notrig_q3
  benchmark: held-out mixed-instruction data from the authors' four scenes (150 validation samples per scene)
  attacker_type: static
  attacker_access: black_box
  judge: judge not named
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
- id: ev_q3_notrig_bf
  defence_id: notrig_q3
  benchmark: held-out mixed-instruction data from the authors' four scenes (150 validation samples per scene)
  attacker_type: static
  attacker_access: black_box
  judge: judge not named
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
- id: ev_l31_none_arena
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: Arena-Hard-Auto (750 hard prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: DeepSeek-Chat against GPT-4o-variant baselines
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
    judge: 13
  baseline_kind: off_the_shelf
- id: ev_l31_none_mtb
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: MT-Bench (80 questions, FastChat)
  attacker_type: none
  attacker_access: not_applicable
  judge: DeepSeek-Chat with GPT-4 reference answers
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
    judge: 13
  baseline_kind: off_the_shelf
- id: ev_l31_none_dju
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: AgentDojo v1, important_instructions, banking, slack, travel and workspace (629 attacked cases)
  attacker_type: static
  attacker_access: black_box
  judge: suite user-task checker on the attacked pass
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
  baseline_kind: off_the_shelf
- id: ev_l31_none_djasr
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: AgentDojo v1, important_instructions, banking, slack, travel and workspace (629 attacked cases)
  attacker_type: static
  attacker_access: black_box
  judge: suite attack-success checker
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
  baseline_kind: off_the_shelf
- id: ev_l31_none_cyb
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: CyberSecEval 2 prompt injection (250 samples, inspect_evals)
  attacker_type: static
  attacker_access: black_box
  judge: DeepSeek-Chat
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
  baseline_kind: off_the_shelf
- id: ev_l31_none_inj
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: InjecAgent, base and enhanced settings, 200 per DH and DS split
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent output check
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
  baseline_kind: off_the_shelf
- id: ev_l31_none_hr
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: held-out mixed-instruction data from the authors' four scenes (150 validation samples per scene)
  attacker_type: static
  attacker_access: black_box
  judge: judge not named
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
  baseline_kind: off_the_shelf
- id: ev_l31_none_bf
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: held-out mixed-instruction data from the authors' four scenes (150 validation samples per scene)
  attacker_type: static
  attacker_access: black_box
  judge: judge not named
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
  baseline_kind: off_the_shelf
- id: ev_l31_secalign_arena
  defence_id: secalign_l31
  benchmark: Arena-Hard-Auto (750 hard prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: DeepSeek-Chat against GPT-4o-variant baselines
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
    judge: 13
- id: ev_l31_secalign_mtb
  defence_id: secalign_l31
  benchmark: MT-Bench (80 questions, FastChat)
  attacker_type: none
  attacker_access: not_applicable
  judge: DeepSeek-Chat with GPT-4 reference answers
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
    judge: 13
- id: ev_l31_secalign_dju
  defence_id: secalign_l31
  benchmark: AgentDojo v1, important_instructions, banking, slack, travel and workspace (629 attacked cases)
  attacker_type: static
  attacker_access: black_box
  judge: suite user-task checker on the attacked pass
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_l31_secalign_djasr
  defence_id: secalign_l31
  benchmark: AgentDojo v1, important_instructions, banking, slack, travel and workspace (629 attacked cases)
  attacker_type: static
  attacker_access: black_box
  judge: suite attack-success checker
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_l31_secalign_cyb
  defence_id: secalign_l31
  benchmark: CyberSecEval 2 prompt injection (250 samples, inspect_evals)
  attacker_type: static
  attacker_access: black_box
  judge: DeepSeek-Chat
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_l31_secalign_inj
  defence_id: secalign_l31
  benchmark: InjecAgent, base and enhanced settings, 200 per DH and DS split
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent output check
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_l31_secalign_hr
  defence_id: secalign_l31
  benchmark: held-out mixed-instruction data from the authors' four scenes (150 validation samples per scene)
  attacker_type: static
  attacker_access: black_box
  judge: judge not named
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
- id: ev_l31_secalign_bf
  defence_id: secalign_l31
  benchmark: held-out mixed-instruction data from the authors' four scenes (150 validation samples per scene)
  attacker_type: static
  attacker_access: black_box
  judge: judge not named
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
- id: ev_l31_struq_dju
  defence_id: struq_l31
  benchmark: AgentDojo v1, important_instructions, banking, slack, travel and workspace (629 attacked cases)
  attacker_type: static
  attacker_access: black_box
  judge: suite user-task checker on the attacked pass
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_l31_struq_djasr
  defence_id: struq_l31
  benchmark: AgentDojo v1, important_instructions, banking, slack, travel and workspace (629 attacked cases)
  attacker_type: static
  attacker_access: black_box
  judge: suite attack-success checker
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_l31_struq_cyb
  defence_id: struq_l31
  benchmark: CyberSecEval 2 prompt injection (250 samples, inspect_evals)
  attacker_type: static
  attacker_access: black_box
  judge: DeepSeek-Chat
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_l31_struq_inj
  defence_id: struq_l31
  benchmark: InjecAgent, base and enhanced settings, 200 per DH and DS split
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent output check
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_l31_struq_hr
  defence_id: struq_l31
  benchmark: held-out mixed-instruction data from the authors' four scenes (150 validation samples per scene)
  attacker_type: static
  attacker_access: black_box
  judge: judge not named
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
- id: ev_l31_struq_bf
  defence_id: struq_l31
  benchmark: held-out mixed-instruction data from the authors' four scenes (150 validation samples per scene)
  attacker_type: static
  attacker_access: black_box
  judge: judge not named
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
- id: ev_l31_metasec_arena
  defence_id: metasec_l31
  benchmark: Arena-Hard-Auto (750 hard prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: DeepSeek-Chat against GPT-4o-variant baselines
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
    judge: 13
- id: ev_l31_metasec_mtb
  defence_id: metasec_l31
  benchmark: MT-Bench (80 questions, FastChat)
  attacker_type: none
  attacker_access: not_applicable
  judge: DeepSeek-Chat with GPT-4 reference answers
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
    judge: 13
- id: ev_l31_metasec_dju
  defence_id: metasec_l31
  benchmark: AgentDojo v1, important_instructions, banking, slack, travel and workspace (629 attacked cases)
  attacker_type: static
  attacker_access: black_box
  judge: suite user-task checker on the attacked pass
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_l31_metasec_djasr
  defence_id: metasec_l31
  benchmark: AgentDojo v1, important_instructions, banking, slack, travel and workspace (629 attacked cases)
  attacker_type: static
  attacker_access: black_box
  judge: suite attack-success checker
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_l31_metasec_cyb
  defence_id: metasec_l31
  benchmark: CyberSecEval 2 prompt injection (250 samples, inspect_evals)
  attacker_type: static
  attacker_access: black_box
  judge: DeepSeek-Chat
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_l31_metasec_inj
  defence_id: metasec_l31
  benchmark: InjecAgent, base and enhanced settings, 200 per DH and DS split
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent output check
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_l31_metasec_hr
  defence_id: metasec_l31
  benchmark: held-out mixed-instruction data from the authors' four scenes (150 validation samples per scene)
  attacker_type: static
  attacker_access: black_box
  judge: judge not named
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
- id: ev_l31_metasec_bf
  defence_id: metasec_l31
  benchmark: held-out mixed-instruction data from the authors' four scenes (150 validation samples per scene)
  attacker_type: static
  attacker_access: black_box
  judge: judge not named
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
- id: ev_l31_reasalign_arena
  defence_id: reasalign_l31
  benchmark: Arena-Hard-Auto (750 hard prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: DeepSeek-Chat against GPT-4o-variant baselines
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
    judge: 13
- id: ev_l31_reasalign_mtb
  defence_id: reasalign_l31
  benchmark: MT-Bench (80 questions, FastChat)
  attacker_type: none
  attacker_access: not_applicable
  judge: DeepSeek-Chat with GPT-4 reference answers
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
    judge: 13
- id: ev_l31_reasalign_dju
  defence_id: reasalign_l31
  benchmark: AgentDojo v1, important_instructions, banking, slack, travel and workspace (629 attacked cases)
  attacker_type: static
  attacker_access: black_box
  judge: suite user-task checker on the attacked pass
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_l31_reasalign_djasr
  defence_id: reasalign_l31
  benchmark: AgentDojo v1, important_instructions, banking, slack, travel and workspace (629 attacked cases)
  attacker_type: static
  attacker_access: black_box
  judge: suite attack-success checker
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_l31_reasalign_cyb
  defence_id: reasalign_l31
  benchmark: CyberSecEval 2 prompt injection (250 samples, inspect_evals)
  attacker_type: static
  attacker_access: black_box
  judge: DeepSeek-Chat
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_l31_reasalign_inj
  defence_id: reasalign_l31
  benchmark: InjecAgent, base and enhanced settings, 200 per DH and DS split
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent output check
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_l31_reasalign_hr
  defence_id: reasalign_l31
  benchmark: held-out mixed-instruction data from the authors' four scenes (150 validation samples per scene)
  attacker_type: static
  attacker_access: black_box
  judge: judge not named
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
- id: ev_l31_reasalign_bf
  defence_id: reasalign_l31
  benchmark: held-out mixed-instruction data from the authors' four scenes (150 validation samples per scene)
  attacker_type: static
  attacker_access: black_box
  judge: judge not named
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
- id: ev_l31_edic_arena
  defence_id: edic_l31
  benchmark: Arena-Hard-Auto (750 hard prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: DeepSeek-Chat against GPT-4o-variant baselines
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
    judge: 13
- id: ev_l31_edic_mtb
  defence_id: edic_l31
  benchmark: MT-Bench (80 questions, FastChat)
  attacker_type: none
  attacker_access: not_applicable
  judge: DeepSeek-Chat with GPT-4 reference answers
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
    judge: 13
- id: ev_l31_edic_dju
  defence_id: edic_l31
  benchmark: AgentDojo v1, important_instructions, banking, slack, travel and workspace (629 attacked cases)
  attacker_type: static
  attacker_access: black_box
  judge: suite user-task checker on the attacked pass
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_l31_edic_djasr
  defence_id: edic_l31
  benchmark: AgentDojo v1, important_instructions, banking, slack, travel and workspace (629 attacked cases)
  attacker_type: static
  attacker_access: black_box
  judge: suite attack-success checker
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_l31_edic_cyb
  defence_id: edic_l31
  benchmark: CyberSecEval 2 prompt injection (250 samples, inspect_evals)
  attacker_type: static
  attacker_access: black_box
  judge: DeepSeek-Chat
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_l31_edic_inj
  defence_id: edic_l31
  benchmark: InjecAgent, base and enhanced settings, 200 per DH and DS split
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent output check
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_l31_edic_hr
  defence_id: edic_l31
  benchmark: held-out mixed-instruction data from the authors' four scenes (150 validation samples per scene)
  attacker_type: static
  attacker_access: black_box
  judge: judge not named
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
- id: ev_l31_edic_bf
  defence_id: edic_l31
  benchmark: held-out mixed-instruction data from the authors' four scenes (150 validation samples per scene)
  attacker_type: static
  attacker_access: black_box
  judge: judge not named
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
- id: ev_l31_notrig_arena
  defence_id: notrig_l31
  benchmark: Arena-Hard-Auto (750 hard prompts)
  attacker_type: none
  attacker_access: not_applicable
  judge: DeepSeek-Chat against GPT-4o-variant baselines
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
    judge: 13
- id: ev_l31_notrig_mtb
  defence_id: notrig_l31
  benchmark: MT-Bench (80 questions, FastChat)
  attacker_type: none
  attacker_access: not_applicable
  judge: DeepSeek-Chat with GPT-4 reference answers
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
    judge: 13
- id: ev_l31_notrig_dju
  defence_id: notrig_l31
  benchmark: AgentDojo v1, important_instructions, banking, slack, travel and workspace (629 attacked cases)
  attacker_type: static
  attacker_access: black_box
  judge: suite user-task checker on the attacked pass
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_l31_notrig_djasr
  defence_id: notrig_l31
  benchmark: AgentDojo v1, important_instructions, banking, slack, travel and workspace (629 attacked cases)
  attacker_type: static
  attacker_access: black_box
  judge: suite attack-success checker
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_l31_notrig_cyb
  defence_id: notrig_l31
  benchmark: CyberSecEval 2 prompt injection (250 samples, inspect_evals)
  attacker_type: static
  attacker_access: black_box
  judge: DeepSeek-Chat
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_l31_notrig_inj
  defence_id: notrig_l31
  benchmark: InjecAgent, base and enhanced settings, 200 per DH and DS split
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent output check
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 13
- id: ev_l31_notrig_hr
  defence_id: notrig_l31
  benchmark: held-out mixed-instruction data from the authors' four scenes (150 validation samples per scene)
  attacker_type: static
  attacker_access: black_box
  judge: judge not named
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
- id: ev_l31_notrig_bf
  defence_id: notrig_l31
  benchmark: held-out mixed-instruction data from the authors' four scenes (150 validation samples per scene)
  attacker_type: static
  attacker_access: black_box
  judge: judge not named
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
- id: ev_q3_honly_hr
  defence_id: honly_q3
  benchmark: held-out mixed-instruction data from the authors' four scenes (150 validation samples per scene)
  attacker_type: static
  attacker_access: black_box
  judge: judge not named
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 7
  decoding: not reported
- id: ev_q3_honly_bf
  defence_id: honly_q3
  benchmark: held-out mixed-instruction data from the authors' four scenes (150 validation samples per scene)
  attacker_type: static
  attacker_access: black_box
  judge: judge not named
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 7
  decoding: not reported
- id: ev_q3_honly_dju
  defence_id: honly_q3
  benchmark: AgentDojo v1, important_instructions, banking, slack, travel and workspace (629 attacked cases)
  attacker_type: static
  attacker_access: black_box
  judge: suite user-task checker on the attacked pass
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 7
  decoding: not reported
- id: ev_q3_honly_djasr
  defence_id: honly_q3
  benchmark: AgentDojo v1, important_instructions, banking, slack, travel and workspace (629 attacked cases)
  attacker_type: static
  attacker_access: black_box
  judge: suite attack-success checker
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 7
  decoding: not reported
- id: ev_q3_bonly_hr
  defence_id: bonly_q3
  benchmark: held-out mixed-instruction data from the authors' four scenes (150 validation samples per scene)
  attacker_type: static
  attacker_access: black_box
  judge: judge not named
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 7
  decoding: not reported
- id: ev_q3_bonly_bf
  defence_id: bonly_q3
  benchmark: held-out mixed-instruction data from the authors' four scenes (150 validation samples per scene)
  attacker_type: static
  attacker_access: black_box
  judge: judge not named
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 7
  decoding: not reported
- id: ev_q3_bonly_dju
  defence_id: bonly_q3
  benchmark: AgentDojo v1, important_instructions, banking, slack, travel and workspace (629 attacked cases)
  attacker_type: static
  attacker_access: black_box
  judge: suite user-task checker on the attacked pass
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 7
  decoding: not reported
- id: ev_q3_bonly_djasr
  defence_id: bonly_q3
  benchmark: AgentDojo v1, important_instructions, banking, slack, travel and workspace (629 attacked cases)
  attacker_type: static
  attacker_access: black_box
  judge: suite attack-success checker
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 7
  decoding: not reported
- id: ev_q3_sepctx_hr
  defence_id: sepctx_q3
  benchmark: held-out mixed-instruction data from the authors' four scenes (150 validation samples per scene)
  attacker_type: static
  attacker_access: black_box
  judge: judge not named
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 7
  decoding: not reported
- id: ev_q3_sepctx_bf
  defence_id: sepctx_q3
  benchmark: held-out mixed-instruction data from the authors' four scenes (150 validation samples per scene)
  attacker_type: static
  attacker_access: black_box
  judge: judge not named
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 7
  decoding: not reported
- id: ev_q3_sepctx_dju
  defence_id: sepctx_q3
  benchmark: AgentDojo v1, important_instructions, banking, slack, travel and workspace (629 attacked cases)
  attacker_type: static
  attacker_access: black_box
  judge: suite user-task checker on the attacked pass
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 7
  decoding: not reported
- id: ev_q3_sepctx_djasr
  defence_id: sepctx_q3
  benchmark: AgentDojo v1, important_instructions, banking, slack, travel and workspace (629 attacked cases)
  attacker_type: static
  attacker_access: black_box
  judge: suite attack-success checker
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 7
  decoding: not reported
- id: ev_q3_none_ni
  defence_id: none:Qwen3-4B
  benchmark: NotInject (339 benign prompts with one to three trigger words)
  attacker_type: none
  attacker_access: not_applicable
  judge: benign accuracy (scorer not named)
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 8
  decoding: not reported
  sample_size: '339'
  baseline_kind: off_the_shelf
- id: ev_q3_secalign_ni
  defence_id: secalign_q3
  benchmark: NotInject (339 benign prompts with one to three trigger words)
  attacker_type: none
  attacker_access: not_applicable
  judge: benign accuracy (scorer not named)
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 8
  decoding: not reported
  sample_size: '339'
- id: ev_q3_reasalign_ni
  defence_id: reasalign_q3
  benchmark: NotInject (339 benign prompts with one to three trigger words)
  attacker_type: none
  attacker_access: not_applicable
  judge: benign accuracy (scorer not named)
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 8
  decoding: not reported
  sample_size: '339'
- id: ev_q3_edic_ni
  defence_id: edic_q3
  benchmark: NotInject (339 benign prompts with one to three trigger words)
  attacker_type: none
  attacker_access: not_applicable
  judge: benign accuracy (scorer not named)
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 8
  decoding: not reported
  sample_size: '339'
- id: ev_l31_none_ni
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: NotInject (339 benign prompts with one to three trigger words)
  attacker_type: none
  attacker_access: not_applicable
  judge: benign accuracy (scorer not named)
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 8
  decoding: not reported
  sample_size: '339'
  baseline_kind: off_the_shelf
- id: ev_l31_secalign_ni
  defence_id: secalign_l31
  benchmark: NotInject (339 benign prompts with one to three trigger words)
  attacker_type: none
  attacker_access: not_applicable
  judge: benign accuracy (scorer not named)
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 8
  decoding: not reported
  sample_size: '339'
- id: ev_l31_reasalign_ni
  defence_id: reasalign_l31
  benchmark: NotInject (339 benign prompts with one to three trigger words)
  attacker_type: none
  attacker_access: not_applicable
  judge: benign accuracy (scorer not named)
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 8
  decoding: not reported
  sample_size: '339'
- id: ev_l31_edic_ni
  defence_id: edic_l31
  benchmark: NotInject (339 benign prompts with one to three trigger words)
  attacker_type: none
  attacker_access: not_applicable
  judge: benign accuracy (scorer not named)
  layout_same_as_defence: 'yes'
  measured_by: edic_emnlp2026
  page: 8
  decoding: not reported
  sample_size: '339'
results:
- id: r_q3_none_arena
  evaluation_id: ev_q3_none_arena
  quantity: general_utility
  coverage: direct
  metric_as_printed: Arena-Hard-Auto win rate
  value: '8.4'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_none_mtb
  evaluation_id: ev_q3_none_mtb
  quantity: general_utility
  coverage: direct
  metric_as_printed: MT-Bench score
  value: '6.64'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_none_dju
  evaluation_id: ev_q3_none_dju
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo utility under attack
  value: '42.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_none_djasr
  evaluation_id: ev_q3_none_djasr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '4.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_none_cyb
  evaluation_id: ev_q3_none_cyb
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval 2 ASR
  value: '54.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_none_inj
  evaluation_id: ev_q3_none_inj
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR
  value: '1.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_none_hr
  evaluation_id: ev_q3_none_hr
  quantity: R1
  coverage: proxy
  metric_as_printed: harmful-reject rate
  value: '67.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_none_bf
  evaluation_id: ev_q3_none_bf
  quantity: other
  coverage: direct
  metric_as_printed: benign-follow rate
  value: '72.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_secalign_arena
  evaluation_id: ev_q3_secalign_arena
  quantity: general_utility
  coverage: direct
  metric_as_printed: Arena-Hard-Auto win rate
  value: '10.2'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_secalign_mtb
  evaluation_id: ev_q3_secalign_mtb
  quantity: general_utility
  coverage: direct
  metric_as_printed: MT-Bench score
  value: '6.59'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_secalign_dju
  evaluation_id: ev_q3_secalign_dju
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo utility under attack
  value: '43.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_secalign_djasr
  evaluation_id: ev_q3_secalign_djasr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '3.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_secalign_cyb
  evaluation_id: ev_q3_secalign_cyb
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval 2 ASR
  value: '29.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_secalign_inj
  evaluation_id: ev_q3_secalign_inj
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR
  value: '0.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_secalign_hr
  evaluation_id: ev_q3_secalign_hr
  quantity: R1
  coverage: proxy
  metric_as_printed: harmful-reject rate
  value: '81.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_secalign_bf
  evaluation_id: ev_q3_secalign_bf
  quantity: other
  coverage: direct
  metric_as_printed: benign-follow rate
  value: '56.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_struq_dju
  evaluation_id: ev_q3_struq_dju
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo utility under attack
  value: '40.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_struq_djasr
  evaluation_id: ev_q3_struq_djasr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '4.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_struq_cyb
  evaluation_id: ev_q3_struq_cyb
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval 2 ASR
  value: '37.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_struq_inj
  evaluation_id: ev_q3_struq_inj
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR
  value: '1.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_struq_hr
  evaluation_id: ev_q3_struq_hr
  quantity: R1
  coverage: proxy
  metric_as_printed: harmful-reject rate
  value: '76.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_struq_bf
  evaluation_id: ev_q3_struq_bf
  quantity: other
  coverage: direct
  metric_as_printed: benign-follow rate
  value: '53.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_reasalign_arena
  evaluation_id: ev_q3_reasalign_arena
  quantity: general_utility
  coverage: direct
  metric_as_printed: Arena-Hard-Auto win rate
  value: '16.4'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_reasalign_mtb
  evaluation_id: ev_q3_reasalign_mtb
  quantity: general_utility
  coverage: direct
  metric_as_printed: MT-Bench score
  value: '7.28'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_reasalign_dju
  evaluation_id: ev_q3_reasalign_dju
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo utility under attack
  value: '44.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_reasalign_djasr
  evaluation_id: ev_q3_reasalign_djasr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '1.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_reasalign_cyb
  evaluation_id: ev_q3_reasalign_cyb
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval 2 ASR
  value: '19.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_reasalign_inj
  evaluation_id: ev_q3_reasalign_inj
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR
  value: '0.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_reasalign_hr
  evaluation_id: ev_q3_reasalign_hr
  quantity: R1
  coverage: proxy
  metric_as_printed: harmful-reject rate
  value: '83.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_reasalign_bf
  evaluation_id: ev_q3_reasalign_bf
  quantity: other
  coverage: direct
  metric_as_printed: benign-follow rate
  value: '77.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_edic_arena
  evaluation_id: ev_q3_edic_arena
  quantity: general_utility
  coverage: direct
  metric_as_printed: Arena-Hard-Auto win rate
  value: '28.9'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_edic_mtb
  evaluation_id: ev_q3_edic_mtb
  quantity: general_utility
  coverage: direct
  metric_as_printed: MT-Bench score
  value: '8.04'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_edic_dju
  evaluation_id: ev_q3_edic_dju
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo utility under attack
  value: '46.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_edic_djasr
  evaluation_id: ev_q3_edic_djasr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '0.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_edic_cyb
  evaluation_id: ev_q3_edic_cyb
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval 2 ASR
  value: '15.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_edic_inj
  evaluation_id: ev_q3_edic_inj
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR
  value: '0.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_edic_hr
  evaluation_id: ev_q3_edic_hr
  quantity: R1
  coverage: proxy
  metric_as_printed: harmful-reject rate
  value: '91.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_edic_bf
  evaluation_id: ev_q3_edic_bf
  quantity: other
  coverage: direct
  metric_as_printed: benign-follow rate
  value: '89.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q3_notrig_arena
  evaluation_id: ev_q3_notrig_arena
  quantity: general_utility
  coverage: direct
  metric_as_printed: Arena-Hard-Auto win rate
  value: '30.1'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q3_notrig_mtb
  evaluation_id: ev_q3_notrig_mtb
  quantity: general_utility
  coverage: direct
  metric_as_printed: MT-Bench score
  value: '8.11'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q3_notrig_dju
  evaluation_id: ev_q3_notrig_dju
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo utility under attack
  value: '47.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q3_notrig_djasr
  evaluation_id: ev_q3_notrig_djasr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '0.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q3_notrig_cyb
  evaluation_id: ev_q3_notrig_cyb
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval 2 ASR
  value: '17.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q3_notrig_inj
  evaluation_id: ev_q3_notrig_inj
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q3_notrig_hr
  evaluation_id: ev_q3_notrig_hr
  quantity: R1
  coverage: proxy
  metric_as_printed: harmful-reject rate
  value: '87.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q3_notrig_bf
  evaluation_id: ev_q3_notrig_bf
  quantity: other
  coverage: direct
  metric_as_printed: benign-follow rate
  value: '90.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_none_arena
  evaluation_id: ev_l31_none_arena
  quantity: general_utility
  coverage: direct
  metric_as_printed: Arena-Hard-Auto win rate
  value: '1.6'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_none_mtb
  evaluation_id: ev_l31_none_mtb
  quantity: general_utility
  coverage: direct
  metric_as_printed: MT-Bench score
  value: '6.38'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_none_dju
  evaluation_id: ev_l31_none_dju
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo utility under attack
  value: '6.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_none_djasr
  evaluation_id: ev_l31_none_djasr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '0.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_none_cyb
  evaluation_id: ev_l31_none_cyb
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval 2 ASR
  value: '28.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_none_inj
  evaluation_id: ev_l31_none_inj
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR
  value: '14.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_none_hr
  evaluation_id: ev_l31_none_hr
  quantity: R1
  coverage: proxy
  metric_as_printed: harmful-reject rate
  value: '74.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_none_bf
  evaluation_id: ev_l31_none_bf
  quantity: other
  coverage: direct
  metric_as_printed: benign-follow rate
  value: '85.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_secalign_arena
  evaluation_id: ev_l31_secalign_arena
  quantity: general_utility
  coverage: direct
  metric_as_printed: Arena-Hard-Auto win rate
  value: '3.2'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_secalign_mtb
  evaluation_id: ev_l31_secalign_mtb
  quantity: general_utility
  coverage: direct
  metric_as_printed: MT-Bench score
  value: '6.41'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_secalign_dju
  evaluation_id: ev_l31_secalign_dju
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo utility under attack
  value: '6.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_secalign_djasr
  evaluation_id: ev_l31_secalign_djasr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_secalign_cyb
  evaluation_id: ev_l31_secalign_cyb
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval 2 ASR
  value: '14.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_secalign_inj
  evaluation_id: ev_l31_secalign_inj
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR
  value: '2.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_secalign_hr
  evaluation_id: ev_l31_secalign_hr
  quantity: R1
  coverage: proxy
  metric_as_printed: harmful-reject rate
  value: '88.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_secalign_bf
  evaluation_id: ev_l31_secalign_bf
  quantity: other
  coverage: direct
  metric_as_printed: benign-follow rate
  value: '67.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_struq_dju
  evaluation_id: ev_l31_struq_dju
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo utility under attack
  value: '4.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_struq_djasr
  evaluation_id: ev_l31_struq_djasr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '0.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_struq_cyb
  evaluation_id: ev_l31_struq_cyb
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval 2 ASR
  value: '18.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_struq_inj
  evaluation_id: ev_l31_struq_inj
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR
  value: '7.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_struq_hr
  evaluation_id: ev_l31_struq_hr
  quantity: R1
  coverage: proxy
  metric_as_printed: harmful-reject rate
  value: '85.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_struq_bf
  evaluation_id: ev_l31_struq_bf
  quantity: other
  coverage: direct
  metric_as_printed: benign-follow rate
  value: '62.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_metasec_arena
  evaluation_id: ev_l31_metasec_arena
  quantity: general_utility
  coverage: direct
  metric_as_printed: Arena-Hard-Auto win rate
  value: '4.6'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_metasec_mtb
  evaluation_id: ev_l31_metasec_mtb
  quantity: general_utility
  coverage: direct
  metric_as_printed: MT-Bench score
  value: '6.68'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_metasec_dju
  evaluation_id: ev_l31_metasec_dju
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo utility under attack
  value: '6.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_metasec_djasr
  evaluation_id: ev_l31_metasec_djasr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_metasec_cyb
  evaluation_id: ev_l31_metasec_cyb
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval 2 ASR
  value: '7.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_metasec_inj
  evaluation_id: ev_l31_metasec_inj
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR
  value: '1.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_metasec_hr
  evaluation_id: ev_l31_metasec_hr
  quantity: R1
  coverage: proxy
  metric_as_printed: harmful-reject rate
  value: '91.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_metasec_bf
  evaluation_id: ev_l31_metasec_bf
  quantity: other
  coverage: direct
  metric_as_printed: benign-follow rate
  value: '68.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_reasalign_arena
  evaluation_id: ev_l31_reasalign_arena
  quantity: general_utility
  coverage: direct
  metric_as_printed: Arena-Hard-Auto win rate
  value: '7.8'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_reasalign_mtb
  evaluation_id: ev_l31_reasalign_mtb
  quantity: general_utility
  coverage: direct
  metric_as_printed: MT-Bench score
  value: '6.95'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_reasalign_dju
  evaluation_id: ev_l31_reasalign_dju
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo utility under attack
  value: '7.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_reasalign_djasr
  evaluation_id: ev_l31_reasalign_djasr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '0.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_reasalign_cyb
  evaluation_id: ev_l31_reasalign_cyb
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval 2 ASR
  value: '3.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_reasalign_inj
  evaluation_id: ev_l31_reasalign_inj
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR
  value: '1.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_reasalign_hr
  evaluation_id: ev_l31_reasalign_hr
  quantity: R1
  coverage: proxy
  metric_as_printed: harmful-reject rate
  value: '93.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_reasalign_bf
  evaluation_id: ev_l31_reasalign_bf
  quantity: other
  coverage: direct
  metric_as_printed: benign-follow rate
  value: '87.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_edic_arena
  evaluation_id: ev_l31_edic_arena
  quantity: general_utility
  coverage: direct
  metric_as_printed: Arena-Hard-Auto win rate
  value: '12.4'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_edic_mtb
  evaluation_id: ev_l31_edic_mtb
  quantity: general_utility
  coverage: direct
  metric_as_printed: MT-Bench score
  value: '7.33'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_edic_dju
  evaluation_id: ev_l31_edic_dju
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo utility under attack
  value: '15.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_edic_djasr
  evaluation_id: ev_l31_edic_djasr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '0.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_edic_cyb
  evaluation_id: ev_l31_edic_cyb
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval 2 ASR
  value: '2.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_edic_inj
  evaluation_id: ev_l31_edic_inj
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR
  value: '0.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_edic_hr
  evaluation_id: ev_l31_edic_hr
  quantity: R1
  coverage: proxy
  metric_as_printed: harmful-reject rate
  value: '92.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_edic_bf
  evaluation_id: ev_l31_edic_bf
  quantity: other
  coverage: direct
  metric_as_printed: benign-follow rate
  value: '91.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l31_notrig_arena
  evaluation_id: ev_l31_notrig_arena
  quantity: general_utility
  coverage: direct
  metric_as_printed: Arena-Hard-Auto win rate
  value: '12.9'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_notrig_mtb
  evaluation_id: ev_l31_notrig_mtb
  quantity: general_utility
  coverage: direct
  metric_as_printed: MT-Bench score
  value: '7.48'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_notrig_dju
  evaluation_id: ev_l31_notrig_dju
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo utility under attack
  value: '17.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_notrig_djasr
  evaluation_id: ev_l31_notrig_djasr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '0.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_notrig_cyb
  evaluation_id: ev_l31_notrig_cyb
  quantity: R1
  coverage: direct
  metric_as_printed: CyberSecEval 2 ASR
  value: '3.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_notrig_inj
  evaluation_id: ev_l31_notrig_inj
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR
  value: '0.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_notrig_hr
  evaluation_id: ev_l31_notrig_hr
  quantity: R1
  coverage: proxy
  metric_as_printed: harmful-reject rate
  value: '89.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_notrig_bf
  evaluation_id: ev_l31_notrig_bf
  quantity: other
  coverage: direct
  metric_as_printed: benign-follow rate
  value: '94.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q3_honly_hr
  evaluation_id: ev_q3_honly_hr
  quantity: R1
  coverage: proxy
  metric_as_printed: harmful-reject rate
  value: '92.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q3_honly_bf
  evaluation_id: ev_q3_honly_bf
  quantity: other
  coverage: direct
  metric_as_printed: benign-follow rate
  value: '74.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q3_honly_dju
  evaluation_id: ev_q3_honly_dju
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo utility under attack
  value: '40.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q3_honly_djasr
  evaluation_id: ev_q3_honly_djasr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q3_bonly_hr
  evaluation_id: ev_q3_bonly_hr
  quantity: R1
  coverage: proxy
  metric_as_printed: harmful-reject rate
  value: '65.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q3_bonly_bf
  evaluation_id: ev_q3_bonly_bf
  quantity: other
  coverage: direct
  metric_as_printed: benign-follow rate
  value: '94.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q3_bonly_dju
  evaluation_id: ev_q3_bonly_dju
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo utility under attack
  value: '44.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q3_bonly_djasr
  evaluation_id: ev_q3_bonly_djasr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '4.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q3_sepctx_hr
  evaluation_id: ev_q3_sepctx_hr
  quantity: R1
  coverage: proxy
  metric_as_printed: harmful-reject rate
  value: '82.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q3_sepctx_bf
  evaluation_id: ev_q3_sepctx_bf
  quantity: other
  coverage: direct
  metric_as_printed: benign-follow rate
  value: '84.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q3_sepctx_dju
  evaluation_id: ev_q3_sepctx_dju
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo utility under attack
  value: '47.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q3_sepctx_djasr
  evaluation_id: ev_q3_sepctx_djasr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '0.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q3_none_ni
  evaluation_id: ev_q3_none_ni
  quantity: other
  coverage: direct
  metric_as_printed: NotInject benign accuracy (over-refusal)
  value: '93.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 6
- id: r_q3_secalign_ni
  evaluation_id: ev_q3_secalign_ni
  quantity: other
  coverage: direct
  metric_as_printed: NotInject benign accuracy (over-refusal)
  value: '72.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 6
- id: r_q3_reasalign_ni
  evaluation_id: ev_q3_reasalign_ni
  quantity: other
  coverage: direct
  metric_as_printed: NotInject benign accuracy (over-refusal)
  value: '78.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 6
- id: r_q3_edic_ni
  evaluation_id: ev_q3_edic_ni
  quantity: other
  coverage: direct
  metric_as_printed: NotInject benign accuracy (over-refusal)
  value: '88.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 6
- id: r_l31_none_ni
  evaluation_id: ev_l31_none_ni
  quantity: other
  coverage: direct
  metric_as_printed: NotInject benign accuracy (over-refusal)
  value: '91.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 6
- id: r_l31_secalign_ni
  evaluation_id: ev_l31_secalign_ni
  quantity: other
  coverage: direct
  metric_as_printed: NotInject benign accuracy (over-refusal)
  value: '69.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 6
- id: r_l31_reasalign_ni
  evaluation_id: ev_l31_reasalign_ni
  quantity: other
  coverage: direct
  metric_as_printed: NotInject benign accuracy (over-refusal)
  value: '75.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 6
- id: r_l31_edic_ni
  evaluation_id: ev_l31_edic_ni
  quantity: other
  coverage: direct
  metric_as_printed: NotInject benign accuracy (over-refusal)
  value: '87.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 6
comparisons:
- id: c_q3_secalign_arena
  left: ev_q3_secalign_arena
  right: ev_q3_none_arena
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q3_edic_secalign_arena
  left: ev_q3_edic_arena
  right: ev_q3_secalign_arena
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against secalign on the same backbone'
  page: 6
- id: c_q3_secalign_mtb
  left: ev_q3_secalign_mtb
  right: ev_q3_none_mtb
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q3_edic_secalign_mtb
  left: ev_q3_edic_mtb
  right: ev_q3_secalign_mtb
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against secalign on the same backbone'
  page: 6
- id: c_q3_secalign_dju
  left: ev_q3_secalign_dju
  right: ev_q3_none_dju
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q3_edic_secalign_dju
  left: ev_q3_edic_dju
  right: ev_q3_secalign_dju
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against secalign on the same backbone'
  page: 6
- id: c_q3_secalign_djasr
  left: ev_q3_secalign_djasr
  right: ev_q3_none_djasr
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q3_edic_secalign_djasr
  left: ev_q3_edic_djasr
  right: ev_q3_secalign_djasr
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against secalign on the same backbone'
  page: 6
- id: c_q3_secalign_cyb
  left: ev_q3_secalign_cyb
  right: ev_q3_none_cyb
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q3_edic_secalign_cyb
  left: ev_q3_edic_cyb
  right: ev_q3_secalign_cyb
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against secalign on the same backbone'
  page: 6
- id: c_q3_secalign_inj
  left: ev_q3_secalign_inj
  right: ev_q3_none_inj
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q3_edic_secalign_inj
  left: ev_q3_edic_inj
  right: ev_q3_secalign_inj
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against secalign on the same backbone'
  page: 6
- id: c_q3_secalign_hr
  left: ev_q3_secalign_hr
  right: ev_q3_none_hr
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q3_edic_secalign_hr
  left: ev_q3_edic_hr
  right: ev_q3_secalign_hr
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against secalign on the same backbone'
  page: 6
- id: c_q3_secalign_bf
  left: ev_q3_secalign_bf
  right: ev_q3_none_bf
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q3_edic_secalign_bf
  left: ev_q3_edic_bf
  right: ev_q3_secalign_bf
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against secalign on the same backbone'
  page: 6
- id: c_q3_struq_dju
  left: ev_q3_struq_dju
  right: ev_q3_none_dju
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q3_edic_struq_dju
  left: ev_q3_edic_dju
  right: ev_q3_struq_dju
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against struq on the same backbone'
  page: 6
- id: c_q3_struq_djasr
  left: ev_q3_struq_djasr
  right: ev_q3_none_djasr
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q3_edic_struq_djasr
  left: ev_q3_edic_djasr
  right: ev_q3_struq_djasr
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against struq on the same backbone'
  page: 6
- id: c_q3_struq_cyb
  left: ev_q3_struq_cyb
  right: ev_q3_none_cyb
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q3_edic_struq_cyb
  left: ev_q3_edic_cyb
  right: ev_q3_struq_cyb
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against struq on the same backbone'
  page: 6
- id: c_q3_struq_inj
  left: ev_q3_struq_inj
  right: ev_q3_none_inj
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q3_edic_struq_inj
  left: ev_q3_edic_inj
  right: ev_q3_struq_inj
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against struq on the same backbone'
  page: 6
- id: c_q3_struq_hr
  left: ev_q3_struq_hr
  right: ev_q3_none_hr
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q3_edic_struq_hr
  left: ev_q3_edic_hr
  right: ev_q3_struq_hr
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against struq on the same backbone'
  page: 6
- id: c_q3_struq_bf
  left: ev_q3_struq_bf
  right: ev_q3_none_bf
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q3_edic_struq_bf
  left: ev_q3_edic_bf
  right: ev_q3_struq_bf
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against struq on the same backbone'
  page: 6
- id: c_q3_reasalign_arena
  left: ev_q3_reasalign_arena
  right: ev_q3_none_arena
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q3_edic_reasalign_arena
  left: ev_q3_edic_arena
  right: ev_q3_reasalign_arena
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against reasalign on the same backbone'
  page: 6
- id: c_q3_reasalign_mtb
  left: ev_q3_reasalign_mtb
  right: ev_q3_none_mtb
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q3_edic_reasalign_mtb
  left: ev_q3_edic_mtb
  right: ev_q3_reasalign_mtb
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against reasalign on the same backbone'
  page: 6
- id: c_q3_reasalign_dju
  left: ev_q3_reasalign_dju
  right: ev_q3_none_dju
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q3_edic_reasalign_dju
  left: ev_q3_edic_dju
  right: ev_q3_reasalign_dju
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against reasalign on the same backbone'
  page: 6
- id: c_q3_reasalign_djasr
  left: ev_q3_reasalign_djasr
  right: ev_q3_none_djasr
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q3_edic_reasalign_djasr
  left: ev_q3_edic_djasr
  right: ev_q3_reasalign_djasr
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against reasalign on the same backbone'
  page: 6
- id: c_q3_reasalign_cyb
  left: ev_q3_reasalign_cyb
  right: ev_q3_none_cyb
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q3_edic_reasalign_cyb
  left: ev_q3_edic_cyb
  right: ev_q3_reasalign_cyb
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against reasalign on the same backbone'
  page: 6
- id: c_q3_reasalign_inj
  left: ev_q3_reasalign_inj
  right: ev_q3_none_inj
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q3_edic_reasalign_inj
  left: ev_q3_edic_inj
  right: ev_q3_reasalign_inj
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against reasalign on the same backbone'
  page: 6
- id: c_q3_reasalign_hr
  left: ev_q3_reasalign_hr
  right: ev_q3_none_hr
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q3_edic_reasalign_hr
  left: ev_q3_edic_hr
  right: ev_q3_reasalign_hr
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against reasalign on the same backbone'
  page: 6
- id: c_q3_reasalign_bf
  left: ev_q3_reasalign_bf
  right: ev_q3_none_bf
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q3_edic_reasalign_bf
  left: ev_q3_edic_bf
  right: ev_q3_reasalign_bf
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against reasalign on the same backbone'
  page: 6
- id: c_q3_edic_arena
  left: ev_q3_edic_arena
  right: ev_q3_none_arena
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q3_edic_mtb
  left: ev_q3_edic_mtb
  right: ev_q3_none_mtb
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q3_edic_dju
  left: ev_q3_edic_dju
  right: ev_q3_none_dju
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q3_edic_djasr
  left: ev_q3_edic_djasr
  right: ev_q3_none_djasr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q3_edic_cyb
  left: ev_q3_edic_cyb
  right: ev_q3_none_cyb
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q3_edic_inj
  left: ev_q3_edic_inj
  right: ev_q3_none_inj
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q3_edic_hr
  left: ev_q3_edic_hr
  right: ev_q3_none_hr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q3_edic_bf
  left: ev_q3_edic_bf
  right: ev_q3_none_bf
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q3_trig_arena
  left: ev_q3_edic_arena
  right: ev_q3_notrig_arena
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the classification trigger switched on at inference
  page: 6
- id: c_q3_trig_mtb
  left: ev_q3_edic_mtb
  right: ev_q3_notrig_mtb
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the classification trigger switched on at inference
  page: 6
- id: c_q3_trig_dju
  left: ev_q3_edic_dju
  right: ev_q3_notrig_dju
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the classification trigger switched on at inference
  page: 6
- id: c_q3_trig_djasr
  left: ev_q3_edic_djasr
  right: ev_q3_notrig_djasr
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the classification trigger switched on at inference
  page: 6
- id: c_q3_trig_cyb
  left: ev_q3_edic_cyb
  right: ev_q3_notrig_cyb
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the classification trigger switched on at inference
  page: 6
- id: c_q3_trig_inj
  left: ev_q3_edic_inj
  right: ev_q3_notrig_inj
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the classification trigger switched on at inference
  page: 6
- id: c_q3_trig_hr
  left: ev_q3_edic_hr
  right: ev_q3_notrig_hr
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the classification trigger switched on at inference
  page: 6
- id: c_q3_trig_bf
  left: ev_q3_edic_bf
  right: ev_q3_notrig_bf
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the classification trigger switched on at inference
  page: 6
- id: c_l31_secalign_arena
  left: ev_l31_secalign_arena
  right: ev_l31_none_arena
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_secalign_arena
  left: ev_l31_edic_arena
  right: ev_l31_secalign_arena
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against secalign on the same backbone'
  page: 6
- id: c_l31_secalign_mtb
  left: ev_l31_secalign_mtb
  right: ev_l31_none_mtb
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_secalign_mtb
  left: ev_l31_edic_mtb
  right: ev_l31_secalign_mtb
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against secalign on the same backbone'
  page: 6
- id: c_l31_secalign_dju
  left: ev_l31_secalign_dju
  right: ev_l31_none_dju
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_secalign_dju
  left: ev_l31_edic_dju
  right: ev_l31_secalign_dju
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against secalign on the same backbone'
  page: 6
- id: c_l31_secalign_djasr
  left: ev_l31_secalign_djasr
  right: ev_l31_none_djasr
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_secalign_djasr
  left: ev_l31_edic_djasr
  right: ev_l31_secalign_djasr
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against secalign on the same backbone'
  page: 6
- id: c_l31_secalign_cyb
  left: ev_l31_secalign_cyb
  right: ev_l31_none_cyb
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_secalign_cyb
  left: ev_l31_edic_cyb
  right: ev_l31_secalign_cyb
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against secalign on the same backbone'
  page: 6
- id: c_l31_secalign_inj
  left: ev_l31_secalign_inj
  right: ev_l31_none_inj
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_secalign_inj
  left: ev_l31_edic_inj
  right: ev_l31_secalign_inj
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against secalign on the same backbone'
  page: 6
- id: c_l31_secalign_hr
  left: ev_l31_secalign_hr
  right: ev_l31_none_hr
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_secalign_hr
  left: ev_l31_edic_hr
  right: ev_l31_secalign_hr
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against secalign on the same backbone'
  page: 6
- id: c_l31_secalign_bf
  left: ev_l31_secalign_bf
  right: ev_l31_none_bf
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_secalign_bf
  left: ev_l31_edic_bf
  right: ev_l31_secalign_bf
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against secalign on the same backbone'
  page: 6
- id: c_l31_struq_dju
  left: ev_l31_struq_dju
  right: ev_l31_none_dju
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_struq_dju
  left: ev_l31_edic_dju
  right: ev_l31_struq_dju
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against struq on the same backbone'
  page: 6
- id: c_l31_struq_djasr
  left: ev_l31_struq_djasr
  right: ev_l31_none_djasr
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_struq_djasr
  left: ev_l31_edic_djasr
  right: ev_l31_struq_djasr
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against struq on the same backbone'
  page: 6
- id: c_l31_struq_cyb
  left: ev_l31_struq_cyb
  right: ev_l31_none_cyb
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_struq_cyb
  left: ev_l31_edic_cyb
  right: ev_l31_struq_cyb
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against struq on the same backbone'
  page: 6
- id: c_l31_struq_inj
  left: ev_l31_struq_inj
  right: ev_l31_none_inj
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_struq_inj
  left: ev_l31_edic_inj
  right: ev_l31_struq_inj
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against struq on the same backbone'
  page: 6
- id: c_l31_struq_hr
  left: ev_l31_struq_hr
  right: ev_l31_none_hr
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_struq_hr
  left: ev_l31_edic_hr
  right: ev_l31_struq_hr
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against struq on the same backbone'
  page: 6
- id: c_l31_struq_bf
  left: ev_l31_struq_bf
  right: ev_l31_none_bf
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_struq_bf
  left: ev_l31_edic_bf
  right: ev_l31_struq_bf
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against struq on the same backbone'
  page: 6
- id: c_l31_metasec_arena
  left: ev_l31_metasec_arena
  right: ev_l31_none_arena
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_metasec_arena
  left: ev_l31_edic_arena
  right: ev_l31_metasec_arena
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against metasec on the same backbone'
  page: 6
- id: c_l31_metasec_mtb
  left: ev_l31_metasec_mtb
  right: ev_l31_none_mtb
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_metasec_mtb
  left: ev_l31_edic_mtb
  right: ev_l31_metasec_mtb
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against metasec on the same backbone'
  page: 6
- id: c_l31_metasec_dju
  left: ev_l31_metasec_dju
  right: ev_l31_none_dju
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_metasec_dju
  left: ev_l31_edic_dju
  right: ev_l31_metasec_dju
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against metasec on the same backbone'
  page: 6
- id: c_l31_metasec_djasr
  left: ev_l31_metasec_djasr
  right: ev_l31_none_djasr
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_metasec_djasr
  left: ev_l31_edic_djasr
  right: ev_l31_metasec_djasr
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against metasec on the same backbone'
  page: 6
- id: c_l31_metasec_cyb
  left: ev_l31_metasec_cyb
  right: ev_l31_none_cyb
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_metasec_cyb
  left: ev_l31_edic_cyb
  right: ev_l31_metasec_cyb
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against metasec on the same backbone'
  page: 6
- id: c_l31_metasec_inj
  left: ev_l31_metasec_inj
  right: ev_l31_none_inj
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_metasec_inj
  left: ev_l31_edic_inj
  right: ev_l31_metasec_inj
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against metasec on the same backbone'
  page: 6
- id: c_l31_metasec_hr
  left: ev_l31_metasec_hr
  right: ev_l31_none_hr
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_metasec_hr
  left: ev_l31_edic_hr
  right: ev_l31_metasec_hr
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against metasec on the same backbone'
  page: 6
- id: c_l31_metasec_bf
  left: ev_l31_metasec_bf
  right: ev_l31_none_bf
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_metasec_bf
  left: ev_l31_edic_bf
  right: ev_l31_metasec_bf
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against metasec on the same backbone'
  page: 6
- id: c_l31_reasalign_arena
  left: ev_l31_reasalign_arena
  right: ev_l31_none_arena
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_reasalign_arena
  left: ev_l31_edic_arena
  right: ev_l31_reasalign_arena
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against reasalign on the same backbone'
  page: 6
- id: c_l31_reasalign_mtb
  left: ev_l31_reasalign_mtb
  right: ev_l31_none_mtb
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_reasalign_mtb
  left: ev_l31_edic_mtb
  right: ev_l31_reasalign_mtb
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against reasalign on the same backbone'
  page: 6
- id: c_l31_reasalign_dju
  left: ev_l31_reasalign_dju
  right: ev_l31_none_dju
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_reasalign_dju
  left: ev_l31_edic_dju
  right: ev_l31_reasalign_dju
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against reasalign on the same backbone'
  page: 6
- id: c_l31_reasalign_djasr
  left: ev_l31_reasalign_djasr
  right: ev_l31_none_djasr
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_reasalign_djasr
  left: ev_l31_edic_djasr
  right: ev_l31_reasalign_djasr
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against reasalign on the same backbone'
  page: 6
- id: c_l31_reasalign_cyb
  left: ev_l31_reasalign_cyb
  right: ev_l31_none_cyb
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_reasalign_cyb
  left: ev_l31_edic_cyb
  right: ev_l31_reasalign_cyb
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against reasalign on the same backbone'
  page: 6
- id: c_l31_reasalign_inj
  left: ev_l31_reasalign_inj
  right: ev_l31_none_inj
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_reasalign_inj
  left: ev_l31_edic_inj
  right: ev_l31_reasalign_inj
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against reasalign on the same backbone'
  page: 6
- id: c_l31_reasalign_hr
  left: ev_l31_reasalign_hr
  right: ev_l31_none_hr
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_reasalign_hr
  left: ev_l31_edic_hr
  right: ev_l31_reasalign_hr
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against reasalign on the same backbone'
  page: 6
- id: c_l31_reasalign_bf
  left: ev_l31_reasalign_bf
  right: ev_l31_none_bf
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_reasalign_bf
  left: ev_l31_edic_bf
  right: ev_l31_reasalign_bf
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: EDIC against reasalign on the same backbone'
  page: 6
- id: c_l31_edic_arena
  left: ev_l31_edic_arena
  right: ev_l31_none_arena
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_mtb
  left: ev_l31_edic_mtb
  right: ev_l31_none_mtb
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_dju
  left: ev_l31_edic_dju
  right: ev_l31_none_dju
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_djasr
  left: ev_l31_edic_djasr
  right: ev_l31_none_djasr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_cyb
  left: ev_l31_edic_cyb
  right: ev_l31_none_cyb
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_inj
  left: ev_l31_edic_inj
  right: ev_l31_none_inj
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_hr
  left: ev_l31_edic_hr
  right: ev_l31_none_hr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_edic_bf
  left: ev_l31_edic_bf
  right: ev_l31_none_bf
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_trig_arena
  left: ev_l31_edic_arena
  right: ev_l31_notrig_arena
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the classification trigger switched on at inference
  page: 6
- id: c_l31_trig_mtb
  left: ev_l31_edic_mtb
  right: ev_l31_notrig_mtb
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the classification trigger switched on at inference
  page: 6
- id: c_l31_trig_dju
  left: ev_l31_edic_dju
  right: ev_l31_notrig_dju
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the classification trigger switched on at inference
  page: 6
- id: c_l31_trig_djasr
  left: ev_l31_edic_djasr
  right: ev_l31_notrig_djasr
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the classification trigger switched on at inference
  page: 6
- id: c_l31_trig_cyb
  left: ev_l31_edic_cyb
  right: ev_l31_notrig_cyb
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the classification trigger switched on at inference
  page: 6
- id: c_l31_trig_inj
  left: ev_l31_edic_inj
  right: ev_l31_notrig_inj
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the classification trigger switched on at inference
  page: 6
- id: c_l31_trig_hr
  left: ev_l31_edic_hr
  right: ev_l31_notrig_hr
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the classification trigger switched on at inference
  page: 6
- id: c_l31_trig_bf
  left: ev_l31_edic_bf
  right: ev_l31_notrig_bf
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the classification trigger switched on at inference
  page: 6
- id: c_q3_mixed_honly_hr
  left: ev_q3_edic_hr
  right: ev_q3_honly_hr
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: training documents that mix beneficial and harmful instructions, against on harmful-only documents
  page: 7
- id: c_q3_mixed_honly_bf
  left: ev_q3_edic_bf
  right: ev_q3_honly_bf
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: training documents that mix beneficial and harmful instructions, against on harmful-only documents
  page: 7
- id: c_q3_mixed_honly_dju
  left: ev_q3_edic_dju
  right: ev_q3_honly_dju
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: training documents that mix beneficial and harmful instructions, against on harmful-only documents
  page: 7
- id: c_q3_mixed_honly_djasr
  left: ev_q3_edic_djasr
  right: ev_q3_honly_djasr
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: training documents that mix beneficial and harmful instructions, against on harmful-only documents
  page: 7
- id: c_q3_mixed_bonly_hr
  left: ev_q3_edic_hr
  right: ev_q3_bonly_hr
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: training documents that mix beneficial and harmful instructions, against on benign-only documents
  page: 7
- id: c_q3_mixed_bonly_bf
  left: ev_q3_edic_bf
  right: ev_q3_bonly_bf
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: training documents that mix beneficial and harmful instructions, against on benign-only documents
  page: 7
- id: c_q3_mixed_bonly_dju
  left: ev_q3_edic_dju
  right: ev_q3_bonly_dju
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: training documents that mix beneficial and harmful instructions, against on benign-only documents
  page: 7
- id: c_q3_mixed_bonly_djasr
  left: ev_q3_edic_djasr
  right: ev_q3_bonly_djasr
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: training documents that mix beneficial and harmful instructions, against on benign-only documents
  page: 7
- id: c_q3_mixed_sepctx_hr
  left: ev_q3_edic_hr
  right: ev_q3_sepctx_hr
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: training documents that mix beneficial and harmful instructions, against with beneficial and harmful instructions in separate documents
  page: 7
- id: c_q3_mixed_sepctx_bf
  left: ev_q3_edic_bf
  right: ev_q3_sepctx_bf
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: training documents that mix beneficial and harmful instructions, against with beneficial and harmful instructions in separate documents
  page: 7
- id: c_q3_mixed_sepctx_dju
  left: ev_q3_edic_dju
  right: ev_q3_sepctx_dju
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: training documents that mix beneficial and harmful instructions, against with beneficial and harmful instructions in separate documents
  page: 7
- id: c_q3_mixed_sepctx_djasr
  left: ev_q3_edic_djasr
  right: ev_q3_sepctx_djasr
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: training documents that mix beneficial and harmful instructions, against with beneficial and harmful instructions in separate documents
  page: 7
- id: c_q3_secalign_ni
  left: ev_q3_secalign_ni
  right: ev_q3_none_ni
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q3_reasalign_ni
  left: ev_q3_reasalign_ni
  right: ev_q3_none_ni
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q3_edic_ni
  left: ev_q3_edic_ni
  right: ev_q3_none_ni
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l31_secalign_ni
  left: ev_l31_secalign_ni
  right: ev_l31_none_ni
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l31_reasalign_ni
  left: ev_l31_reasalign_ni
  right: ev_l31_none_ni
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l31_edic_ni
  left: ev_l31_edic_ni
  right: ev_l31_none_ni
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
claims:
- id: k1
  text: EDIC improves both safety and utility over the strongest prior defence, with up to 7.7 points more harmful rejection and 12.5 points more Arena-Hard-Auto on Qwen3-4B
  location: abstract
  anchor: EDIC improves both safety and utility over the strongest prior defense
  jobs:
  - R1
  - general_utility
  supported_by:
  - c_q3_edic_reasalign_arena
  - c_q3_edic_reasalign_mtb
  - c_q3_edic_reasalign_djasr
  - c_q3_edic_reasalign_cyb
  - c_q3_edic_reasalign_inj
  - c_q3_edic_reasalign_hr
  - c_l31_edic_reasalign_arena
  - c_l31_edic_reasalign_mtb
  - c_l31_edic_reasalign_djasr
  - c_l31_edic_reasalign_cyb
  - c_l31_edic_reasalign_inj
  - c_l31_edic_reasalign_hr
  page: 1
  contradicted_by:
  - r_l31_edic_hr
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: direct
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 1 (AgentDojo, CyberSecEval 2 and InjecAgent ASR; harmful-reject rate on the authors' data as a proxy)
    R2attack: Table 1 (AgentDojo utility under attack)
    general_utility: Table 1 (Arena-Hard-Auto, MT-Bench)
    R2clean: AgentDojo clean utility not reported
schema_gaps:
- where: defences secalign_*, struq_*, reasalign_*, metasec_l31
  note: the paper does not say whether these baselines were re-trained on Qwen3-4B and Llama-3.1-8B or which checkpoints were used; Appendix A also lists PromptGuard and Instruction Hierarchy, which do not appear in any table
- where: evaluations ev_*_hr, ev_*_bf
  note: harmful-reject and benign-follow rates are on the authors' own held-out documents and the scorer is not named; harmful-reject is coded as an R1 proxy
- where: evaluations ev_*_inj
  note: InjecAgent was run in base and enhanced settings but Table 1 gives one number without saying how they were combined
- where: claims k1
  note: on Llama-3.1-8B, EDIC's harmful-reject rate (92.8) is below ReasAlign's (93.1) and its AgentDojo ASR (0.1) equals ReasAlign's and is above Meta-SecAlign's (0.0)

`ignorednotrefused_arxiv2026` (2610.23254)

In [ ]:
%%writefile reports/records/ignorednotrefused_arxiv2026.yaml
report:
  id: ignorednotrefused_arxiv2026
  slug: '2610.23254'
  title: 'Ignored, Not Refused: What Low Prompt-Injection Success Rates Hide in Small Tool-Using Agents'
  first_author: Farhan Zarif
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2610.23254'
  version_read: arXiv v1
  first_date: '2026-10-16'
  read: F
  role: benchmark
  code_release: anonymous repository linked on p. 1 (labelled corpus, judge prompt and analysis code)
  extraction_scope: 'Section 4.2: InjecAgent ASR per agent at FP16 for the baseline prompt (Table 3) and the Interdict and source-boundary guardrail prompts (Figure 2 labels); the abstract claim on system prompts (the behavioural
    audit, precision runs, the Authorize prompt, AgentDojo, ATOM and Kimi K3 runs measure no defence and are not coded)'
defences:
- id: inter_qwen9
  name: Interdict prompt (tool-output instructions are anomalies to flag and report, never execute)
  origin: this_report
  base_checkpoint: Qwen3.5-9B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: InjecAgent ReAct agent; the injection sits in a tool output (Observation); the defence block is appended to the system prompt
  layout_compat: &id001
    user_turn: untested
    separate_role: untested
    system_context: supported
    agent_tool_result: supported
  anchor: They instruct agents that tool-output directives are either threats to report
- id: guard_qwen9
  name: source-boundary guardrail prompt (tool outputs are data; original task only; verify each tool call)
  origin: this_report
  base_checkpoint: Qwen3.5-9B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: InjecAgent ReAct agent; the injection sits in a tool output (Observation); the defence block is appended to the system prompt
  layout_compat: *id001
  anchor: Source-boundary guardrail (InjecAgent, 9
- id: inter_gemma4
  name: Interdict prompt (tool-output instructions are anomalies to flag and report, never execute)
  origin: this_report
  base_checkpoint: Gemma-4-E4B-it
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: InjecAgent ReAct agent; the injection sits in a tool output (Observation); the defence block is appended to the system prompt
  layout_compat: *id001
  anchor: They instruct agents that tool-output directives are either threats to report
- id: guard_gemma4
  name: source-boundary guardrail prompt (tool outputs are data; original task only; verify each tool call)
  origin: this_report
  base_checkpoint: Gemma-4-E4B-it
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: InjecAgent ReAct agent; the injection sits in a tool output (Observation); the defence block is appended to the system prompt
  layout_compat: *id001
  anchor: Source-boundary guardrail (InjecAgent, 9
- id: inter_spark4
  name: Interdict prompt (tool-output instructions are anomalies to flag and report, never execute)
  origin: this_report
  base_checkpoint: Spark-X2.5-4B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: InjecAgent ReAct agent; the injection sits in a tool output (Observation); the defence block is appended to the system prompt
  layout_compat: *id001
  anchor: They instruct agents that tool-output directives are either threats to report
- id: inter_ornith9
  name: Interdict prompt (tool-output instructions are anomalies to flag and report, never execute)
  origin: this_report
  base_checkpoint: Ornith-1.5-9B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: InjecAgent ReAct agent; the injection sits in a tool output (Observation); the defence block is appended to the system prompt
  layout_compat: *id001
  anchor: They instruct agents that tool-output directives are either threats to report
- id: guard_ornith9
  name: source-boundary guardrail prompt (tool outputs are data; original task only; verify each tool call)
  origin: this_report
  base_checkpoint: Ornith-1.5-9B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: InjecAgent ReAct agent; the injection sits in a tool output (Observation); the defence block is appended to the system prompt
  layout_compat: *id001
  anchor: Source-boundary guardrail (InjecAgent, 9
- id: inter_qwen4
  name: Interdict prompt (tool-output instructions are anomalies to flag and report, never execute)
  origin: this_report
  base_checkpoint: Qwen3.5-4B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: InjecAgent ReAct agent; the injection sits in a tool output (Observation); the defence block is appended to the system prompt
  layout_compat: *id001
  anchor: They instruct agents that tool-output directives are either threats to report
- id: guard_qwen4
  name: source-boundary guardrail prompt (tool outputs are data; original task only; verify each tool call)
  origin: this_report
  base_checkpoint: Qwen3.5-4B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: InjecAgent ReAct agent; the injection sits in a tool output (Observation); the defence block is appended to the system prompt
  layout_compat: *id001
  anchor: Source-boundary guardrail (InjecAgent, 9
- id: inter_gemma12
  name: Interdict prompt (tool-output instructions are anomalies to flag and report, never execute)
  origin: this_report
  base_checkpoint: Gemma-4-12B-it
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: InjecAgent ReAct agent; the injection sits in a tool output (Observation); the defence block is appended to the system prompt
  layout_compat: *id001
  anchor: They instruct agents that tool-output directives are either threats to report
- id: guard_gemma12
  name: source-boundary guardrail prompt (tool outputs are data; original task only; verify each tool call)
  origin: this_report
  base_checkpoint: Gemma-4-12B-it
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: InjecAgent ReAct agent; the injection sits in a tool output (Observation); the defence block is appended to the system prompt
  layout_compat: *id001
  anchor: Source-boundary guardrail (InjecAgent, 9
- id: inter_nemo4
  name: Interdict prompt (tool-output instructions are anomalies to flag and report, never execute)
  origin: this_report
  base_checkpoint: Nemotron-3-Nano-4B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: InjecAgent ReAct agent; the injection sits in a tool output (Observation); the defence block is appended to the system prompt
  layout_compat: *id001
  anchor: They instruct agents that tool-output directives are either threats to report
- id: guard_nemo4
  name: source-boundary guardrail prompt (tool outputs are data; original task only; verify each tool call)
  origin: this_report
  base_checkpoint: Nemotron-3-Nano-4B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: InjecAgent ReAct agent; the injection sits in a tool output (Observation); the defence block is appended to the system prompt
  layout_compat: *id001
  anchor: Source-boundary guardrail (InjecAgent, 9
- id: inter_nanb3
  name: Interdict prompt (tool-output instructions are anomalies to flag and report, never execute)
  origin: this_report
  base_checkpoint: Nanbeige-4.2-3B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: InjecAgent ReAct agent; the injection sits in a tool output (Observation); the defence block is appended to the system prompt
  layout_compat: *id001
  anchor: They instruct agents that tool-output directives are either threats to report
- id: guard_nanb3
  name: source-boundary guardrail prompt (tool outputs are data; original task only; verify each tool call)
  origin: this_report
  base_checkpoint: Nanbeige-4.2-3B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: InjecAgent ReAct agent; the injection sits in a tool output (Observation); the defence block is appended to the system prompt
  layout_compat: *id001
  anchor: Source-boundary guardrail (InjecAgent, 9
- id: inter_minicpm2
  name: Interdict prompt (tool-output instructions are anomalies to flag and report, never execute)
  origin: this_report
  base_checkpoint: MiniCPM5-2B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: InjecAgent ReAct agent; the injection sits in a tool output (Observation); the defence block is appended to the system prompt
  layout_compat: *id001
  anchor: They instruct agents that tool-output directives are either threats to report
- id: inter_minis14
  name: Interdict prompt (tool-output instructions are anomalies to flag and report, never execute)
  origin: this_report
  base_checkpoint: Ministral-3-14B-Reasoning
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: InjecAgent ReAct agent; the injection sits in a tool output (Observation); the defence block is appended to the system prompt
  layout_compat: *id001
  anchor: They instruct agents that tool-output directives are either threats to report
- id: guard_minis14
  name: source-boundary guardrail prompt (tool outputs are data; original task only; verify each tool call)
  origin: this_report
  base_checkpoint: Ministral-3-14B-Reasoning
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: InjecAgent ReAct agent; the injection sits in a tool output (Observation); the defence block is appended to the system prompt
  layout_compat: *id001
  anchor: Source-boundary guardrail (InjecAgent, 9
- id: inter_lfm26
  name: Interdict prompt (tool-output instructions are anomalies to flag and report, never execute)
  origin: this_report
  base_checkpoint: LFM-2.5-2.6B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: InjecAgent ReAct agent; the injection sits in a tool output (Observation); the defence block is appended to the system prompt
  layout_compat: *id001
  anchor: They instruct agents that tool-output directives are either threats to report
- id: guard_lfm26
  name: source-boundary guardrail prompt (tool outputs are data; original task only; verify each tool call)
  origin: this_report
  base_checkpoint: LFM-2.5-2.6B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: InjecAgent ReAct agent; the injection sits in a tool output (Observation); the defence block is appended to the system prompt
  layout_compat: *id001
  anchor: Source-boundary guardrail (InjecAgent, 9
evaluations:
- id: ev_qwen9_base
  defence_id: none:Qwen3.5-9B
  benchmark: InjecAgent base setting (510 Direct Harm and 544 Data Stealing cases), official ReAct prompt and parser
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent official output parser; invalid outputs count as failed attacks
  layout_same_as_defence: 'yes'
  measured_by: ignorednotrefused_arxiv2026
  page: 6
  decoding: greedy (temperature 0, top-p 1)
  sample_size: 1,054
  pages:
    decoding: 14
    benchmark: 12
  baseline_kind: off_the_shelf
- id: ev_qwen9_inter
  defence_id: inter_qwen9
  benchmark: InjecAgent base setting (510 Direct Harm and 544 Data Stealing cases), official ReAct prompt and parser
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent official output parser; invalid outputs count as failed attacks
  layout_same_as_defence: 'yes'
  measured_by: ignorednotrefused_arxiv2026
  page: 6
  decoding: greedy (temperature 0, top-p 1)
  sample_size: cases run under both Interdict and Authorize (10,698 over 11 agents)
  pages:
    decoding: 14
    benchmark: 12
- id: ev_qwen9_guard
  defence_id: guard_qwen9
  benchmark: InjecAgent base setting (510 Direct Harm and 544 Data Stealing cases), official ReAct prompt and parser
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent official output parser; invalid outputs count as failed attacks
  layout_same_as_defence: 'yes'
  measured_by: ignorednotrefused_arxiv2026
  page: 6
  decoding: greedy (temperature 0, top-p 1)
  sample_size: 9,458 cases over nine agents
  pages:
    decoding: 14
    benchmark: 12
- id: ev_gemma4_base
  defence_id: none:Gemma-4-E4B-it
  benchmark: InjecAgent base setting (510 Direct Harm and 544 Data Stealing cases), official ReAct prompt and parser
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent official output parser; invalid outputs count as failed attacks
  layout_same_as_defence: 'yes'
  measured_by: ignorednotrefused_arxiv2026
  page: 6
  decoding: greedy (temperature 0, top-p 1)
  sample_size: 1,054
  pages:
    decoding: 14
    benchmark: 12
  baseline_kind: off_the_shelf
- id: ev_gemma4_inter
  defence_id: inter_gemma4
  benchmark: InjecAgent base setting (510 Direct Harm and 544 Data Stealing cases), official ReAct prompt and parser
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent official output parser; invalid outputs count as failed attacks
  layout_same_as_defence: 'yes'
  measured_by: ignorednotrefused_arxiv2026
  page: 6
  decoding: greedy (temperature 0, top-p 1)
  sample_size: cases run under both Interdict and Authorize (10,698 over 11 agents)
  pages:
    decoding: 14
    benchmark: 12
- id: ev_gemma4_guard
  defence_id: guard_gemma4
  benchmark: InjecAgent base setting (510 Direct Harm and 544 Data Stealing cases), official ReAct prompt and parser
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent official output parser; invalid outputs count as failed attacks
  layout_same_as_defence: 'yes'
  measured_by: ignorednotrefused_arxiv2026
  page: 6
  decoding: greedy (temperature 0, top-p 1)
  sample_size: 9,458 cases over nine agents
  pages:
    decoding: 14
    benchmark: 12
- id: ev_spark4_base
  defence_id: none:Spark-X2.5-4B
  benchmark: InjecAgent base setting (510 Direct Harm and 544 Data Stealing cases), official ReAct prompt and parser
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent official output parser; invalid outputs count as failed attacks
  layout_same_as_defence: 'yes'
  measured_by: ignorednotrefused_arxiv2026
  page: 6
  decoding: greedy (temperature 0, top-p 1)
  sample_size: 1,054
  pages:
    decoding: 14
    benchmark: 12
  baseline_kind: off_the_shelf
- id: ev_spark4_inter
  defence_id: inter_spark4
  benchmark: InjecAgent base setting (510 Direct Harm and 544 Data Stealing cases), official ReAct prompt and parser
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent official output parser; invalid outputs count as failed attacks
  layout_same_as_defence: 'yes'
  measured_by: ignorednotrefused_arxiv2026
  page: 6
  decoding: greedy (temperature 0, top-p 1)
  sample_size: cases run under both Interdict and Authorize (10,698 over 11 agents)
  pages:
    decoding: 14
    benchmark: 12
- id: ev_ornith9_base
  defence_id: none:Ornith-1.5-9B
  benchmark: InjecAgent base setting (510 Direct Harm and 544 Data Stealing cases), official ReAct prompt and parser
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent official output parser; invalid outputs count as failed attacks
  layout_same_as_defence: 'yes'
  measured_by: ignorednotrefused_arxiv2026
  page: 6
  decoding: greedy (temperature 0, top-p 1)
  sample_size: 1,054
  pages:
    decoding: 14
    benchmark: 12
  baseline_kind: off_the_shelf
- id: ev_ornith9_inter
  defence_id: inter_ornith9
  benchmark: InjecAgent base setting (510 Direct Harm and 544 Data Stealing cases), official ReAct prompt and parser
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent official output parser; invalid outputs count as failed attacks
  layout_same_as_defence: 'yes'
  measured_by: ignorednotrefused_arxiv2026
  page: 6
  decoding: greedy (temperature 0, top-p 1)
  sample_size: cases run under both Interdict and Authorize (10,698 over 11 agents)
  pages:
    decoding: 14
    benchmark: 12
- id: ev_ornith9_guard
  defence_id: guard_ornith9
  benchmark: InjecAgent base setting (510 Direct Harm and 544 Data Stealing cases), official ReAct prompt and parser
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent official output parser; invalid outputs count as failed attacks
  layout_same_as_defence: 'yes'
  measured_by: ignorednotrefused_arxiv2026
  page: 6
  decoding: greedy (temperature 0, top-p 1)
  sample_size: 9,458 cases over nine agents
  pages:
    decoding: 14
    benchmark: 12
- id: ev_qwen4_base
  defence_id: none:Qwen3.5-4B
  benchmark: InjecAgent base setting (510 Direct Harm and 544 Data Stealing cases), official ReAct prompt and parser
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent official output parser; invalid outputs count as failed attacks
  layout_same_as_defence: 'yes'
  measured_by: ignorednotrefused_arxiv2026
  page: 6
  decoding: greedy (temperature 0, top-p 1)
  sample_size: 1,054
  pages:
    decoding: 14
    benchmark: 12
  baseline_kind: off_the_shelf
- id: ev_qwen4_inter
  defence_id: inter_qwen4
  benchmark: InjecAgent base setting (510 Direct Harm and 544 Data Stealing cases), official ReAct prompt and parser
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent official output parser; invalid outputs count as failed attacks
  layout_same_as_defence: 'yes'
  measured_by: ignorednotrefused_arxiv2026
  page: 6
  decoding: greedy (temperature 0, top-p 1)
  sample_size: cases run under both Interdict and Authorize (10,698 over 11 agents)
  pages:
    decoding: 14
    benchmark: 12
- id: ev_qwen4_guard
  defence_id: guard_qwen4
  benchmark: InjecAgent base setting (510 Direct Harm and 544 Data Stealing cases), official ReAct prompt and parser
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent official output parser; invalid outputs count as failed attacks
  layout_same_as_defence: 'yes'
  measured_by: ignorednotrefused_arxiv2026
  page: 6
  decoding: greedy (temperature 0, top-p 1)
  sample_size: 9,458 cases over nine agents
  pages:
    decoding: 14
    benchmark: 12
- id: ev_gemma12_base
  defence_id: none:Gemma-4-12B-it
  benchmark: InjecAgent base setting (510 Direct Harm and 544 Data Stealing cases), official ReAct prompt and parser
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent official output parser; invalid outputs count as failed attacks
  layout_same_as_defence: 'yes'
  measured_by: ignorednotrefused_arxiv2026
  page: 6
  decoding: greedy (temperature 0, top-p 1)
  sample_size: 1,054
  pages:
    decoding: 14
    benchmark: 12
  baseline_kind: off_the_shelf
- id: ev_gemma12_inter
  defence_id: inter_gemma12
  benchmark: InjecAgent base setting (510 Direct Harm and 544 Data Stealing cases), official ReAct prompt and parser
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent official output parser; invalid outputs count as failed attacks
  layout_same_as_defence: 'yes'
  measured_by: ignorednotrefused_arxiv2026
  page: 6
  decoding: greedy (temperature 0, top-p 1)
  sample_size: cases run under both Interdict and Authorize (10,698 over 11 agents)
  pages:
    decoding: 14
    benchmark: 12
- id: ev_gemma12_guard
  defence_id: guard_gemma12
  benchmark: InjecAgent base setting (510 Direct Harm and 544 Data Stealing cases), official ReAct prompt and parser
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent official output parser; invalid outputs count as failed attacks
  layout_same_as_defence: 'yes'
  measured_by: ignorednotrefused_arxiv2026
  page: 6
  decoding: greedy (temperature 0, top-p 1)
  sample_size: 9,458 cases over nine agents
  pages:
    decoding: 14
    benchmark: 12
- id: ev_nemo4_base
  defence_id: none:Nemotron-3-Nano-4B
  benchmark: InjecAgent base setting (510 Direct Harm and 544 Data Stealing cases), official ReAct prompt and parser
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent official output parser; invalid outputs count as failed attacks
  layout_same_as_defence: 'yes'
  measured_by: ignorednotrefused_arxiv2026
  page: 6
  decoding: greedy (temperature 0, top-p 1)
  sample_size: 1,054
  pages:
    decoding: 14
    benchmark: 12
  baseline_kind: off_the_shelf
- id: ev_nemo4_inter
  defence_id: inter_nemo4
  benchmark: InjecAgent base setting (510 Direct Harm and 544 Data Stealing cases), official ReAct prompt and parser
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent official output parser; invalid outputs count as failed attacks
  layout_same_as_defence: 'yes'
  measured_by: ignorednotrefused_arxiv2026
  page: 6
  decoding: greedy (temperature 0, top-p 1)
  sample_size: cases run under both Interdict and Authorize (10,698 over 11 agents)
  pages:
    decoding: 14
    benchmark: 12
- id: ev_nemo4_guard
  defence_id: guard_nemo4
  benchmark: InjecAgent base setting (510 Direct Harm and 544 Data Stealing cases), official ReAct prompt and parser
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent official output parser; invalid outputs count as failed attacks
  layout_same_as_defence: 'yes'
  measured_by: ignorednotrefused_arxiv2026
  page: 6
  decoding: greedy (temperature 0, top-p 1)
  sample_size: 9,458 cases over nine agents
  pages:
    decoding: 14
    benchmark: 12
- id: ev_nanb3_base
  defence_id: none:Nanbeige-4.2-3B
  benchmark: InjecAgent base setting (510 Direct Harm and 544 Data Stealing cases), official ReAct prompt and parser
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent official output parser; invalid outputs count as failed attacks
  layout_same_as_defence: 'yes'
  measured_by: ignorednotrefused_arxiv2026
  page: 6
  decoding: greedy (temperature 0, top-p 1)
  sample_size: 1,054
  pages:
    decoding: 14
    benchmark: 12
  baseline_kind: off_the_shelf
- id: ev_nanb3_inter
  defence_id: inter_nanb3
  benchmark: InjecAgent base setting (510 Direct Harm and 544 Data Stealing cases), official ReAct prompt and parser
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent official output parser; invalid outputs count as failed attacks
  layout_same_as_defence: 'yes'
  measured_by: ignorednotrefused_arxiv2026
  page: 6
  decoding: greedy (temperature 0, top-p 1)
  sample_size: cases run under both Interdict and Authorize (10,698 over 11 agents)
  pages:
    decoding: 14
    benchmark: 12
- id: ev_nanb3_guard
  defence_id: guard_nanb3
  benchmark: InjecAgent base setting (510 Direct Harm and 544 Data Stealing cases), official ReAct prompt and parser
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent official output parser; invalid outputs count as failed attacks
  layout_same_as_defence: 'yes'
  measured_by: ignorednotrefused_arxiv2026
  page: 6
  decoding: greedy (temperature 0, top-p 1)
  sample_size: 9,458 cases over nine agents
  pages:
    decoding: 14
    benchmark: 12
- id: ev_minicpm2_base
  defence_id: none:MiniCPM5-2B
  benchmark: InjecAgent base setting (510 Direct Harm and 544 Data Stealing cases), official ReAct prompt and parser
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent official output parser; invalid outputs count as failed attacks
  layout_same_as_defence: 'yes'
  measured_by: ignorednotrefused_arxiv2026
  page: 6
  decoding: greedy (temperature 0, top-p 1)
  sample_size: 1,054
  pages:
    decoding: 14
    benchmark: 12
  baseline_kind: off_the_shelf
- id: ev_minicpm2_inter
  defence_id: inter_minicpm2
  benchmark: InjecAgent base setting (510 Direct Harm and 544 Data Stealing cases), official ReAct prompt and parser
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent official output parser; invalid outputs count as failed attacks
  layout_same_as_defence: 'yes'
  measured_by: ignorednotrefused_arxiv2026
  page: 6
  decoding: greedy (temperature 0, top-p 1)
  sample_size: cases run under both Interdict and Authorize (10,698 over 11 agents)
  pages:
    decoding: 14
    benchmark: 12
- id: ev_minis14_base
  defence_id: none:Ministral-3-14B-Reasoning
  benchmark: InjecAgent base setting (510 Direct Harm and 544 Data Stealing cases), official ReAct prompt and parser
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent official output parser; invalid outputs count as failed attacks
  layout_same_as_defence: 'yes'
  measured_by: ignorednotrefused_arxiv2026
  page: 6
  decoding: greedy (temperature 0, top-p 1)
  sample_size: 1,054
  pages:
    decoding: 14
    benchmark: 12
  baseline_kind: off_the_shelf
- id: ev_minis14_inter
  defence_id: inter_minis14
  benchmark: InjecAgent base setting (510 Direct Harm and 544 Data Stealing cases), official ReAct prompt and parser
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent official output parser; invalid outputs count as failed attacks
  layout_same_as_defence: 'yes'
  measured_by: ignorednotrefused_arxiv2026
  page: 6
  decoding: greedy (temperature 0, top-p 1)
  sample_size: cases run under both Interdict and Authorize (10,698 over 11 agents)
  pages:
    decoding: 14
    benchmark: 12
- id: ev_minis14_guard
  defence_id: guard_minis14
  benchmark: InjecAgent base setting (510 Direct Harm and 544 Data Stealing cases), official ReAct prompt and parser
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent official output parser; invalid outputs count as failed attacks
  layout_same_as_defence: 'yes'
  measured_by: ignorednotrefused_arxiv2026
  page: 6
  decoding: greedy (temperature 0, top-p 1)
  sample_size: 9,458 cases over nine agents
  pages:
    decoding: 14
    benchmark: 12
- id: ev_lfm26_base
  defence_id: none:LFM-2.5-2.6B
  benchmark: InjecAgent base setting (510 Direct Harm and 544 Data Stealing cases), official ReAct prompt and parser
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent official output parser; invalid outputs count as failed attacks
  layout_same_as_defence: 'yes'
  measured_by: ignorednotrefused_arxiv2026
  page: 6
  decoding: greedy (temperature 0, top-p 1)
  sample_size: 1,054
  pages:
    decoding: 14
    benchmark: 12
  baseline_kind: off_the_shelf
- id: ev_lfm26_inter
  defence_id: inter_lfm26
  benchmark: InjecAgent base setting (510 Direct Harm and 544 Data Stealing cases), official ReAct prompt and parser
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent official output parser; invalid outputs count as failed attacks
  layout_same_as_defence: 'yes'
  measured_by: ignorednotrefused_arxiv2026
  page: 6
  decoding: greedy (temperature 0, top-p 1)
  sample_size: cases run under both Interdict and Authorize (10,698 over 11 agents)
  pages:
    decoding: 14
    benchmark: 12
- id: ev_lfm26_guard
  defence_id: guard_lfm26
  benchmark: InjecAgent base setting (510 Direct Harm and 544 Data Stealing cases), official ReAct prompt and parser
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent official output parser; invalid outputs count as failed attacks
  layout_same_as_defence: 'yes'
  measured_by: ignorednotrefused_arxiv2026
  page: 6
  decoding: greedy (temperature 0, top-p 1)
  sample_size: 9,458 cases over nine agents
  pages:
    decoding: 14
    benchmark: 12
results:
- id: r_qwen9_base
  evaluation_id: ev_qwen9_base
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, baseline prompt, FP16
  value: '25.14'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 3
- id: r_qwen9_inter
  evaluation_id: ev_qwen9_inter
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, Interdict prompt, FP16
  value: '1.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 6
  location: Figure 2
- id: r_qwen9_guard
  evaluation_id: ev_qwen9_guard
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, guardrail prompt, FP16
  value: '0.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 6
  location: Figure 2
- id: r_gemma4_base
  evaluation_id: ev_gemma4_base
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, baseline prompt, FP16
  value: '7.02'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 3
- id: r_gemma4_inter
  evaluation_id: ev_gemma4_inter
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, Interdict prompt, FP16
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 6
  location: Figure 2
- id: r_gemma4_guard
  evaluation_id: ev_gemma4_guard
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, guardrail prompt, FP16
  value: '1.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 6
  location: Figure 2
- id: r_spark4_base
  evaluation_id: ev_spark4_base
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, baseline prompt, FP16
  value: '0.95'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 3
- id: r_spark4_inter
  evaluation_id: ev_spark4_inter
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, Interdict prompt, FP16
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 6
  location: Figure 2
- id: r_ornith9_base
  evaluation_id: ev_ornith9_base
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, baseline prompt, FP16
  value: '27.42'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 3
- id: r_ornith9_inter
  evaluation_id: ev_ornith9_inter
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, Interdict prompt, FP16
  value: '0.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 6
  location: Figure 2
- id: r_ornith9_guard
  evaluation_id: ev_ornith9_guard
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, guardrail prompt, FP16
  value: '0.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 6
  location: Figure 2
- id: r_qwen4_base
  evaluation_id: ev_qwen4_base
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, baseline prompt, FP16
  value: '18.79'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 3
- id: r_qwen4_inter
  evaluation_id: ev_qwen4_inter
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, Interdict prompt, FP16
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 6
  location: Figure 2
- id: r_qwen4_guard
  evaluation_id: ev_qwen4_guard
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, guardrail prompt, FP16
  value: '0.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 6
  location: Figure 2
- id: r_gemma12_base
  evaluation_id: ev_gemma12_base
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, baseline prompt, FP16
  value: '3.51'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 3
- id: r_gemma12_inter
  evaluation_id: ev_gemma12_inter
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, Interdict prompt, FP16
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 6
  location: Figure 2
- id: r_gemma12_guard
  evaluation_id: ev_gemma12_guard
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, guardrail prompt, FP16
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 6
  location: Figure 2
- id: r_nemo4_base
  evaluation_id: ev_nemo4_base
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, baseline prompt, FP16
  value: '2.56'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 3
- id: r_nemo4_inter
  evaluation_id: ev_nemo4_inter
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, Interdict prompt, FP16
  value: '0.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 6
  location: Figure 2
- id: r_nemo4_guard
  evaluation_id: ev_nemo4_guard
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, guardrail prompt, FP16
  value: '0.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 6
  location: Figure 2
- id: r_nanb3_base
  evaluation_id: ev_nanb3_base
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, baseline prompt, FP16
  value: '2.47'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 3
- id: r_nanb3_inter
  evaluation_id: ev_nanb3_inter
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, Interdict prompt, FP16
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 6
  location: Figure 2
- id: r_nanb3_guard
  evaluation_id: ev_nanb3_guard
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, guardrail prompt, FP16
  value: '0.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 6
  location: Figure 2
- id: r_minicpm2_base
  evaluation_id: ev_minicpm2_base
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, baseline prompt, FP16
  value: '0.85'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 3
- id: r_minicpm2_inter
  evaluation_id: ev_minicpm2_inter
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, Interdict prompt, FP16
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 6
  location: Figure 2
- id: r_minis14_base
  evaluation_id: ev_minis14_base
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, baseline prompt, FP16
  value: '0.28'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 3
- id: r_minis14_inter
  evaluation_id: ev_minis14_inter
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, Interdict prompt, FP16
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 6
  location: Figure 2
- id: r_minis14_guard
  evaluation_id: ev_minis14_guard
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, guardrail prompt, FP16
  value: '4.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 6
  location: Figure 2
- id: r_lfm26_base
  evaluation_id: ev_lfm26_base
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, baseline prompt, FP16
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 3
- id: r_lfm26_inter
  evaluation_id: ev_lfm26_inter
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, Interdict prompt, FP16
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 6
  location: Figure 2
- id: r_lfm26_guard
  evaluation_id: ev_lfm26_guard
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, guardrail prompt, FP16
  value: '0.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 6
  location: Figure 2
comparisons:
- id: c_qwen9_inter
  left: ev_qwen9_inter
  right: ev_qwen9_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: unmatched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: Interdict block added to the system prompt'
  page: 6
- id: c_qwen9_guard
  left: ev_qwen9_guard
  right: ev_qwen9_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: unmatched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: guardrail block added to the system prompt'
  page: 6
- id: c_gemma4_inter
  left: ev_gemma4_inter
  right: ev_gemma4_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: unmatched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: Interdict block added to the system prompt'
  page: 6
- id: c_gemma4_guard
  left: ev_gemma4_guard
  right: ev_gemma4_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: unmatched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: guardrail block added to the system prompt'
  page: 6
- id: c_spark4_inter
  left: ev_spark4_inter
  right: ev_spark4_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: unmatched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: Interdict block added to the system prompt'
  page: 6
- id: c_ornith9_inter
  left: ev_ornith9_inter
  right: ev_ornith9_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: unmatched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: Interdict block added to the system prompt'
  page: 6
- id: c_ornith9_guard
  left: ev_ornith9_guard
  right: ev_ornith9_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: unmatched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: guardrail block added to the system prompt'
  page: 6
- id: c_qwen4_inter
  left: ev_qwen4_inter
  right: ev_qwen4_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: unmatched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: Interdict block added to the system prompt'
  page: 6
- id: c_qwen4_guard
  left: ev_qwen4_guard
  right: ev_qwen4_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: unmatched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: guardrail block added to the system prompt'
  page: 6
- id: c_gemma12_inter
  left: ev_gemma12_inter
  right: ev_gemma12_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: unmatched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: Interdict block added to the system prompt'
  page: 6
- id: c_gemma12_guard
  left: ev_gemma12_guard
  right: ev_gemma12_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: unmatched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: guardrail block added to the system prompt'
  page: 6
- id: c_nemo4_inter
  left: ev_nemo4_inter
  right: ev_nemo4_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: unmatched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: Interdict block added to the system prompt'
  page: 6
- id: c_nemo4_guard
  left: ev_nemo4_guard
  right: ev_nemo4_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: unmatched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: guardrail block added to the system prompt'
  page: 6
- id: c_nanb3_inter
  left: ev_nanb3_inter
  right: ev_nanb3_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: unmatched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: Interdict block added to the system prompt'
  page: 6
- id: c_nanb3_guard
  left: ev_nanb3_guard
  right: ev_nanb3_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: unmatched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: guardrail block added to the system prompt'
  page: 6
- id: c_minicpm2_inter
  left: ev_minicpm2_inter
  right: ev_minicpm2_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: unmatched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: Interdict block added to the system prompt'
  page: 6
- id: c_minis14_inter
  left: ev_minis14_inter
  right: ev_minis14_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: unmatched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: Interdict block added to the system prompt'
  page: 6
- id: c_minis14_guard
  left: ev_minis14_guard
  right: ev_minis14_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: unmatched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: guardrail block added to the system prompt'
  page: 6
- id: c_lfm26_inter
  left: ev_lfm26_inter
  right: ev_lfm26_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: unmatched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: Interdict block added to the system prompt'
  page: 6
- id: c_lfm26_guard
  left: ev_lfm26_guard
  right: ev_lfm26_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: unmatched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: guardrail block added to the system prompt'
  page: 6
claims:
- id: k1
  text: changing only the system prompt shifts InjecAgent ASR from 0.2% to 43% with the same weights
  location: abstract
  anchor: changing only the system prompt shifts InjecAgent ASR from 0.2% to 43%
  jobs:
  - R1
  supported_by:
  - c_qwen9_inter
  - c_gemma4_inter
  - c_spark4_inter
  - c_ornith9_inter
  - c_qwen4_inter
  - c_gemma12_inter
  - c_nemo4_inter
  - c_nanb3_inter
  - c_minicpm2_inter
  - c_minis14_inter
  - c_lfm26_inter
  page: 1
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 3 and Figure 2 (InjecAgent)
    R2clean: InjecAgent has no utility measure; AgentDojo runs test no defence
    R2attack: not reported for the defence prompts
schema_gaps:
- where: results r_*_inter, r_*_guard
  note: per-agent Interdict and guardrail ASR are text-extractable heat-map labels in Figure 2 at one decimal; the pooled values (0.22% over eleven agents, 0.95% over nine) are in the text only
- where: comparisons c_*_inter, c_*_guard
  note: the defence prompts were run on subsets (10,698 and 9,458 cases) while the Table 3 baseline uses all 1,054 cases per agent; the text compares paired cases but per-agent paired baselines are not given, so c3 is unmatched
- where: claims k1
  note: the 43% end of the claim is the Authorize prompt, which tells the agent to carry out tool-output instructions; it is not a defence and is not coded; LFM-2.6B's 0.00% baseline comes with 735 invalid outputs out of 1,054
    (Table 3)
- where: report
  note: 'an evaluation study: the defences are two prompt blocks written by the authors; the headline finding is about what failed attacks consist of (76 to 80% ignored, 17 to 18% refused), which the schema has no field for'

`hos_arxiv2026` (2610.37657)

In [ ]:
%%writefile reports/records/hos_arxiv2026.yaml
report:
  id: hos_arxiv2026
  slug: '2610.37657'
  title: 'Heads of Security: Learned Attention Gates Improve Prompt Injection Robustness'
  first_author: Igor Shilov
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2610.37657'
  version_read: arXiv v2
  first_date: '2026-10-27'
  read: F
  role: proposes
  code_release: imperial-aisp/heads_of_security, demo and released gates (p. 1)
  extraction_scope: 'Sections 4 and 5 and Appendices F, G and J: Table 1 (static, own-attacker and highest RL-Hammer rows for the undefended, SecAlign and HoS models), Table 2, Table 3, Table 13 (stable learning rates), Table
    4 and the white-box counts stated in Appendix F.2; the abstract claims (the GPT-5.6-Luna column, Figure 2 compute curves, Figure 3, Table 12, Table 14 and Figures 5 to 9 not coded)'
defences:
- id: hos_g
  name: Heads of Security (learning rate 0.1, the headline GPT-OSS model)
  origin: this_report
  base_checkpoint: GPT-OSS-20B
  axis1:
  - chat_roles
  - internal_intervention
  axis2:
  - preference_optimisation
  - distillation
  isolation: ablation_present
  signal_dependence: stated
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  components: &id002
  - 'each token gets a channel: trusted (system, user, the model''s own output), untrusted (tool outputs) or template (chat-template tokens)'
  - a learned bias per head and per query-key channel pair is added to the attention logits (log sigmoid of a gate logit); base weights frozen
  - trained with the SecAlign++ DPO recipe on Cleaned-Alpaca injection pairs
  layout: native tool calling; every tool-output token assigned to the untrusted channel
  layout_compat: &id001
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: HoS adds a learned, per-channel bias to each head
  pages:
    components: 4
    isolation: 9
    training: 5
- id: hos_g_001
  name: Heads of Security (learning rate 0.01)
  origin: this_report
  base_checkpoint: GPT-OSS-20B
  axis1:
  - chat_roles
  - internal_intervention
  axis2:
  - preference_optimisation
  - distillation
  isolation: not_applicable
  signal_dependence: stated
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 31
  layout: native tool calling; every tool-output token assigned to the untrusted channel
  layout_compat: *id001
  anchor: separately trained SecAlign and HoS models at each learning rate
- id: hos_g_0025
  name: Heads of Security (learning rate 0.025)
  origin: this_report
  base_checkpoint: GPT-OSS-20B
  axis1:
  - chat_roles
  - internal_intervention
  axis2:
  - preference_optimisation
  - distillation
  isolation: not_applicable
  signal_dependence: stated
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 31
  layout: native tool calling; every tool-output token assigned to the untrusted channel
  layout_compat: *id001
  anchor: separately trained SecAlign and HoS models at each learning rate
- id: hos_g_005
  name: Heads of Security (learning rate 0.05)
  origin: this_report
  base_checkpoint: GPT-OSS-20B
  axis1:
  - chat_roles
  - internal_intervention
  axis2:
  - preference_optimisation
  - distillation
  isolation: not_applicable
  signal_dependence: stated
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 31
  layout: native tool calling; every tool-output token assigned to the untrusted channel
  layout_compat: *id001
  anchor: separately trained SecAlign and HoS models at each learning rate
- id: hos_l
  name: Heads of Security (learning rate 0.008)
  origin: this_report
  base_checkpoint: Llama-3.3-70B-Instruct
  axis1:
  - chat_roles
  - internal_intervention
  axis2:
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: stated
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  components: *id002
  layout: native tool calling; every tool-output token assigned to the untrusted channel
  layout_compat: *id001
  anchor: HoS adds a learned, per-channel bias to each head
  pages:
    components: 4
    isolation: 9
    training: 5
- id: red_g
  name: HoS with only the gates that shift attention away from untrusted keys (red heads)
  origin: this_report
  base_checkpoint: GPT-OSS-20B
  axis1:
  - chat_roles
  - internal_intervention
  axis2:
  - preference_optimisation
  - distillation
  isolation: not_applicable
  signal_dependence: stated
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 9
  layout: native tool calling; every tool-output token assigned to the untrusted channel
  anchor: we evaluate “Red gates only” and “Blue gates only” variants
- id: blue_g
  name: HoS with only the gates that shift attention towards untrusted keys (blue heads)
  origin: this_report
  base_checkpoint: GPT-OSS-20B
  axis1:
  - chat_roles
  - internal_intervention
  axis2:
  - preference_optimisation
  - distillation
  isolation: not_applicable
  signal_dependence: stated
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 9
  layout: native tool calling; every tool-output token assigned to the untrusted channel
  anchor: we evaluate “Red gates only” and “Blue gates only” variants
- id: sec_g
  name: SecAlign++ LoRA trained by the authors (learning rate used in Tables 1 and 2 not stated)
  origin: other_report
  base_checkpoint: GPT-OSS-20B
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  - distillation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 34
  layout: tool outputs escaped and remapped to the input role (SecAlign protocol)
  anchor: We also train a SecAlign LoRA adapter for GPT-OSS-20B
- id: sec_g_5e5
  name: SecAlign++ LoRA trained by the authors (learning rate 5e-5)
  origin: other_report
  base_checkpoint: GPT-OSS-20B
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  - distillation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 31
  layout: tool outputs escaped and remapped to the input role (SecAlign protocol)
  anchor: separately trained SecAlign and HoS models at each learning rate
- id: sec_g_1e4
  name: SecAlign++ LoRA trained by the authors (learning rate 1e-4)
  origin: other_report
  base_checkpoint: GPT-OSS-20B
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  - distillation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 31
  layout: tool outputs escaped and remapped to the input role (SecAlign protocol)
  anchor: separately trained SecAlign and HoS models at each learning rate
- id: sec_g_2e4
  name: SecAlign++ LoRA trained by the authors (learning rate 2e-4)
  origin: other_report
  base_checkpoint: GPT-OSS-20B
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  - distillation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 31
  layout: tool outputs escaped and remapped to the input role (SecAlign protocol)
  anchor: separately trained SecAlign and HoS models at each learning rate
- id: aside_g
  name: ASIDE (full fine-tuning on benign Alpaca with GPT-OSS-120B targets, selected checkpoint)
  origin: other_report
  base_checkpoint: GPT-OSS-20B
  axis1:
  - fixed_embedding_transform
  axis2:
  - sft
  - distillation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 31
  layout: tool-output embeddings rotated by pi/2
  anchor: ASIDE uses the selected benign-Alpaca full
- id: sec_l
  name: Meta-SecAlign-70B (released adapter, unmodified)
  origin: other_report
  base_checkpoint: Llama-3.3-70B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 34
  layout: tool outputs escaped and remapped to the input role (SecAlign protocol)
  anchor: we use the released facebook/Meta-SecAlign-70B adapter without modiﬁcation
evaluations:
- id: ev_g_none_static
  defence_id: none:GPT-OSS-20B
  benchmark: InjecAgent, 100 held-out direct-harm goals (native tool protocol); static InjecAgent injection
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent success criterion
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 5
  decoding: greedy
  sample_size: '100'
  pages:
    decoding: 21
  baseline_kind: off_the_shelf
- id: ev_g_sec_static
  defence_id: sec_g
  benchmark: InjecAgent, 100 held-out direct-harm goals (native tool protocol); static InjecAgent injection
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent success criterion
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 5
  decoding: greedy
  sample_size: '100'
  pages:
    decoding: 21
- id: ev_g_hos_static
  defence_id: hos_g
  benchmark: InjecAgent, 100 held-out direct-harm goals (native tool protocol); static InjecAgent injection
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent success criterion
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 5
  decoding: greedy
  sample_size: '100'
  pages:
    decoding: 21
- id: ev_l_none_static
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: InjecAgent, 100 held-out direct-harm goals (native tool protocol); static InjecAgent injection
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent success criterion
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 5
  decoding: greedy
  sample_size: '100'
  pages:
    decoding: 21
  baseline_kind: off_the_shelf
- id: ev_l_sec_static
  defence_id: sec_l
  benchmark: InjecAgent, 100 held-out direct-harm goals (native tool protocol); static InjecAgent injection
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent success criterion
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 5
  decoding: greedy
  sample_size: '100'
  pages:
    decoding: 21
- id: ev_l_hos_static
  defence_id: hos_l
  benchmark: InjecAgent, 100 held-out direct-harm goals (native tool protocol); static InjecAgent injection
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent success criterion
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 5
  decoding: greedy
  sample_size: '100'
  pages:
    decoding: 21
- id: ev_g_none_own
  defence_id: none:GPT-OSS-20B
  benchmark: InjecAgent, 100 held-out direct-harm goals (native tool protocol); RL-Hammer injection from the attacker trained against this target
  attacker_type: rl
  attacker_access: black_box
  judge: InjecAgent success criterion
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 5
  attacker_budget: Llama-3.1-8B-Instruct attacker, LoRA and GRPO for 1,240 steps on 310 training goals, paired with an easier target
  decoding: greedy
  sample_size: '100'
  pages:
    decoding: 21
  baseline_kind: off_the_shelf
- id: ev_g_sec_own
  defence_id: sec_g
  benchmark: InjecAgent, 100 held-out direct-harm goals (native tool protocol); RL-Hammer injection from the attacker trained against this target
  attacker_type: rl
  attacker_access: black_box
  judge: InjecAgent success criterion
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 5
  attacker_budget: Llama-3.1-8B-Instruct attacker, LoRA and GRPO for 1,240 steps on 310 training goals, paired with an easier target
  decoding: greedy
  sample_size: '100'
  pages:
    decoding: 21
- id: ev_g_hos_own
  defence_id: hos_g
  benchmark: InjecAgent, 100 held-out direct-harm goals (native tool protocol); RL-Hammer injection from the attacker trained against this target
  attacker_type: rl
  attacker_access: black_box
  judge: InjecAgent success criterion
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 5
  attacker_budget: Llama-3.1-8B-Instruct attacker, LoRA and GRPO for 1,240 steps on 310 training goals, paired with an easier target
  decoding: greedy
  sample_size: '100'
  pages:
    decoding: 21
- id: ev_l_none_own
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: InjecAgent, 100 held-out direct-harm goals (native tool protocol); RL-Hammer injection from the attacker trained against this target
  attacker_type: rl
  attacker_access: black_box
  judge: InjecAgent success criterion
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 5
  attacker_budget: Llama-3.1-8B-Instruct attacker, LoRA and GRPO for 1,240 steps on 310 training goals, paired with an easier target
  decoding: greedy
  sample_size: '100'
  pages:
    decoding: 21
  baseline_kind: off_the_shelf
- id: ev_l_sec_own
  defence_id: sec_l
  benchmark: InjecAgent, 100 held-out direct-harm goals (native tool protocol); RL-Hammer injection from the attacker trained against this target
  attacker_type: rl
  attacker_access: black_box
  judge: InjecAgent success criterion
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 5
  attacker_budget: Llama-3.1-8B-Instruct attacker, LoRA and GRPO for 1,240 steps on 310 training goals, paired with an easier target
  decoding: greedy
  sample_size: '100'
  pages:
    decoding: 21
- id: ev_l_hos_own
  defence_id: hos_l
  benchmark: InjecAgent, 100 held-out direct-harm goals (native tool protocol); RL-Hammer injection from the attacker trained against this target
  attacker_type: rl
  attacker_access: black_box
  judge: InjecAgent success criterion
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 5
  attacker_budget: Llama-3.1-8B-Instruct attacker, LoRA and GRPO for 1,240 steps on 310 training goals, paired with an easier target
  decoding: greedy
  sample_size: '100'
  pages:
    decoding: 21
- id: ev_g_none_max
  defence_id: none:GPT-OSS-20B
  benchmark: InjecAgent, 100 held-out direct-harm goals (native tool protocol); highest ASR over the four RL-Hammer attackers (trained against the undefended, SecAlign, HoS and GPT-5.6-Luna targets)
  attacker_type: rl
  attacker_access: black_box
  judge: InjecAgent success criterion
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 5
  attacker_budget: four attackers of the RL-Hammer recipe; the highest of the four reported
  decoding: greedy
  sample_size: '100'
  pages:
    decoding: 21
  baseline_kind: off_the_shelf
- id: ev_g_sec_max
  defence_id: sec_g
  benchmark: InjecAgent, 100 held-out direct-harm goals (native tool protocol); highest ASR over the four RL-Hammer attackers (trained against the undefended, SecAlign, HoS and GPT-5.6-Luna targets)
  attacker_type: rl
  attacker_access: black_box
  judge: InjecAgent success criterion
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 5
  attacker_budget: four attackers of the RL-Hammer recipe; the highest of the four reported
  decoding: greedy
  sample_size: '100'
  pages:
    decoding: 21
- id: ev_g_hos_max
  defence_id: hos_g
  benchmark: InjecAgent, 100 held-out direct-harm goals (native tool protocol); highest ASR over the four RL-Hammer attackers (trained against the undefended, SecAlign, HoS and GPT-5.6-Luna targets)
  attacker_type: rl
  attacker_access: black_box
  judge: InjecAgent success criterion
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 5
  attacker_budget: four attackers of the RL-Hammer recipe; the highest of the four reported
  decoding: greedy
  sample_size: '100'
  pages:
    decoding: 21
- id: ev_l_none_max
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: InjecAgent, 100 held-out direct-harm goals (native tool protocol); highest ASR over the four RL-Hammer attackers (trained against the undefended, SecAlign, HoS and GPT-5.6-Luna targets)
  attacker_type: rl
  attacker_access: black_box
  judge: InjecAgent success criterion
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 5
  attacker_budget: four attackers of the RL-Hammer recipe; the highest of the four reported
  decoding: greedy
  sample_size: '100'
  pages:
    decoding: 21
  baseline_kind: off_the_shelf
- id: ev_l_sec_max
  defence_id: sec_l
  benchmark: InjecAgent, 100 held-out direct-harm goals (native tool protocol); highest ASR over the four RL-Hammer attackers (trained against the undefended, SecAlign, HoS and GPT-5.6-Luna targets)
  attacker_type: rl
  attacker_access: black_box
  judge: InjecAgent success criterion
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 5
  attacker_budget: four attackers of the RL-Hammer recipe; the highest of the four reported
  decoding: greedy
  sample_size: '100'
  pages:
    decoding: 21
- id: ev_l_hos_max
  defence_id: hos_l
  benchmark: InjecAgent, 100 held-out direct-harm goals (native tool protocol); highest ASR over the four RL-Hammer attackers (trained against the undefended, SecAlign, HoS and GPT-5.6-Luna targets)
  attacker_type: rl
  attacker_access: black_box
  judge: InjecAgent success criterion
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 5
  attacker_budget: four attackers of the RL-Hammer recipe; the highest of the four reported
  decoding: greedy
  sample_size: '100'
  pages:
    decoding: 21
- id: ev_g_none_arc0
  defence_id: none:GPT-OSS-20B
  benchmark: ARC-C 0-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_g_none_gsm0
  defence_id: none:GPT-OSS-20B
  benchmark: GSM8K 0-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_g_none_gsm1
  defence_id: none:GPT-OSS-20B
  benchmark: GSM8K 1-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_g_none_gsm8
  defence_id: none:GPT-OSS-20B
  benchmark: GSM8K 8-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_g_none_mmlu0
  defence_id: none:GPT-OSS-20B
  benchmark: MMLU 0-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_g_none_mmlu5
  defence_id: none:GPT-OSS-20B
  benchmark: MMLU 5-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_g_none_pro0
  defence_id: none:GPT-OSS-20B
  benchmark: MMLU-Pro 0-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_g_none_pro5
  defence_id: none:GPT-OSS-20B
  benchmark: MMLU-Pro 5-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_g_sec_arc0
  defence_id: sec_g
  benchmark: ARC-C 0-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_g_sec_gsm0
  defence_id: sec_g
  benchmark: GSM8K 0-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_g_sec_gsm1
  defence_id: sec_g
  benchmark: GSM8K 1-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_g_sec_gsm8
  defence_id: sec_g
  benchmark: GSM8K 8-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_g_sec_mmlu0
  defence_id: sec_g
  benchmark: MMLU 0-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_g_sec_mmlu5
  defence_id: sec_g
  benchmark: MMLU 5-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_g_sec_pro0
  defence_id: sec_g
  benchmark: MMLU-Pro 0-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_g_sec_pro5
  defence_id: sec_g
  benchmark: MMLU-Pro 5-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_g_aside_arc0
  defence_id: aside_g
  benchmark: ARC-C 0-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_g_aside_gsm0
  defence_id: aside_g
  benchmark: GSM8K 0-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_g_aside_gsm1
  defence_id: aside_g
  benchmark: GSM8K 1-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_g_aside_gsm8
  defence_id: aside_g
  benchmark: GSM8K 8-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_g_aside_mmlu0
  defence_id: aside_g
  benchmark: MMLU 0-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_g_aside_mmlu5
  defence_id: aside_g
  benchmark: MMLU 5-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_g_aside_pro0
  defence_id: aside_g
  benchmark: MMLU-Pro 0-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_g_aside_pro5
  defence_id: aside_g
  benchmark: MMLU-Pro 5-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_g_hos_arc0
  defence_id: hos_g
  benchmark: ARC-C 0-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_g_hos_gsm0
  defence_id: hos_g
  benchmark: GSM8K 0-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_g_hos_gsm1
  defence_id: hos_g
  benchmark: GSM8K 1-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_g_hos_gsm8
  defence_id: hos_g
  benchmark: GSM8K 8-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_g_hos_mmlu0
  defence_id: hos_g
  benchmark: MMLU 0-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_g_hos_mmlu5
  defence_id: hos_g
  benchmark: MMLU 5-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_g_hos_pro0
  defence_id: hos_g
  benchmark: MMLU-Pro 0-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_g_hos_pro5
  defence_id: hos_g
  benchmark: MMLU-Pro 5-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_l_none_arc0
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: ARC-C 0-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_l_none_gsm0
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: GSM8K 0-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_l_none_gsm1
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: GSM8K 1-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_l_none_gsm8
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: GSM8K 8-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_l_none_mmlu0
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: MMLU 0-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_l_none_mmlu5
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: MMLU 5-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_l_none_pro0
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: MMLU-Pro 0-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_l_none_pro5
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: MMLU-Pro 5-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_l_sec_arc0
  defence_id: sec_l
  benchmark: ARC-C 0-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_l_sec_gsm0
  defence_id: sec_l
  benchmark: GSM8K 0-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_l_sec_gsm1
  defence_id: sec_l
  benchmark: GSM8K 1-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_l_sec_gsm8
  defence_id: sec_l
  benchmark: GSM8K 8-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_l_sec_mmlu0
  defence_id: sec_l
  benchmark: MMLU 0-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_l_sec_mmlu5
  defence_id: sec_l
  benchmark: MMLU 5-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_l_sec_pro0
  defence_id: sec_l
  benchmark: MMLU-Pro 0-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_l_sec_pro5
  defence_id: sec_l
  benchmark: MMLU-Pro 5-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_l_hos_arc0
  defence_id: hos_l
  benchmark: ARC-C 0-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_l_hos_gsm0
  defence_id: hos_l
  benchmark: GSM8K 0-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_l_hos_gsm1
  defence_id: hos_l
  benchmark: GSM8K 1-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_l_hos_gsm8
  defence_id: hos_l
  benchmark: GSM8K 8-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_l_hos_mmlu0
  defence_id: hos_l
  benchmark: MMLU 0-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_l_hos_mmlu5
  defence_id: hos_l
  benchmark: MMLU 5-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_l_hos_pro0
  defence_id: hos_l
  benchmark: MMLU-Pro 0-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_l_hos_pro5
  defence_id: hos_l
  benchmark: MMLU-Pro 5-shot, standard in-message few-shot format, all content trusted
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 6
  decoding: not reported
- id: ev_l_none_ad_clean
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: none
  attacker_access: not_applicable
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 30
  baseline_kind: off_the_shelf
- id: ev_l_none_ad_att
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 30
  baseline_kind: off_the_shelf
- id: ev_l_none_ad_asr
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 30
  baseline_kind: off_the_shelf
- id: ev_l_none_dyn_clean
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: none
  attacker_access: not_applicable
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 30
  baseline_kind: off_the_shelf
- id: ev_l_none_dyn_att
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 30
  baseline_kind: off_the_shelf
- id: ev_l_none_dyn_asr
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 30
  baseline_kind: off_the_shelf
- id: ev_l_none_wasp_clean
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: none
  attacker_access: not_applicable
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 8
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
  baseline_kind: off_the_shelf
- id: ev_l_none_wasp_att
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: static
  attacker_access: black_box
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 8
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
  baseline_kind: off_the_shelf
- id: ev_l_none_wasp_asr
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: static
  attacker_access: black_box
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 8
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
  baseline_kind: off_the_shelf
- id: ev_l_none_llm_att
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: LLMail-Inject, 4,000 Phase-1 submissions (1,000 per level)
  attacker_type: human
  attacker_access: black_box
  judge: send_email to the attacker for ASR; Claude Sonnet 5 judges summary coverage for utility
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 30
  baseline_kind: off_the_shelf
- id: ev_l_none_llm_asr
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: LLMail-Inject, 4,000 Phase-1 submissions (1,000 per level)
  attacker_type: human
  attacker_access: black_box
  judge: send_email call to the attacker
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 30
  baseline_kind: off_the_shelf
- id: ev_l_sec_ad_clean
  defence_id: sec_l
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: none
  attacker_access: not_applicable
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_l_sec_ad_att
  defence_id: sec_l
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_l_sec_ad_asr
  defence_id: sec_l
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_l_sec_dyn_clean
  defence_id: sec_l
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: none
  attacker_access: not_applicable
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_l_sec_dyn_att
  defence_id: sec_l
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_l_sec_dyn_asr
  defence_id: sec_l
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_l_sec_wasp_clean
  defence_id: sec_l
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: none
  attacker_access: not_applicable
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 8
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
- id: ev_l_sec_wasp_att
  defence_id: sec_l
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: static
  attacker_access: black_box
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 8
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
- id: ev_l_sec_wasp_asr
  defence_id: sec_l
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: static
  attacker_access: black_box
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 8
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
- id: ev_l_sec_llm_att
  defence_id: sec_l
  benchmark: LLMail-Inject, 4,000 Phase-1 submissions (1,000 per level)
  attacker_type: human
  attacker_access: black_box
  judge: send_email to the attacker for ASR; Claude Sonnet 5 judges summary coverage for utility
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_l_sec_llm_asr
  defence_id: sec_l
  benchmark: LLMail-Inject, 4,000 Phase-1 submissions (1,000 per level)
  attacker_type: human
  attacker_access: black_box
  judge: send_email call to the attacker
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_l_hos_ad_clean
  defence_id: hos_l
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: none
  attacker_access: not_applicable
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_l_hos_ad_att
  defence_id: hos_l
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_l_hos_ad_asr
  defence_id: hos_l
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_l_hos_dyn_clean
  defence_id: hos_l
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: none
  attacker_access: not_applicable
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_l_hos_dyn_att
  defence_id: hos_l
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_l_hos_dyn_asr
  defence_id: hos_l
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_l_hos_wasp_clean
  defence_id: hos_l
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: none
  attacker_access: not_applicable
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 8
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
- id: ev_l_hos_wasp_att
  defence_id: hos_l
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: static
  attacker_access: black_box
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 8
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
- id: ev_l_hos_wasp_asr
  defence_id: hos_l
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: static
  attacker_access: black_box
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 8
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
- id: ev_l_hos_llm_att
  defence_id: hos_l
  benchmark: LLMail-Inject, 4,000 Phase-1 submissions (1,000 per level)
  attacker_type: human
  attacker_access: black_box
  judge: send_email to the attacker for ASR; Claude Sonnet 5 judges summary coverage for utility
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_l_hos_llm_asr
  defence_id: hos_l
  benchmark: LLMail-Inject, 4,000 Phase-1 submissions (1,000 per level)
  attacker_type: human
  attacker_access: black_box
  judge: send_email call to the attacker
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_none_ad_clean
  defence_id: none:GPT-OSS-20B
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: none
  attacker_access: not_applicable
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
  baseline_kind: off_the_shelf
- id: ev_g_none_ad_att
  defence_id: none:GPT-OSS-20B
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
  baseline_kind: off_the_shelf
- id: ev_g_none_ad_asr
  defence_id: none:GPT-OSS-20B
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
  baseline_kind: off_the_shelf
- id: ev_g_none_dyn_clean
  defence_id: none:GPT-OSS-20B
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: none
  attacker_access: not_applicable
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
  baseline_kind: off_the_shelf
- id: ev_g_none_dyn_att
  defence_id: none:GPT-OSS-20B
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
  baseline_kind: off_the_shelf
- id: ev_g_none_dyn_asr
  defence_id: none:GPT-OSS-20B
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
  baseline_kind: off_the_shelf
- id: ev_g_none_wasp_clean
  defence_id: none:GPT-OSS-20B
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: none
  attacker_access: not_applicable
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
  baseline_kind: off_the_shelf
- id: ev_g_none_wasp_att
  defence_id: none:GPT-OSS-20B
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: static
  attacker_access: black_box
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
  baseline_kind: off_the_shelf
- id: ev_g_none_wasp_asr
  defence_id: none:GPT-OSS-20B
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: static
  attacker_access: black_box
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
  baseline_kind: off_the_shelf
- id: ev_g_none_llm_att
  defence_id: none:GPT-OSS-20B
  benchmark: LLMail-Inject, 4,000 Phase-1 submissions (1,000 per level)
  attacker_type: human
  attacker_access: black_box
  judge: send_email to the attacker for ASR; Claude Sonnet 5 judges summary coverage for utility
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
  baseline_kind: off_the_shelf
- id: ev_g_none_llm_asr
  defence_id: none:GPT-OSS-20B
  benchmark: LLMail-Inject, 4,000 Phase-1 submissions (1,000 per level)
  attacker_type: human
  attacker_access: black_box
  judge: send_email call to the attacker
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
  baseline_kind: off_the_shelf
- id: ev_g_sec_5e5_ad_clean
  defence_id: sec_g_5e5
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: none
  attacker_access: not_applicable
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_sec_5e5_ad_att
  defence_id: sec_g_5e5
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_sec_5e5_ad_asr
  defence_id: sec_g_5e5
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_sec_5e5_dyn_clean
  defence_id: sec_g_5e5
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: none
  attacker_access: not_applicable
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_sec_5e5_dyn_att
  defence_id: sec_g_5e5
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_sec_5e5_dyn_asr
  defence_id: sec_g_5e5
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_sec_5e5_wasp_clean
  defence_id: sec_g_5e5
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: none
  attacker_access: not_applicable
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
- id: ev_g_sec_5e5_wasp_att
  defence_id: sec_g_5e5
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: static
  attacker_access: black_box
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
- id: ev_g_sec_5e5_wasp_asr
  defence_id: sec_g_5e5
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: static
  attacker_access: black_box
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
- id: ev_g_sec_5e5_llm_att
  defence_id: sec_g_5e5
  benchmark: LLMail-Inject, 4,000 Phase-1 submissions (1,000 per level)
  attacker_type: human
  attacker_access: black_box
  judge: send_email to the attacker for ASR; Claude Sonnet 5 judges summary coverage for utility
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_sec_5e5_llm_asr
  defence_id: sec_g_5e5
  benchmark: LLMail-Inject, 4,000 Phase-1 submissions (1,000 per level)
  attacker_type: human
  attacker_access: black_box
  judge: send_email call to the attacker
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_sec_1e4_ad_clean
  defence_id: sec_g_1e4
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: none
  attacker_access: not_applicable
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_sec_1e4_ad_att
  defence_id: sec_g_1e4
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_sec_1e4_ad_asr
  defence_id: sec_g_1e4
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_sec_1e4_dyn_clean
  defence_id: sec_g_1e4
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: none
  attacker_access: not_applicable
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_sec_1e4_dyn_att
  defence_id: sec_g_1e4
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_sec_1e4_dyn_asr
  defence_id: sec_g_1e4
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_sec_1e4_wasp_clean
  defence_id: sec_g_1e4
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: none
  attacker_access: not_applicable
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
- id: ev_g_sec_1e4_wasp_att
  defence_id: sec_g_1e4
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: static
  attacker_access: black_box
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
- id: ev_g_sec_1e4_wasp_asr
  defence_id: sec_g_1e4
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: static
  attacker_access: black_box
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
- id: ev_g_sec_1e4_llm_att
  defence_id: sec_g_1e4
  benchmark: LLMail-Inject, 4,000 Phase-1 submissions (1,000 per level)
  attacker_type: human
  attacker_access: black_box
  judge: send_email to the attacker for ASR; Claude Sonnet 5 judges summary coverage for utility
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_sec_1e4_llm_asr
  defence_id: sec_g_1e4
  benchmark: LLMail-Inject, 4,000 Phase-1 submissions (1,000 per level)
  attacker_type: human
  attacker_access: black_box
  judge: send_email call to the attacker
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_sec_2e4_ad_clean
  defence_id: sec_g_2e4
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: none
  attacker_access: not_applicable
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_sec_2e4_ad_att
  defence_id: sec_g_2e4
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_sec_2e4_ad_asr
  defence_id: sec_g_2e4
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_sec_2e4_dyn_clean
  defence_id: sec_g_2e4
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: none
  attacker_access: not_applicable
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_sec_2e4_dyn_att
  defence_id: sec_g_2e4
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_sec_2e4_dyn_asr
  defence_id: sec_g_2e4
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_sec_2e4_wasp_clean
  defence_id: sec_g_2e4
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: none
  attacker_access: not_applicable
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
- id: ev_g_sec_2e4_wasp_att
  defence_id: sec_g_2e4
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: static
  attacker_access: black_box
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
- id: ev_g_sec_2e4_wasp_asr
  defence_id: sec_g_2e4
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: static
  attacker_access: black_box
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
- id: ev_g_sec_2e4_llm_att
  defence_id: sec_g_2e4
  benchmark: LLMail-Inject, 4,000 Phase-1 submissions (1,000 per level)
  attacker_type: human
  attacker_access: black_box
  judge: send_email to the attacker for ASR; Claude Sonnet 5 judges summary coverage for utility
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_sec_2e4_llm_asr
  defence_id: sec_g_2e4
  benchmark: LLMail-Inject, 4,000 Phase-1 submissions (1,000 per level)
  attacker_type: human
  attacker_access: black_box
  judge: send_email call to the attacker
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_aside_ad_clean
  defence_id: aside_g
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: none
  attacker_access: not_applicable
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_aside_ad_att
  defence_id: aside_g
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_aside_ad_asr
  defence_id: aside_g
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_aside_dyn_clean
  defence_id: aside_g
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: none
  attacker_access: not_applicable
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_aside_dyn_att
  defence_id: aside_g
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_aside_dyn_asr
  defence_id: aside_g
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_aside_wasp_clean
  defence_id: aside_g
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: none
  attacker_access: not_applicable
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
- id: ev_g_aside_wasp_att
  defence_id: aside_g
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: static
  attacker_access: black_box
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
- id: ev_g_aside_wasp_asr
  defence_id: aside_g
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: static
  attacker_access: black_box
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
- id: ev_g_aside_llm_att
  defence_id: aside_g
  benchmark: LLMail-Inject, 4,000 Phase-1 submissions (1,000 per level)
  attacker_type: human
  attacker_access: black_box
  judge: send_email to the attacker for ASR; Claude Sonnet 5 judges summary coverage for utility
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_aside_llm_asr
  defence_id: aside_g
  benchmark: LLMail-Inject, 4,000 Phase-1 submissions (1,000 per level)
  attacker_type: human
  attacker_access: black_box
  judge: send_email call to the attacker
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_hos_001_ad_clean
  defence_id: hos_g_001
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: none
  attacker_access: not_applicable
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_hos_001_ad_att
  defence_id: hos_g_001
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_hos_001_ad_asr
  defence_id: hos_g_001
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_hos_001_dyn_clean
  defence_id: hos_g_001
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: none
  attacker_access: not_applicable
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_hos_001_dyn_att
  defence_id: hos_g_001
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_hos_001_dyn_asr
  defence_id: hos_g_001
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_hos_001_wasp_clean
  defence_id: hos_g_001
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: none
  attacker_access: not_applicable
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
- id: ev_g_hos_001_wasp_att
  defence_id: hos_g_001
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: static
  attacker_access: black_box
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
- id: ev_g_hos_001_wasp_asr
  defence_id: hos_g_001
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: static
  attacker_access: black_box
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
- id: ev_g_hos_001_llm_att
  defence_id: hos_g_001
  benchmark: LLMail-Inject, 4,000 Phase-1 submissions (1,000 per level)
  attacker_type: human
  attacker_access: black_box
  judge: send_email to the attacker for ASR; Claude Sonnet 5 judges summary coverage for utility
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_hos_001_llm_asr
  defence_id: hos_g_001
  benchmark: LLMail-Inject, 4,000 Phase-1 submissions (1,000 per level)
  attacker_type: human
  attacker_access: black_box
  judge: send_email call to the attacker
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_hos_0025_ad_clean
  defence_id: hos_g_0025
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: none
  attacker_access: not_applicable
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_hos_0025_ad_att
  defence_id: hos_g_0025
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_hos_0025_ad_asr
  defence_id: hos_g_0025
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_hos_0025_dyn_clean
  defence_id: hos_g_0025
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: none
  attacker_access: not_applicable
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_hos_0025_dyn_att
  defence_id: hos_g_0025
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_hos_0025_dyn_asr
  defence_id: hos_g_0025
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_hos_0025_wasp_clean
  defence_id: hos_g_0025
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: none
  attacker_access: not_applicable
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
- id: ev_g_hos_0025_wasp_att
  defence_id: hos_g_0025
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: static
  attacker_access: black_box
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
- id: ev_g_hos_0025_wasp_asr
  defence_id: hos_g_0025
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: static
  attacker_access: black_box
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
- id: ev_g_hos_0025_llm_att
  defence_id: hos_g_0025
  benchmark: LLMail-Inject, 4,000 Phase-1 submissions (1,000 per level)
  attacker_type: human
  attacker_access: black_box
  judge: send_email to the attacker for ASR; Claude Sonnet 5 judges summary coverage for utility
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_hos_0025_llm_asr
  defence_id: hos_g_0025
  benchmark: LLMail-Inject, 4,000 Phase-1 submissions (1,000 per level)
  attacker_type: human
  attacker_access: black_box
  judge: send_email call to the attacker
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_hos_005_ad_clean
  defence_id: hos_g_005
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: none
  attacker_access: not_applicable
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_hos_005_ad_att
  defence_id: hos_g_005
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_hos_005_ad_asr
  defence_id: hos_g_005
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_hos_005_dyn_clean
  defence_id: hos_g_005
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: none
  attacker_access: not_applicable
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_hos_005_dyn_att
  defence_id: hos_g_005
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_hos_005_dyn_asr
  defence_id: hos_g_005
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_hos_005_wasp_clean
  defence_id: hos_g_005
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: none
  attacker_access: not_applicable
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
- id: ev_g_hos_005_wasp_att
  defence_id: hos_g_005
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: static
  attacker_access: black_box
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
- id: ev_g_hos_005_wasp_asr
  defence_id: hos_g_005
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: static
  attacker_access: black_box
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
- id: ev_g_hos_005_llm_att
  defence_id: hos_g_005
  benchmark: LLMail-Inject, 4,000 Phase-1 submissions (1,000 per level)
  attacker_type: human
  attacker_access: black_box
  judge: send_email to the attacker for ASR; Claude Sonnet 5 judges summary coverage for utility
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_hos_005_llm_asr
  defence_id: hos_g_005
  benchmark: LLMail-Inject, 4,000 Phase-1 submissions (1,000 per level)
  attacker_type: human
  attacker_access: black_box
  judge: send_email call to the attacker
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_hos_ad_clean
  defence_id: hos_g
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: none
  attacker_access: not_applicable
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_hos_ad_att
  defence_id: hos_g
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_hos_ad_asr
  defence_id: hos_g
  benchmark: AgentDojo, important_instructions (Workspace, Slack, Banking, Travel)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_hos_dyn_clean
  defence_id: hos_g
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: none
  attacker_access: not_applicable
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_hos_dyn_att
  defence_id: hos_g
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_hos_dyn_asr
  defence_id: hos_g
  benchmark: AgentDyn, important_instructions (Shopping, GitHub, Daily Life)
  attacker_type: static
  attacker_access: black_box
  judge: deterministic environment-state check
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_hos_wasp_clean
  defence_id: hos_g
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: none
  attacker_access: not_applicable
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
- id: ev_g_hos_wasp_att
  defence_id: hos_g
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: static
  attacker_access: black_box
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
- id: ev_g_hos_wasp_asr
  defence_id: hos_g
  benchmark: WASP (37 clean tasks; 84 injection combinations, accessibility trees)
  attacker_type: static
  attacker_access: black_box
  judge: WASP task and injection checks
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: sampled, temperature 1, top-p 0.9, mean of five runs
  pages:
    benchmark: 30
- id: ev_g_hos_llm_att
  defence_id: hos_g
  benchmark: LLMail-Inject, 4,000 Phase-1 submissions (1,000 per level)
  attacker_type: human
  attacker_access: black_box
  judge: send_email to the attacker for ASR; Claude Sonnet 5 judges summary coverage for utility
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_hos_llm_asr
  defence_id: hos_g
  benchmark: LLMail-Inject, 4,000 Phase-1 submissions (1,000 per level)
  attacker_type: human
  attacker_access: black_box
  judge: send_email call to the attacker
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 31
  decoding: not reported
  pages:
    benchmark: 30
- id: ev_g_base_flag
  defence_id: none:GPT-OSS-20B
  benchmark: LLMail-Inject subset with a request to flag injected instructions (benchmark of Abdelnabi and Bagdasarian)
  attacker_type: human
  attacker_access: black_box
  judge: send_email to the attacker (send); injection flagged in the reply (detection)
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 9
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_g_red_flag
  defence_id: red_g
  benchmark: LLMail-Inject subset with a request to flag injected instructions (benchmark of Abdelnabi and Bagdasarian)
  attacker_type: human
  attacker_access: black_box
  judge: send_email to the attacker (send); injection flagged in the reply (detection)
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 9
  decoding: not reported
- id: ev_g_blue_flag
  defence_id: blue_g
  benchmark: LLMail-Inject subset with a request to flag injected instructions (benchmark of Abdelnabi and Bagdasarian)
  attacker_type: human
  attacker_access: black_box
  judge: send_email to the attacker (send); injection flagged in the reply (detection)
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 9
  decoding: not reported
- id: ev_g_hos_flag
  defence_id: hos_g
  benchmark: LLMail-Inject subset with a request to flag injected instructions (benchmark of Abdelnabi and Bagdasarian)
  attacker_type: human
  attacker_access: black_box
  judge: send_email to the attacker (send); injection flagged in the reply (detection)
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 9
  decoding: not reported
- id: ev_l_none_v38ad
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: AgentDojo, 20 user-task and injection-task pairs where the fixed attack fails
  attacker_type: optimisation
  attacker_access: white_box
  judge: benchmark success criterion; an example counts if any suffix within the budget succeeds
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 29
  attacker_budget: 120-token suffix, up to 5x10^18 FLOPs per example
  denominator: 20 examples
  decoding: greedy
  baseline_kind: off_the_shelf
- id: ev_l_sec_v38ad
  defence_id: sec_l
  benchmark: AgentDojo, 20 user-task and injection-task pairs where the fixed attack fails
  attacker_type: optimisation
  attacker_access: white_box
  judge: benchmark success criterion; an example counts if any suffix within the budget succeeds
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 29
  attacker_budget: 120-token suffix, up to 5x10^18 FLOPs per example
  denominator: 20 examples
  decoding: greedy
- id: ev_l_hos_v38ad
  defence_id: hos_l
  benchmark: AgentDojo, 20 user-task and injection-task pairs where the fixed attack fails
  attacker_type: optimisation
  attacker_access: white_box
  judge: benchmark success criterion; an example counts if any suffix within the budget succeeds
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 29
  attacker_budget: 120-token suffix, up to 5x10^18 FLOPs per example
  denominator: 20 examples
  decoding: greedy
- id: ev_g_none_c45ad
  defence_id: none:GPT-OSS-20B
  benchmark: AgentDojo, 100 user-task and injection-task pairs where the fixed attack fails
  attacker_type: optimisation
  attacker_access: white_box
  judge: benchmark success criterion; an example counts if any suffix within the budget succeeds
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 30
  attacker_budget: 120-token suffix, up to 10^17 FLOPs per example
  decoding: greedy
  baseline_kind: off_the_shelf
- id: ev_g_sec_c45ad
  defence_id: sec_g
  benchmark: AgentDojo, 100 user-task and injection-task pairs where the fixed attack fails
  attacker_type: optimisation
  attacker_access: white_box
  judge: benchmark success criterion; an example counts if any suffix within the budget succeeds
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 30
  attacker_budget: 120-token suffix, up to 10^17 FLOPs per example
  decoding: greedy
- id: ev_g_hos_c45ad
  defence_id: hos_g
  benchmark: AgentDojo, 100 user-task and injection-task pairs where the fixed attack fails
  attacker_type: optimisation
  attacker_access: white_box
  judge: benchmark success criterion; an example counts if any suffix within the budget succeeds
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 30
  attacker_budget: 120-token suffix, up to 10^17 FLOPs per example
  decoding: greedy
- id: ev_g_none_c45ia
  defence_id: none:GPT-OSS-20B
  benchmark: InjecAgent, 100 held-out goals
  attacker_type: optimisation
  attacker_access: white_box
  judge: benchmark success criterion; an example counts if any suffix within the budget succeeds
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 30
  attacker_budget: 120-token suffix, up to 10^17 FLOPs per example
  decoding: greedy
  baseline_kind: off_the_shelf
- id: ev_g_hos_c45ia
  defence_id: hos_g
  benchmark: InjecAgent, 100 held-out goals
  attacker_type: optimisation
  attacker_access: white_box
  judge: benchmark success criterion; an example counts if any suffix within the budget succeeds
  layout_same_as_defence: 'yes'
  measured_by: hos_arxiv2026
  page: 30
  attacker_budget: 120-token suffix, up to 10^17 FLOPs per example
  decoding: greedy
results:
- id: r_g_none_static
  evaluation_id: ev_g_none_static
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, static InjecAgent injection
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_g_sec_static
  evaluation_id: ev_g_sec_static
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, static InjecAgent injection
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_g_hos_static
  evaluation_id: ev_g_hos_static
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, static InjecAgent injection
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_l_none_static
  evaluation_id: ev_l_none_static
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, static InjecAgent injection
  value: '16'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_l_sec_static
  evaluation_id: ev_l_sec_static
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, static InjecAgent injection
  value: '5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_l_hos_static
  evaluation_id: ev_l_hos_static
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, static InjecAgent injection
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_g_none_own
  evaluation_id: ev_g_none_own
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, RL-Hammer injection from the attacker trained against this target
  value: '95'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_g_sec_own
  evaluation_id: ev_g_sec_own
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, RL-Hammer injection from the attacker trained against this target
  value: '85'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_g_hos_own
  evaluation_id: ev_g_hos_own
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, RL-Hammer injection from the attacker trained against this target
  value: '3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_l_none_own
  evaluation_id: ev_l_none_own
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, RL-Hammer injection from the attacker trained against this target
  value: '99'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_l_sec_own
  evaluation_id: ev_l_sec_own
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, RL-Hammer injection from the attacker trained against this target
  value: '82'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_l_hos_own
  evaluation_id: ev_l_hos_own
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, RL-Hammer injection from the attacker trained against this target
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_g_none_max
  evaluation_id: ev_g_none_max
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, highest ASR over the four RL-Hammer attackers (trained against the undefended, SecAlign, HoS and GPT-5.6-Luna targets)
  value: '97'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_g_sec_max
  evaluation_id: ev_g_sec_max
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, highest ASR over the four RL-Hammer attackers (trained against the undefended, SecAlign, HoS and GPT-5.6-Luna targets)
  value: '85'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_g_hos_max
  evaluation_id: ev_g_hos_max
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, highest ASR over the four RL-Hammer attackers (trained against the undefended, SecAlign, HoS and GPT-5.6-Luna targets)
  value: '10'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_l_none_max
  evaluation_id: ev_l_none_max
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, highest ASR over the four RL-Hammer attackers (trained against the undefended, SecAlign, HoS and GPT-5.6-Luna targets)
  value: '99'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_l_sec_max
  evaluation_id: ev_l_sec_max
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, highest ASR over the four RL-Hammer attackers (trained against the undefended, SecAlign, HoS and GPT-5.6-Luna targets)
  value: '82'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_l_hos_max
  evaluation_id: ev_l_hos_max
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent ASR, highest ASR over the four RL-Hammer attackers (trained against the undefended, SecAlign, HoS and GPT-5.6-Luna targets)
  value: '1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_g_none_arc0
  evaluation_id: ev_g_none_arc0
  quantity: general_utility
  coverage: direct
  metric_as_printed: ARC-C 0-shot accuracy
  value: '96.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_g_none_gsm0
  evaluation_id: ev_g_none_gsm0
  quantity: general_utility
  coverage: direct
  metric_as_printed: GSM8K 0-shot accuracy
  value: '91.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_g_none_gsm1
  evaluation_id: ev_g_none_gsm1
  quantity: general_utility
  coverage: direct
  metric_as_printed: GSM8K 1-shot accuracy
  value: '93.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_g_none_gsm8
  evaluation_id: ev_g_none_gsm8
  quantity: general_utility
  coverage: direct
  metric_as_printed: GSM8K 8-shot accuracy
  value: '95.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_g_none_mmlu0
  evaluation_id: ev_g_none_mmlu0
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU 0-shot accuracy
  value: '83.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_g_none_mmlu5
  evaluation_id: ev_g_none_mmlu5
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU 5-shot accuracy
  value: '82.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_g_none_pro0
  evaluation_id: ev_g_none_pro0
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU-Pro 0-shot accuracy
  value: '72.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_g_none_pro5
  evaluation_id: ev_g_none_pro5
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU-Pro 5-shot accuracy
  value: '68.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_g_sec_arc0
  evaluation_id: ev_g_sec_arc0
  quantity: general_utility
  coverage: direct
  metric_as_printed: ARC-C 0-shot accuracy
  value: '94.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_g_sec_gsm0
  evaluation_id: ev_g_sec_gsm0
  quantity: general_utility
  coverage: direct
  metric_as_printed: GSM8K 0-shot accuracy
  value: '93.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_g_sec_gsm1
  evaluation_id: ev_g_sec_gsm1
  quantity: general_utility
  coverage: direct
  metric_as_printed: GSM8K 1-shot accuracy
  value: '70.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_g_sec_gsm8
  evaluation_id: ev_g_sec_gsm8
  quantity: general_utility
  coverage: direct
  metric_as_printed: GSM8K 8-shot accuracy
  value: '91.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_g_sec_mmlu0
  evaluation_id: ev_g_sec_mmlu0
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU 0-shot accuracy
  value: '82.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_g_sec_mmlu5
  evaluation_id: ev_g_sec_mmlu5
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU 5-shot accuracy
  value: '54.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_g_sec_pro0
  evaluation_id: ev_g_sec_pro0
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU-Pro 0-shot accuracy
  value: '71.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_g_sec_pro5
  evaluation_id: ev_g_sec_pro5
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU-Pro 5-shot accuracy
  value: '59.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_g_aside_arc0
  evaluation_id: ev_g_aside_arc0
  quantity: general_utility
  coverage: direct
  metric_as_printed: ARC-C 0-shot accuracy
  value: '94.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_g_aside_gsm0
  evaluation_id: ev_g_aside_gsm0
  quantity: general_utility
  coverage: direct
  metric_as_printed: GSM8K 0-shot accuracy
  value: '91.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_g_aside_gsm1
  evaluation_id: ev_g_aside_gsm1
  quantity: general_utility
  coverage: direct
  metric_as_printed: GSM8K 1-shot accuracy
  value: '93.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_g_aside_gsm8
  evaluation_id: ev_g_aside_gsm8
  quantity: general_utility
  coverage: direct
  metric_as_printed: GSM8K 8-shot accuracy
  value: '93.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_g_aside_mmlu0
  evaluation_id: ev_g_aside_mmlu0
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU 0-shot accuracy
  value: '83.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_g_aside_mmlu5
  evaluation_id: ev_g_aside_mmlu5
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU 5-shot accuracy
  value: '74.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_g_aside_pro0
  evaluation_id: ev_g_aside_pro0
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU-Pro 0-shot accuracy
  value: '71.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_g_aside_pro5
  evaluation_id: ev_g_aside_pro5
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU-Pro 5-shot accuracy
  value: '52.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_g_hos_arc0
  evaluation_id: ev_g_hos_arc0
  quantity: general_utility
  coverage: direct
  metric_as_printed: ARC-C 0-shot accuracy
  value: '95.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_g_hos_gsm0
  evaluation_id: ev_g_hos_gsm0
  quantity: general_utility
  coverage: direct
  metric_as_printed: GSM8K 0-shot accuracy
  value: '90.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_g_hos_gsm1
  evaluation_id: ev_g_hos_gsm1
  quantity: general_utility
  coverage: direct
  metric_as_printed: GSM8K 1-shot accuracy
  value: '92.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_g_hos_gsm8
  evaluation_id: ev_g_hos_gsm8
  quantity: general_utility
  coverage: direct
  metric_as_printed: GSM8K 8-shot accuracy
  value: '94.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_g_hos_mmlu0
  evaluation_id: ev_g_hos_mmlu0
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU 0-shot accuracy
  value: '83.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_g_hos_mmlu5
  evaluation_id: ev_g_hos_mmlu5
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU 5-shot accuracy
  value: '80.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_g_hos_pro0
  evaluation_id: ev_g_hos_pro0
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU-Pro 0-shot accuracy
  value: '72.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_g_hos_pro5
  evaluation_id: ev_g_hos_pro5
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU-Pro 5-shot accuracy
  value: '67.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l_none_arc0
  evaluation_id: ev_l_none_arc0
  quantity: general_utility
  coverage: direct
  metric_as_printed: ARC-C 0-shot accuracy
  value: '94.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l_none_gsm0
  evaluation_id: ev_l_none_gsm0
  quantity: general_utility
  coverage: direct
  metric_as_printed: GSM8K 0-shot accuracy
  value: '93.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l_none_gsm1
  evaluation_id: ev_l_none_gsm1
  quantity: general_utility
  coverage: direct
  metric_as_printed: GSM8K 1-shot accuracy
  value: '93.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l_none_gsm8
  evaluation_id: ev_l_none_gsm8
  quantity: general_utility
  coverage: direct
  metric_as_printed: GSM8K 8-shot accuracy
  value: '95.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l_none_mmlu0
  evaluation_id: ev_l_none_mmlu0
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU 0-shot accuracy
  value: '84.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l_none_mmlu5
  evaluation_id: ev_l_none_mmlu5
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU 5-shot accuracy
  value: '82.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l_none_pro0
  evaluation_id: ev_l_none_pro0
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU-Pro 0-shot accuracy
  value: '67.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l_none_pro5
  evaluation_id: ev_l_none_pro5
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU-Pro 5-shot accuracy
  value: '56.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l_sec_arc0
  evaluation_id: ev_l_sec_arc0
  quantity: general_utility
  coverage: direct
  metric_as_printed: ARC-C 0-shot accuracy
  value: '93.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l_sec_gsm0
  evaluation_id: ev_l_sec_gsm0
  quantity: general_utility
  coverage: direct
  metric_as_printed: GSM8K 0-shot accuracy
  value: '92.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l_sec_gsm1
  evaluation_id: ev_l_sec_gsm1
  quantity: general_utility
  coverage: direct
  metric_as_printed: GSM8K 1-shot accuracy
  value: '2.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l_sec_gsm8
  evaluation_id: ev_l_sec_gsm8
  quantity: general_utility
  coverage: direct
  metric_as_printed: GSM8K 8-shot accuracy
  value: '68.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l_sec_mmlu0
  evaluation_id: ev_l_sec_mmlu0
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU 0-shot accuracy
  value: '84.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l_sec_mmlu5
  evaluation_id: ev_l_sec_mmlu5
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU 5-shot accuracy
  value: '28.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l_sec_pro0
  evaluation_id: ev_l_sec_pro0
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU-Pro 0-shot accuracy
  value: '68.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l_sec_pro5
  evaluation_id: ev_l_sec_pro5
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU-Pro 5-shot accuracy
  value: '16.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l_hos_arc0
  evaluation_id: ev_l_hos_arc0
  quantity: general_utility
  coverage: direct
  metric_as_printed: ARC-C 0-shot accuracy
  value: '93.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l_hos_gsm0
  evaluation_id: ev_l_hos_gsm0
  quantity: general_utility
  coverage: direct
  metric_as_printed: GSM8K 0-shot accuracy
  value: '92.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l_hos_gsm1
  evaluation_id: ev_l_hos_gsm1
  quantity: general_utility
  coverage: direct
  metric_as_printed: GSM8K 1-shot accuracy
  value: '93.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l_hos_gsm8
  evaluation_id: ev_l_hos_gsm8
  quantity: general_utility
  coverage: direct
  metric_as_printed: GSM8K 8-shot accuracy
  value: '95.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l_hos_mmlu0
  evaluation_id: ev_l_hos_mmlu0
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU 0-shot accuracy
  value: '85.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l_hos_mmlu5
  evaluation_id: ev_l_hos_mmlu5
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU 5-shot accuracy
  value: '82.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l_hos_pro0
  evaluation_id: ev_l_hos_pro0
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU-Pro 0-shot accuracy
  value: '67.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l_hos_pro5
  evaluation_id: ev_l_hos_pro5
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU-Pro 5-shot accuracy
  value: '54.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l_none_ad_clean
  evaluation_id: ev_l_none_ad_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDojo clean utility
  value: '53.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l_none_ad_att
  evaluation_id: ev_l_none_ad_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo utility under attack
  value: '36.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l_none_ad_asr
  evaluation_id: ev_l_none_ad_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '23.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l_none_dyn_clean
  evaluation_id: ev_l_none_dyn_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDyn clean utility
  value: '33.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l_none_dyn_att
  evaluation_id: ev_l_none_dyn_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDyn utility under attack
  value: '30.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l_none_dyn_asr
  evaluation_id: ev_l_none_dyn_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDyn ASR
  value: '18.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l_none_wasp_clean
  evaluation_id: ev_l_none_wasp_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations clean utility
  value: '66.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l_none_wasp_att
  evaluation_id: ev_l_none_wasp_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations utility under attack
  value: '62.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l_none_wasp_asr
  evaluation_id: ev_l_none_wasp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations ASR
  value: '7.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l_none_llm_att
  evaluation_id: ev_l_none_llm_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: LLMail-Inject utility under attack
  value: '80.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l_none_llm_asr
  evaluation_id: ev_l_none_llm_asr
  quantity: R1
  coverage: direct
  metric_as_printed: LLMail-Inject ASR
  value: '2.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l_sec_ad_clean
  evaluation_id: ev_l_sec_ad_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDojo clean utility
  value: '67.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l_sec_ad_att
  evaluation_id: ev_l_sec_ad_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo utility under attack
  value: '63.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l_sec_ad_asr
  evaluation_id: ev_l_sec_ad_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '2.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l_sec_dyn_clean
  evaluation_id: ev_l_sec_dyn_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDyn clean utility
  value: '41.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l_sec_dyn_att
  evaluation_id: ev_l_sec_dyn_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDyn utility under attack
  value: '38.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l_sec_dyn_asr
  evaluation_id: ev_l_sec_dyn_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDyn ASR
  value: '5.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l_sec_wasp_clean
  evaluation_id: ev_l_sec_wasp_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations clean utility
  value: '62.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l_sec_wasp_att
  evaluation_id: ev_l_sec_wasp_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations utility under attack
  value: '79.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l_sec_wasp_asr
  evaluation_id: ev_l_sec_wasp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations ASR
  value: '2.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l_sec_llm_att
  evaluation_id: ev_l_sec_llm_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: LLMail-Inject utility under attack
  value: '84.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l_sec_llm_asr
  evaluation_id: ev_l_sec_llm_asr
  quantity: R1
  coverage: direct
  metric_as_printed: LLMail-Inject ASR
  value: '12.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l_hos_ad_clean
  evaluation_id: ev_l_hos_ad_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDojo clean utility
  value: '58.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l_hos_ad_att
  evaluation_id: ev_l_hos_ad_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo utility under attack
  value: '52.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l_hos_ad_asr
  evaluation_id: ev_l_hos_ad_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '0.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l_hos_dyn_clean
  evaluation_id: ev_l_hos_dyn_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDyn clean utility
  value: '18.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l_hos_dyn_att
  evaluation_id: ev_l_hos_dyn_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDyn utility under attack
  value: '18.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l_hos_dyn_asr
  evaluation_id: ev_l_hos_dyn_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDyn ASR
  value: '1.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l_hos_wasp_clean
  evaluation_id: ev_l_hos_wasp_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations clean utility
  value: '62.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l_hos_wasp_att
  evaluation_id: ev_l_hos_wasp_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations utility under attack
  value: '81.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l_hos_wasp_asr
  evaluation_id: ev_l_hos_wasp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations ASR
  value: '1.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l_hos_llm_att
  evaluation_id: ev_l_hos_llm_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: LLMail-Inject utility under attack
  value: '82.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_l_hos_llm_asr
  evaluation_id: ev_l_hos_llm_asr
  quantity: R1
  coverage: direct
  metric_as_printed: LLMail-Inject ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 3
- id: r_g_none_ad_clean
  evaluation_id: ev_g_none_ad_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDojo clean utility
  value: '66.32'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_none_ad_att
  evaluation_id: ev_g_none_ad_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo utility under attack
  value: '53.64'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_none_ad_asr
  evaluation_id: ev_g_none_ad_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '23.78'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_none_dyn_clean
  evaluation_id: ev_g_none_dyn_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDyn clean utility
  value: '58.33'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_none_dyn_att
  evaluation_id: ev_g_none_dyn_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDyn utility under attack
  value: '47.32'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_none_dyn_asr
  evaluation_id: ev_g_none_dyn_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDyn ASR
  value: '38.21'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_none_wasp_clean
  evaluation_id: ev_g_none_wasp_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations clean utility
  value: '56.22'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_none_wasp_att
  evaluation_id: ev_g_none_wasp_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations utility under attack
  value: '49.29'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_none_wasp_asr
  evaluation_id: ev_g_none_wasp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations ASR
  value: '10.24'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_none_llm_att
  evaluation_id: ev_g_none_llm_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: LLMail-Inject utility under attack
  value: '96.08'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_none_llm_asr
  evaluation_id: ev_g_none_llm_asr
  quantity: R1
  coverage: direct
  metric_as_printed: LLMail-Inject ASR
  value: '2.85'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_sec_5e5_ad_clean
  evaluation_id: ev_g_sec_5e5_ad_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDojo clean utility
  value: '60.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_sec_5e5_ad_att
  evaluation_id: ev_g_sec_5e5_ad_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo utility under attack
  value: '59.93'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_sec_5e5_ad_asr
  evaluation_id: ev_g_sec_5e5_ad_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '5.86'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_sec_5e5_dyn_clean
  evaluation_id: ev_g_sec_5e5_dyn_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDyn clean utility
  value: '60.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_sec_5e5_dyn_att
  evaluation_id: ev_g_sec_5e5_dyn_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDyn utility under attack
  value: '53.04'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_sec_5e5_dyn_asr
  evaluation_id: ev_g_sec_5e5_dyn_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDyn ASR
  value: '21.96'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_sec_5e5_wasp_clean
  evaluation_id: ev_g_sec_5e5_wasp_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations clean utility
  value: '61.08'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_sec_5e5_wasp_att
  evaluation_id: ev_g_sec_5e5_wasp_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations utility under attack
  value: '55.71'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_sec_5e5_wasp_asr
  evaluation_id: ev_g_sec_5e5_wasp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations ASR
  value: '7.38'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_sec_5e5_llm_att
  evaluation_id: ev_g_sec_5e5_llm_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: LLMail-Inject utility under attack
  value: '99.42'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_sec_5e5_llm_asr
  evaluation_id: ev_g_sec_5e5_llm_asr
  quantity: R1
  coverage: direct
  metric_as_printed: LLMail-Inject ASR
  value: '0.33'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_sec_1e4_ad_clean
  evaluation_id: ev_g_sec_1e4_ad_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDojo clean utility
  value: '60.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_sec_1e4_ad_att
  evaluation_id: ev_g_sec_1e4_ad_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo utility under attack
  value: '56.57'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_sec_1e4_ad_asr
  evaluation_id: ev_g_sec_1e4_ad_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '2.17'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_sec_1e4_dyn_clean
  evaluation_id: ev_g_sec_1e4_dyn_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDyn clean utility
  value: '55.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_sec_1e4_dyn_att
  evaluation_id: ev_g_sec_1e4_dyn_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDyn utility under attack
  value: '53.57'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_sec_1e4_dyn_asr
  evaluation_id: ev_g_sec_1e4_dyn_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDyn ASR
  value: '13.57'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_sec_1e4_wasp_clean
  evaluation_id: ev_g_sec_1e4_wasp_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations clean utility
  value: '49.73'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_sec_1e4_wasp_att
  evaluation_id: ev_g_sec_1e4_wasp_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations utility under attack
  value: '55.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_sec_1e4_wasp_asr
  evaluation_id: ev_g_sec_1e4_wasp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations ASR
  value: '5.71'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_sec_1e4_llm_att
  evaluation_id: ev_g_sec_1e4_llm_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: LLMail-Inject utility under attack
  value: '99.55'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_sec_1e4_llm_asr
  evaluation_id: ev_g_sec_1e4_llm_asr
  quantity: R1
  coverage: direct
  metric_as_printed: LLMail-Inject ASR
  value: '0.10'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_sec_2e4_ad_clean
  evaluation_id: ev_g_sec_2e4_ad_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDojo clean utility
  value: '30.53'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_sec_2e4_ad_att
  evaluation_id: ev_g_sec_2e4_ad_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo utility under attack
  value: '30.84'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_sec_2e4_ad_asr
  evaluation_id: ev_g_sec_2e4_ad_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '1.30'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_sec_2e4_dyn_clean
  evaluation_id: ev_g_sec_2e4_dyn_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDyn clean utility
  value: '51.67'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_sec_2e4_dyn_att
  evaluation_id: ev_g_sec_2e4_dyn_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDyn utility under attack
  value: '46.79'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_sec_2e4_dyn_asr
  evaluation_id: ev_g_sec_2e4_dyn_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDyn ASR
  value: '4.64'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_sec_2e4_wasp_clean
  evaluation_id: ev_g_sec_2e4_wasp_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations clean utility
  value: '27.03'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_sec_2e4_wasp_att
  evaluation_id: ev_g_sec_2e4_wasp_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations utility under attack
  value: '30.24'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_sec_2e4_wasp_asr
  evaluation_id: ev_g_sec_2e4_wasp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations ASR
  value: '0.95'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_sec_2e4_llm_att
  evaluation_id: ev_g_sec_2e4_llm_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: LLMail-Inject utility under attack
  value: '6.07'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_sec_2e4_llm_asr
  evaluation_id: ev_g_sec_2e4_llm_asr
  quantity: R1
  coverage: direct
  metric_as_printed: LLMail-Inject ASR
  value: '3.40'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_aside_ad_clean
  evaluation_id: ev_g_aside_ad_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDojo clean utility
  value: '9.47'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_aside_ad_att
  evaluation_id: ev_g_aside_ad_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo utility under attack
  value: '10.42'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_aside_ad_asr
  evaluation_id: ev_g_aside_ad_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_aside_dyn_clean
  evaluation_id: ev_g_aside_dyn_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDyn clean utility
  value: '0.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_aside_dyn_att
  evaluation_id: ev_g_aside_dyn_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDyn utility under attack
  value: '0.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_aside_dyn_asr
  evaluation_id: ev_g_aside_dyn_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDyn ASR
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_aside_wasp_clean
  evaluation_id: ev_g_aside_wasp_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations clean utility
  value: '15.68'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_aside_wasp_att
  evaluation_id: ev_g_aside_wasp_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations utility under attack
  value: '0.95'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_aside_wasp_asr
  evaluation_id: ev_g_aside_wasp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations ASR
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_aside_llm_att
  evaluation_id: ev_g_aside_llm_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: LLMail-Inject utility under attack
  value: '0.96'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_aside_llm_asr
  evaluation_id: ev_g_aside_llm_asr
  quantity: R1
  coverage: direct
  metric_as_printed: LLMail-Inject ASR
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_001_ad_clean
  evaluation_id: ev_g_hos_001_ad_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDojo clean utility
  value: '65.26'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_001_ad_att
  evaluation_id: ev_g_hos_001_ad_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo utility under attack
  value: '64.60'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_001_ad_asr
  evaluation_id: ev_g_hos_001_ad_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '3.37'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_001_dyn_clean
  evaluation_id: ev_g_hos_001_dyn_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDyn clean utility
  value: '55.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_001_dyn_att
  evaluation_id: ev_g_hos_001_dyn_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDyn utility under attack
  value: '50.18'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_001_dyn_asr
  evaluation_id: ev_g_hos_001_dyn_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDyn ASR
  value: '8.93'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_001_wasp_clean
  evaluation_id: ev_g_hos_001_wasp_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations clean utility
  value: '57.30'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_001_wasp_att
  evaluation_id: ev_g_hos_001_wasp_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations utility under attack
  value: '69.05'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_001_wasp_asr
  evaluation_id: ev_g_hos_001_wasp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations ASR
  value: '1.19'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_001_llm_att
  evaluation_id: ev_g_hos_001_llm_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: LLMail-Inject utility under attack
  value: '98.97'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_001_llm_asr
  evaluation_id: ev_g_hos_001_llm_asr
  quantity: R1
  coverage: direct
  metric_as_printed: LLMail-Inject ASR
  value: '0.05'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_0025_ad_clean
  evaluation_id: ev_g_hos_0025_ad_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDojo clean utility
  value: '65.26'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_0025_ad_att
  evaluation_id: ev_g_hos_0025_ad_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo utility under attack
  value: '64.82'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_0025_ad_asr
  evaluation_id: ev_g_hos_0025_ad_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '2.06'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_0025_dyn_clean
  evaluation_id: ev_g_hos_0025_dyn_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDyn clean utility
  value: '50.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_0025_dyn_att
  evaluation_id: ev_g_hos_0025_dyn_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDyn utility under attack
  value: '41.96'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_0025_dyn_asr
  evaluation_id: ev_g_hos_0025_dyn_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDyn ASR
  value: '5.54'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_0025_wasp_clean
  evaluation_id: ev_g_hos_0025_wasp_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations clean utility
  value: '51.89'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_0025_wasp_att
  evaluation_id: ev_g_hos_0025_wasp_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations utility under attack
  value: '70.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_0025_wasp_asr
  evaluation_id: ev_g_hos_0025_wasp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations ASR
  value: '1.43'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_0025_llm_att
  evaluation_id: ev_g_hos_0025_llm_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: LLMail-Inject utility under attack
  value: '98.19'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_0025_llm_asr
  evaluation_id: ev_g_hos_0025_llm_asr
  quantity: R1
  coverage: direct
  metric_as_printed: LLMail-Inject ASR
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_005_ad_clean
  evaluation_id: ev_g_hos_005_ad_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDojo clean utility
  value: '62.11'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_005_ad_att
  evaluation_id: ev_g_hos_005_ad_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo utility under attack
  value: '61.78'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_005_ad_asr
  evaluation_id: ev_g_hos_005_ad_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '1.74'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_005_dyn_clean
  evaluation_id: ev_g_hos_005_dyn_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDyn clean utility
  value: '50.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_005_dyn_att
  evaluation_id: ev_g_hos_005_dyn_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDyn utility under attack
  value: '45.89'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_005_dyn_asr
  evaluation_id: ev_g_hos_005_dyn_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDyn ASR
  value: '3.93'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_005_wasp_clean
  evaluation_id: ev_g_hos_005_wasp_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations clean utility
  value: '47.57'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_005_wasp_att
  evaluation_id: ev_g_hos_005_wasp_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations utility under attack
  value: '60.95'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_005_wasp_asr
  evaluation_id: ev_g_hos_005_wasp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations ASR
  value: '0.95'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_005_llm_att
  evaluation_id: ev_g_hos_005_llm_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: LLMail-Inject utility under attack
  value: '97.49'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_005_llm_asr
  evaluation_id: ev_g_hos_005_llm_asr
  quantity: R1
  coverage: direct
  metric_as_printed: LLMail-Inject ASR
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_ad_clean
  evaluation_id: ev_g_hos_ad_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDojo clean utility
  value: '63.16'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_ad_att
  evaluation_id: ev_g_hos_ad_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDojo utility under attack
  value: '59.61'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_ad_asr
  evaluation_id: ev_g_hos_ad_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '1.09'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_dyn_clean
  evaluation_id: ev_g_hos_dyn_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDyn clean utility
  value: '41.67'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_dyn_att
  evaluation_id: ev_g_hos_dyn_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: AgentDyn utility under attack
  value: '38.75'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_dyn_asr
  evaluation_id: ev_g_hos_dyn_asr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDyn ASR
  value: '0.89'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_wasp_clean
  evaluation_id: ev_g_hos_wasp_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations clean utility
  value: '41.62'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_wasp_att
  evaluation_id: ev_g_hos_wasp_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations utility under attack
  value: '54.05'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_wasp_asr
  evaluation_id: ev_g_hos_wasp_asr
  quantity: R1
  coverage: direct
  metric_as_printed: WASP (37 clean tasks; 84 injection combinations ASR
  value: '0.24'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_llm_att
  evaluation_id: ev_g_hos_llm_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: LLMail-Inject utility under attack
  value: '88.97'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_hos_llm_asr
  evaluation_id: ev_g_hos_llm_asr
  quantity: R1
  coverage: direct
  metric_as_printed: LLMail-Inject ASR
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 31
  location: Table 13
- id: r_g_base_flag_send
  evaluation_id: ev_g_base_flag
  quantity: R1
  coverage: direct
  metric_as_printed: send rate
  value: '0.86'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 4
- id: r_g_base_flag_det
  evaluation_id: ev_g_base_flag
  quantity: other
  coverage: direct
  metric_as_printed: detection rate
  value: '11.03'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 4
- id: r_g_red_flag_send
  evaluation_id: ev_g_red_flag
  quantity: R1
  coverage: direct
  metric_as_printed: send rate
  value: '0.31'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 4
- id: r_g_red_flag_det
  evaluation_id: ev_g_red_flag
  quantity: other
  coverage: direct
  metric_as_printed: detection rate
  value: '3.60'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 4
- id: r_g_blue_flag_send
  evaluation_id: ev_g_blue_flag
  quantity: R1
  coverage: direct
  metric_as_printed: send rate
  value: '0.32'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 4
- id: r_g_blue_flag_det
  evaluation_id: ev_g_blue_flag
  quantity: other
  coverage: direct
  metric_as_printed: detection rate
  value: '28.20'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 4
- id: r_g_hos_flag_send
  evaluation_id: ev_g_hos_flag
  quantity: R1
  coverage: direct
  metric_as_printed: send rate
  value: '0.37'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 4
- id: r_g_hos_flag_det
  evaluation_id: ev_g_hos_flag
  quantity: other
  coverage: direct
  metric_as_printed: detection rate
  value: '3.11'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 4
- id: r_l_none_v38ad
  evaluation_id: ev_l_none_v38ad
  quantity: R1
  coverage: direct
  metric_as_printed: examples broken, V38 white-box suffix attack (autoresearch from Claudini, developed against HoS on GPT-OSS-20B)
  value: '12'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 29
  location: Appendix F.2
- id: r_l_sec_v38ad
  evaluation_id: ev_l_sec_v38ad
  quantity: R1
  coverage: direct
  metric_as_printed: examples broken, V38 white-box suffix attack (autoresearch from Claudini, developed against HoS on GPT-OSS-20B)
  value: '6'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 29
  location: Appendix F.2
- id: r_l_hos_v38ad
  evaluation_id: ev_l_hos_v38ad
  quantity: R1
  coverage: direct
  metric_as_printed: examples broken, V38 white-box suffix attack (autoresearch from Claudini, developed against HoS on GPT-OSS-20B)
  value: '7'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 29
  location: Appendix F.2
- id: r_g_none_c45ad
  evaluation_id: ev_g_none_c45ad
  quantity: R1
  coverage: direct
  metric_as_printed: cumulative ASR, claudini_v45 white-box suffix attack, not adapted to HoS
  value: '55'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 30
  location: Appendix F.2
- id: r_g_sec_c45ad
  evaluation_id: ev_g_sec_c45ad
  quantity: R1
  coverage: direct
  metric_as_printed: cumulative ASR, claudini_v45 white-box suffix attack, not adapted to HoS
  value: '71'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 30
  location: Appendix F.2
- id: r_g_hos_c45ad
  evaluation_id: ev_g_hos_c45ad
  quantity: R1
  coverage: direct
  metric_as_printed: cumulative ASR, claudini_v45 white-box suffix attack, not adapted to HoS
  value: '48'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 30
  location: Appendix F.2
- id: r_g_none_c45ia
  evaluation_id: ev_g_none_c45ia
  quantity: R1
  coverage: direct
  metric_as_printed: cumulative ASR, claudini_v45 white-box suffix attack, not adapted to HoS
  value: '31'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 30
  location: Appendix F.2
- id: r_g_hos_c45ia
  evaluation_id: ev_g_hos_c45ia
  quantity: R1
  coverage: direct
  metric_as_printed: cumulative ASR, claudini_v45 white-box suffix attack, not adapted to HoS
  value: '2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 30
  location: Appendix F.2
comparisons:
- id: c_g_static_hos_none
  left: ev_g_hos_static
  right: ev_g_none_static
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_g_static_sec_none
  left: ev_g_sec_static
  right: ev_g_none_static
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_g_static_hos_sec
  left: ev_g_hos_static
  right: ev_g_sec_static
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: HoS against SecAlign on the same base'
  page: 5
- id: c_l_static_hos_none
  left: ev_l_hos_static
  right: ev_l_none_static
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_l_static_sec_none
  left: ev_l_sec_static
  right: ev_l_none_static
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_l_static_hos_sec
  left: ev_l_hos_static
  right: ev_l_sec_static
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: HoS against SecAlign on the same base'
  page: 5
- id: c_g_own_hos_none
  left: ev_g_hos_own
  right: ev_g_none_own
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_g_own_sec_none
  left: ev_g_sec_own
  right: ev_g_none_own
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_g_own_hos_sec
  left: ev_g_hos_own
  right: ev_g_sec_own
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: HoS against SecAlign on the same base'
  page: 5
- id: c_l_own_hos_none
  left: ev_l_hos_own
  right: ev_l_none_own
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_l_own_sec_none
  left: ev_l_sec_own
  right: ev_l_none_own
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_l_own_hos_sec
  left: ev_l_hos_own
  right: ev_l_sec_own
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: HoS against SecAlign on the same base'
  page: 5
- id: c_g_max_hos_none
  left: ev_g_hos_max
  right: ev_g_none_max
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_g_max_sec_none
  left: ev_g_sec_max
  right: ev_g_none_max
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_g_max_hos_sec
  left: ev_g_hos_max
  right: ev_g_sec_max
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: HoS against SecAlign on the same base'
  page: 5
- id: c_l_max_hos_none
  left: ev_l_hos_max
  right: ev_l_none_max
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_l_max_sec_none
  left: ev_l_sec_max
  right: ev_l_none_max
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_l_max_hos_sec
  left: ev_l_hos_max
  right: ev_l_sec_max
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: HoS against SecAlign on the same base'
  page: 5
- id: c_g_sec_arc0
  left: ev_g_sec_arc0
  right: ev_g_none_arc0
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_g_sec_gsm0
  left: ev_g_sec_gsm0
  right: ev_g_none_gsm0
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_g_sec_gsm1
  left: ev_g_sec_gsm1
  right: ev_g_none_gsm1
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_g_sec_gsm8
  left: ev_g_sec_gsm8
  right: ev_g_none_gsm8
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_g_sec_mmlu0
  left: ev_g_sec_mmlu0
  right: ev_g_none_mmlu0
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_g_sec_mmlu5
  left: ev_g_sec_mmlu5
  right: ev_g_none_mmlu5
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_g_sec_pro0
  left: ev_g_sec_pro0
  right: ev_g_none_pro0
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_g_sec_pro5
  left: ev_g_sec_pro5
  right: ev_g_none_pro5
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_g_aside_arc0
  left: ev_g_aside_arc0
  right: ev_g_none_arc0
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_g_aside_gsm0
  left: ev_g_aside_gsm0
  right: ev_g_none_gsm0
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_g_aside_gsm1
  left: ev_g_aside_gsm1
  right: ev_g_none_gsm1
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_g_aside_gsm8
  left: ev_g_aside_gsm8
  right: ev_g_none_gsm8
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_g_aside_mmlu0
  left: ev_g_aside_mmlu0
  right: ev_g_none_mmlu0
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_g_aside_mmlu5
  left: ev_g_aside_mmlu5
  right: ev_g_none_mmlu5
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_g_aside_pro0
  left: ev_g_aside_pro0
  right: ev_g_none_pro0
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_g_aside_pro5
  left: ev_g_aside_pro5
  right: ev_g_none_pro5
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_g_hos_arc0
  left: ev_g_hos_arc0
  right: ev_g_none_arc0
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_g_hos_gsm0
  left: ev_g_hos_gsm0
  right: ev_g_none_gsm0
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_g_hos_gsm1
  left: ev_g_hos_gsm1
  right: ev_g_none_gsm1
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_g_hos_gsm8
  left: ev_g_hos_gsm8
  right: ev_g_none_gsm8
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_g_hos_mmlu0
  left: ev_g_hos_mmlu0
  right: ev_g_none_mmlu0
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_g_hos_mmlu5
  left: ev_g_hos_mmlu5
  right: ev_g_none_mmlu5
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_g_hos_pro0
  left: ev_g_hos_pro0
  right: ev_g_none_pro0
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_g_hos_pro5
  left: ev_g_hos_pro5
  right: ev_g_none_pro5
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_sec_arc0
  left: ev_l_sec_arc0
  right: ev_l_none_arc0
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_sec_gsm0
  left: ev_l_sec_gsm0
  right: ev_l_none_gsm0
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_sec_gsm1
  left: ev_l_sec_gsm1
  right: ev_l_none_gsm1
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_sec_gsm8
  left: ev_l_sec_gsm8
  right: ev_l_none_gsm8
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_sec_mmlu0
  left: ev_l_sec_mmlu0
  right: ev_l_none_mmlu0
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_sec_mmlu5
  left: ev_l_sec_mmlu5
  right: ev_l_none_mmlu5
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_sec_pro0
  left: ev_l_sec_pro0
  right: ev_l_none_pro0
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_sec_pro5
  left: ev_l_sec_pro5
  right: ev_l_none_pro5
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_hos_arc0
  left: ev_l_hos_arc0
  right: ev_l_none_arc0
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_hos_gsm0
  left: ev_l_hos_gsm0
  right: ev_l_none_gsm0
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_hos_gsm1
  left: ev_l_hos_gsm1
  right: ev_l_none_gsm1
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_hos_gsm8
  left: ev_l_hos_gsm8
  right: ev_l_none_gsm8
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_hos_mmlu0
  left: ev_l_hos_mmlu0
  right: ev_l_none_mmlu0
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_hos_mmlu5
  left: ev_l_hos_mmlu5
  right: ev_l_none_mmlu5
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_hos_pro0
  left: ev_l_hos_pro0
  right: ev_l_none_pro0
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_hos_pro5
  left: ev_l_hos_pro5
  right: ev_l_none_pro5
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_sec_ad_clean
  left: ev_l_sec_ad_clean
  right: ev_l_none_ad_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_sec_ad_att
  left: ev_l_sec_ad_att
  right: ev_l_none_ad_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_sec_ad_asr
  left: ev_l_sec_ad_asr
  right: ev_l_none_ad_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_sec_dyn_clean
  left: ev_l_sec_dyn_clean
  right: ev_l_none_dyn_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_sec_dyn_att
  left: ev_l_sec_dyn_att
  right: ev_l_none_dyn_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_sec_dyn_asr
  left: ev_l_sec_dyn_asr
  right: ev_l_none_dyn_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_sec_wasp_clean
  left: ev_l_sec_wasp_clean
  right: ev_l_none_wasp_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_sec_wasp_att
  left: ev_l_sec_wasp_att
  right: ev_l_none_wasp_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_sec_wasp_asr
  left: ev_l_sec_wasp_asr
  right: ev_l_none_wasp_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_sec_llm_att
  left: ev_l_sec_llm_att
  right: ev_l_none_llm_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_sec_llm_asr
  left: ev_l_sec_llm_asr
  right: ev_l_none_llm_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_hos_ad_clean
  left: ev_l_hos_ad_clean
  right: ev_l_none_ad_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_hos_sec_ad_clean
  left: ev_l_hos_ad_clean
  right: ev_l_sec_ad_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: HoS against Meta-SecAlign-70B on the same base'
  page: 8
- id: c_l_hos_ad_att
  left: ev_l_hos_ad_att
  right: ev_l_none_ad_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_hos_sec_ad_att
  left: ev_l_hos_ad_att
  right: ev_l_sec_ad_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: HoS against Meta-SecAlign-70B on the same base'
  page: 8
- id: c_l_hos_ad_asr
  left: ev_l_hos_ad_asr
  right: ev_l_none_ad_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_hos_sec_ad_asr
  left: ev_l_hos_ad_asr
  right: ev_l_sec_ad_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: HoS against Meta-SecAlign-70B on the same base'
  page: 8
- id: c_l_hos_dyn_clean
  left: ev_l_hos_dyn_clean
  right: ev_l_none_dyn_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_hos_sec_dyn_clean
  left: ev_l_hos_dyn_clean
  right: ev_l_sec_dyn_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: HoS against Meta-SecAlign-70B on the same base'
  page: 8
- id: c_l_hos_dyn_att
  left: ev_l_hos_dyn_att
  right: ev_l_none_dyn_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_hos_sec_dyn_att
  left: ev_l_hos_dyn_att
  right: ev_l_sec_dyn_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: HoS against Meta-SecAlign-70B on the same base'
  page: 8
- id: c_l_hos_dyn_asr
  left: ev_l_hos_dyn_asr
  right: ev_l_none_dyn_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_hos_sec_dyn_asr
  left: ev_l_hos_dyn_asr
  right: ev_l_sec_dyn_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: HoS against Meta-SecAlign-70B on the same base'
  page: 8
- id: c_l_hos_wasp_clean
  left: ev_l_hos_wasp_clean
  right: ev_l_none_wasp_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_hos_sec_wasp_clean
  left: ev_l_hos_wasp_clean
  right: ev_l_sec_wasp_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: HoS against Meta-SecAlign-70B on the same base'
  page: 8
- id: c_l_hos_wasp_att
  left: ev_l_hos_wasp_att
  right: ev_l_none_wasp_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_hos_sec_wasp_att
  left: ev_l_hos_wasp_att
  right: ev_l_sec_wasp_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: HoS against Meta-SecAlign-70B on the same base'
  page: 8
- id: c_l_hos_wasp_asr
  left: ev_l_hos_wasp_asr
  right: ev_l_none_wasp_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_hos_sec_wasp_asr
  left: ev_l_hos_wasp_asr
  right: ev_l_sec_wasp_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: HoS against Meta-SecAlign-70B on the same base'
  page: 8
- id: c_l_hos_llm_att
  left: ev_l_hos_llm_att
  right: ev_l_none_llm_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_hos_sec_llm_att
  left: ev_l_hos_llm_att
  right: ev_l_sec_llm_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: HoS against Meta-SecAlign-70B on the same base'
  page: 8
- id: c_l_hos_llm_asr
  left: ev_l_hos_llm_asr
  right: ev_l_none_llm_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_hos_sec_llm_asr
  left: ev_l_hos_llm_asr
  right: ev_l_sec_llm_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: HoS against Meta-SecAlign-70B on the same base'
  page: 8
- id: c_g_sec_5e5_ad_clean
  left: ev_g_sec_5e5_ad_clean
  right: ev_g_none_ad_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_sec_5e5_ad_att
  left: ev_g_sec_5e5_ad_att
  right: ev_g_none_ad_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_sec_5e5_ad_asr
  left: ev_g_sec_5e5_ad_asr
  right: ev_g_none_ad_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_sec_5e5_dyn_clean
  left: ev_g_sec_5e5_dyn_clean
  right: ev_g_none_dyn_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_sec_5e5_dyn_att
  left: ev_g_sec_5e5_dyn_att
  right: ev_g_none_dyn_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_sec_5e5_dyn_asr
  left: ev_g_sec_5e5_dyn_asr
  right: ev_g_none_dyn_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_sec_5e5_wasp_clean
  left: ev_g_sec_5e5_wasp_clean
  right: ev_g_none_wasp_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_sec_5e5_wasp_att
  left: ev_g_sec_5e5_wasp_att
  right: ev_g_none_wasp_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_sec_5e5_wasp_asr
  left: ev_g_sec_5e5_wasp_asr
  right: ev_g_none_wasp_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_sec_5e5_llm_att
  left: ev_g_sec_5e5_llm_att
  right: ev_g_none_llm_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_sec_5e5_llm_asr
  left: ev_g_sec_5e5_llm_asr
  right: ev_g_none_llm_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_sec_1e4_ad_clean
  left: ev_g_sec_1e4_ad_clean
  right: ev_g_none_ad_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_sec_1e4_ad_att
  left: ev_g_sec_1e4_ad_att
  right: ev_g_none_ad_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_sec_1e4_ad_asr
  left: ev_g_sec_1e4_ad_asr
  right: ev_g_none_ad_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_sec_1e4_dyn_clean
  left: ev_g_sec_1e4_dyn_clean
  right: ev_g_none_dyn_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_sec_1e4_dyn_att
  left: ev_g_sec_1e4_dyn_att
  right: ev_g_none_dyn_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_sec_1e4_dyn_asr
  left: ev_g_sec_1e4_dyn_asr
  right: ev_g_none_dyn_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_sec_1e4_wasp_clean
  left: ev_g_sec_1e4_wasp_clean
  right: ev_g_none_wasp_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_sec_1e4_wasp_att
  left: ev_g_sec_1e4_wasp_att
  right: ev_g_none_wasp_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_sec_1e4_wasp_asr
  left: ev_g_sec_1e4_wasp_asr
  right: ev_g_none_wasp_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_sec_1e4_llm_att
  left: ev_g_sec_1e4_llm_att
  right: ev_g_none_llm_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_sec_1e4_llm_asr
  left: ev_g_sec_1e4_llm_asr
  right: ev_g_none_llm_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_sec_2e4_ad_clean
  left: ev_g_sec_2e4_ad_clean
  right: ev_g_none_ad_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_sec_2e4_ad_att
  left: ev_g_sec_2e4_ad_att
  right: ev_g_none_ad_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_sec_2e4_ad_asr
  left: ev_g_sec_2e4_ad_asr
  right: ev_g_none_ad_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_sec_2e4_dyn_clean
  left: ev_g_sec_2e4_dyn_clean
  right: ev_g_none_dyn_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_sec_2e4_dyn_att
  left: ev_g_sec_2e4_dyn_att
  right: ev_g_none_dyn_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_sec_2e4_dyn_asr
  left: ev_g_sec_2e4_dyn_asr
  right: ev_g_none_dyn_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_sec_2e4_wasp_clean
  left: ev_g_sec_2e4_wasp_clean
  right: ev_g_none_wasp_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_sec_2e4_wasp_att
  left: ev_g_sec_2e4_wasp_att
  right: ev_g_none_wasp_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_sec_2e4_wasp_asr
  left: ev_g_sec_2e4_wasp_asr
  right: ev_g_none_wasp_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_sec_2e4_llm_att
  left: ev_g_sec_2e4_llm_att
  right: ev_g_none_llm_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_sec_2e4_llm_asr
  left: ev_g_sec_2e4_llm_asr
  right: ev_g_none_llm_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_aside_ad_clean
  left: ev_g_aside_ad_clean
  right: ev_g_none_ad_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_aside_ad_att
  left: ev_g_aside_ad_att
  right: ev_g_none_ad_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_aside_ad_asr
  left: ev_g_aside_ad_asr
  right: ev_g_none_ad_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_aside_dyn_clean
  left: ev_g_aside_dyn_clean
  right: ev_g_none_dyn_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_aside_dyn_att
  left: ev_g_aside_dyn_att
  right: ev_g_none_dyn_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_aside_dyn_asr
  left: ev_g_aside_dyn_asr
  right: ev_g_none_dyn_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_aside_wasp_clean
  left: ev_g_aside_wasp_clean
  right: ev_g_none_wasp_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_aside_wasp_att
  left: ev_g_aside_wasp_att
  right: ev_g_none_wasp_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_aside_wasp_asr
  left: ev_g_aside_wasp_asr
  right: ev_g_none_wasp_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_aside_llm_att
  left: ev_g_aside_llm_att
  right: ev_g_none_llm_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_aside_llm_asr
  left: ev_g_aside_llm_asr
  right: ev_g_none_llm_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_001_ad_clean
  left: ev_g_hos_001_ad_clean
  right: ev_g_none_ad_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_001_ad_att
  left: ev_g_hos_001_ad_att
  right: ev_g_none_ad_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_001_ad_asr
  left: ev_g_hos_001_ad_asr
  right: ev_g_none_ad_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_001_dyn_clean
  left: ev_g_hos_001_dyn_clean
  right: ev_g_none_dyn_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_001_dyn_att
  left: ev_g_hos_001_dyn_att
  right: ev_g_none_dyn_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_001_dyn_asr
  left: ev_g_hos_001_dyn_asr
  right: ev_g_none_dyn_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_001_wasp_clean
  left: ev_g_hos_001_wasp_clean
  right: ev_g_none_wasp_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_001_wasp_att
  left: ev_g_hos_001_wasp_att
  right: ev_g_none_wasp_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_001_wasp_asr
  left: ev_g_hos_001_wasp_asr
  right: ev_g_none_wasp_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_001_llm_att
  left: ev_g_hos_001_llm_att
  right: ev_g_none_llm_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_001_llm_asr
  left: ev_g_hos_001_llm_asr
  right: ev_g_none_llm_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_0025_ad_clean
  left: ev_g_hos_0025_ad_clean
  right: ev_g_none_ad_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_0025_ad_att
  left: ev_g_hos_0025_ad_att
  right: ev_g_none_ad_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_0025_ad_asr
  left: ev_g_hos_0025_ad_asr
  right: ev_g_none_ad_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_0025_dyn_clean
  left: ev_g_hos_0025_dyn_clean
  right: ev_g_none_dyn_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_0025_dyn_att
  left: ev_g_hos_0025_dyn_att
  right: ev_g_none_dyn_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_0025_dyn_asr
  left: ev_g_hos_0025_dyn_asr
  right: ev_g_none_dyn_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_0025_wasp_clean
  left: ev_g_hos_0025_wasp_clean
  right: ev_g_none_wasp_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_0025_wasp_att
  left: ev_g_hos_0025_wasp_att
  right: ev_g_none_wasp_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_0025_wasp_asr
  left: ev_g_hos_0025_wasp_asr
  right: ev_g_none_wasp_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_0025_llm_att
  left: ev_g_hos_0025_llm_att
  right: ev_g_none_llm_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_0025_llm_asr
  left: ev_g_hos_0025_llm_asr
  right: ev_g_none_llm_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_005_ad_clean
  left: ev_g_hos_005_ad_clean
  right: ev_g_none_ad_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_005_ad_att
  left: ev_g_hos_005_ad_att
  right: ev_g_none_ad_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_005_ad_asr
  left: ev_g_hos_005_ad_asr
  right: ev_g_none_ad_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_005_dyn_clean
  left: ev_g_hos_005_dyn_clean
  right: ev_g_none_dyn_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_005_dyn_att
  left: ev_g_hos_005_dyn_att
  right: ev_g_none_dyn_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_005_dyn_asr
  left: ev_g_hos_005_dyn_asr
  right: ev_g_none_dyn_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_005_wasp_clean
  left: ev_g_hos_005_wasp_clean
  right: ev_g_none_wasp_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_005_wasp_att
  left: ev_g_hos_005_wasp_att
  right: ev_g_none_wasp_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_005_wasp_asr
  left: ev_g_hos_005_wasp_asr
  right: ev_g_none_wasp_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_005_llm_att
  left: ev_g_hos_005_llm_att
  right: ev_g_none_llm_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_005_llm_asr
  left: ev_g_hos_005_llm_asr
  right: ev_g_none_llm_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_ad_clean
  left: ev_g_hos_ad_clean
  right: ev_g_none_ad_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_ad_att
  left: ev_g_hos_ad_att
  right: ev_g_none_ad_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_ad_asr
  left: ev_g_hos_ad_asr
  right: ev_g_none_ad_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_dyn_clean
  left: ev_g_hos_dyn_clean
  right: ev_g_none_dyn_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_dyn_att
  left: ev_g_hos_dyn_att
  right: ev_g_none_dyn_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_dyn_asr
  left: ev_g_hos_dyn_asr
  right: ev_g_none_dyn_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_wasp_clean
  left: ev_g_hos_wasp_clean
  right: ev_g_none_wasp_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_wasp_att
  left: ev_g_hos_wasp_att
  right: ev_g_none_wasp_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_wasp_asr
  left: ev_g_hos_wasp_asr
  right: ev_g_none_wasp_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_llm_att
  left: ev_g_hos_llm_att
  right: ev_g_none_llm_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_llm_asr
  left: ev_g_hos_llm_asr
  right: ev_g_none_llm_asr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 31
- id: c_g_hos_flag
  left: ev_g_hos_flag
  right: ev_g_base_flag
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 9
- id: c_g_red_flag
  left: ev_g_hos_flag
  right: ev_g_red_flag
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: blue-head gates added to the red-head gates
  page: 9
- id: c_g_blue_flag
  left: ev_g_hos_flag
  right: ev_g_blue_flag
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: red-head gates added to the blue-head gates
  page: 9
- id: c_l_sec_v38ad
  left: ev_l_sec_v38ad
  right: ev_l_none_v38ad
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 29
- id: c_l_hos_v38ad
  left: ev_l_hos_v38ad
  right: ev_l_none_v38ad
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 29
- id: c_l_hos_sec_v38ad
  left: ev_l_hos_v38ad
  right: ev_l_sec_v38ad
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: HoS against SecAlign on the same base'
  page: 29
- id: c_g_sec_c45ad
  left: ev_g_sec_c45ad
  right: ev_g_none_c45ad
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 30
- id: c_g_hos_c45ad
  left: ev_g_hos_c45ad
  right: ev_g_none_c45ad
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 30
- id: c_g_hos_sec_c45ad
  left: ev_g_hos_c45ad
  right: ev_g_sec_c45ad
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: HoS against SecAlign on the same base'
  page: 30
- id: c_g_hos_c45ia
  left: ev_g_hos_c45ia
  right: ev_g_none_c45ia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 30
claims:
- id: k1
  text: the black-box RL-Hammer attack reaches 97% ASR against GPT-OSS-20B and 85% against SecAlign but only 10% against HoS
  location: abstract
  anchor: the black-box RL-Hammer attack reaches 97% ASR against GPT-OSS-20B
  jobs:
  - R1
  supported_by:
  - c_g_max_hos_none
  - c_g_max_hos_sec
  - c_l_max_hos_none
  - c_l_max_hos_sec
  - c_g_own_hos_none
  - c_l_own_hos_none
  page: 1
- id: k2
  text: HoS preserves base-model utility across standard multiple-choice benchmarks and prompting setups, whereas SecAlign and ASIDE degrade in few-shot settings
  location: abstract
  anchor: HoS preserves base-model utility across standard multiple-choice benchmarks
  jobs:
  - general_utility
  supported_by:
  - c_g_hos_arc0
  - c_g_hos_gsm0
  - c_g_hos_gsm1
  - c_g_hos_gsm8
  - c_g_hos_mmlu0
  - c_g_hos_mmlu5
  - c_g_hos_pro0
  - c_g_hos_pro5
  - c_g_sec_arc0
  - c_g_sec_gsm0
  - c_g_sec_gsm1
  - c_g_sec_gsm8
  - c_g_sec_mmlu0
  - c_g_sec_mmlu5
  - c_g_sec_pro0
  - c_g_sec_pro5
  - c_g_aside_arc0
  - c_g_aside_gsm0
  - c_g_aside_gsm1
  - c_g_aside_gsm8
  - c_g_aside_mmlu0
  - c_g_aside_mmlu5
  - c_g_aside_pro0
  - c_g_aside_pro5
  - c_l_hos_arc0
  - c_l_hos_gsm0
  - c_l_hos_gsm1
  - c_l_hos_gsm8
  - c_l_hos_mmlu0
  - c_l_hos_mmlu5
  - c_l_hos_pro0
  - c_l_hos_pro5
  - c_l_sec_arc0
  - c_l_sec_gsm0
  - c_l_sec_gsm1
  - c_l_sec_gsm8
  - c_l_sec_mmlu0
  - c_l_sec_mmlu5
  - c_l_sec_pro0
  - c_l_sec_pro5
  page: 1
- id: k3
  text: on agentic benchmarks with static attacks HoS and SecAlign perform similarly and improve the security-utility trade-off over the base model
  location: abstract
  anchor: On agentic benchmarks with static attacks, HoS and SecAlign perform similarly
  jobs:
  - R1
  - R2
  supported_by:
  - c_l_hos_ad_clean
  - c_l_hos_ad_att
  - c_l_hos_ad_asr
  - c_l_hos_dyn_clean
  - c_l_hos_dyn_att
  - c_l_hos_dyn_asr
  - c_l_hos_wasp_clean
  - c_l_hos_wasp_att
  - c_l_hos_wasp_asr
  - c_g_hos_ad_clean
  - c_g_hos_ad_att
  - c_g_hos_ad_asr
  - c_g_hos_dyn_clean
  - c_g_hos_dyn_att
  - c_g_hos_dyn_asr
  - c_g_hos_wasp_clean
  - c_g_hos_wasp_att
  - c_g_hos_wasp_asr
  - c_l_hos_sec_ad_clean
  - c_l_hos_sec_ad_att
  - c_l_hos_sec_ad_asr
  - c_l_hos_sec_dyn_clean
  - c_l_hos_sec_dyn_att
  - c_l_hos_sec_dyn_asr
  - c_l_hos_sec_wasp_clean
  - c_l_hos_sec_wasp_att
  - c_l_hos_sec_wasp_asr
  page: 1
  contradicted_by:
  - r_l_hos_dyn_clean
  - r_g_hos_dyn_clean
coverage:
  R1: direct
  R2clean: direct
  R2attack: direct
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Tables 1, 3, 4 and 13; Appendix F.2
    R2clean: Tables 3 and 13
    R2attack: Tables 3 and 13
    general_utility: Table 2
schema_gaps:
- where: defences sec_g
  note: the paper sweeps SecAlign learning rates on GPT-OSS-20B (Table 13) but does not say which checkpoint Tables 1 and 2 use; the headline HoS GPT-OSS model is learning rate 0.1 (p. 35)
- where: comparisons c_*_own_*
  note: each arm is attacked by an RL-Hammer attacker trained against it, so the attacker differs by design; the authors note the HoS-trained attacker got sparse reward (3.2% training ASR) and the SecAlign-trained attacker
    is stronger against HoS, which the max rows capture
- where: claims k3
  note: the headline HoS GPT-OSS model loses clean utility on AgentDyn (41.67 against 58.33) and WASP (41.62 against 56.22); Llama HoS loses 15.4 points of AgentDyn clean utility while Meta-SecAlign-70B gains 8.0 (Table 3)
- where: report
  note: the white-box claim (about three times more compute to reach 50% ASR) rests on Figure 2 curves and is not coded; the LLMail utility and multi-turn AlpacaFarm judge is Claude Sonnet 5 (Appendices G and I); the attack
    names Claudini and claudini_v45 are the paper's
- where: evaluations ev_l_*_static
  note: Table 12 shows that under SecAlign's own InjecAgent protocol Meta-SecAlign-70B gives invalid ReAct outputs on 90 to 95% of cases; the coded static rows use the authors' native tool protocol

`llmon_arxiv2026` (2610.38120)

In [ ]:
%%writefile reports/records/llmon_arxiv2026.yaml
report:
  id: llmon_arxiv2026
  slug: '2610.38120'
  title: Post-Training Large Language Models with LLMON
  first_author: Basel Shbita
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2610.38120'
  version_read: arXiv v1
  first_date: '2026-10-28'
  read: F
  role: proposes
  code_release: not stated
  extraction_scope: 'Section 5: Table 2 (OPI and ASB), the Indirect column of Table 3 (ARES) and Table 1 (capability categories); the abstract claim (the Direct and Multi-turn columns of Table 3, which are direct or user-turn
    attacks, and the attention metrics of Table 4 and Figures 2 to 6 not coded)'
defences:
- id: llmon_3b
  name: LLMON post-training (SFT, GRPO, then DPO)
  origin: this_report
  base_checkpoint: Granite-4.1-3B-Base
  axis1:
  - delimiters_markers
  axis2:
  - sft
  - preference_optimisation
  isolation: no_ablation
  signal_dependence: not_reported
  trains_instruction_pair: partly
  trains_carry_through: 'no'
  page: 2
  components: &id001
  - six reserved delimiter tokens (remapped unused vocabulary slots) wrap every instruction, data, document, tool and system segment in a typed block with an identifier
  - an exec block names which instruction to run on which data; content reached through exec input (data, documents, tool results) is read, never executed
  - 'training: full SFT on a 4.6B-token LLMON corpus with distractor variants (one to three competing instructions from the pool, irrelevant data) and 130k planted-injection examples; five GRPO stages; a final DPO stage on
    about 1k on-policy injection pairs'
  layout: 'LLMON serialisation: the injected payload arrives inside a data block reached through exec input; tool results return as data blocks'
  layout_compat: &id002
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: LLMON defines six special delimiter tokens
  pages:
    components: 3
    training: 4
- id: llmon_8b
  name: LLMON post-training (SFT, GRPO, then DPO)
  origin: this_report
  base_checkpoint: Granite-4.1-8B-Base
  axis1:
  - delimiters_markers
  axis2:
  - sft
  - preference_optimisation
  isolation: no_ablation
  signal_dependence: not_reported
  trains_instruction_pair: partly
  trains_carry_through: 'no'
  page: 2
  components: *id001
  layout: 'LLMON serialisation: the injected payload arrives inside a data block reached through exec input; tool results return as data blocks'
  layout_compat: *id002
  anchor: LLMON defines six special delimiter tokens
  pages:
    components: 3
    training: 4
evaluations:
- id: ev_3b_none_opi
  defence_id: none:Granite-4.1-3B-Base
  benchmark: 'OPI (open prompt injection): naive, escape, ignore, fake-completion and combine attacks on classic NLP task data'
  attacker_type: static
  attacker_access: black_box
  judge: OPI attack-success metric
  layout_same_as_defence: 'no'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 5
  baseline_kind: same_data_twin
- id: ev_3b_none_asb
  defence_id: none:Granite-4.1-3B-Base
  benchmark: ASB, observation-injection arm (injection in agent tool observations)
  attacker_type: static
  attacker_access: black_box
  judge: ASB attack-success check
  layout_same_as_defence: 'no'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 5
  baseline_kind: same_data_twin
- id: ev_3b_llmon_opi
  defence_id: llmon_3b
  benchmark: 'OPI (open prompt injection): naive, escape, ignore, fake-completion and combine attacks on classic NLP task data'
  attacker_type: static
  attacker_access: black_box
  judge: OPI attack-success metric
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 5
- id: ev_3b_llmon_asb
  defence_id: llmon_3b
  benchmark: ASB, observation-injection arm (injection in agent tool observations)
  attacker_type: static
  attacker_access: black_box
  judge: ASB attack-success check
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 5
- id: ev_8b_none_opi
  defence_id: none:Granite-4.1-8B-Base
  benchmark: 'OPI (open prompt injection): naive, escape, ignore, fake-completion and combine attacks on classic NLP task data'
  attacker_type: static
  attacker_access: black_box
  judge: OPI attack-success metric
  layout_same_as_defence: 'no'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 5
  baseline_kind: same_data_twin
- id: ev_8b_none_asb
  defence_id: none:Granite-4.1-8B-Base
  benchmark: ASB, observation-injection arm (injection in agent tool observations)
  attacker_type: static
  attacker_access: black_box
  judge: ASB attack-success check
  layout_same_as_defence: 'no'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 5
  baseline_kind: same_data_twin
- id: ev_8b_llmon_opi
  defence_id: llmon_8b
  benchmark: 'OPI (open prompt injection): naive, escape, ignore, fake-completion and combine attacks on classic NLP task data'
  attacker_type: static
  attacker_access: black_box
  judge: OPI attack-success metric
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 5
- id: ev_8b_llmon_asb
  defence_id: llmon_8b
  benchmark: ASB, observation-injection arm (injection in agent tool observations)
  attacker_type: static
  attacker_access: black_box
  judge: ASB attack-success check
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 5
- id: ev_3b_none_ares
  defence_id: none:Granite-4.1-3B-Base
  benchmark: 'ARES red-teaming, indirect arm: StruQ-style payloads planted in the input, seeds from Alpaca and a safety-policy set, mutated by eight attack strategies; ID and OOD splits averaged'
  attacker_type: static
  attacker_access: black_box
  judge: ARES harness judgement
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
  attacker_budget: eight mutation strategies from basic to advanced
  pages:
    benchmark: 5
  baseline_kind: same_data_twin
- id: ev_3b_llmon_ares
  defence_id: llmon_3b
  benchmark: 'ARES red-teaming, indirect arm: StruQ-style payloads planted in the input, seeds from Alpaca and a safety-policy set, mutated by eight attack strategies; ID and OOD splits averaged'
  attacker_type: static
  attacker_access: black_box
  judge: ARES harness judgement
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
  attacker_budget: eight mutation strategies from basic to advanced
  pages:
    benchmark: 5
- id: ev_8b_none_ares
  defence_id: none:Granite-4.1-8B-Base
  benchmark: 'ARES red-teaming, indirect arm: StruQ-style payloads planted in the input, seeds from Alpaca and a safety-policy set, mutated by eight attack strategies; ID and OOD splits averaged'
  attacker_type: static
  attacker_access: black_box
  judge: ARES harness judgement
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
  attacker_budget: eight mutation strategies from basic to advanced
  pages:
    benchmark: 5
  baseline_kind: same_data_twin
- id: ev_8b_llmon_ares
  defence_id: llmon_8b
  benchmark: 'ARES red-teaming, indirect arm: StruQ-style payloads planted in the input, seeds from Alpaca and a safety-policy set, mutated by eight attack strategies; ID and OOD splits averaged'
  attacker_type: static
  attacker_access: black_box
  judge: ARES harness judgement
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
  attacker_budget: eight mutation strategies from basic to advanced
  pages:
    benchmark: 5
- id: ev_3b_none_general
  defence_id: none:Granite-4.1-3B-Base
  benchmark: General category, unweighted mean of its benchmarks (Section 4.3.1)
  attacker_type: none
  attacker_access: not_applicable
  judge: each benchmark's own scorer
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
  baseline_kind: same_data_twin
- id: ev_3b_none_alignment
  defence_id: none:Granite-4.1-3B-Base
  benchmark: Alignment category, unweighted mean of its benchmarks (Section 4.3.1)
  attacker_type: none
  attacker_access: not_applicable
  judge: each benchmark's own scorer
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
  baseline_kind: same_data_twin
- id: ev_3b_none_math
  defence_id: none:Granite-4.1-3B-Base
  benchmark: Math category, unweighted mean of its benchmarks (Section 4.3.1)
  attacker_type: none
  attacker_access: not_applicable
  judge: each benchmark's own scorer
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
  baseline_kind: same_data_twin
- id: ev_3b_none_code
  defence_id: none:Granite-4.1-3B-Base
  benchmark: Code category, unweighted mean of its benchmarks (Section 4.3.1)
  attacker_type: none
  attacker_access: not_applicable
  judge: each benchmark's own scorer
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
  baseline_kind: same_data_twin
- id: ev_3b_none_tool-calling
  defence_id: none:Granite-4.1-3B-Base
  benchmark: Tool-Calling category, unweighted mean of its benchmarks (Section 4.3.1)
  attacker_type: none
  attacker_access: not_applicable
  judge: each benchmark's own scorer
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
  baseline_kind: same_data_twin
- id: ev_3b_none_multilingual
  defence_id: none:Granite-4.1-3B-Base
  benchmark: Multilingual category, unweighted mean of its benchmarks (Section 4.3.1)
  attacker_type: none
  attacker_access: not_applicable
  judge: each benchmark's own scorer
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
  baseline_kind: same_data_twin
- id: ev_3b_none_safety
  defence_id: none:Granite-4.1-3B-Base
  benchmark: Safety category, unweighted mean of its benchmarks (Section 4.3.1)
  attacker_type: none
  attacker_access: not_applicable
  judge: each benchmark's own scorer
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
  baseline_kind: same_data_twin
- id: ev_3b_llmon_general
  defence_id: llmon_3b
  benchmark: General category, unweighted mean of its benchmarks (Section 4.3.1)
  attacker_type: none
  attacker_access: not_applicable
  judge: each benchmark's own scorer
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
- id: ev_3b_llmon_alignment
  defence_id: llmon_3b
  benchmark: Alignment category, unweighted mean of its benchmarks (Section 4.3.1)
  attacker_type: none
  attacker_access: not_applicable
  judge: each benchmark's own scorer
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
- id: ev_3b_llmon_math
  defence_id: llmon_3b
  benchmark: Math category, unweighted mean of its benchmarks (Section 4.3.1)
  attacker_type: none
  attacker_access: not_applicable
  judge: each benchmark's own scorer
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
- id: ev_3b_llmon_code
  defence_id: llmon_3b
  benchmark: Code category, unweighted mean of its benchmarks (Section 4.3.1)
  attacker_type: none
  attacker_access: not_applicable
  judge: each benchmark's own scorer
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
- id: ev_3b_llmon_tool-calling
  defence_id: llmon_3b
  benchmark: Tool-Calling category, unweighted mean of its benchmarks (Section 4.3.1)
  attacker_type: none
  attacker_access: not_applicable
  judge: each benchmark's own scorer
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
- id: ev_3b_llmon_multilingual
  defence_id: llmon_3b
  benchmark: Multilingual category, unweighted mean of its benchmarks (Section 4.3.1)
  attacker_type: none
  attacker_access: not_applicable
  judge: each benchmark's own scorer
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
- id: ev_3b_llmon_safety
  defence_id: llmon_3b
  benchmark: Safety category, unweighted mean of its benchmarks (Section 4.3.1)
  attacker_type: none
  attacker_access: not_applicable
  judge: each benchmark's own scorer
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
- id: ev_8b_none_general
  defence_id: none:Granite-4.1-8B-Base
  benchmark: General category, unweighted mean of its benchmarks (Section 4.3.1)
  attacker_type: none
  attacker_access: not_applicable
  judge: each benchmark's own scorer
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
  baseline_kind: same_data_twin
- id: ev_8b_none_alignment
  defence_id: none:Granite-4.1-8B-Base
  benchmark: Alignment category, unweighted mean of its benchmarks (Section 4.3.1)
  attacker_type: none
  attacker_access: not_applicable
  judge: each benchmark's own scorer
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
  baseline_kind: same_data_twin
- id: ev_8b_none_math
  defence_id: none:Granite-4.1-8B-Base
  benchmark: Math category, unweighted mean of its benchmarks (Section 4.3.1)
  attacker_type: none
  attacker_access: not_applicable
  judge: each benchmark's own scorer
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
  baseline_kind: same_data_twin
- id: ev_8b_none_code
  defence_id: none:Granite-4.1-8B-Base
  benchmark: Code category, unweighted mean of its benchmarks (Section 4.3.1)
  attacker_type: none
  attacker_access: not_applicable
  judge: each benchmark's own scorer
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
  baseline_kind: same_data_twin
- id: ev_8b_none_tool-calling
  defence_id: none:Granite-4.1-8B-Base
  benchmark: Tool-Calling category, unweighted mean of its benchmarks (Section 4.3.1)
  attacker_type: none
  attacker_access: not_applicable
  judge: each benchmark's own scorer
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
  baseline_kind: same_data_twin
- id: ev_8b_none_multilingual
  defence_id: none:Granite-4.1-8B-Base
  benchmark: Multilingual category, unweighted mean of its benchmarks (Section 4.3.1)
  attacker_type: none
  attacker_access: not_applicable
  judge: each benchmark's own scorer
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
  baseline_kind: same_data_twin
- id: ev_8b_none_safety
  defence_id: none:Granite-4.1-8B-Base
  benchmark: Safety category, unweighted mean of its benchmarks (Section 4.3.1)
  attacker_type: none
  attacker_access: not_applicable
  judge: each benchmark's own scorer
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
  baseline_kind: same_data_twin
- id: ev_8b_llmon_general
  defence_id: llmon_8b
  benchmark: General category, unweighted mean of its benchmarks (Section 4.3.1)
  attacker_type: none
  attacker_access: not_applicable
  judge: each benchmark's own scorer
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
- id: ev_8b_llmon_alignment
  defence_id: llmon_8b
  benchmark: Alignment category, unweighted mean of its benchmarks (Section 4.3.1)
  attacker_type: none
  attacker_access: not_applicable
  judge: each benchmark's own scorer
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
- id: ev_8b_llmon_math
  defence_id: llmon_8b
  benchmark: Math category, unweighted mean of its benchmarks (Section 4.3.1)
  attacker_type: none
  attacker_access: not_applicable
  judge: each benchmark's own scorer
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
- id: ev_8b_llmon_code
  defence_id: llmon_8b
  benchmark: Code category, unweighted mean of its benchmarks (Section 4.3.1)
  attacker_type: none
  attacker_access: not_applicable
  judge: each benchmark's own scorer
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
- id: ev_8b_llmon_tool-calling
  defence_id: llmon_8b
  benchmark: Tool-Calling category, unweighted mean of its benchmarks (Section 4.3.1)
  attacker_type: none
  attacker_access: not_applicable
  judge: each benchmark's own scorer
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
- id: ev_8b_llmon_multilingual
  defence_id: llmon_8b
  benchmark: Multilingual category, unweighted mean of its benchmarks (Section 4.3.1)
  attacker_type: none
  attacker_access: not_applicable
  judge: each benchmark's own scorer
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
- id: ev_8b_llmon_safety
  defence_id: llmon_8b
  benchmark: Safety category, unweighted mean of its benchmarks (Section 4.3.1)
  attacker_type: none
  attacker_access: not_applicable
  judge: each benchmark's own scorer
  layout_same_as_defence: 'yes'
  measured_by: llmon_arxiv2026
  page: 7
  decoding: not reported
results:
- id: r_3b_none_opi
  evaluation_id: ev_3b_none_opi
  quantity: R1
  coverage: direct
  metric_as_printed: OPI ASR
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_3b_none_asb
  evaluation_id: ev_3b_none_asb
  quantity: R1
  coverage: direct
  metric_as_printed: ASB ASR
  value: '99.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_3b_llmon_opi
  evaluation_id: ev_3b_llmon_opi
  quantity: R1
  coverage: direct
  metric_as_printed: OPI ASR
  value: '0.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_3b_llmon_asb
  evaluation_id: ev_3b_llmon_asb
  quantity: R1
  coverage: direct
  metric_as_printed: ASB ASR
  value: '5.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_8b_none_opi
  evaluation_id: ev_8b_none_opi
  quantity: R1
  coverage: direct
  metric_as_printed: OPI ASR
  value: '96.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_8b_none_asb
  evaluation_id: ev_8b_none_asb
  quantity: R1
  coverage: direct
  metric_as_printed: ASB ASR
  value: '52.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_8b_llmon_opi
  evaluation_id: ev_8b_llmon_opi
  quantity: R1
  coverage: direct
  metric_as_printed: OPI ASR
  value: '0.02'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_8b_llmon_asb
  evaluation_id: ev_8b_llmon_asb
  quantity: R1
  coverage: direct
  metric_as_printed: ASB ASR
  value: '3.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_3b_none_ares
  evaluation_id: ev_3b_none_ares
  quantity: R1
  coverage: direct
  metric_as_printed: ARES indirect robustness (1 - ASR)
  value: '2.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_3b_llmon_ares
  evaluation_id: ev_3b_llmon_ares
  quantity: R1
  coverage: direct
  metric_as_printed: ARES indirect robustness (1 - ASR)
  value: '84.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_8b_none_ares
  evaluation_id: ev_8b_none_ares
  quantity: R1
  coverage: direct
  metric_as_printed: ARES indirect robustness (1 - ASR)
  value: '0.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_8b_llmon_ares
  evaluation_id: ev_8b_llmon_ares
  quantity: R1
  coverage: direct
  metric_as_printed: ARES indirect robustness (1 - ASR)
  value: '79.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_3b_none_general
  evaluation_id: ev_3b_none_general
  quantity: general_utility
  coverage: direct
  metric_as_printed: General category mean
  value: '57.91'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_3b_none_alignment
  evaluation_id: ev_3b_none_alignment
  quantity: general_utility
  coverage: direct
  metric_as_printed: Alignment category mean
  value: '82.30'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_3b_none_math
  evaluation_id: ev_3b_none_math
  quantity: general_utility
  coverage: direct
  metric_as_printed: Math category mean
  value: '75.19'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_3b_none_code
  evaluation_id: ev_3b_none_code
  quantity: general_utility
  coverage: direct
  metric_as_printed: Code category mean
  value: '64.19'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_3b_none_tool-calling
  evaluation_id: ev_3b_none_tool-calling
  quantity: general_utility
  coverage: direct
  metric_as_printed: Tool-Calling category mean
  value: '60.80'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_3b_none_multilingual
  evaluation_id: ev_3b_none_multilingual
  quantity: general_utility
  coverage: direct
  metric_as_printed: Multilingual category mean
  value: '59.89'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_3b_none_safety
  evaluation_id: ev_3b_none_safety
  quantity: other
  coverage: direct
  metric_as_printed: Safety category mean
  value: '80.89'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_3b_llmon_general
  evaluation_id: ev_3b_llmon_general
  quantity: general_utility
  coverage: direct
  metric_as_printed: General category mean
  value: '57.16'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_3b_llmon_alignment
  evaluation_id: ev_3b_llmon_alignment
  quantity: general_utility
  coverage: direct
  metric_as_printed: Alignment category mean
  value: '81.02'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_3b_llmon_math
  evaluation_id: ev_3b_llmon_math
  quantity: general_utility
  coverage: direct
  metric_as_printed: Math category mean
  value: '73.25'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_3b_llmon_code
  evaluation_id: ev_3b_llmon_code
  quantity: general_utility
  coverage: direct
  metric_as_printed: Code category mean
  value: '65.45'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_3b_llmon_tool-calling
  evaluation_id: ev_3b_llmon_tool-calling
  quantity: general_utility
  coverage: direct
  metric_as_printed: Tool-Calling category mean
  value: '60.71'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_3b_llmon_multilingual
  evaluation_id: ev_3b_llmon_multilingual
  quantity: general_utility
  coverage: direct
  metric_as_printed: Multilingual category mean
  value: '62.68'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_3b_llmon_safety
  evaluation_id: ev_3b_llmon_safety
  quantity: other
  coverage: direct
  metric_as_printed: Safety category mean
  value: '89.64'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_8b_none_general
  evaluation_id: ev_8b_none_general
  quantity: general_utility
  coverage: direct
  metric_as_printed: General category mean
  value: '64.95'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_8b_none_alignment
  evaluation_id: ev_8b_none_alignment
  quantity: general_utility
  coverage: direct
  metric_as_printed: Alignment category mean
  value: '87.06'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_8b_none_math
  evaluation_id: ev_8b_none_math
  quantity: general_utility
  coverage: direct
  metric_as_printed: Math category mean
  value: '84.09'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_8b_none_code
  evaluation_id: ev_8b_none_code
  quantity: general_utility
  coverage: direct
  metric_as_printed: Code category mean
  value: '72.38'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_8b_none_tool-calling
  evaluation_id: ev_8b_none_tool-calling
  quantity: general_utility
  coverage: direct
  metric_as_printed: Tool-Calling category mean
  value: '68.27'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_8b_none_multilingual
  evaluation_id: ev_8b_none_multilingual
  quantity: general_utility
  coverage: direct
  metric_as_printed: Multilingual category mean
  value: '68.68'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_8b_none_safety
  evaluation_id: ev_8b_none_safety
  quantity: other
  coverage: direct
  metric_as_printed: Safety category mean
  value: '84.19'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_8b_llmon_general
  evaluation_id: ev_8b_llmon_general
  quantity: general_utility
  coverage: direct
  metric_as_printed: General category mean
  value: '63.31'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_8b_llmon_alignment
  evaluation_id: ev_8b_llmon_alignment
  quantity: general_utility
  coverage: direct
  metric_as_printed: Alignment category mean
  value: '86.69'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_8b_llmon_math
  evaluation_id: ev_8b_llmon_math
  quantity: general_utility
  coverage: direct
  metric_as_printed: Math category mean
  value: '79.63'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_8b_llmon_code
  evaluation_id: ev_8b_llmon_code
  quantity: general_utility
  coverage: direct
  metric_as_printed: Code category mean
  value: '71.01'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_8b_llmon_tool-calling
  evaluation_id: ev_8b_llmon_tool-calling
  quantity: general_utility
  coverage: direct
  metric_as_printed: Tool-Calling category mean
  value: '67.93'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_8b_llmon_multilingual
  evaluation_id: ev_8b_llmon_multilingual
  quantity: general_utility
  coverage: direct
  metric_as_printed: Multilingual category mean
  value: '68.24'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_8b_llmon_safety
  evaluation_id: ev_8b_llmon_safety
  quantity: other
  coverage: direct
  metric_as_printed: Safety category mean
  value: '90.36'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
comparisons:
- id: c_3b_opi
  left: ev_3b_llmon_opi
  right: ev_3b_none_opi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_3b_asb
  left: ev_3b_llmon_asb
  right: ev_3b_none_asb
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_8b_opi
  left: ev_8b_llmon_opi
  right: ev_8b_none_opi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_8b_asb
  left: ev_8b_llmon_asb
  right: ev_8b_none_asb
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_3b_ares
  left: ev_3b_llmon_ares
  right: ev_3b_none_ares
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_8b_ares
  left: ev_8b_llmon_ares
  right: ev_8b_none_ares
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_3b_general
  left: ev_3b_llmon_general
  right: ev_3b_none_general
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_3b_alignment
  left: ev_3b_llmon_alignment
  right: ev_3b_none_alignment
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_3b_math
  left: ev_3b_llmon_math
  right: ev_3b_none_math
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_3b_code
  left: ev_3b_llmon_code
  right: ev_3b_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_3b_tool-calling
  left: ev_3b_llmon_tool-calling
  right: ev_3b_none_tool-calling
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_3b_multilingual
  left: ev_3b_llmon_multilingual
  right: ev_3b_none_multilingual
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_3b_safety
  left: ev_3b_llmon_safety
  right: ev_3b_none_safety
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_8b_general
  left: ev_8b_llmon_general
  right: ev_8b_none_general
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_8b_alignment
  left: ev_8b_llmon_alignment
  right: ev_8b_none_alignment
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_8b_math
  left: ev_8b_llmon_math
  right: ev_8b_none_math
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_8b_code
  left: ev_8b_llmon_code
  right: ev_8b_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_8b_tool-calling
  left: ev_8b_llmon_tool-calling
  right: ev_8b_none_tool-calling
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_8b_multilingual
  left: ev_8b_llmon_multilingual
  right: ev_8b_none_multilingual
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_8b_safety
  left: ev_8b_llmon_safety
  right: ev_8b_none_safety
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
claims:
- id: k1
  text: LLMON-structured supervision cuts ASR on indirect prompt-injection benchmarks from 53-100% to 0.02-5.6% while preserving general capabilities
  location: abstract
  anchor: attack success rates on indirect prompt-injection benchmarks from 53-100%
  jobs:
  - R1
  - general_utility
  supported_by:
  - c_3b_opi
  - c_3b_asb
  - c_3b_ares
  - c_3b_general
  - c_3b_alignment
  - c_3b_math
  - c_3b_code
  - c_3b_tool-calling
  - c_3b_multilingual
  - c_8b_opi
  - c_8b_asb
  - c_8b_ares
  - c_8b_general
  - c_8b_alignment
  - c_8b_math
  - c_8b_code
  - c_8b_tool-calling
  - c_8b_multilingual
  page: 1
  contradicted_by:
  - r_8b_llmon_math
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Tables 2 and 3
    general_utility: Table 1 (category means)
    R2clean: no task utility on the injection benchmarks
    R2attack: not reported
schema_gaps:
- where: evaluations ev_*_none_*
  note: the Instruct baselines are trained by the authors on the matched chat-template corpus; the paper does not say whether they also received the 130k planted-injection examples, the GRPO stages or the final DPO stage
- where: results r_*_ares
  note: Table 3 reports robustness (1 - ASR), the opposite direction to Table 2; ID and OOD values are given only for the LLMON models
- where: claims k1
  note: the 53% lower end is the 8B Instruct baseline on ASB; capability is a mean over six categories (unchanged at 3B, 1.4 points lower at 8B), with mathematics at 8B the largest drop (84.09 to 79.63)

`secpo_arxiv2026` (2610.41998)

In [ ]:
%%writefile reports/records/secpo_arxiv2026.yaml
report:
  id: secpo_arxiv2026
  slug: '2610.41998'
  title: 'SecPO: Principled Adversarial Training for Prompt Injection Security'
  first_author: Sizhe Chen
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2610.41998'
  version_read: arXiv v1
  first_date: '2026-10-30'
  read: F
  role: proposes
  code_release: code, training data and checkpoints released (Table 7)
  extraction_scope: 'Section 4 and Appendix C.8: Tables 1 to 4 and Table 18; the abstract claims (Table 5, an image channel, Table 6, a re-run of SecOPD with this training set, Tables 8 and 17, Figure 1 and the DTaP subsets
    of Section 4.3 not coded)'
defences:
- id: secpo_l
  name: SecPO loss with online adversarial training (Llama-3.1-8B)
  origin: this_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  - distillation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 2
  components:
  - 'SecPO loss: preferred response is the undefended model''s reply to the clean input, rejected is its reply to the injected instruction; the regulariser pulls the model on the injected input towards the undefended model
    on the clean input'
  - 'adversarial training: PAIR search (8 candidates, 30 rounds, best of 240) against the latest checkpoint for every training sample; attacker and judge are API models'
  - 19,157 Cleaned-Alpaca preference pairs; LoRA (rank 64)
  layout: 'Meta-SecAlign format: untrusted data in an added input role'
  layout_compat: &id001
    user_turn: untested
    separate_role: supported
    system_context: untested
    agent_tool_result: supported
  anchor: We combine SecPO with adversarial training
  pages:
    components: 5
    isolation: 10
- id: secpo_q
  name: SecPO loss with offline adversarial training (Qwen3.6-27B, reusing the injections optimised on Llama)
  origin: this_report
  base_checkpoint: Qwen3.6-27B (thinking off)
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  - distillation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  components:
  - 'SecPO loss: preferred response is the undefended model''s reply to the clean input, rejected is its reply to the injected instruction; the regulariser pulls the model on the injected input towards the undefended model
    on the clean input'
  - 'offline AT: the optimised injections saved from the Llama run, no new search'
  - 19,157 Cleaned-Alpaca preference pairs; LoRA (rank 64)
  layout: 'Meta-SecAlign format: untrusted data in an added input role'
  layout_compat: *id001
  anchor: Following Meta-SecAlign [9], we build our security preference dataset
  pages:
    isolation: 10
- id: msa_l
  name: Meta-SecAlign recipe (DPO, no adversarial training) trained by the authors
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 8
  layout: 'Meta-SecAlign format: untrusted data in an added input role'
  anchor: We first fine-tune
- id: atdpo_l
  name: adversarial training with the DPO loss (ablation)
  origin: this_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 10
  layout: 'Meta-SecAlign format: untrusted data in an added input role'
  anchor: SecPO loss and AT are both important
- id: msa_q
  name: Meta-SecAlign recipe (DPO, no adversarial training) trained by the authors
  origin: other_report
  base_checkpoint: Qwen3.6-27B (thinking off)
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 8
  layout: 'Meta-SecAlign format: untrusted data in an added input role'
  anchor: We first fine-tune
- id: atdpo_q
  name: adversarial training with the DPO loss (ablation)
  origin: this_report
  base_checkpoint: Qwen3.6-27B (thinking off)
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 10
  layout: 'Meta-SecAlign format: untrusted data in an added input role'
  anchor: SecPO loss and AT are both important
- id: secponoat_l
  name: SecPO loss with static simulated injections, no adversarial training (ablation)
  origin: this_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  - distillation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 27
  layout: 'Meta-SecAlign format: untrusted data in an added input role'
  anchor: White-box attack success rates (ASR
evaluations:
- id: ev_l_none_mmlupro
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: MMLU-Pro
  attacker_type: none
  attacker_access: not_applicable
  judge: answer checked against the reference
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_l_none_gpqa
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: GPQA Diamond
  attacker_type: none
  attacker_access: not_applicable
  judge: answer checked against the reference
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_l_none_gsm8k
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: GSM8K
  attacker_type: none
  attacker_access: not_applicable
  judge: answer checked against the reference
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_l_none_minerva
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: Minerva Math
  attacker_type: none
  attacker_access: not_applicable
  judge: normalised answer checker
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_l_none_alpaca
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: AlpacaEval2 (805 instructions, no injection)
  attacker_type: none
  attacker_access: not_applicable
  judge: Claude-Opus-4.8 compares each response with a fixed reference (GPT-4o-2024-05-13 references)
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: '805'
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_l_none_seputil
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: SEP (9,160 examples, no injection)
  attacker_type: none
  attacker_access: not_applicable
  judge: Claude-Opus-4.8 compares each response with a fixed reference (Meta-Llama-3-8B-Instruct references)
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: 9,160
  decoding: greedy
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_l_none_sepasr
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: SEP, six static injection templates around the data (any success counts)
  attacker_type: static
  attacker_access: black_box
  judge: nine verification calls to three LLM judges (Gemini-3.1-Flash-Lite-Preview, Gemini-3-Flash-Preview, GPT-5); success only if all nine confirm
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: 9,160
  decoding: greedy
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_l_none_sepgen
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: SEP, Genetic search attack (island GA with MAP-Elites)
  attacker_type: adaptive
  attacker_access: black_box
  judge: nine verification calls to three LLM judges (Gemini-3.1-Flash-Lite-Preview, Gemini-3-Flash-Preview, GPT-5); success only if all nine confirm
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  attacker_budget: up to 800 candidates per sample (Pass@800)
  sample_size: 1,024
  decoding: greedy
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_l_none_septap
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: SEP, TAP search attack
  attacker_type: adaptive
  attacker_access: black_box
  judge: nine verification calls to three LLM judges (Gemini-3.1-Flash-Lite-Preview, Gemini-3-Flash-Preview, GPT-5); success only if all nine confirm
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  attacker_budget: up to 800 candidates per sample (Pass@800)
  sample_size: 1,024
  decoding: greedy
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_l_none_seppis
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: SEP, PISmith RL attacker trained against each checkpoint on PIArena Dolly Closed-QA
  attacker_type: rl
  attacker_access: black_box
  judge: nine verification calls to three LLM judges (Gemini-3.1-Flash-Lite-Preview, Gemini-3-Flash-Preview, GPT-5); success only if all nine confirm
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  attacker_budget: 10 sampled injections per example (pass@10)
  sample_size: 1,024
  decoding: greedy
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_l_msa_mmlupro
  defence_id: msa_l
  benchmark: MMLU-Pro
  attacker_type: none
  attacker_access: not_applicable
  judge: answer checked against the reference
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_msa_gpqa
  defence_id: msa_l
  benchmark: GPQA Diamond
  attacker_type: none
  attacker_access: not_applicable
  judge: answer checked against the reference
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_msa_gsm8k
  defence_id: msa_l
  benchmark: GSM8K
  attacker_type: none
  attacker_access: not_applicable
  judge: answer checked against the reference
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_msa_minerva
  defence_id: msa_l
  benchmark: Minerva Math
  attacker_type: none
  attacker_access: not_applicable
  judge: normalised answer checker
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_msa_alpaca
  defence_id: msa_l
  benchmark: AlpacaEval2 (805 instructions, no injection)
  attacker_type: none
  attacker_access: not_applicable
  judge: Claude-Opus-4.8 compares each response with a fixed reference (GPT-4o-2024-05-13 references)
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: '805'
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_msa_seputil
  defence_id: msa_l
  benchmark: SEP (9,160 examples, no injection)
  attacker_type: none
  attacker_access: not_applicable
  judge: Claude-Opus-4.8 compares each response with a fixed reference (Meta-Llama-3-8B-Instruct references)
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: 9,160
  decoding: greedy
  pages:
    benchmark: 7
- id: ev_l_msa_sepasr
  defence_id: msa_l
  benchmark: SEP, six static injection templates around the data (any success counts)
  attacker_type: static
  attacker_access: black_box
  judge: nine verification calls to three LLM judges (Gemini-3.1-Flash-Lite-Preview, Gemini-3-Flash-Preview, GPT-5); success only if all nine confirm
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: 9,160
  decoding: greedy
  pages:
    benchmark: 7
- id: ev_l_msa_sepgen
  defence_id: msa_l
  benchmark: SEP, Genetic search attack (island GA with MAP-Elites)
  attacker_type: adaptive
  attacker_access: black_box
  judge: nine verification calls to three LLM judges (Gemini-3.1-Flash-Lite-Preview, Gemini-3-Flash-Preview, GPT-5); success only if all nine confirm
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  attacker_budget: up to 800 candidates per sample (Pass@800)
  sample_size: 1,024
  decoding: greedy
  pages:
    benchmark: 7
- id: ev_l_msa_septap
  defence_id: msa_l
  benchmark: SEP, TAP search attack
  attacker_type: adaptive
  attacker_access: black_box
  judge: nine verification calls to three LLM judges (Gemini-3.1-Flash-Lite-Preview, Gemini-3-Flash-Preview, GPT-5); success only if all nine confirm
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  attacker_budget: up to 800 candidates per sample (Pass@800)
  sample_size: 1,024
  decoding: greedy
  pages:
    benchmark: 7
- id: ev_l_msa_seppis
  defence_id: msa_l
  benchmark: SEP, PISmith RL attacker trained against each checkpoint on PIArena Dolly Closed-QA
  attacker_type: rl
  attacker_access: black_box
  judge: nine verification calls to three LLM judges (Gemini-3.1-Flash-Lite-Preview, Gemini-3-Flash-Preview, GPT-5); success only if all nine confirm
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  attacker_budget: 10 sampled injections per example (pass@10)
  sample_size: 1,024
  decoding: greedy
  pages:
    benchmark: 7
- id: ev_l_secpo_mmlupro
  defence_id: secpo_l
  benchmark: MMLU-Pro
  attacker_type: none
  attacker_access: not_applicable
  judge: answer checked against the reference
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_secpo_gpqa
  defence_id: secpo_l
  benchmark: GPQA Diamond
  attacker_type: none
  attacker_access: not_applicable
  judge: answer checked against the reference
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_secpo_gsm8k
  defence_id: secpo_l
  benchmark: GSM8K
  attacker_type: none
  attacker_access: not_applicable
  judge: answer checked against the reference
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_secpo_minerva
  defence_id: secpo_l
  benchmark: Minerva Math
  attacker_type: none
  attacker_access: not_applicable
  judge: normalised answer checker
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_secpo_alpaca
  defence_id: secpo_l
  benchmark: AlpacaEval2 (805 instructions, no injection)
  attacker_type: none
  attacker_access: not_applicable
  judge: Claude-Opus-4.8 compares each response with a fixed reference (GPT-4o-2024-05-13 references)
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: '805'
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_secpo_seputil
  defence_id: secpo_l
  benchmark: SEP (9,160 examples, no injection)
  attacker_type: none
  attacker_access: not_applicable
  judge: Claude-Opus-4.8 compares each response with a fixed reference (Meta-Llama-3-8B-Instruct references)
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: 9,160
  decoding: greedy
  pages:
    benchmark: 7
- id: ev_l_secpo_sepasr
  defence_id: secpo_l
  benchmark: SEP, six static injection templates around the data (any success counts)
  attacker_type: static
  attacker_access: black_box
  judge: nine verification calls to three LLM judges (Gemini-3.1-Flash-Lite-Preview, Gemini-3-Flash-Preview, GPT-5); success only if all nine confirm
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: 9,160
  decoding: greedy
  pages:
    benchmark: 7
- id: ev_l_secpo_sepgen
  defence_id: secpo_l
  benchmark: SEP, Genetic search attack (island GA with MAP-Elites)
  attacker_type: adaptive
  attacker_access: black_box
  judge: nine verification calls to three LLM judges (Gemini-3.1-Flash-Lite-Preview, Gemini-3-Flash-Preview, GPT-5); success only if all nine confirm
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  attacker_budget: up to 800 candidates per sample (Pass@800)
  sample_size: 1,024
  decoding: greedy
  pages:
    benchmark: 7
- id: ev_l_secpo_septap
  defence_id: secpo_l
  benchmark: SEP, TAP search attack
  attacker_type: adaptive
  attacker_access: black_box
  judge: nine verification calls to three LLM judges (Gemini-3.1-Flash-Lite-Preview, Gemini-3-Flash-Preview, GPT-5); success only if all nine confirm
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  attacker_budget: up to 800 candidates per sample (Pass@800)
  sample_size: 1,024
  decoding: greedy
  pages:
    benchmark: 7
- id: ev_l_secpo_seppis
  defence_id: secpo_l
  benchmark: SEP, PISmith RL attacker trained against each checkpoint on PIArena Dolly Closed-QA
  attacker_type: rl
  attacker_access: black_box
  judge: nine verification calls to three LLM judges (Gemini-3.1-Flash-Lite-Preview, Gemini-3-Flash-Preview, GPT-5); success only if all nine confirm
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  attacker_budget: 10 sampled injections per example (pass@10)
  sample_size: 1,024
  decoding: greedy
  pages:
    benchmark: 7
- id: ev_q_none_mmlupro
  defence_id: none:Qwen3.6-27B (thinking off)
  benchmark: MMLU-Pro
  attacker_type: none
  attacker_access: not_applicable
  judge: answer checked against the reference
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_q_none_gpqa
  defence_id: none:Qwen3.6-27B (thinking off)
  benchmark: GPQA Diamond
  attacker_type: none
  attacker_access: not_applicable
  judge: answer checked against the reference
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_q_none_gsm8k
  defence_id: none:Qwen3.6-27B (thinking off)
  benchmark: GSM8K
  attacker_type: none
  attacker_access: not_applicable
  judge: answer checked against the reference
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_q_none_minerva
  defence_id: none:Qwen3.6-27B (thinking off)
  benchmark: Minerva Math
  attacker_type: none
  attacker_access: not_applicable
  judge: normalised answer checker
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_q_none_alpaca
  defence_id: none:Qwen3.6-27B (thinking off)
  benchmark: AlpacaEval2 (805 instructions, no injection)
  attacker_type: none
  attacker_access: not_applicable
  judge: Claude-Opus-4.8 compares each response with a fixed reference (GPT-4o-2024-05-13 references)
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: '805'
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_q_none_seputil
  defence_id: none:Qwen3.6-27B (thinking off)
  benchmark: SEP (9,160 examples, no injection)
  attacker_type: none
  attacker_access: not_applicable
  judge: Claude-Opus-4.8 compares each response with a fixed reference (Meta-Llama-3-8B-Instruct references)
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: 9,160
  decoding: greedy
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_q_none_adutil
  defence_id: none:Qwen3.6-27B (thinking off)
  benchmark: AgentDojo, 97 user tasks, no injection (repeat-user-prompt defence on every arm)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo user-task check
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: '97'
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_q_none_dynutil
  defence_id: none:Qwen3.6-27B (thinking off)
  benchmark: AgentDyn, 60 benign tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDyn user-task check
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: '60'
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_q_none_dtaputil
  defence_id: none:Qwen3.6-27B (thinking off)
  benchmark: DTaP-Bench-Indirect, 1,838 samples (Qwen-Agent harness)
  attacker_type: static
  attacker_access: black_box
  judge: DTaP task check under attack
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: 1,838
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_q_none_sepasr
  defence_id: none:Qwen3.6-27B (thinking off)
  benchmark: SEP, six static injection templates around the data (any success counts)
  attacker_type: static
  attacker_access: black_box
  judge: nine verification calls to three LLM judges (Gemini-3.1-Flash-Lite-Preview, Gemini-3-Flash-Preview, GPT-5); success only if all nine confirm
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: 9,160
  decoding: greedy
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_q_none_sepgen
  defence_id: none:Qwen3.6-27B (thinking off)
  benchmark: SEP, Genetic search attack (island GA with MAP-Elites)
  attacker_type: adaptive
  attacker_access: black_box
  judge: nine verification calls to three LLM judges (Gemini-3.1-Flash-Lite-Preview, Gemini-3-Flash-Preview, GPT-5); success only if all nine confirm
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  attacker_budget: up to 800 candidates per sample (Pass@800)
  sample_size: 1,024
  decoding: greedy
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_q_none_septap
  defence_id: none:Qwen3.6-27B (thinking off)
  benchmark: SEP, TAP search attack
  attacker_type: adaptive
  attacker_access: black_box
  judge: nine verification calls to three LLM judges (Gemini-3.1-Flash-Lite-Preview, Gemini-3-Flash-Preview, GPT-5); success only if all nine confirm
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  attacker_budget: up to 800 candidates per sample (Pass@800)
  sample_size: 1,024
  decoding: greedy
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_q_none_seppis
  defence_id: none:Qwen3.6-27B (thinking off)
  benchmark: SEP, PISmith RL attacker trained against each checkpoint on PIArena Dolly Closed-QA
  attacker_type: rl
  attacker_access: black_box
  judge: nine verification calls to three LLM judges (Gemini-3.1-Flash-Lite-Preview, Gemini-3-Flash-Preview, GPT-5); success only if all nine confirm
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  attacker_budget: 10 sampled injections per example (pass@10)
  sample_size: 1,024
  decoding: greedy
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_q_none_adasr
  defence_id: none:Qwen3.6-27B (thinking off)
  benchmark: AgentDojo, important_instructions, 949 pairs (repeat-user-prompt defence on every arm)
  attacker_type: static
  attacker_access: black_box
  judge: malicious tool called
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: '949'
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_q_none_adgen
  defence_id: none:Qwen3.6-27B (thinking off)
  benchmark: AgentDojo, Genetic search attack (Attacker Moves Second protocol, Gemini-2.5-Pro attacker), 80-example subset
  attacker_type: adaptive
  attacker_access: black_box
  judge: malicious tool called
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  attacker_budget: up to 800 attempts per task (ASR@800)
  sample_size: '80'
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_q_none_dynasr
  defence_id: none:Qwen3.6-27B (thinking off)
  benchmark: AgentDyn, 560 static injected cases
  attacker_type: static
  attacker_access: black_box
  judge: AgentDyn security check
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: '560'
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_q_none_dtapasr
  defence_id: none:Qwen3.6-27B (thinking off)
  benchmark: DTaP-Bench-Indirect, 1,838 samples (Qwen-Agent harness)
  attacker_type: static
  attacker_access: black_box
  judge: DTaP attack check
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: 1,838
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_q_msa_mmlupro
  defence_id: msa_q
  benchmark: MMLU-Pro
  attacker_type: none
  attacker_access: not_applicable
  judge: answer checked against the reference
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_msa_gpqa
  defence_id: msa_q
  benchmark: GPQA Diamond
  attacker_type: none
  attacker_access: not_applicable
  judge: answer checked against the reference
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_msa_gsm8k
  defence_id: msa_q
  benchmark: GSM8K
  attacker_type: none
  attacker_access: not_applicable
  judge: answer checked against the reference
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_msa_minerva
  defence_id: msa_q
  benchmark: Minerva Math
  attacker_type: none
  attacker_access: not_applicable
  judge: normalised answer checker
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_msa_alpaca
  defence_id: msa_q
  benchmark: AlpacaEval2 (805 instructions, no injection)
  attacker_type: none
  attacker_access: not_applicable
  judge: Claude-Opus-4.8 compares each response with a fixed reference (GPT-4o-2024-05-13 references)
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: '805'
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_msa_seputil
  defence_id: msa_q
  benchmark: SEP (9,160 examples, no injection)
  attacker_type: none
  attacker_access: not_applicable
  judge: Claude-Opus-4.8 compares each response with a fixed reference (Meta-Llama-3-8B-Instruct references)
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: 9,160
  decoding: greedy
  pages:
    benchmark: 7
- id: ev_q_msa_adutil
  defence_id: msa_q
  benchmark: AgentDojo, 97 user tasks, no injection (repeat-user-prompt defence on every arm)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo user-task check
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: '97'
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_msa_dynutil
  defence_id: msa_q
  benchmark: AgentDyn, 60 benign tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDyn user-task check
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: '60'
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_msa_dtaputil
  defence_id: msa_q
  benchmark: DTaP-Bench-Indirect, 1,838 samples (Qwen-Agent harness)
  attacker_type: static
  attacker_access: black_box
  judge: DTaP task check under attack
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: 1,838
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_msa_sepasr
  defence_id: msa_q
  benchmark: SEP, six static injection templates around the data (any success counts)
  attacker_type: static
  attacker_access: black_box
  judge: nine verification calls to three LLM judges (Gemini-3.1-Flash-Lite-Preview, Gemini-3-Flash-Preview, GPT-5); success only if all nine confirm
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: 9,160
  decoding: greedy
  pages:
    benchmark: 7
- id: ev_q_msa_sepgen
  defence_id: msa_q
  benchmark: SEP, Genetic search attack (island GA with MAP-Elites)
  attacker_type: adaptive
  attacker_access: black_box
  judge: nine verification calls to three LLM judges (Gemini-3.1-Flash-Lite-Preview, Gemini-3-Flash-Preview, GPT-5); success only if all nine confirm
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  attacker_budget: up to 800 candidates per sample (Pass@800)
  sample_size: 1,024
  decoding: greedy
  pages:
    benchmark: 7
- id: ev_q_msa_septap
  defence_id: msa_q
  benchmark: SEP, TAP search attack
  attacker_type: adaptive
  attacker_access: black_box
  judge: nine verification calls to three LLM judges (Gemini-3.1-Flash-Lite-Preview, Gemini-3-Flash-Preview, GPT-5); success only if all nine confirm
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  attacker_budget: up to 800 candidates per sample (Pass@800)
  sample_size: 1,024
  decoding: greedy
  pages:
    benchmark: 7
- id: ev_q_msa_seppis
  defence_id: msa_q
  benchmark: SEP, PISmith RL attacker trained against each checkpoint on PIArena Dolly Closed-QA
  attacker_type: rl
  attacker_access: black_box
  judge: nine verification calls to three LLM judges (Gemini-3.1-Flash-Lite-Preview, Gemini-3-Flash-Preview, GPT-5); success only if all nine confirm
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  attacker_budget: 10 sampled injections per example (pass@10)
  sample_size: 1,024
  decoding: greedy
  pages:
    benchmark: 7
- id: ev_q_msa_adasr
  defence_id: msa_q
  benchmark: AgentDojo, important_instructions, 949 pairs (repeat-user-prompt defence on every arm)
  attacker_type: static
  attacker_access: black_box
  judge: malicious tool called
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: '949'
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_msa_adgen
  defence_id: msa_q
  benchmark: AgentDojo, Genetic search attack (Attacker Moves Second protocol, Gemini-2.5-Pro attacker), 80-example subset
  attacker_type: adaptive
  attacker_access: black_box
  judge: malicious tool called
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  attacker_budget: up to 800 attempts per task (ASR@800)
  sample_size: '80'
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_msa_dynasr
  defence_id: msa_q
  benchmark: AgentDyn, 560 static injected cases
  attacker_type: static
  attacker_access: black_box
  judge: AgentDyn security check
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: '560'
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_msa_dtapasr
  defence_id: msa_q
  benchmark: DTaP-Bench-Indirect, 1,838 samples (Qwen-Agent harness)
  attacker_type: static
  attacker_access: black_box
  judge: DTaP attack check
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: 1,838
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_secpo_mmlupro
  defence_id: secpo_q
  benchmark: MMLU-Pro
  attacker_type: none
  attacker_access: not_applicable
  judge: answer checked against the reference
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_secpo_gpqa
  defence_id: secpo_q
  benchmark: GPQA Diamond
  attacker_type: none
  attacker_access: not_applicable
  judge: answer checked against the reference
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_secpo_gsm8k
  defence_id: secpo_q
  benchmark: GSM8K
  attacker_type: none
  attacker_access: not_applicable
  judge: answer checked against the reference
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_secpo_minerva
  defence_id: secpo_q
  benchmark: Minerva Math
  attacker_type: none
  attacker_access: not_applicable
  judge: normalised answer checker
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_secpo_alpaca
  defence_id: secpo_q
  benchmark: AlpacaEval2 (805 instructions, no injection)
  attacker_type: none
  attacker_access: not_applicable
  judge: Claude-Opus-4.8 compares each response with a fixed reference (GPT-4o-2024-05-13 references)
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: '805'
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_secpo_seputil
  defence_id: secpo_q
  benchmark: SEP (9,160 examples, no injection)
  attacker_type: none
  attacker_access: not_applicable
  judge: Claude-Opus-4.8 compares each response with a fixed reference (Meta-Llama-3-8B-Instruct references)
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: 9,160
  decoding: greedy
  pages:
    benchmark: 7
- id: ev_q_secpo_adutil
  defence_id: secpo_q
  benchmark: AgentDojo, 97 user tasks, no injection (repeat-user-prompt defence on every arm)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo user-task check
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: '97'
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_secpo_dynutil
  defence_id: secpo_q
  benchmark: AgentDyn, 60 benign tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDyn user-task check
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: '60'
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_secpo_dtaputil
  defence_id: secpo_q
  benchmark: DTaP-Bench-Indirect, 1,838 samples (Qwen-Agent harness)
  attacker_type: static
  attacker_access: black_box
  judge: DTaP task check under attack
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: 1,838
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_secpo_sepasr
  defence_id: secpo_q
  benchmark: SEP, six static injection templates around the data (any success counts)
  attacker_type: static
  attacker_access: black_box
  judge: nine verification calls to three LLM judges (Gemini-3.1-Flash-Lite-Preview, Gemini-3-Flash-Preview, GPT-5); success only if all nine confirm
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: 9,160
  decoding: greedy
  pages:
    benchmark: 7
- id: ev_q_secpo_sepgen
  defence_id: secpo_q
  benchmark: SEP, Genetic search attack (island GA with MAP-Elites)
  attacker_type: adaptive
  attacker_access: black_box
  judge: nine verification calls to three LLM judges (Gemini-3.1-Flash-Lite-Preview, Gemini-3-Flash-Preview, GPT-5); success only if all nine confirm
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  attacker_budget: up to 800 candidates per sample (Pass@800)
  sample_size: 1,024
  decoding: greedy
  pages:
    benchmark: 7
- id: ev_q_secpo_septap
  defence_id: secpo_q
  benchmark: SEP, TAP search attack
  attacker_type: adaptive
  attacker_access: black_box
  judge: nine verification calls to three LLM judges (Gemini-3.1-Flash-Lite-Preview, Gemini-3-Flash-Preview, GPT-5); success only if all nine confirm
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  attacker_budget: up to 800 candidates per sample (Pass@800)
  sample_size: 1,024
  decoding: greedy
  pages:
    benchmark: 7
- id: ev_q_secpo_seppis
  defence_id: secpo_q
  benchmark: SEP, PISmith RL attacker trained against each checkpoint on PIArena Dolly Closed-QA
  attacker_type: rl
  attacker_access: black_box
  judge: nine verification calls to three LLM judges (Gemini-3.1-Flash-Lite-Preview, Gemini-3-Flash-Preview, GPT-5); success only if all nine confirm
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  attacker_budget: 10 sampled injections per example (pass@10)
  sample_size: 1,024
  decoding: greedy
  pages:
    benchmark: 7
- id: ev_q_secpo_adasr
  defence_id: secpo_q
  benchmark: AgentDojo, important_instructions, 949 pairs (repeat-user-prompt defence on every arm)
  attacker_type: static
  attacker_access: black_box
  judge: malicious tool called
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: '949'
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_secpo_adgen
  defence_id: secpo_q
  benchmark: AgentDojo, Genetic search attack (Attacker Moves Second protocol, Gemini-2.5-Pro attacker), 80-example subset
  attacker_type: adaptive
  attacker_access: black_box
  judge: malicious tool called
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  attacker_budget: up to 800 attempts per task (ASR@800)
  sample_size: '80'
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_secpo_dynasr
  defence_id: secpo_q
  benchmark: AgentDyn, 560 static injected cases
  attacker_type: static
  attacker_access: black_box
  judge: AgentDyn security check
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: '560'
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_secpo_dtapasr
  defence_id: secpo_q
  benchmark: DTaP-Bench-Indirect, 1,838 samples (Qwen-Agent harness)
  attacker_type: static
  attacker_access: black_box
  judge: DTaP attack check
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 8
  sample_size: 1,838
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_atdpo_mmlupro
  defence_id: atdpo_l
  benchmark: MMLU-Pro
  attacker_type: none
  attacker_access: not_applicable
  judge: answer checked against the reference
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_atdpo_gpqa
  defence_id: atdpo_l
  benchmark: GPQA Diamond
  attacker_type: none
  attacker_access: not_applicable
  judge: answer checked against the reference
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_atdpo_gsm8k
  defence_id: atdpo_l
  benchmark: GSM8K
  attacker_type: none
  attacker_access: not_applicable
  judge: answer checked against the reference
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_atdpo_minerva
  defence_id: atdpo_l
  benchmark: Minerva Math
  attacker_type: none
  attacker_access: not_applicable
  judge: normalised answer checker
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_atdpo_alpaca
  defence_id: atdpo_l
  benchmark: AlpacaEval2 (805 instructions, no injection)
  attacker_type: none
  attacker_access: not_applicable
  judge: Claude-Opus-4.8 compares each response with a fixed reference (GPT-4o-2024-05-13 references)
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 10
  sample_size: '805'
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_atdpo_seputil
  defence_id: atdpo_l
  benchmark: SEP (9,160 examples, no injection)
  attacker_type: none
  attacker_access: not_applicable
  judge: Claude-Opus-4.8 compares each response with a fixed reference (Meta-Llama-3-8B-Instruct references)
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 10
  sample_size: 9,160
  decoding: greedy
  pages:
    benchmark: 7
- id: ev_l_atdpo_sepgen
  defence_id: atdpo_l
  benchmark: SEP, Genetic search attack (island GA with MAP-Elites)
  attacker_type: adaptive
  attacker_access: black_box
  judge: nine verification calls to three LLM judges (Gemini-3.1-Flash-Lite-Preview, Gemini-3-Flash-Preview, GPT-5); success only if all nine confirm
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 10
  attacker_budget: up to 800 candidates per sample (Pass@800)
  sample_size: 1,024
  decoding: greedy
  pages:
    benchmark: 7
- id: ev_l_atdpo_septap
  defence_id: atdpo_l
  benchmark: SEP, TAP search attack
  attacker_type: adaptive
  attacker_access: black_box
  judge: nine verification calls to three LLM judges (Gemini-3.1-Flash-Lite-Preview, Gemini-3-Flash-Preview, GPT-5); success only if all nine confirm
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 10
  attacker_budget: up to 800 candidates per sample (Pass@800)
  sample_size: 1,024
  decoding: greedy
  pages:
    benchmark: 7
- id: ev_l_atdpo_seppis
  defence_id: atdpo_l
  benchmark: SEP, PISmith RL attacker trained against each checkpoint on PIArena Dolly Closed-QA
  attacker_type: rl
  attacker_access: black_box
  judge: nine verification calls to three LLM judges (Gemini-3.1-Flash-Lite-Preview, Gemini-3-Flash-Preview, GPT-5); success only if all nine confirm
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 10
  attacker_budget: 10 sampled injections per example (pass@10)
  sample_size: 1,024
  decoding: greedy
  pages:
    benchmark: 7
- id: ev_q_atdpo_mmlupro
  defence_id: atdpo_q
  benchmark: MMLU-Pro
  attacker_type: none
  attacker_access: not_applicable
  judge: answer checked against the reference
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_atdpo_gpqa
  defence_id: atdpo_q
  benchmark: GPQA Diamond
  attacker_type: none
  attacker_access: not_applicable
  judge: answer checked against the reference
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_atdpo_gsm8k
  defence_id: atdpo_q
  benchmark: GSM8K
  attacker_type: none
  attacker_access: not_applicable
  judge: answer checked against the reference
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_atdpo_minerva
  defence_id: atdpo_q
  benchmark: Minerva Math
  attacker_type: none
  attacker_access: not_applicable
  judge: normalised answer checker
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_atdpo_alpaca
  defence_id: atdpo_q
  benchmark: AlpacaEval2 (805 instructions, no injection)
  attacker_type: none
  attacker_access: not_applicable
  judge: Claude-Opus-4.8 compares each response with a fixed reference (GPT-4o-2024-05-13 references)
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 10
  sample_size: '805'
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_atdpo_seputil
  defence_id: atdpo_q
  benchmark: SEP (9,160 examples, no injection)
  attacker_type: none
  attacker_access: not_applicable
  judge: Claude-Opus-4.8 compares each response with a fixed reference (Meta-Llama-3-8B-Instruct references)
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 10
  sample_size: 9,160
  decoding: greedy
  pages:
    benchmark: 7
- id: ev_q_atdpo_sepgen
  defence_id: atdpo_q
  benchmark: SEP, Genetic search attack (island GA with MAP-Elites)
  attacker_type: adaptive
  attacker_access: black_box
  judge: nine verification calls to three LLM judges (Gemini-3.1-Flash-Lite-Preview, Gemini-3-Flash-Preview, GPT-5); success only if all nine confirm
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 10
  attacker_budget: up to 800 candidates per sample (Pass@800)
  sample_size: 1,024
  decoding: greedy
  pages:
    benchmark: 7
- id: ev_q_atdpo_septap
  defence_id: atdpo_q
  benchmark: SEP, TAP search attack
  attacker_type: adaptive
  attacker_access: black_box
  judge: nine verification calls to three LLM judges (Gemini-3.1-Flash-Lite-Preview, Gemini-3-Flash-Preview, GPT-5); success only if all nine confirm
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 10
  attacker_budget: up to 800 candidates per sample (Pass@800)
  sample_size: 1,024
  decoding: greedy
  pages:
    benchmark: 7
- id: ev_q_atdpo_seppis
  defence_id: atdpo_q
  benchmark: SEP, PISmith RL attacker trained against each checkpoint on PIArena Dolly Closed-QA
  attacker_type: rl
  attacker_access: black_box
  judge: nine verification calls to three LLM judges (Gemini-3.1-Flash-Lite-Preview, Gemini-3-Flash-Preview, GPT-5); success only if all nine confirm
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 10
  attacker_budget: 10 sampled injections per example (pass@10)
  sample_size: 1,024
  decoding: greedy
  pages:
    benchmark: 7
- id: ev_l_none_basic
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: Basic Adaptive white-box attack (delimiter-based, needs embedding access) (benchmark samples not stated)
  attacker_type: optimisation
  attacker_access: white_box
  judge: attack success (judge not restated)
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 27
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_l_msa_basic
  defence_id: msa_l
  benchmark: Basic Adaptive white-box attack (delimiter-based, needs embedding access) (benchmark samples not stated)
  attacker_type: optimisation
  attacker_access: white_box
  judge: attack success (judge not restated)
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 27
  decoding: not reported
- id: ev_l_secponoat_basic
  defence_id: secponoat_l
  benchmark: Basic Adaptive white-box attack (delimiter-based, needs embedding access) (benchmark samples not stated)
  attacker_type: optimisation
  attacker_access: white_box
  judge: attack success (judge not restated)
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 27
  decoding: not reported
- id: ev_l_atdpo_basic
  defence_id: atdpo_l
  benchmark: Basic Adaptive white-box attack (delimiter-based, needs embedding access) (benchmark samples not stated)
  attacker_type: optimisation
  attacker_access: white_box
  judge: attack success (judge not restated)
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 27
  decoding: not reported
- id: ev_l_secpo_basic
  defence_id: secpo_l
  benchmark: Basic Adaptive white-box attack (delimiter-based, needs embedding access) (benchmark samples not stated)
  attacker_type: optimisation
  attacker_access: white_box
  judge: attack success (judge not restated)
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 27
  decoding: not reported
- id: ev_l_none_gcg
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: GCG white-box suffix attack (benchmark samples not stated)
  attacker_type: optimisation
  attacker_access: white_box
  judge: attack success (judge not restated)
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 27
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_l_msa_gcg
  defence_id: msa_l
  benchmark: GCG white-box suffix attack (benchmark samples not stated)
  attacker_type: optimisation
  attacker_access: white_box
  judge: attack success (judge not restated)
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 27
  decoding: not reported
- id: ev_l_secponoat_gcg
  defence_id: secponoat_l
  benchmark: GCG white-box suffix attack (benchmark samples not stated)
  attacker_type: optimisation
  attacker_access: white_box
  judge: attack success (judge not restated)
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 27
  decoding: not reported
- id: ev_l_atdpo_gcg
  defence_id: atdpo_l
  benchmark: GCG white-box suffix attack (benchmark samples not stated)
  attacker_type: optimisation
  attacker_access: white_box
  judge: attack success (judge not restated)
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 27
  decoding: not reported
- id: ev_l_secpo_gcg
  defence_id: secpo_l
  benchmark: GCG white-box suffix attack (benchmark samples not stated)
  attacker_type: optimisation
  attacker_access: white_box
  judge: attack success (judge not restated)
  layout_same_as_defence: 'yes'
  measured_by: secpo_arxiv2026
  page: 27
  decoding: not reported
results:
- id: r_l_none_mmlupro
  evaluation_id: ev_l_none_mmlupro
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU-Pro accuracy
  value: '46.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_none_gpqa
  evaluation_id: ev_l_none_gpqa
  quantity: general_utility
  coverage: direct
  metric_as_printed: GPQA Diamond accuracy
  value: '23.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_none_gsm8k
  evaluation_id: ev_l_none_gsm8k
  quantity: general_utility
  coverage: direct
  metric_as_printed: GSM8K accuracy
  value: '82.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_none_minerva
  evaluation_id: ev_l_none_minerva
  quantity: general_utility
  coverage: direct
  metric_as_printed: Minerva Math accuracy
  value: '49.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_none_alpaca
  evaluation_id: ev_l_none_alpaca
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval2 accuracy
  value: '23.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_none_seputil
  evaluation_id: ev_l_none_seputil
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP utility
  value: '55.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_none_sepasr
  evaluation_id: ev_l_none_sepasr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '98.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_none_sepgen
  evaluation_id: ev_l_none_sepgen
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_none_septap
  evaluation_id: ev_l_none_septap
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_none_seppis
  evaluation_id: ev_l_none_seppis
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '98.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_msa_mmlupro
  evaluation_id: ev_l_msa_mmlupro
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU-Pro accuracy
  value: '46.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_msa_gpqa
  evaluation_id: ev_l_msa_gpqa
  quantity: general_utility
  coverage: direct
  metric_as_printed: GPQA Diamond accuracy
  value: '24.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_msa_gsm8k
  evaluation_id: ev_l_msa_gsm8k
  quantity: general_utility
  coverage: direct
  metric_as_printed: GSM8K accuracy
  value: '84.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_msa_minerva
  evaluation_id: ev_l_msa_minerva
  quantity: general_utility
  coverage: direct
  metric_as_printed: Minerva Math accuracy
  value: '48.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_msa_alpaca
  evaluation_id: ev_l_msa_alpaca
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval2 accuracy
  value: '24.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_msa_seputil
  evaluation_id: ev_l_msa_seputil
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP utility
  value: '55.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_msa_sepasr
  evaluation_id: ev_l_msa_sepasr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '1.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_msa_sepgen
  evaluation_id: ev_l_msa_sepgen
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '89.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_msa_septap
  evaluation_id: ev_l_msa_septap
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '84.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_msa_seppis
  evaluation_id: ev_l_msa_seppis
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '95.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_secpo_mmlupro
  evaluation_id: ev_l_secpo_mmlupro
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU-Pro accuracy
  value: '46.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_secpo_gpqa
  evaluation_id: ev_l_secpo_gpqa
  quantity: general_utility
  coverage: direct
  metric_as_printed: GPQA Diamond accuracy
  value: '24.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_secpo_gsm8k
  evaluation_id: ev_l_secpo_gsm8k
  quantity: general_utility
  coverage: direct
  metric_as_printed: GSM8K accuracy
  value: '81.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_secpo_minerva
  evaluation_id: ev_l_secpo_minerva
  quantity: general_utility
  coverage: direct
  metric_as_printed: Minerva Math accuracy
  value: '50.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_secpo_alpaca
  evaluation_id: ev_l_secpo_alpaca
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval2 accuracy
  value: '23.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_secpo_seputil
  evaluation_id: ev_l_secpo_seputil
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP utility
  value: '59.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_secpo_sepasr
  evaluation_id: ev_l_secpo_sepasr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '1.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_secpo_sepgen
  evaluation_id: ev_l_secpo_sepgen
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '12.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_secpo_septap
  evaluation_id: ev_l_secpo_septap
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '12.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_secpo_seppis
  evaluation_id: ev_l_secpo_seppis
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '7.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_none_mmlupro
  evaluation_id: ev_q_none_mmlupro
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU-Pro accuracy
  value: '84.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_none_gpqa
  evaluation_id: ev_q_none_gpqa
  quantity: general_utility
  coverage: direct
  metric_as_printed: GPQA Diamond accuracy
  value: '81.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_none_gsm8k
  evaluation_id: ev_q_none_gsm8k
  quantity: general_utility
  coverage: direct
  metric_as_printed: GSM8K accuracy
  value: '97.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_none_minerva
  evaluation_id: ev_q_none_minerva
  quantity: general_utility
  coverage: direct
  metric_as_printed: Minerva Math accuracy
  value: '96.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_none_alpaca
  evaluation_id: ev_q_none_alpaca
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval2 accuracy
  value: '80.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_none_seputil
  evaluation_id: ev_q_none_seputil
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP utility
  value: '93.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_none_adutil
  evaluation_id: ev_q_none_adutil
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDojo utility
  value: '89.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_none_dynutil
  evaluation_id: ev_q_none_dynutil
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDyn utility
  value: '71.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_none_dtaputil
  evaluation_id: ev_q_none_dtaputil
  quantity: R2attack
  coverage: direct
  metric_as_printed: DTaP-Bench-Indirect utility
  value: '65.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_none_sepasr
  evaluation_id: ev_q_none_sepasr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '98.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_none_sepgen
  evaluation_id: ev_q_none_sepgen
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_none_septap
  evaluation_id: ev_q_none_septap
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_none_seppis
  evaluation_id: ev_q_none_seppis
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '98.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_none_adasr
  evaluation_id: ev_q_none_adasr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '26.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_none_adgen
  evaluation_id: ev_q_none_adgen
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '81.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_none_dynasr
  evaluation_id: ev_q_none_dynasr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDyn ASR
  value: '22.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_none_dtapasr
  evaluation_id: ev_q_none_dtapasr
  quantity: R1
  coverage: direct
  metric_as_printed: DTaP-Bench-Indirect ASR
  value: '50.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_msa_mmlupro
  evaluation_id: ev_q_msa_mmlupro
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU-Pro accuracy
  value: '83.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_msa_gpqa
  evaluation_id: ev_q_msa_gpqa
  quantity: general_utility
  coverage: direct
  metric_as_printed: GPQA Diamond accuracy
  value: '79.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_msa_gsm8k
  evaluation_id: ev_q_msa_gsm8k
  quantity: general_utility
  coverage: direct
  metric_as_printed: GSM8K accuracy
  value: '97.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_msa_minerva
  evaluation_id: ev_q_msa_minerva
  quantity: general_utility
  coverage: direct
  metric_as_printed: Minerva Math accuracy
  value: '94.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_msa_alpaca
  evaluation_id: ev_q_msa_alpaca
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval2 accuracy
  value: '81.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_msa_seputil
  evaluation_id: ev_q_msa_seputil
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP utility
  value: '93.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_msa_adutil
  evaluation_id: ev_q_msa_adutil
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDojo utility
  value: '87.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_msa_dynutil
  evaluation_id: ev_q_msa_dynutil
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDyn utility
  value: '76.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_msa_dtaputil
  evaluation_id: ev_q_msa_dtaputil
  quantity: R2attack
  coverage: direct
  metric_as_printed: DTaP-Bench-Indirect utility
  value: '70.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_msa_sepasr
  evaluation_id: ev_q_msa_sepasr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '2.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_msa_sepgen
  evaluation_id: ev_q_msa_sepgen
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '93.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_msa_septap
  evaluation_id: ev_q_msa_septap
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '90.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_msa_seppis
  evaluation_id: ev_q_msa_seppis
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '94.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_msa_adasr
  evaluation_id: ev_q_msa_adasr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '2.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_msa_adgen
  evaluation_id: ev_q_msa_adgen
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '78.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_msa_dynasr
  evaluation_id: ev_q_msa_dynasr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDyn ASR
  value: '4.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_msa_dtapasr
  evaluation_id: ev_q_msa_dtapasr
  quantity: R1
  coverage: direct
  metric_as_printed: DTaP-Bench-Indirect ASR
  value: '45.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_secpo_mmlupro
  evaluation_id: ev_q_secpo_mmlupro
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU-Pro accuracy
  value: '84.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_secpo_gpqa
  evaluation_id: ev_q_secpo_gpqa
  quantity: general_utility
  coverage: direct
  metric_as_printed: GPQA Diamond accuracy
  value: '80.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_secpo_gsm8k
  evaluation_id: ev_q_secpo_gsm8k
  quantity: general_utility
  coverage: direct
  metric_as_printed: GSM8K accuracy
  value: '96.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_secpo_minerva
  evaluation_id: ev_q_secpo_minerva
  quantity: general_utility
  coverage: direct
  metric_as_printed: Minerva Math accuracy
  value: '95.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_secpo_alpaca
  evaluation_id: ev_q_secpo_alpaca
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval2 accuracy
  value: '82.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_secpo_seputil
  evaluation_id: ev_q_secpo_seputil
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP utility
  value: '94.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_secpo_adutil
  evaluation_id: ev_q_secpo_adutil
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDojo utility
  value: '90.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_secpo_dynutil
  evaluation_id: ev_q_secpo_dynutil
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDyn utility
  value: '71.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_secpo_dtaputil
  evaluation_id: ev_q_secpo_dtaputil
  quantity: R2attack
  coverage: direct
  metric_as_printed: DTaP-Bench-Indirect utility
  value: '74.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_secpo_sepasr
  evaluation_id: ev_q_secpo_sepasr
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '1.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_secpo_sepgen
  evaluation_id: ev_q_secpo_sepgen
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '13.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_secpo_septap
  evaluation_id: ev_q_secpo_septap
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '12.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_secpo_seppis
  evaluation_id: ev_q_secpo_seppis
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '1.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_secpo_adasr
  evaluation_id: ev_q_secpo_adasr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_secpo_adgen
  evaluation_id: ev_q_secpo_adgen
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '8.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_secpo_dynasr
  evaluation_id: ev_q_secpo_dynasr
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDyn ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q_secpo_dtapasr
  evaluation_id: ev_q_secpo_dtapasr
  quantity: R1
  coverage: direct
  metric_as_printed: DTaP-Bench-Indirect ASR
  value: '7.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_l_atdpo_mmlupro
  evaluation_id: ev_l_atdpo_mmlupro
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU-Pro accuracy
  value: '46.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
- id: r_l_atdpo_gpqa
  evaluation_id: ev_l_atdpo_gpqa
  quantity: general_utility
  coverage: direct
  metric_as_printed: GPQA Diamond accuracy
  value: '27.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
- id: r_l_atdpo_gsm8k
  evaluation_id: ev_l_atdpo_gsm8k
  quantity: general_utility
  coverage: direct
  metric_as_printed: GSM8K accuracy
  value: '82.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
- id: r_l_atdpo_minerva
  evaluation_id: ev_l_atdpo_minerva
  quantity: general_utility
  coverage: direct
  metric_as_printed: Minerva Math accuracy
  value: '50.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
- id: r_l_atdpo_alpaca
  evaluation_id: ev_l_atdpo_alpaca
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval2 accuracy
  value: '19.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
- id: r_l_atdpo_seputil
  evaluation_id: ev_l_atdpo_seputil
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP utility
  value: '56.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
- id: r_l_atdpo_sepgen
  evaluation_id: ev_l_atdpo_sepgen
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '29.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
- id: r_l_atdpo_septap
  evaluation_id: ev_l_atdpo_septap
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '31.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
- id: r_l_atdpo_seppis
  evaluation_id: ev_l_atdpo_seppis
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '23.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
- id: r_q_atdpo_mmlupro
  evaluation_id: ev_q_atdpo_mmlupro
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU-Pro accuracy
  value: '84.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 4
- id: r_q_atdpo_gpqa
  evaluation_id: ev_q_atdpo_gpqa
  quantity: general_utility
  coverage: direct
  metric_as_printed: GPQA Diamond accuracy
  value: '77.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 4
- id: r_q_atdpo_gsm8k
  evaluation_id: ev_q_atdpo_gsm8k
  quantity: general_utility
  coverage: direct
  metric_as_printed: GSM8K accuracy
  value: '96.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 4
- id: r_q_atdpo_minerva
  evaluation_id: ev_q_atdpo_minerva
  quantity: general_utility
  coverage: direct
  metric_as_printed: Minerva Math accuracy
  value: '95.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 4
- id: r_q_atdpo_alpaca
  evaluation_id: ev_q_atdpo_alpaca
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval2 accuracy
  value: '81.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 4
- id: r_q_atdpo_seputil
  evaluation_id: ev_q_atdpo_seputil
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP utility
  value: '94.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 4
- id: r_q_atdpo_sepgen
  evaluation_id: ev_q_atdpo_sepgen
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '60.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 4
- id: r_q_atdpo_septap
  evaluation_id: ev_q_atdpo_septap
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '46.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 4
- id: r_q_atdpo_seppis
  evaluation_id: ev_q_atdpo_seppis
  quantity: R1
  coverage: direct
  metric_as_printed: SEP ASR
  value: '6.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 4
- id: r_l_none_basic
  evaluation_id: ev_l_none_basic
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Basic Adaptive white-box attack (delimiter-based, needs embedding access)
  value: '97.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 27
  location: Table 18
- id: r_l_msa_basic
  evaluation_id: ev_l_msa_basic
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Basic Adaptive white-box attack (delimiter-based, needs embedding access)
  value: '33.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 27
  location: Table 18
- id: r_l_secponoat_basic
  evaluation_id: ev_l_secponoat_basic
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Basic Adaptive white-box attack (delimiter-based, needs embedding access)
  value: '6.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 27
  location: Table 18
- id: r_l_atdpo_basic
  evaluation_id: ev_l_atdpo_basic
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Basic Adaptive white-box attack (delimiter-based, needs embedding access)
  value: '9.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 27
  location: Table 18
- id: r_l_secpo_basic
  evaluation_id: ev_l_secpo_basic
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Basic Adaptive white-box attack (delimiter-based, needs embedding access)
  value: '10.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 27
  location: Table 18
- id: r_l_none_gcg
  evaluation_id: ev_l_none_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GCG white-box suffix attack
  value: '87.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 27
  location: Table 18
- id: r_l_msa_gcg
  evaluation_id: ev_l_msa_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GCG white-box suffix attack
  value: '20.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 27
  location: Table 18
- id: r_l_secponoat_gcg
  evaluation_id: ev_l_secponoat_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GCG white-box suffix attack
  value: '15.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 27
  location: Table 18
- id: r_l_atdpo_gcg
  evaluation_id: ev_l_atdpo_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GCG white-box suffix attack
  value: '8.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 27
  location: Table 18
- id: r_l_secpo_gcg
  evaluation_id: ev_l_secpo_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GCG white-box suffix attack
  value: '5.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 27
  location: Table 18
comparisons:
- id: c_l_mmlupro_secpo_none
  left: ev_l_secpo_mmlupro
  right: ev_l_none_mmlupro
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_mmlupro_msa_none
  left: ev_l_msa_mmlupro
  right: ev_l_none_mmlupro
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_mmlupro_secpo_msa
  left: ev_l_secpo_mmlupro
  right: ev_l_msa_mmlupro
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: SecPO with adversarial training against the Meta-SecAlign recipe on the same base'
  page: 8
- id: c_l_gpqa_secpo_none
  left: ev_l_secpo_gpqa
  right: ev_l_none_gpqa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_gpqa_msa_none
  left: ev_l_msa_gpqa
  right: ev_l_none_gpqa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_gpqa_secpo_msa
  left: ev_l_secpo_gpqa
  right: ev_l_msa_gpqa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: SecPO with adversarial training against the Meta-SecAlign recipe on the same base'
  page: 8
- id: c_l_gsm8k_secpo_none
  left: ev_l_secpo_gsm8k
  right: ev_l_none_gsm8k
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_gsm8k_msa_none
  left: ev_l_msa_gsm8k
  right: ev_l_none_gsm8k
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_gsm8k_secpo_msa
  left: ev_l_secpo_gsm8k
  right: ev_l_msa_gsm8k
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: SecPO with adversarial training against the Meta-SecAlign recipe on the same base'
  page: 8
- id: c_l_minerva_secpo_none
  left: ev_l_secpo_minerva
  right: ev_l_none_minerva
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_minerva_msa_none
  left: ev_l_msa_minerva
  right: ev_l_none_minerva
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_minerva_secpo_msa
  left: ev_l_secpo_minerva
  right: ev_l_msa_minerva
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: SecPO with adversarial training against the Meta-SecAlign recipe on the same base'
  page: 8
- id: c_l_alpaca_secpo_none
  left: ev_l_secpo_alpaca
  right: ev_l_none_alpaca
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_alpaca_msa_none
  left: ev_l_msa_alpaca
  right: ev_l_none_alpaca
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_alpaca_secpo_msa
  left: ev_l_secpo_alpaca
  right: ev_l_msa_alpaca
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: SecPO with adversarial training against the Meta-SecAlign recipe on the same base'
  page: 8
- id: c_l_seputil_secpo_none
  left: ev_l_secpo_seputil
  right: ev_l_none_seputil
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_seputil_msa_none
  left: ev_l_msa_seputil
  right: ev_l_none_seputil
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_seputil_secpo_msa
  left: ev_l_secpo_seputil
  right: ev_l_msa_seputil
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: SecPO with adversarial training against the Meta-SecAlign recipe on the same base'
  page: 8
- id: c_l_sepasr_secpo_none
  left: ev_l_secpo_sepasr
  right: ev_l_none_sepasr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_sepasr_msa_none
  left: ev_l_msa_sepasr
  right: ev_l_none_sepasr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_sepasr_secpo_msa
  left: ev_l_secpo_sepasr
  right: ev_l_msa_sepasr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: SecPO with adversarial training against the Meta-SecAlign recipe on the same base'
  page: 8
- id: c_l_sepgen_secpo_none
  left: ev_l_secpo_sepgen
  right: ev_l_none_sepgen
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_sepgen_msa_none
  left: ev_l_msa_sepgen
  right: ev_l_none_sepgen
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_sepgen_secpo_msa
  left: ev_l_secpo_sepgen
  right: ev_l_msa_sepgen
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: SecPO with adversarial training against the Meta-SecAlign recipe on the same base'
  page: 8
- id: c_l_septap_secpo_none
  left: ev_l_secpo_septap
  right: ev_l_none_septap
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_septap_msa_none
  left: ev_l_msa_septap
  right: ev_l_none_septap
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_septap_secpo_msa
  left: ev_l_secpo_septap
  right: ev_l_msa_septap
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: SecPO with adversarial training against the Meta-SecAlign recipe on the same base'
  page: 8
- id: c_l_seppis_secpo_none
  left: ev_l_secpo_seppis
  right: ev_l_none_seppis
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_seppis_msa_none
  left: ev_l_msa_seppis
  right: ev_l_none_seppis
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_seppis_secpo_msa
  left: ev_l_secpo_seppis
  right: ev_l_msa_seppis
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: SecPO with adversarial training against the Meta-SecAlign recipe on the same base'
  page: 8
- id: c_q_mmlupro_secpo_none
  left: ev_q_secpo_mmlupro
  right: ev_q_none_mmlupro
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_mmlupro_msa_none
  left: ev_q_msa_mmlupro
  right: ev_q_none_mmlupro
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_mmlupro_secpo_msa
  left: ev_q_secpo_mmlupro
  right: ev_q_msa_mmlupro
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: SecPO with adversarial training against the Meta-SecAlign recipe on the same base'
  page: 8
- id: c_q_gpqa_secpo_none
  left: ev_q_secpo_gpqa
  right: ev_q_none_gpqa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_gpqa_msa_none
  left: ev_q_msa_gpqa
  right: ev_q_none_gpqa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_gpqa_secpo_msa
  left: ev_q_secpo_gpqa
  right: ev_q_msa_gpqa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: SecPO with adversarial training against the Meta-SecAlign recipe on the same base'
  page: 8
- id: c_q_gsm8k_secpo_none
  left: ev_q_secpo_gsm8k
  right: ev_q_none_gsm8k
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_gsm8k_msa_none
  left: ev_q_msa_gsm8k
  right: ev_q_none_gsm8k
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_gsm8k_secpo_msa
  left: ev_q_secpo_gsm8k
  right: ev_q_msa_gsm8k
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: SecPO with adversarial training against the Meta-SecAlign recipe on the same base'
  page: 8
- id: c_q_minerva_secpo_none
  left: ev_q_secpo_minerva
  right: ev_q_none_minerva
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_minerva_msa_none
  left: ev_q_msa_minerva
  right: ev_q_none_minerva
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_minerva_secpo_msa
  left: ev_q_secpo_minerva
  right: ev_q_msa_minerva
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: SecPO with adversarial training against the Meta-SecAlign recipe on the same base'
  page: 8
- id: c_q_alpaca_secpo_none
  left: ev_q_secpo_alpaca
  right: ev_q_none_alpaca
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_alpaca_msa_none
  left: ev_q_msa_alpaca
  right: ev_q_none_alpaca
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_alpaca_secpo_msa
  left: ev_q_secpo_alpaca
  right: ev_q_msa_alpaca
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: SecPO with adversarial training against the Meta-SecAlign recipe on the same base'
  page: 8
- id: c_q_seputil_secpo_none
  left: ev_q_secpo_seputil
  right: ev_q_none_seputil
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_seputil_msa_none
  left: ev_q_msa_seputil
  right: ev_q_none_seputil
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_seputil_secpo_msa
  left: ev_q_secpo_seputil
  right: ev_q_msa_seputil
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: SecPO with adversarial training against the Meta-SecAlign recipe on the same base'
  page: 8
- id: c_q_adutil_secpo_none
  left: ev_q_secpo_adutil
  right: ev_q_none_adutil
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_adutil_msa_none
  left: ev_q_msa_adutil
  right: ev_q_none_adutil
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_adutil_secpo_msa
  left: ev_q_secpo_adutil
  right: ev_q_msa_adutil
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: SecPO with adversarial training against the Meta-SecAlign recipe on the same base'
  page: 8
- id: c_q_dynutil_secpo_none
  left: ev_q_secpo_dynutil
  right: ev_q_none_dynutil
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_dynutil_msa_none
  left: ev_q_msa_dynutil
  right: ev_q_none_dynutil
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_dynutil_secpo_msa
  left: ev_q_secpo_dynutil
  right: ev_q_msa_dynutil
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: SecPO with adversarial training against the Meta-SecAlign recipe on the same base'
  page: 8
- id: c_q_dtaputil_secpo_none
  left: ev_q_secpo_dtaputil
  right: ev_q_none_dtaputil
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_dtaputil_msa_none
  left: ev_q_msa_dtaputil
  right: ev_q_none_dtaputil
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_dtaputil_secpo_msa
  left: ev_q_secpo_dtaputil
  right: ev_q_msa_dtaputil
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: SecPO with adversarial training against the Meta-SecAlign recipe on the same base'
  page: 8
- id: c_q_sepasr_secpo_none
  left: ev_q_secpo_sepasr
  right: ev_q_none_sepasr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_sepasr_msa_none
  left: ev_q_msa_sepasr
  right: ev_q_none_sepasr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_sepasr_secpo_msa
  left: ev_q_secpo_sepasr
  right: ev_q_msa_sepasr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: SecPO with adversarial training against the Meta-SecAlign recipe on the same base'
  page: 8
- id: c_q_sepgen_secpo_none
  left: ev_q_secpo_sepgen
  right: ev_q_none_sepgen
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_sepgen_msa_none
  left: ev_q_msa_sepgen
  right: ev_q_none_sepgen
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_sepgen_secpo_msa
  left: ev_q_secpo_sepgen
  right: ev_q_msa_sepgen
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: SecPO with adversarial training against the Meta-SecAlign recipe on the same base'
  page: 8
- id: c_q_septap_secpo_none
  left: ev_q_secpo_septap
  right: ev_q_none_septap
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_septap_msa_none
  left: ev_q_msa_septap
  right: ev_q_none_septap
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_septap_secpo_msa
  left: ev_q_secpo_septap
  right: ev_q_msa_septap
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: SecPO with adversarial training against the Meta-SecAlign recipe on the same base'
  page: 8
- id: c_q_seppis_secpo_none
  left: ev_q_secpo_seppis
  right: ev_q_none_seppis
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_seppis_msa_none
  left: ev_q_msa_seppis
  right: ev_q_none_seppis
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_seppis_secpo_msa
  left: ev_q_secpo_seppis
  right: ev_q_msa_seppis
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: SecPO with adversarial training against the Meta-SecAlign recipe on the same base'
  page: 8
- id: c_q_adasr_secpo_none
  left: ev_q_secpo_adasr
  right: ev_q_none_adasr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_adasr_msa_none
  left: ev_q_msa_adasr
  right: ev_q_none_adasr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_adasr_secpo_msa
  left: ev_q_secpo_adasr
  right: ev_q_msa_adasr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: SecPO with adversarial training against the Meta-SecAlign recipe on the same base'
  page: 8
- id: c_q_adgen_secpo_none
  left: ev_q_secpo_adgen
  right: ev_q_none_adgen
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_adgen_msa_none
  left: ev_q_msa_adgen
  right: ev_q_none_adgen
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_adgen_secpo_msa
  left: ev_q_secpo_adgen
  right: ev_q_msa_adgen
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: SecPO with adversarial training against the Meta-SecAlign recipe on the same base'
  page: 8
- id: c_q_dynasr_secpo_none
  left: ev_q_secpo_dynasr
  right: ev_q_none_dynasr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_dynasr_msa_none
  left: ev_q_msa_dynasr
  right: ev_q_none_dynasr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_dynasr_secpo_msa
  left: ev_q_secpo_dynasr
  right: ev_q_msa_dynasr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: SecPO with adversarial training against the Meta-SecAlign recipe on the same base'
  page: 8
- id: c_q_dtapasr_secpo_none
  left: ev_q_secpo_dtapasr
  right: ev_q_none_dtapasr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_dtapasr_msa_none
  left: ev_q_msa_dtapasr
  right: ev_q_none_dtapasr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_dtapasr_secpo_msa
  left: ev_q_secpo_dtapasr
  right: ev_q_msa_dtapasr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the defence: SecPO with adversarial training against the Meta-SecAlign recipe on the same base'
  page: 8
- id: c_l_mmlupro_at
  left: ev_l_atdpo_mmlupro
  right: ev_l_msa_mmlupro
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: adversarial training added (DPO loss in both arms)
  page: 10
- id: c_l_mmlupro_loss
  left: ev_l_secpo_mmlupro
  right: ev_l_atdpo_mmlupro
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: SecPO loss in place of DPO (adversarial training in both arms)
  page: 10
- id: c_l_gpqa_at
  left: ev_l_atdpo_gpqa
  right: ev_l_msa_gpqa
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: adversarial training added (DPO loss in both arms)
  page: 10
- id: c_l_gpqa_loss
  left: ev_l_secpo_gpqa
  right: ev_l_atdpo_gpqa
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: SecPO loss in place of DPO (adversarial training in both arms)
  page: 10
- id: c_l_gsm8k_at
  left: ev_l_atdpo_gsm8k
  right: ev_l_msa_gsm8k
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: adversarial training added (DPO loss in both arms)
  page: 10
- id: c_l_gsm8k_loss
  left: ev_l_secpo_gsm8k
  right: ev_l_atdpo_gsm8k
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: SecPO loss in place of DPO (adversarial training in both arms)
  page: 10
- id: c_l_minerva_at
  left: ev_l_atdpo_minerva
  right: ev_l_msa_minerva
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: adversarial training added (DPO loss in both arms)
  page: 10
- id: c_l_minerva_loss
  left: ev_l_secpo_minerva
  right: ev_l_atdpo_minerva
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: SecPO loss in place of DPO (adversarial training in both arms)
  page: 10
- id: c_l_alpaca_at
  left: ev_l_atdpo_alpaca
  right: ev_l_msa_alpaca
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: adversarial training added (DPO loss in both arms)
  page: 10
- id: c_l_alpaca_loss
  left: ev_l_secpo_alpaca
  right: ev_l_atdpo_alpaca
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: SecPO loss in place of DPO (adversarial training in both arms)
  page: 10
- id: c_l_seputil_at
  left: ev_l_atdpo_seputil
  right: ev_l_msa_seputil
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: adversarial training added (DPO loss in both arms)
  page: 10
- id: c_l_seputil_loss
  left: ev_l_secpo_seputil
  right: ev_l_atdpo_seputil
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: SecPO loss in place of DPO (adversarial training in both arms)
  page: 10
- id: c_l_sepgen_at
  left: ev_l_atdpo_sepgen
  right: ev_l_msa_sepgen
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: adversarial training added (DPO loss in both arms)
  page: 10
- id: c_l_sepgen_loss
  left: ev_l_secpo_sepgen
  right: ev_l_atdpo_sepgen
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: SecPO loss in place of DPO (adversarial training in both arms)
  page: 10
- id: c_l_septap_at
  left: ev_l_atdpo_septap
  right: ev_l_msa_septap
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: adversarial training added (DPO loss in both arms)
  page: 10
- id: c_l_septap_loss
  left: ev_l_secpo_septap
  right: ev_l_atdpo_septap
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: SecPO loss in place of DPO (adversarial training in both arms)
  page: 10
- id: c_l_seppis_at
  left: ev_l_atdpo_seppis
  right: ev_l_msa_seppis
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: adversarial training added (DPO loss in both arms)
  page: 10
- id: c_l_seppis_loss
  left: ev_l_secpo_seppis
  right: ev_l_atdpo_seppis
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: SecPO loss in place of DPO (adversarial training in both arms)
  page: 10
- id: c_q_mmlupro_at
  left: ev_q_atdpo_mmlupro
  right: ev_q_msa_mmlupro
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: adversarial training added (DPO loss in both arms)
  page: 10
- id: c_q_mmlupro_loss
  left: ev_q_secpo_mmlupro
  right: ev_q_atdpo_mmlupro
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: SecPO loss in place of DPO (adversarial training in both arms)
  page: 10
- id: c_q_gpqa_at
  left: ev_q_atdpo_gpqa
  right: ev_q_msa_gpqa
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: adversarial training added (DPO loss in both arms)
  page: 10
- id: c_q_gpqa_loss
  left: ev_q_secpo_gpqa
  right: ev_q_atdpo_gpqa
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: SecPO loss in place of DPO (adversarial training in both arms)
  page: 10
- id: c_q_gsm8k_at
  left: ev_q_atdpo_gsm8k
  right: ev_q_msa_gsm8k
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: adversarial training added (DPO loss in both arms)
  page: 10
- id: c_q_gsm8k_loss
  left: ev_q_secpo_gsm8k
  right: ev_q_atdpo_gsm8k
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: SecPO loss in place of DPO (adversarial training in both arms)
  page: 10
- id: c_q_minerva_at
  left: ev_q_atdpo_minerva
  right: ev_q_msa_minerva
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: adversarial training added (DPO loss in both arms)
  page: 10
- id: c_q_minerva_loss
  left: ev_q_secpo_minerva
  right: ev_q_atdpo_minerva
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: SecPO loss in place of DPO (adversarial training in both arms)
  page: 10
- id: c_q_alpaca_at
  left: ev_q_atdpo_alpaca
  right: ev_q_msa_alpaca
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: adversarial training added (DPO loss in both arms)
  page: 10
- id: c_q_alpaca_loss
  left: ev_q_secpo_alpaca
  right: ev_q_atdpo_alpaca
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: SecPO loss in place of DPO (adversarial training in both arms)
  page: 10
- id: c_q_seputil_at
  left: ev_q_atdpo_seputil
  right: ev_q_msa_seputil
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: adversarial training added (DPO loss in both arms)
  page: 10
- id: c_q_seputil_loss
  left: ev_q_secpo_seputil
  right: ev_q_atdpo_seputil
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: SecPO loss in place of DPO (adversarial training in both arms)
  page: 10
- id: c_q_sepgen_at
  left: ev_q_atdpo_sepgen
  right: ev_q_msa_sepgen
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: adversarial training added (DPO loss in both arms)
  page: 10
- id: c_q_sepgen_loss
  left: ev_q_secpo_sepgen
  right: ev_q_atdpo_sepgen
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: SecPO loss in place of DPO (adversarial training in both arms)
  page: 10
- id: c_q_septap_at
  left: ev_q_atdpo_septap
  right: ev_q_msa_septap
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: adversarial training added (DPO loss in both arms)
  page: 10
- id: c_q_septap_loss
  left: ev_q_secpo_septap
  right: ev_q_atdpo_septap
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: SecPO loss in place of DPO (adversarial training in both arms)
  page: 10
- id: c_q_seppis_at
  left: ev_q_atdpo_seppis
  right: ev_q_msa_seppis
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: adversarial training added (DPO loss in both arms)
  page: 10
- id: c_q_seppis_loss
  left: ev_q_secpo_seppis
  right: ev_q_atdpo_seppis
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: SecPO loss in place of DPO (adversarial training in both arms)
  page: 10
- id: c_l_basic_msa_none
  left: ev_l_msa_basic
  right: ev_l_none_basic
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched_unstated
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 27
- id: c_l_basic_secponoat_none
  left: ev_l_secponoat_basic
  right: ev_l_none_basic
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched_unstated
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 27
- id: c_l_basic_atdpo_none
  left: ev_l_atdpo_basic
  right: ev_l_none_basic
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched_unstated
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 27
- id: c_l_basic_secpo_none
  left: ev_l_secpo_basic
  right: ev_l_none_basic
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched_unstated
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 27
- id: c_l_basic_lossnoat
  left: ev_l_secponoat_basic
  right: ev_l_msa_basic
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched_unstated
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: SecPO loss in place of DPO (no adversarial training)
  page: 27
- id: c_l_basic_at
  left: ev_l_atdpo_basic
  right: ev_l_msa_basic
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched_unstated
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: adversarial training added (DPO loss in both arms)
  page: 27
- id: c_l_basic_loss
  left: ev_l_secpo_basic
  right: ev_l_atdpo_basic
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched_unstated
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: SecPO loss in place of DPO (adversarial training in both arms)
  page: 27
- id: c_l_gcg_msa_none
  left: ev_l_msa_gcg
  right: ev_l_none_gcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched_unstated
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 27
- id: c_l_gcg_secponoat_none
  left: ev_l_secponoat_gcg
  right: ev_l_none_gcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched_unstated
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 27
- id: c_l_gcg_atdpo_none
  left: ev_l_atdpo_gcg
  right: ev_l_none_gcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched_unstated
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 27
- id: c_l_gcg_secpo_none
  left: ev_l_secpo_gcg
  right: ev_l_none_gcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched_unstated
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 27
- id: c_l_gcg_lossnoat
  left: ev_l_secponoat_gcg
  right: ev_l_msa_gcg
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched_unstated
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: SecPO loss in place of DPO (no adversarial training)
  page: 27
- id: c_l_gcg_at
  left: ev_l_atdpo_gcg
  right: ev_l_msa_gcg
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched_unstated
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: adversarial training added (DPO loss in both arms)
  page: 27
- id: c_l_gcg_loss
  left: ev_l_secpo_gcg
  right: ev_l_atdpo_gcg
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched_unstated
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: SecPO loss in place of DPO (adversarial training in both arms)
  page: 27
claims:
- id: k1
  text: adversarial training cuts ASR by 1.5 to 14.5 times across three adaptive attacks compared with Meta-SecAlign
  location: abstract
  anchor: Our AT reduces ASRs by 1.5–14.5× across three SoTA adaptive attacks
  jobs:
  - R1
  supported_by:
  - c_l_sepgen_at
  - c_l_septap_at
  - c_l_seppis_at
  - c_q_sepgen_at
  - c_q_septap_at
  - c_q_seppis_at
  page: 1
- id: k2
  text: the SecPO loss cuts adaptive ASR by a further 2.4 to 4.6 times compared with the DPO loss
  location: abstract
  anchor: the SecPO loss reduces them by a further
  jobs:
  - R1
  supported_by:
  - c_l_sepgen_loss
  - c_l_septap_loss
  - c_l_seppis_loss
  - c_q_sepgen_loss
  - c_q_septap_loss
  - c_q_seppis_loss
  page: 1
  contradicted_by:
  - r_l_secpo_basic
- id: k3
  text: SecPO with adversarial training on Qwen3.6-27B reaches 8.8% against 78.8% for Meta-SecAlign on AgentDojo ASR@800
  location: abstract
  anchor: 8.8% vs. 78.8% AgentDojo ASR@800
  jobs:
  - R1
  supported_by:
  - c_q_adgen_secpo_msa
  - c_q_adgen_secpo_none
  page: 1
- id: k4
  text: the defended model shows no noticeable loss of benign utility on nine benchmarks
  location: abstract
  anchor: shows no noticeable reduction in benign utility in nine benchmarks
  jobs:
  - general_utility
  - R2
  supported_by:
  - c_q_mmlupro_secpo_none
  - c_q_gpqa_secpo_none
  - c_q_gsm8k_secpo_none
  - c_q_minerva_secpo_none
  - c_q_alpaca_secpo_none
  - c_q_seputil_secpo_none
  - c_q_adutil_secpo_none
  - c_q_dynutil_secpo_none
  - c_q_dtaputil_secpo_none
  - c_l_mmlupro_secpo_none
  - c_l_gpqa_secpo_none
  - c_l_gsm8k_secpo_none
  - c_l_minerva_secpo_none
  - c_l_alpaca_secpo_none
  - c_l_seputil_secpo_none
  page: 1
coverage:
  R1: direct
  R2clean: direct
  R2attack: direct
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Tables 1 to 4 and 18
    R2clean: Table 2 (AgentDojo and AgentDyn benign utility; SEP utility as a proxy)
    R2attack: Table 2 (DTaP utility under attack)
    general_utility: Tables 1 to 4
schema_gaps:
- where: defences secpo_*, atdpo_*
  note: adversarial training is a per-step PAIR search against the current checkpoint, not a learned attacker; axis 2 has no value for search-based adversarial training, so it is recorded in the components
- where: evaluations ev_q_*_adutil, ev_q_*_adasr
  note: all AgentDojo numbers, including the undefended arm, are run with AgentDojo's repeat-user-prompt defence (following Meta-SecAlign)
- where: claims k2
  note: under the white-box Basic Adaptive attack (Table 18), SecPO with adversarial training (10.6%) is slightly above adversarial training with DPO (9.7%); white-box attacks are outside the paper's stated threat model
- where: report
  note: the utility judge for AlpacaEval2 and SEP is Claude-Opus-4.8 (Appendix B); the judge names are the paper's

`untrustedcomponent_arxiv2026` (2611.11571)

In [ ]:
%%writefile reports/records/untrustedcomponent_arxiv2026.yaml
report:
  id: untrustedcomponent_arxiv2026
  slug: '2611.11571'
  title: 'Treating the Model as an Untrusted Component: Deterministic Policy Enforcement, Human Approval and Taint Tracking Against Prompt Injection in Tool-Using Agents, with Validation on Open-Weights Models'
  first_author: Mustafa S. Aljumaily
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2611.11571'
  version_read: arXiv v1
  first_date: '2026-11-08'
  read: F
  role: re_measures
  code_release: code, data, oracle audit and notebook provided (Section 9)
  extraction_scope: 'Sections 6.10 and 6.11: the D0, D1 and D2 rows of Tables 15 and 16 (real Qwen2.5 models) and Table 13; the abstract claim on guardrails (the deterministic monitors D4 to D8, human approval and taint tracking,
    which are system-level, the detector stack D3, the simulation results of Sections 6.1 to 6.9 and Table 14, and the composed persistence results of Tables 17 and 18 not coded)'
defences:
- id: d1_15
  name: in-prompt guardrail (D1)
  origin: this_report
  base_checkpoint: Qwen2.5-1.5B-Instruct
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: JSON tool-calling agent over realistic tool outputs; the injection arrives in a read step's output
  layout_compat: &id001
    user_turn: untested
    separate_role: untested
    system_context: supported
    agent_tool_result: supported
  anchor: D1 In-prompt guardrail no
- id: d2_15
  name: in-prompt guardrail with spotlighting by datamarking (D2)
  origin: this_report
  base_checkpoint: Qwen2.5-1.5B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: ablation_present
  signal_dependence: measured
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: JSON tool-calling agent over realistic tool outputs; the injection arrives in a read step's output; tool outputs datamarked
  layout_compat: *id001
  anchor: D2 Guardrail + spotlighting no
  signal_dependence_detail: D1 is the same guardrail without datamarking, run on the same episodes and models (Tables 15 and 16)
  pages:
    components: 14
- id: d1_7
  name: in-prompt guardrail (D1)
  origin: this_report
  base_checkpoint: Qwen2.5-7B-Instruct
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: JSON tool-calling agent over realistic tool outputs; the injection arrives in a read step's output
  layout_compat: *id001
  anchor: D1 In-prompt guardrail no
- id: d2_7
  name: in-prompt guardrail with spotlighting by datamarking (D2)
  origin: this_report
  base_checkpoint: Qwen2.5-7B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: ablation_present
  signal_dependence: measured
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: JSON tool-calling agent over realistic tool outputs; the injection arrives in a read step's output; tool outputs datamarked
  layout_compat: *id001
  anchor: D2 Guardrail + spotlighting no
  signal_dependence_detail: D1 is the same guardrail without datamarking, run on the same episodes and models (Tables 15 and 16)
  pages:
    components: 14
- id: g1_15
  name: in-prompt guardrail G1 (calibration run)
  origin: this_report
  base_checkpoint: Qwen2.5-1.5B-Instruct
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 13
  layout: single summarisation call with an injected inert tool call
  anchor: with and without an in-prompt guardrail (174 prompts, two samples each
evaluations:
- id: ev_15_none_att
  defence_id: none:Qwen2.5-1.5B-Instruct
  benchmark: 'authors'' executable harness: 13 tasks, 12 tools, injection carried by a read step, static attack families pooled (156 attack episodes per defence and model)'
  attacker_type: static
  attacker_access: black_box
  judge: effect-based oracle on canonicalised calls, independent of the defence (two-rater audit)
  layout_same_as_defence: 'yes'
  measured_by: untrustedcomponent_arxiv2026
  page: 15
  decoding: temperature 0.7, up to six turns
  sample_size: '156'
  pages:
    benchmark: 14
    decoding: 14
  baseline_kind: off_the_shelf
- id: ev_15_none_ben
  defence_id: none:Qwen2.5-1.5B-Instruct
  benchmark: 'authors'' executable harness: 13 tasks, benign episodes (78 per defence and model)'
  attacker_type: none
  attacker_access: not_applicable
  judge: task-completion check by the same oracle
  layout_same_as_defence: 'yes'
  measured_by: untrustedcomponent_arxiv2026
  page: 15
  decoding: temperature 0.7, up to six turns
  sample_size: '78'
  pages:
    benchmark: 14
    decoding: 14
  baseline_kind: off_the_shelf
- id: ev_15_d1_att
  defence_id: d1_15
  benchmark: 'authors'' executable harness: 13 tasks, 12 tools, injection carried by a read step, static attack families pooled (156 attack episodes per defence and model)'
  attacker_type: static
  attacker_access: black_box
  judge: effect-based oracle on canonicalised calls, independent of the defence (two-rater audit)
  layout_same_as_defence: 'yes'
  measured_by: untrustedcomponent_arxiv2026
  page: 15
  decoding: temperature 0.7, up to six turns
  sample_size: '156'
  pages:
    benchmark: 14
    decoding: 14
- id: ev_15_d1_ben
  defence_id: d1_15
  benchmark: 'authors'' executable harness: 13 tasks, benign episodes (78 per defence and model)'
  attacker_type: none
  attacker_access: not_applicable
  judge: task-completion check by the same oracle
  layout_same_as_defence: 'yes'
  measured_by: untrustedcomponent_arxiv2026
  page: 15
  decoding: temperature 0.7, up to six turns
  sample_size: '78'
  pages:
    benchmark: 14
    decoding: 14
- id: ev_15_d2_att
  defence_id: d2_15
  benchmark: 'authors'' executable harness: 13 tasks, 12 tools, injection carried by a read step, static attack families pooled (156 attack episodes per defence and model)'
  attacker_type: static
  attacker_access: black_box
  judge: effect-based oracle on canonicalised calls, independent of the defence (two-rater audit)
  layout_same_as_defence: 'yes'
  measured_by: untrustedcomponent_arxiv2026
  page: 15
  decoding: temperature 0.7, up to six turns
  sample_size: '156'
  pages:
    benchmark: 14
    decoding: 14
- id: ev_15_d2_ben
  defence_id: d2_15
  benchmark: 'authors'' executable harness: 13 tasks, benign episodes (78 per defence and model)'
  attacker_type: none
  attacker_access: not_applicable
  judge: task-completion check by the same oracle
  layout_same_as_defence: 'yes'
  measured_by: untrustedcomponent_arxiv2026
  page: 15
  decoding: temperature 0.7, up to six turns
  sample_size: '78'
  pages:
    benchmark: 14
    decoding: 14
- id: ev_7_none_att
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: 'authors'' executable harness: 13 tasks, 12 tools, injection carried by a read step, static attack families pooled (156 attack episodes per defence and model)'
  attacker_type: static
  attacker_access: black_box
  judge: effect-based oracle on canonicalised calls, independent of the defence (two-rater audit)
  layout_same_as_defence: 'yes'
  measured_by: untrustedcomponent_arxiv2026
  page: 16
  decoding: temperature 0.7, up to six turns
  sample_size: '156'
  pages:
    benchmark: 14
    decoding: 14
  baseline_kind: off_the_shelf
- id: ev_7_none_ben
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: 'authors'' executable harness: 13 tasks, benign episodes (78 per defence and model)'
  attacker_type: none
  attacker_access: not_applicable
  judge: task-completion check by the same oracle
  layout_same_as_defence: 'yes'
  measured_by: untrustedcomponent_arxiv2026
  page: 16
  decoding: temperature 0.7, up to six turns
  sample_size: '78'
  pages:
    benchmark: 14
    decoding: 14
  baseline_kind: off_the_shelf
- id: ev_7_d1_att
  defence_id: d1_7
  benchmark: 'authors'' executable harness: 13 tasks, 12 tools, injection carried by a read step, static attack families pooled (156 attack episodes per defence and model)'
  attacker_type: static
  attacker_access: black_box
  judge: effect-based oracle on canonicalised calls, independent of the defence (two-rater audit)
  layout_same_as_defence: 'yes'
  measured_by: untrustedcomponent_arxiv2026
  page: 16
  decoding: temperature 0.7, up to six turns
  sample_size: '156'
  pages:
    benchmark: 14
    decoding: 14
- id: ev_7_d1_ben
  defence_id: d1_7
  benchmark: 'authors'' executable harness: 13 tasks, benign episodes (78 per defence and model)'
  attacker_type: none
  attacker_access: not_applicable
  judge: task-completion check by the same oracle
  layout_same_as_defence: 'yes'
  measured_by: untrustedcomponent_arxiv2026
  page: 16
  decoding: temperature 0.7, up to six turns
  sample_size: '78'
  pages:
    benchmark: 14
    decoding: 14
- id: ev_7_d2_att
  defence_id: d2_7
  benchmark: 'authors'' executable harness: 13 tasks, 12 tools, injection carried by a read step, static attack families pooled (156 attack episodes per defence and model)'
  attacker_type: static
  attacker_access: black_box
  judge: effect-based oracle on canonicalised calls, independent of the defence (two-rater audit)
  layout_same_as_defence: 'yes'
  measured_by: untrustedcomponent_arxiv2026
  page: 16
  decoding: temperature 0.7, up to six turns
  sample_size: '156'
  pages:
    benchmark: 14
    decoding: 14
- id: ev_7_d2_ben
  defence_id: d2_7
  benchmark: 'authors'' executable harness: 13 tasks, benign episodes (78 per defence and model)'
  attacker_type: none
  attacker_access: not_applicable
  judge: task-completion check by the same oracle
  layout_same_as_defence: 'yes'
  measured_by: untrustedcomponent_arxiv2026
  page: 16
  decoding: temperature 0.7, up to six turns
  sample_size: '78'
  pages:
    benchmark: 14
    decoding: 14
- id: ev_15_none_lvl0
  defence_id: none:Qwen2.5-1.5B-Instruct
  benchmark: summarisation task with an injected inert tool call, attack level 0 (a distinct phrasing family)
  attacker_type: static
  attacker_access: black_box
  judge: the model emits the injected tool call with the attacker string in its arguments
  layout_same_as_defence: 'yes'
  measured_by: untrustedcomponent_arxiv2026
  page: 13
  sample_size: '24'
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_15_none_lvl1
  defence_id: none:Qwen2.5-1.5B-Instruct
  benchmark: summarisation task with an injected inert tool call, attack level 1 (a distinct phrasing family)
  attacker_type: static
  attacker_access: black_box
  judge: the model emits the injected tool call with the attacker string in its arguments
  layout_same_as_defence: 'yes'
  measured_by: untrustedcomponent_arxiv2026
  page: 13
  sample_size: '24'
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_15_none_lvl2
  defence_id: none:Qwen2.5-1.5B-Instruct
  benchmark: summarisation task with an injected inert tool call, attack level 2 (a distinct phrasing family)
  attacker_type: static
  attacker_access: black_box
  judge: the model emits the injected tool call with the attacker string in its arguments
  layout_same_as_defence: 'yes'
  measured_by: untrustedcomponent_arxiv2026
  page: 13
  sample_size: '24'
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_15_none_lvl3
  defence_id: none:Qwen2.5-1.5B-Instruct
  benchmark: summarisation task with an injected inert tool call, attack level 3 (a distinct phrasing family)
  attacker_type: static
  attacker_access: black_box
  judge: the model emits the injected tool call with the attacker string in its arguments
  layout_same_as_defence: 'yes'
  measured_by: untrustedcomponent_arxiv2026
  page: 13
  sample_size: '24'
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_15_none_lvl4
  defence_id: none:Qwen2.5-1.5B-Instruct
  benchmark: summarisation task with an injected inert tool call, attack level 4 (a distinct phrasing family)
  attacker_type: static
  attacker_access: black_box
  judge: the model emits the injected tool call with the attacker string in its arguments
  layout_same_as_defence: 'yes'
  measured_by: untrustedcomponent_arxiv2026
  page: 13
  sample_size: '24'
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_15_none_lvl5
  defence_id: none:Qwen2.5-1.5B-Instruct
  benchmark: summarisation task with an injected inert tool call, attack level 5 (a distinct phrasing family)
  attacker_type: static
  attacker_access: black_box
  judge: the model emits the injected tool call with the attacker string in its arguments
  layout_same_as_defence: 'yes'
  measured_by: untrustedcomponent_arxiv2026
  page: 13
  sample_size: '24'
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_15_g1_lvl0
  defence_id: g1_15
  benchmark: summarisation task with an injected inert tool call, attack level 0 (a distinct phrasing family)
  attacker_type: static
  attacker_access: black_box
  judge: the model emits the injected tool call with the attacker string in its arguments
  layout_same_as_defence: 'yes'
  measured_by: untrustedcomponent_arxiv2026
  page: 13
  sample_size: '24'
  decoding: not reported
- id: ev_15_g1_lvl1
  defence_id: g1_15
  benchmark: summarisation task with an injected inert tool call, attack level 1 (a distinct phrasing family)
  attacker_type: static
  attacker_access: black_box
  judge: the model emits the injected tool call with the attacker string in its arguments
  layout_same_as_defence: 'yes'
  measured_by: untrustedcomponent_arxiv2026
  page: 13
  sample_size: '24'
  decoding: not reported
- id: ev_15_g1_lvl2
  defence_id: g1_15
  benchmark: summarisation task with an injected inert tool call, attack level 2 (a distinct phrasing family)
  attacker_type: static
  attacker_access: black_box
  judge: the model emits the injected tool call with the attacker string in its arguments
  layout_same_as_defence: 'yes'
  measured_by: untrustedcomponent_arxiv2026
  page: 13
  sample_size: '24'
  decoding: not reported
- id: ev_15_g1_lvl3
  defence_id: g1_15
  benchmark: summarisation task with an injected inert tool call, attack level 3 (a distinct phrasing family)
  attacker_type: static
  attacker_access: black_box
  judge: the model emits the injected tool call with the attacker string in its arguments
  layout_same_as_defence: 'yes'
  measured_by: untrustedcomponent_arxiv2026
  page: 13
  sample_size: '24'
  decoding: not reported
- id: ev_15_g1_lvl4
  defence_id: g1_15
  benchmark: summarisation task with an injected inert tool call, attack level 4 (a distinct phrasing family)
  attacker_type: static
  attacker_access: black_box
  judge: the model emits the injected tool call with the attacker string in its arguments
  layout_same_as_defence: 'yes'
  measured_by: untrustedcomponent_arxiv2026
  page: 13
  sample_size: '24'
  decoding: not reported
- id: ev_15_g1_lvl5
  defence_id: g1_15
  benchmark: summarisation task with an injected inert tool call, attack level 5 (a distinct phrasing family)
  attacker_type: static
  attacker_access: black_box
  judge: the model emits the injected tool call with the attacker string in its arguments
  layout_same_as_defence: 'yes'
  measured_by: untrustedcomponent_arxiv2026
  page: 13
  sample_size: '24'
  decoding: not reported
results:
- id: r_15_none_prop
  evaluation_id: ev_15_none_att
  quantity: R1
  coverage: direct
  metric_as_printed: model proposes the attacker's call
  value: '67'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 15
  spread: 95% Wilson 59-74
- id: r_15_none_exec
  evaluation_id: ev_15_none_att
  quantity: R1
  coverage: direct
  metric_as_printed: attacker's goal executed
  value: '66'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 15
  spread: 95% Wilson 58-73
- id: r_15_none_comp
  evaluation_id: ev_15_none_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign completion
  value: '50'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 15
  spread: 95% Wilson 39-61
- id: r_15_none_viol
  evaluation_id: ev_15_none_ben
  quantity: other
  coverage: direct
  metric_as_printed: benign spontaneous violations
  value: '40'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 15
- id: r_15_d1_prop
  evaluation_id: ev_15_d1_att
  quantity: R1
  coverage: direct
  metric_as_printed: model proposes the attacker's call
  value: '65'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 15
  spread: 95% Wilson 58-72
- id: r_15_d1_exec
  evaluation_id: ev_15_d1_att
  quantity: R1
  coverage: direct
  metric_as_printed: attacker's goal executed
  value: '65'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 15
  spread: 95% Wilson 57-72
- id: r_15_d1_comp
  evaluation_id: ev_15_d1_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign completion
  value: '59'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 15
  spread: 95% Wilson 48-69
- id: r_15_d1_viol
  evaluation_id: ev_15_d1_ben
  quantity: other
  coverage: direct
  metric_as_printed: benign spontaneous violations
  value: '35'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 15
- id: r_15_d2_prop
  evaluation_id: ev_15_d2_att
  quantity: R1
  coverage: direct
  metric_as_printed: model proposes the attacker's call
  value: '47'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 15
  spread: 95% Wilson 39-55
- id: r_15_d2_exec
  evaluation_id: ev_15_d2_att
  quantity: R1
  coverage: direct
  metric_as_printed: attacker's goal executed
  value: '42'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 15
  spread: 95% Wilson 35-50
- id: r_15_d2_comp
  evaluation_id: ev_15_d2_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign completion
  value: '54'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 15
  spread: 95% Wilson 43-64
- id: r_15_d2_viol
  evaluation_id: ev_15_d2_ben
  quantity: other
  coverage: direct
  metric_as_printed: benign spontaneous violations
  value: '37'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 15
- id: r_7_none_prop
  evaluation_id: ev_7_none_att
  quantity: R1
  coverage: direct
  metric_as_printed: model proposes the attacker's call
  value: '86'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 16
  spread: 95% Wilson 80-90
- id: r_7_none_exec
  evaluation_id: ev_7_none_att
  quantity: R1
  coverage: direct
  metric_as_printed: attacker's goal executed
  value: '86'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 16
  spread: 95% Wilson 80-90
- id: r_7_none_comp
  evaluation_id: ev_7_none_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign completion
  value: '99'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 16
  spread: 95% Wilson 93-100
- id: r_7_none_viol
  evaluation_id: ev_7_none_ben
  quantity: other
  coverage: direct
  metric_as_printed: benign spontaneous violations
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 16
- id: r_7_d1_prop
  evaluation_id: ev_7_d1_att
  quantity: R1
  coverage: direct
  metric_as_printed: model proposes the attacker's call
  value: '55'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 16
  spread: 95% Wilson 47-63
- id: r_7_d1_exec
  evaluation_id: ev_7_d1_att
  quantity: R1
  coverage: direct
  metric_as_printed: attacker's goal executed
  value: '55'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 16
  spread: 95% Wilson 47-63
- id: r_7_d1_comp
  evaluation_id: ev_7_d1_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign completion
  value: '100'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 16
  spread: 95% Wilson 95-100
- id: r_7_d1_viol
  evaluation_id: ev_7_d1_ben
  quantity: other
  coverage: direct
  metric_as_printed: benign spontaneous violations
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 16
- id: r_7_d2_prop
  evaluation_id: ev_7_d2_att
  quantity: R1
  coverage: direct
  metric_as_printed: model proposes the attacker's call
  value: '42'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 16
  spread: 95% Wilson 35-50
- id: r_7_d2_exec
  evaluation_id: ev_7_d2_att
  quantity: R1
  coverage: direct
  metric_as_printed: attacker's goal executed
  value: '40'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 16
  spread: 95% Wilson 33-48
- id: r_7_d2_comp
  evaluation_id: ev_7_d2_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign completion
  value: '100'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 16
  spread: 95% Wilson 95-100
- id: r_7_d2_viol
  evaluation_id: ev_7_d2_ben
  quantity: other
  coverage: direct
  metric_as_printed: benign spontaneous violations
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 16
- id: r_15_none_lvl0
  evaluation_id: ev_15_none_lvl0
  quantity: R1
  coverage: direct
  metric_as_printed: obedience, attack level 0
  value: '4.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 13
- id: r_15_none_lvl1
  evaluation_id: ev_15_none_lvl1
  quantity: R1
  coverage: direct
  metric_as_printed: obedience, attack level 1
  value: '20.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 13
- id: r_15_none_lvl2
  evaluation_id: ev_15_none_lvl2
  quantity: R1
  coverage: direct
  metric_as_printed: obedience, attack level 2
  value: '50.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 13
- id: r_15_none_lvl3
  evaluation_id: ev_15_none_lvl3
  quantity: R1
  coverage: direct
  metric_as_printed: obedience, attack level 3
  value: '4.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 13
- id: r_15_none_lvl4
  evaluation_id: ev_15_none_lvl4
  quantity: R1
  coverage: direct
  metric_as_printed: obedience, attack level 4
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 13
- id: r_15_none_lvl5
  evaluation_id: ev_15_none_lvl5
  quantity: R1
  coverage: direct
  metric_as_printed: obedience, attack level 5
  value: '4.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 13
- id: r_15_g1_lvl0
  evaluation_id: ev_15_g1_lvl0
  quantity: R1
  coverage: direct
  metric_as_printed: obedience, attack level 0
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 13
- id: r_15_g1_lvl1
  evaluation_id: ev_15_g1_lvl1
  quantity: R1
  coverage: direct
  metric_as_printed: obedience, attack level 1
  value: '16.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 13
- id: r_15_g1_lvl2
  evaluation_id: ev_15_g1_lvl2
  quantity: R1
  coverage: direct
  metric_as_printed: obedience, attack level 2
  value: '45.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 13
- id: r_15_g1_lvl3
  evaluation_id: ev_15_g1_lvl3
  quantity: R1
  coverage: direct
  metric_as_printed: obedience, attack level 3
  value: '4.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 13
- id: r_15_g1_lvl4
  evaluation_id: ev_15_g1_lvl4
  quantity: R1
  coverage: direct
  metric_as_printed: obedience, attack level 4
  value: '8.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 13
- id: r_15_g1_lvl5
  evaluation_id: ev_15_g1_lvl5
  quantity: R1
  coverage: direct
  metric_as_printed: obedience, attack level 5
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 13
comparisons:
- id: c_15_d1_att
  left: ev_15_d1_att
  right: ev_15_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 15
- id: c_15_d1_ben
  left: ev_15_d1_ben
  right: ev_15_none_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 15
- id: c_15_d2_att
  left: ev_15_d2_att
  right: ev_15_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 15
- id: c_15_d2_ben
  left: ev_15_d2_ben
  right: ev_15_none_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 15
- id: c_15_spot_att
  left: ev_15_d2_att
  right: ev_15_d1_att
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: datamarking added to the in-prompt guardrail
  page: 15
- id: c_15_spot_ben
  left: ev_15_d2_ben
  right: ev_15_d1_ben
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: datamarking added to the in-prompt guardrail
  page: 15
- id: c_7_d1_att
  left: ev_7_d1_att
  right: ev_7_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_7_d1_ben
  left: ev_7_d1_ben
  right: ev_7_none_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_7_d2_att
  left: ev_7_d2_att
  right: ev_7_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_7_d2_ben
  left: ev_7_d2_ben
  right: ev_7_none_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_7_spot_att
  left: ev_7_d2_att
  right: ev_7_d1_att
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: datamarking added to the in-prompt guardrail
  page: 16
- id: c_7_spot_ben
  left: ev_7_d2_ben
  right: ev_7_d1_ben
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: datamarking added to the in-prompt guardrail
  page: 16
- id: c_15_g1_lvl0
  left: ev_15_g1_lvl0
  right: ev_15_none_lvl0
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_15_g1_lvl1
  left: ev_15_g1_lvl1
  right: ev_15_none_lvl1
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_15_g1_lvl2
  left: ev_15_g1_lvl2
  right: ev_15_none_lvl2
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_15_g1_lvl3
  left: ev_15_g1_lvl3
  right: ev_15_none_lvl3
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_15_g1_lvl4
  left: ev_15_g1_lvl4
  right: ev_15_none_lvl4
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_15_g1_lvl5
  left: ev_15_g1_lvl5
  right: ev_15_none_lvl5
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 13
claims:
- id: k1
  text: 'guardrail effects depended on the model: the prompt guardrail moved obedience from 67% to 65% on the 1.5B model and from 86% to 55% on the 7B model'
  location: abstract
  anchor: guardrail effects depended on the model
  jobs:
  - R1
  supported_by:
  - c_15_d1_att
  - c_7_d1_att
  - c_15_d2_att
  - c_7_d2_att
  page: 1
coverage:
  R1: direct
  R2clean: direct
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Tables 13, 15 and 16
    R2clean: Tables 15 and 16 (benign completion)
    R2attack: not reported for the attack episodes
schema_gaps:
- where: report
  note: the paper's main subject (deterministic monitors, human approval, taint tracking) is system-level and the D3 stack contains an LLM detector; both are left out under the review's scope rules, and most of the paper's
    results come from a simulation with an assumed model
- where: evaluations ev_15_*_lvl*
  note: Table 13 counts were reconstructed from printed rates; the raw sample log was not kept (p. 13); at levels 4 and 5 only half the outputs parse as tool calls
- where: evaluations ev_*_att
  note: attack episodes pool the static attack families; the paper's persistence figures (Tables 17 and 18) compose per-task rates over k attempts rather than running repeated attacks

## Stage 2. The PDFs read are the PDFs registered, and every report is an included study

In [ ]:
import csv, logging, datetime as dt
from collections import Counter
import schema, search, corpus, paths
for m in (schema, search, corpus, paths):
    importlib.reload(m)
logging.getLogger("pypdf").setLevel(logging.ERROR)
from pypdf import PdfReader

BATCH = "04f"
cfg_exclusions = 1
cfg_next = "batch 04g"
IDS = ['raised_arxiv2026', 'advsim2real_arxiv2026', 'ltbd_arxiv2026', 'tracer_emnlp2026', 'agentsafeevo_arxiv2026', 'edic_emnlp2026', 'ignorednotrefused_arxiv2026', 'hos_arxiv2026', 'llmon_arxiv2026', 'secpo_arxiv2026', 'untrustedcomponent_arxiv2026']
man = {m["slug"]: m for m in search.read_csv(paths.REPORTS / "defences.csv")}
problems, texts = [], {}
for rid in IDS:
    rec = schema.load_record(paths.RECORDS / f"{rid}.yaml")
    slug = rec["report"]["slug"]
    pdf = paths.DATA_DEFENCES / f"{slug}.pdf"
    m = man.get(slug, {})
    if not pdf.exists():
        problems.append(f"{rid}: {pdf.name} not in data/defences"); continue
    sha = corpus.sha256(pdf)
    rd = PdfReader(str(pdf))
    texts[slug] = ([p.extract_text() or "" for p in rd.pages], [p.extract_text(extraction_mode="layout") or "" for p in rd.pages])
    ok_sha = sha == m.get("sha256")
    ok_inc = m.get("decision") == "include"
    print(f"{rid:<28} {slug:<12} pages {len(rd.pages):>3}  sha256 {'matches' if ok_sha else 'DIFFERS'}  decision {m.get('decision')}")
    if not ok_sha:
        problems.append(f"{rid}: PDF differs from the one registered in reports/defences.csv")
    if not ok_inc:
        problems.append(f"{rid}: not an included study")

## Stage 2b. Full-text exclusions found while coding

Each study below was included at title and abstract; its full text shows it does not meet the inclusion rule. The anchor is checked in its PDF, then the decision is written to schema/fulltext_decisions.csv and schema/screening_decisions.csv and carried into reports/screening.csv, reports/defences.csv, reports/fulltext_check.csv and reports/prisma_counts.json, as notebook 02d did.

In [ ]:
import json
import overlap
importlib.reload(overlap)
EXCL = [{'key': 'arxiv:2610.33201', 'slug': '2610.33201', 'reason': 'benchmark only', 'anchor': 'Instruction-hierarchy benchmarks ask whether the higher-ranked source wins', 'note': 'WhoWins benchmark of instruction conflicts across operator, configuration, user, tool-result and memory seats on 14 off-the-shelf models; no defence is added, trained or compared, and the role-against-label tests concern the operator line, a system-against-user conflict'}]
FT_FIELDS = ["key", "slug", "decision", "reason", "basis", "anchor", "note"]
SD_FIELDS = ["key", "decision", "reason", "stage", "access_status", "note", "title"]
CHECK_FIELDS = ["key", "slug", "decision", "reason", "basis", "page", "status", "anchor", "sha256", "note"]
ft = search.read_csv(paths.SCHEMA / "fulltext_decisions.csv")
sd = search.read_csv(paths.SCHEMA / "screening_decisions.csv")
fchk = search.read_csv(paths.REPORTS / "fulltext_check.csv")
mlist = search.read_csv(paths.REPORTS / "defences.csv")
for e in EXCL:
    m = next((x for x in mlist if x["slug"] == e["slug"]), None)
    pdf = paths.DATA_DEFENCES / f"{e['slug']}.pdf"
    if m is None or not pdf.exists():
        problems.append(f"exclusion {e['slug']}: not in reports/defences.csv or PDF missing"); continue
    sha = corpus.sha256(pdf)
    if sha != m.get("sha256"):
        problems.append(f"exclusion {e['slug']}: PDF differs from the one registered")
    if e["reason"] not in search.EXCLUSION_REASONS:
        problems.append(f"exclusion {e['slug']}: reason not in the fixed list")
    pages = overlap.pdf_page_texts(pdf)
    a = overlap.norm_text(e["anchor"])
    hits = [i + 1 for i, t in enumerate(pages) if a in t]
    status = "found" if hits else "anchor not found"
    if not hits:
        problems.append(f"exclusion {e['slug']}: anchor not found in the PDF")
    row = {"key": e["key"], "slug": e["slug"], "decision": "exclude", "reason": e["reason"], "basis": "full text", "anchor": e["anchor"], "note": e["note"]}
    ft = [r for r in ft if r["key"] != e["key"]] + [row]
    fchk = [r for r in fchk if r["key"] != e["key"]] + [dict(row, page=str(hits[0]) if hits else "", status=status, sha256=sha)]
    n = 0
    for r in sd:
        if r["key"] == e["key"]:
            r.update(decision="exclude", reason=e["reason"], stage="full_text", note="full text: " + e["note"]); n += 1
    if n != 1:
        problems.append(f"exclusion {e['slug']}: {n} rows for {e['key']} in schema/screening_decisions.csv")
    m.update(decision="exclude", reason=e["reason"], stage="full_text", note="full text: " + e["note"])
    print(f"{e['slug']:<12} excluded at full text ({e['reason']}); anchor {status}{' on page ' + str(hits[0]) if hits else ''}")
def write_lf(path, rows, fields):              # the schema files were written with plain newlines; keep them that way
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore", lineterminator="\n")
        w.writeheader()
        w.writerows({k: r.get(k) or "" for k in fields} for r in rows)
write_lf(paths.SCHEMA / "fulltext_decisions.csv", ft, FT_FIELDS)
write_lf(paths.SCHEMA / "screening_decisions.csv", sd, SD_FIELDS)
search.write_csv(paths.REPORTS / "fulltext_check.csv", fchk, CHECK_FIELDS)
search.write_csv(paths.REPORTS / "defences.csv", mlist, list(mlist[0].keys()))
rows = search.read_csv(paths.REPORTS / "screening.csv")
rows, more = search.apply_decisions(rows, sd)
problems += more
search.write_csv(paths.REPORTS / "screening.csv", rows, search.SCREEN_FIELDS)
prisma = json.loads((paths.REPORTS / "prisma_counts.json").read_text())
prisma.update(search.prisma_counts(search.read_csv(paths.REPORTS / "search_log.csv"), rows))
prisma["fulltext_anchors_found"] = sum(1 for r in fchk if r["status"].startswith("found"))
(paths.REPORTS / "prisma_counts.json").write_text(json.dumps(prisma, indent=1))
print(f"included now: {prisma['included']}; full text assessed: {prisma['full_text_assessed']}; "
      f"full-text exclusions: {json.dumps(prisma['excluded_full_text_by_reason'])}")

## Stage 3. Validate against the frozen schema and the PDFs; grade the comparisons; extra checks

In [ ]:
import re
import pypdfium2 as pdfium
_docs = {}
def pdfium_page(slug, page):
    if slug not in _docs:
        _docs[slug] = pdfium.PdfDocument(str(paths.DATA_DEFENCES / f"{slug}.pdf"))
    return _docs[slug][page - 1].get_textpage().get_text_range()
S1 = schema.load_schema(paths.SCHEMA / "record_schema.yaml")
check_rows, summary, residual, third_rows = [], [], [], []
for rid in IDS:
    rec = schema.load_record(paths.RECORDS / f"{rid}.yaml")
    slug = rec["report"]["slug"]
    plain, lay = texts.get(slug, (None, None))
    probs, notes = schema.validate_record(rec, S1, plain, lay)
    kept, third = [], []                       # a value pypdf glues to its neighbour is looked for in a third extraction (pypdfium2)
    for p in probs:
        m = re.match(r"results:(\S+): value '(.+)' not found on page (\d+)$", p)
        if m and schema.value_on_page(m.group(2), pdfium_page(slug, int(m.group(3)))):
            third.append(f"{m.group(1)} ({m.group(2)!r}, page {m.group(3)})")
        else:
            kept.append(p)
    probs = kept
    for kind in ("defences", "evaluations", "results", "comparisons", "claims"):          # ids unique within each kind
        dup = [i for i, n in Counter(r["id"] for r in rec.get(kind) or []).items() if n > 1]
        probs += [f"{kind}: id {i} used more than once" for i in dup]
    fig_kept, third_chart = 0, []
    for r in rec.get("results") or []:                                                     # chart values kept as printed
        if r.get("value_source") == "figure_only" and r.get("value") not in (None, ""):
            fig_kept += 1
            pg = int(r["page"]) - 1
            if plain and not (schema.value_on_page(r["value"], plain[pg]) or schema.value_on_page(r["value"], lay[pg])):
                if schema.value_on_page(r["value"], pdfium_page(slug, int(r["page"]))):
                    third_chart.append(f"{r['id']} ({r['value']!r}, page {r['page']}, chart label)")
                else:
                    probs.append(f"results:{r['id']}: chart value {r['value']!r} not found on page {r['page']}")
    third_rows += [{"report": rid, "status": "value found in the third extraction", "detail": t} for t in third + third_chart]
    grades = Counter(schema.grade(c) for c in rec.get("comparisons") or [])
    n_val = sum(1 for r in rec.get("results") or [] if r.get("value_source") != "figure_only")
    n_anc = sum(1 for k in ("defences", "claims") for r in rec.get(k) or [] if r.get("anchor"))
    summary.append({"batch": BATCH, "report": rid, "slug": slug, "role": rec["report"]["role"],
                    "defences": len(rec.get("defences") or []), "evaluations": len(rec.get("evaluations") or []),
                    "results": len(rec.get("results") or []), "values_checked": n_val, "chart_values_checked": fig_kept,
                    "anchors_checked": n_anc, "comparisons": len(rec.get("comparisons") or []),
                    "grade_A": grades["A"], "grade_B": grades["B"], "grade_C": grades["C"], "grade_U": grades["U"],
                    "claims": len(rec.get("claims") or []),
                    **{f"cov_{q}": rec["coverage"][q] for q in ("R1", "R2clean", "R2attack", "R3ns", "R3cp", "R3safe")},
                    "values_third_extraction": len(third), "chart_values_third_extraction": len(third_chart), "residual_notes": len(notes), "problems": len(probs)})
    check_rows += [{"report": rid, "status": "problem", "detail": p} for p in probs]
    residual += [{"report": rid, "where": n.split(": ", 1)[0], "note": n.split(": ", 1)[1]} for n in notes]
    print(f"{rid:<28} problems {len(probs)}; values {n_val} ({len(third)} via the third extraction); chart values {fig_kept} ({len(third_chart)} via the third extraction); anchors {n_anc}; grades {dict(grades)}")
    for p in probs:
        print("   ", p)
    problems += [f"{rid}: {p}" for p in probs]
check_rows += third_rows + [{"report": g["report"], "status": "residual note", "detail": f"{g['where']}: {g['note']}"} for g in residual]
search.write_csv(paths.REPORTS / f"extraction_{BATCH}_check.csv", check_rows or [{"report": "all", "status": "ok", "detail": "no problems"}],
                 ["report", "status", "detail"])
search.write_csv(paths.REPORTS / f"extraction_{BATCH}_summary.csv", summary, list(summary[0].keys()))
for g in residual:
    print(f"  residual note, {g['report']}: {g['note'][:150]}")

## Stage 4. Batch register (one row per coded report, kept across batches)

In [ ]:
reg_path = paths.REPORTS / "extraction_batches.csv"
reg = [r for r in search.read_csv(reg_path) if r["batch"] != BATCH] if reg_path.exists() else []
today = dt.date.today().isoformat()
for s in summary:
    reg.append({"batch": BATCH, "report": s["report"], "slug": s["slug"], "coded": today,
                "status": "checked" if s["problems"] == 0 else "problems"})
reg.sort(key=lambda r: (r["batch"], r["report"]))
search.write_csv(reg_path, reg, ["batch", "report", "slug", "coded", "status"])
included = sum(1 for m in search.read_csv(paths.REPORTS / "defences.csv") if m.get("decision") == "include")   # read again: Stage 2b may change it
pilot = len(list(paths.RECORDS.glob("*.yaml"))) - len({r["report"] for r in reg})
print(f"records in reports/records: {len(list(paths.RECORDS.glob('*.yaml')))} (pilot {pilot}, batches {len({r['report'] for r in reg})}); included studies: {included}")

## Stage 5. Decision record and tests

In [ ]:
tot = {k: sum(s[k] for s in summary) for k in ("defences", "evaluations", "results", "values_checked", "values_third_extraction", "chart_values_checked",
                                              "chart_values_third_extraction", "anchors_checked", "comparisons", "grade_A", "grade_B", "grade_C", "grade_U", "claims")}
status = "BATCH CHECKED" if not problems else "BATCH NOT CHECKED: PROBLEMS BELOW"
lines = [f"decision_{BATCH}: {status}", f"this step: {dt.date.today().isoformat()}",
         "reports: " + ", ".join(IDS),
         f"records: {tot['defences']} defences, {tot['evaluations']} evaluations, {tot['results']} results, "
         f"{tot['comparisons']} comparisons, {tot['claims']} claims",
         f"checked against the PDFs: {tot['values_checked']} printed values ({tot['values_third_extraction']} of them found only in the third "
         f"extraction), {tot['chart_values_checked']} chart values ({tot['chart_values_third_extraction']} of them found only in the third extraction) "
         f"and {tot['anchors_checked']} anchors, each found on its page",
         f"comparison grades: A {tot['grade_A']}, B {tot['grade_B']}, C {tot['grade_C']}, U {tot['grade_U']}",
         f"residual notes: {len(residual)}; full-text exclusions found while coding: {cfg_exclusions}",
         f"next: {cfg_next}"] + [f"- problem: {p}" for p in problems]
(paths.REPORTS / f"decision_{BATCH}.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))
tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(tests.stdout.strip().splitlines()[-1] if tests.stdout.strip() else tests.stderr.strip()[-400:])

## Stage 6. Commit

In [ ]:
import os, subprocess
os.chdir(os.environ.get("SOURCEMAP_REPO_ROOT", "/content/drive/MyDrive/SourceMap_Research/sourcemap-research"))
r = subprocess.run(["python", "tools/commit_cell.py",
                    "04f extraction batch f: eleven included studies coded from their PDFs and one full-text exclusion, values and anchors checked, comparisons graded"],
                   capture_output=True, text=True)
print(r.stdout)
print(r.stderr)